<a href="https://colab.research.google.com/github/mardocheeogecime-gif/Rede_Colabora-o_Acad-mica_ECI_UFMG/blob/main/LAB_BIBLIOMETRIC_MEASURES_OF_EPISTEMIC_CHANGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# LAB SCIENTOMETRICS 2026
# ETAPA 01 — Auditoria seletiva de artefatos
# Ambiente: CPU

from pathlib import Path
import pandas as pd
from datetime import datetime

ROOT = Path("/content/gdrive/MyDrive")
BASE = ROOT / "CAPES_Epistemic_Change"

LAB = BASE / "08_epistemic_measurement_validation"
INVENTORY_DIR = LAB / "01_inventory"
INVENTORY_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "09_ablacao_M1_M4",
    "15_calibracao_drift",
    "16_validacao_split_half",
    "19_auditoria_final_temporal",
    "21_novidade_semantica_prior_only",
    "22_frontier_novelty_3y",
    "23_recombination_frontier3y",
    "24_ETS_prospectivo_3y",
    "27_ETS_dimensoes_finais",
    "28_modelo_processual_ETS",
    "29_auditoria_hubness_semantic_exposure",
    "30_diffusion_conditional_final",
    "31_sensibilidade_diffusion_k",
    "32_robustez_familias_ETS",
    "33_translation_estimand_check",
    "34_manuscript_assets",
    "35_manuscript_assets_final_colab",
]

records = []
MAX_ITEMS = 40

for folder in TARGETS:
    path = BASE / folder

    if not path.is_dir():
        print("MISSING:", folder)
        continue

    print(f"\n=== {folder} ===")

    # Somente o primeiro nível da pasta.
    # Sem rglob e sem abrir os arquivos.
    count = 0

    for item in path.iterdir():
        if count >= MAX_ITEMS:
            print("... limite de exibição atingido")
            break

        is_dir = item.is_dir()

        record = {
            "source_folder": folder,
            "name": item.name,
            "type": "directory" if is_dir else "file",
            "path": str(item),
        }

        records.append(record)

        symbol = "DIR " if is_dir else "FILE"
        print(symbol, item.name)

        count += 1

df = pd.DataFrame(records)

output = INVENTORY_DIR / "selective_inventory.csv"
df.to_csv(output, index=False)

print("\n" + "=" * 60)
print("AUDITORIA CONCLUÍDA")
print("Entradas identificadas:", len(df))
print("Inventário salvo:", output)
print("Horário:", datetime.now().isoformat())


MISSING: 09_ablacao_M1_M4
MISSING: 15_calibracao_drift
MISSING: 16_validacao_split_half
MISSING: 19_auditoria_final_temporal
MISSING: 21_novidade_semantica_prior_only
MISSING: 22_frontier_novelty_3y
MISSING: 23_recombination_frontier3y
MISSING: 24_ETS_prospectivo_3y
MISSING: 27_ETS_dimensoes_finais
MISSING: 28_modelo_processual_ETS
MISSING: 29_auditoria_hubness_semantic_exposure
MISSING: 30_diffusion_conditional_final
MISSING: 31_sensibilidade_diffusion_k
MISSING: 32_robustez_familias_ETS
MISSING: 33_translation_estimand_check
MISSING: 34_manuscript_assets
MISSING: 35_manuscript_assets_final_colab

AUDITORIA CONCLUÍDA
Entradas identificadas: 0
Inventário salvo: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/01_inventory/selective_inventory.csv
Horário: 2026-10-21T01:40:26.779751


In [4]:

# LAB SCIENTOMETRICS 2026
# ETAPA 02 — Auditoria de schemas e configurações
# CPU | Sem carregar tabelas nacionais em memória

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import json

BASE = Path(
    "/content/gdrive/MyDrive/CAPES_Epistemic_Change"
)

LAB = BASE / "08_epistemic_measurement_validation"
OUT = LAB / "01_inventory"
OUT.mkdir(parents=True, exist_ok=True)

FILES = [
    "09_ablacao_M1_M4/CAPES_ABLACAO_M1_M4_2013_2024.parquet",
    "21_novidade_semantica_prior_only/CAPES_M4_NOVIDADE_PRIOR_ONLY_2014_2024.parquet",
    "22_frontier_novelty_3y/CAPES_M4_FRONTIER_NOVELTY_3Y_2016_2024.parquet",
    "23_recombination_frontier3y/CAPES_M4_RECOMBINATION_FRONTIER3Y_2016_2024.parquet",
    "24_ETS_prospectivo_3y/CAPES_ETS_PROSPECTIVO_3Y_2016_2021.parquet",
    "27_ETS_dimensoes_finais/CAPES_ETS_MAIN44_DIMENSOES_FINAIS_2016_2021.parquet",
    "29_auditoria_hubness_semantic_exposure/CAPES_ETS_MAIN44_COM_HUBNESS_2016_2021.parquet",
]

schemas = []

print("=" * 65)
print("PARQUET SCHEMAS")
print("=" * 65)

for rel in FILES:
    path = BASE / rel

    if not path.is_file():
        print("MISSING:", rel)
        continue

    pf = pq.ParquetFile(path)

    print("\nFILE:", rel)
    print("ROWS:", pf.metadata.num_rows)
    print("ROW GROUPS:", pf.metadata.num_row_groups)
    print("COLUMNS:", pf.metadata.num_columns)

    for field in pf.schema_arrow:
        schemas.append({
            "file": rel,
            "column": field.name,
            "dtype": str(field.type),
            "rows": pf.metadata.num_rows,
        })
        print(" -", field.name, ":", field.type)

pd.DataFrame(schemas).to_csv(
    OUT / "parquet_schema_audit.csv",
    index=False
)

CSV_FILES = [
    "16_validacao_split_half/validacao_split_half_539_transicoes.csv",
    "19_auditoria_final_temporal/candidatos_temporais_finais.csv",
    "29_auditoria_hubness_semantic_exposure/correlacoes_hubness_emergence_diffusion.csv",
    "32_robustez_familias_ETS/robustez_familias_coeficientes.csv",
    "33_translation_estimand_check/translation_estimand_comparison.csv",
]

print("\n" + "=" * 65)
print("CSV HEADERS AND FIRST ROWS")
print("=" * 65)

for rel in CSV_FILES:
    path = BASE / rel

    if not path.is_file():
        continue

    print("\nFILE:", rel)

    try:
        sample = pd.read_csv(path, nrows=3)
        print(sample.to_string(index=False))
    except Exception as e:
        print("READ ERROR:", repr(e))

JSON_FILES = [
    "21_novidade_semantica_prior_only/configuracao_novidade_prior_only.json",
    "22_frontier_novelty_3y/configuracao_frontier_novelty_3y.json",
    "24_ETS_prospectivo_3y/configuracao_ETS_prospectivo_3y.json",
]

print("\n" + "=" * 65)
print("EXPERIMENT CONFIGURATIONS")
print("=" * 65)

for rel in JSON_FILES:
    path = BASE / rel

    if not path.is_file():
        continue

    print("\nCONFIG:", rel)

    with path.open(encoding="utf-8") as f:
        obj = json.load(f)

    print(json.dumps(
        obj,
        ensure_ascii=False,
        indent=2,
        default=str
    )[:5000])

print("\nAUDITORIA CONCLUÍDA.")
print("Schema salvo em:",
      OUT / "parquet_schema_audit.csv")


PARQUET SCHEMAS

FILE: 09_ablacao_M1_M4/CAPES_ABLACAO_M1_M4_2013_2024.parquet
ROWS: 993086
ROW GROUPS: 107
COLUMNS: 19
 - ID_TRABALHO : string
 - ANO : int32
 - NM_AREA_AVALIACAO : string
 - NM_GRAU_ACADEMICO : string
 - NM_IDIOMA : string
 - NM_GRANDE_AREA_CONHECIMENTO : string
 - CD_PROGRAMA : string
 - SG_ENTIDADE_ENSINO : string
 - COS_M1_M2 : float
 - COS_M1_M3 : float
 - COS_M1_M4 : float
 - COS_M2_M3 : float
 - COS_M2_M4 : float
 - COS_M3_M4 : float
 - DELTA_K_FROM_T : float
 - DELTA_A_FROM_T : float
 - DELTA_AK_FROM_T : float
 - DELTA_A_GIVEN_K : float
 - DELTA_K_GIVEN_A : float

FILE: 21_novidade_semantica_prior_only/CAPES_M4_NOVIDADE_PRIOR_ONLY_2014_2024.parquet
ROWS: 925655
ROW GROUPS: 11
COLUMNS: 21
 - ID_TRABALHO : string
 - ANO : int32
 - AREA_HARMONIZADA : string
 - CD_PROGRAMA : string
 - NM_GRAU_ACADEMICO : string
 - GLOBAL_ROW : int64
 - N_REFERENCIA_PASSADA : int64
 - SIM_TOP1 : float
 - SIM_TOP10_MEDIA : float
 - SIM_TOP10_MEDIANA : float
 - SIM_TOP50_MEDIA : float


In [5]:

# ETAPA 03 — Auditoria seletiva das fontes textuais
# CPU | Somente metadados de arquivos e schemas

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path("/content/gdrive/MyDrive")
BASE = ROOT / "CAPES_Epistemic_Change"

TARGETS = [
    BASE / "07_embeddings_nacionais",
    BASE / "08_auditoria_embeddings",
    BASE / "12_indice_analitico_harmonizado",
    BASE / "20_benchmark_ann_gpu",
    ROOT / "CAPES_Sucupira_Producoes_2013_2024_FINAL",
]

MAX_ENTRIES_PER_LEVEL = 35
MAX_SCHEMA_FILES = 12

def list_limited(path, depth=2):
    if not path.is_dir():
        print("MISSING:", path)
        return []

    found = []
    frontier = [(path, 0)]

    while frontier:
        current, level = frontier.pop(0)

        try:
            entries = []
            for i, p in enumerate(current.iterdir()):
                if i >= MAX_ENTRIES_PER_LEVEL:
                    print("LIMIT:", current)
                    break
                entries.append(p)
        except Exception as exc:
            print("READ ERROR:", current, str(exc))
            continue

        for p in entries:
            found.append(p)
            print("  " * level,
                  "DIR " if p.is_dir() else "FILE",
                  p.name)

            if p.is_dir() and level + 1 < depth:
                frontier.append((p, level + 1))

    return found

all_items = []

for target in TARGETS:
    print("\n=== ", target.name, " ===")
    all_items.extend(list_limited(target))

# Inspecionar somente schemas Parquet candidatos.
candidates = [
    p for p in all_items
    if p.is_file() and p.suffix.lower() == ".parquet"
]

print("\n=== PARQUET CANDIDATES ===")
print("Identificados:", len(candidates))

for p in candidates[:MAX_SCHEMA_FILES]:
    try:
        pf = pq.ParquetFile(p)
        print("\nFILE:", p.name)
        print("ROWS:", pf.metadata.num_rows)
        print("COLUMNS:", pf.schema_arrow.names)
    except Exception as exc:
        print("ERROR:", p.name, str(exc))

print("\nETAPA 03 CONCLUÍDA")



===  07_embeddings_nacionais  ===
 DIR  checkpoints
 DIR  manifestos
 DIR  M1
 DIR  M2
 DIR  M3
 DIR  M4
   FILE configuracao_embeddings.json
   FILE manifesto_embeddings_M1_M4.csv
LIMIT: /content/gdrive/MyDrive/CAPES_Epistemic_Change/07_embeddings_nacionais/M1
   FILE M1_2013_embeddings_float16.npy
   FILE M1_2013_ids.parquet
   FILE M1_2013_metadata.json
   FILE M1_2014_embeddings_float16.npy
   FILE M1_2014_ids.parquet
   FILE M1_2014_metadata.json
   FILE M1_2015_embeddings_float16.npy
   FILE M1_2015_ids.parquet
   FILE M1_2015_metadata.json
   FILE M1_2016_embeddings_float16.npy
   FILE M1_2016_ids.parquet
   FILE M1_2016_metadata.json
   FILE M1_2017_embeddings_float16.npy
   FILE M1_2017_ids.parquet
   FILE M1_2017_metadata.json
   FILE M1_2018_embeddings_float16.npy
   FILE M1_2018_ids.parquet
   FILE M1_2018_metadata.json
   FILE M1_2019_embeddings_float16.npy
   FILE M1_2019_ids.parquet
   FILE M1_2019_metadata.json
   FILE M1_2020_embeddings_float16.npy
   FILE M1_2020_ids

In [6]:

# SCIENTOMETRICS 2026 — ETAPA 04
# Auditoria do corpus textual original
# CPU | Sem carregar os milhões de registros

from pathlib import Path
import pyarrow.parquet as pq
import json
import pandas as pd

ROOT = Path("/content/gdrive/MyDrive")
BASE = ROOT / "CAPES_Epistemic_Change"
SRC = ROOT / "CAPES_Sucupira_Producoes_2013_2024_FINAL"

LAB = BASE / "08_epistemic_measurement_validation"
OUT = LAB / "01_inventory"
OUT.mkdir(parents=True, exist_ok=True)

INDEX = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

KEYWORDS = [
    "TITULO", "TITLE",
    "RESUMO", "ABSTRACT",
    "PALAVRA", "KEYWORD",
    "TEXTO", "TEXT",
    "REFERENCIA", "CITACAO",
    "DOI", "URL",
    "ID_TRABALHO"
]

# 1. Esquema do índice analítico
print("=" * 65)
print("INDICE ANALITICO HARMONIZADO")
print("=" * 65)

if INDEX.is_file():
    pf = pq.ParquetFile(INDEX)

    print("ROWS:", pf.metadata.num_rows)
    print("COLUMNS:", pf.metadata.num_columns)

    for field in pf.schema_arrow:
        print(field.name, ":", field.type)

    matched = [
        field.name
        for field in pf.schema_arrow
        if any(
            term in field.name.upper()
            for term in KEYWORDS
        )
    ]

    print("\nCAMPOS CANDIDATOS:", matched)

else:
    print("INDICE NAO ENCONTRADO")

# 2. Localizar arquivos candidatos sem varredura recursiva
TARGET_DIRS = [
    SRC / "06_parquet",
    SRC / "08_duckdb",
    SRC / "10_producao_2024_final",
    BASE / "05_transferencia_colab",
    BASE / "12_indice_analitico_harmonizado",
]

candidates = []

for folder in TARGET_DIRS:
    print("\n=== ", folder.name, " ===")

    if not folder.is_dir():
        print("MISSING")
        continue

    entries = list(folder.iterdir())

    print("Entradas diretas:", len(entries))

    for p in entries[:60]:
        print(
            "DIR " if p.is_dir() else "FILE",
            p.name
        )

        if p.is_file() and p.suffix.lower() in [
            ".parquet", ".duckdb", ".csv"
        ]:
            candidates.append(p)

# 3. Schemas de arquivos Parquet candidatos
print("\n" + "=" * 65)
print("PARQUET SCHEMAS CANDIDATOS")
print("=" * 65)

results = []

for path in candidates[:20]:
    if path.suffix.lower() != ".parquet":
        continue

    try:
        pf = pq.ParquetFile(path)
        cols = pf.schema_arrow.names

        hits = [
            c for c in cols
            if any(
                k in c.upper()
                for k in KEYWORDS
            )
        ]

        print("\nFILE:", path.name)
        print("ROWS:", pf.metadata.num_rows)
        print("TEXT/ID COLUMNS:", hits)

        results.append({
            "path": str(path),
            "rows": pf.metadata.num_rows,
            "matching_columns": json.dumps(hits)
        })

    except Exception as exc:
        print("ERROR:", path.name, repr(exc))

pd.DataFrame(results).to_csv(
    OUT / "source_text_schema_candidates.csv",
    index=False
)

# 4. Verificações diretas de integridade
print("\n" + "=" * 65)
print("EMBEDDING FILE CHECK")
print("=" * 65)

for mode in ["M1", "M2", "M3", "M4"]:
    for year in [2015, 2021, 2024]:
        f = (
            BASE / "07_embeddings_nacionais"
            / mode
            / f"{mode}_{year}_embeddings_float16.npy"
        )
        print(
            mode, year,
            "FOUND" if f.is_file() else "MISSING"
        )

print("\nETAPA 04 CONCLUIDA")


INDICE ANALITICO HARMONIZADO
ROWS: 993086
COLUMNS: 23
ID_TRABALHO : string
ANO : int32
NM_AREA_AVALIACAO : string
NM_GRANDE_AREA_CONHECIMENTO : string
NM_GRAU_ACADEMICO : string
NM_IDIOMA : string
CD_PROGRAMA : string
NM_PROGRAMA : string
SG_ENTIDADE_ENSINO : string
NM_ENTIDADE_ENSINO : string
SG_UF_IES : string
NM_UF_IES : string
NM_REGIAO : string
ID_AREA_CONCENTRACAO : string
NM_AREA_CONCENTRACAO : string
ID_LINHA_PESQUISA : string
NM_LINHA_PESQUISA : string
AREA_ORIGINAL_NORM : string
AREA_HARMONIZADA : string
ID_COMPONENTE_AREA : string
COMPONENTE_MULTIRROTULO : bool
AREA_BRUTA_ESTAVEL_12_ANOS : bool
ROTULO_FOI_HARMONIZADO : bool

CAMPOS CANDIDATOS: ['ID_TRABALHO']

===  06_parquet  ===
Entradas diretas: 0

===  08_duckdb  ===
Entradas diretas: 0

===  10_producao_2024_final  ===
Entradas diretas: 8
DIR  01_particoes
DIR  02_controle
DIR  03_parquet
DIR  04_auditoria
DIR  05_duckdb
DIR  06_reparo_persistentes_v2
DIR  07_reparo_por_subtipo
DIR  08_scan_size1

===  05_transferencia_

In [7]:

# SCIENTOMETRICS 2026 — ETAPA 05
# Auditoria diagnóstica de qualidade textual
# CPU | Sem carregar o corpus inteiro

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import json
import re

ROOT = Path("/content/gdrive/MyDrive")
BASE = ROOT / "CAPES_Epistemic_Change"
SOURCE = BASE / "05_transferencia_colab"

OUT = BASE / "08_epistemic_measurement_validation" / "01_inventory"
OUT.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(20261020)
SAMPLE_PER_YEAR = 400

TEXT_COLS = [
    "ID_TRABALHO",
    "TITULO",
    "RESUMO",
    "PALAVRAS_CHAVE"
]

def valid_text(s):
    if pd.isna(s):
        return False
    s = str(s).strip()
    return bool(s) and s.lower() not in {
        "nan", "none", "null", "não informado",
        "nao informado", "sem informação",
        "sem informacao"
    }

samples = []
audit = []

for year in range(2013, 2025):

    path = SOURCE / f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    pf = pq.ParquetFile(path)

    # Escolher grupos de linhas sem ler o arquivo inteiro.
    group = int(RNG.integers(0, pf.num_row_groups))

    tbl = pf.read_row_group(
        group,
        columns=TEXT_COLS
    )

    df = tbl.to_pandas()

    n = min(SAMPLE_PER_YEAR, len(df))
    df = df.sample(
        n=n,
        random_state=20261020 + year
    ).copy()

    df["ANO_AUDITORIA"] = year

    for col in ["TITULO", "RESUMO", "PALAVRAS_CHAVE"]:
        df[f"{col}_VALIDO"] = df[col].map(valid_text)
        df[f"{col}_CARACTERES"] = df[col].fillna("").astype(str).str.len()

    audit.append({
        "ANO": year,
        "N_TOTAL_ARQUIVO": pf.metadata.num_rows,
        "N_AMOSTRA": len(df),
        "ROW_GROUP_AMOSTRADO": group,
        "TITULO_VALIDO_PCT": 100 * df["TITULO_VALIDO"].mean(),
        "RESUMO_VALIDO_PCT": 100 * df["RESUMO_VALIDO"].mean(),
        "PALAVRAS_VALIDAS_PCT": 100 * df["PALAVRAS_CHAVE_VALIDO"].mean(),
        "RESUMO_MEDIANA_CARACTERES":
            df.loc[df["RESUMO_VALIDO"], "RESUMO_CARACTERES"].median(),
        "IDS_DUPLICADOS_AMOSTRA":
            int(df["ID_TRABALHO"].duplicated().sum())
    })

    # Guardar somente informações diagnósticas.
    samples.append(df[[
        "ID_TRABALHO", "ANO_AUDITORIA",
        "TITULO_VALIDO", "RESUMO_VALIDO",
        "PALAVRAS_CHAVE_VALIDO",
        "TITULO_CARACTERES",
        "RESUMO_CARACTERES",
        "PALAVRAS_CHAVE_CARACTERES"
    ]])

    print(f"{year}: {len(df)} documentos auditados")

summary = pd.DataFrame(audit)
sample_ids = pd.concat(samples, ignore_index=True)

# Vinculação amostral com índice harmonizado.
INDEX = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

ids = pq.read_table(
    INDEX,
    columns=["ID_TRABALHO", "ANO", "AREA_HARMONIZADA"]
).to_pandas()

duplicate_index_ids = int(ids["ID_TRABALHO"].duplicated().sum())

matched = sample_ids.merge(
    ids,
    on="ID_TRABALHO",
    how="left",
    indicator=True,
    validate="many_to_one" if duplicate_index_ids == 0 else "many_to_many"
)

print("\n=== RESUMO DE QUALIDADE ===")
print(summary.to_string(index=False))

print("\n=== VINCULACAO ===")
print("Duplicatas ID no indice:", duplicate_index_ids)
print(matched["_merge"].value_counts().to_string())

if duplicate_index_ids == 0:
    print("Taxa de correspondencia:",
          round(100 * (matched["_merge"] == "both").mean(), 2), "%")

summary.to_csv(OUT / "text_quality_diagnostic.csv", index=False)
sample_ids.to_parquet(OUT / "text_diagnostic_sample_ids.parquet", index=False)

print("\nArquivos salvos em:", OUT)
print("ETAPA 05 CONCLUIDA")


2013: 400 documentos auditados
2014: 400 documentos auditados
2015: 400 documentos auditados
2016: 400 documentos auditados
2017: 400 documentos auditados
2018: 400 documentos auditados
2019: 400 documentos auditados
2020: 400 documentos auditados
2021: 400 documentos auditados
2022: 400 documentos auditados
2023: 400 documentos auditados
2024: 400 documentos auditados

=== RESUMO DE QUALIDADE ===
 ANO  N_TOTAL_ARQUIVO  N_AMOSTRA  ROW_GROUP_AMOSTRADO  TITULO_VALIDO_PCT  RESUMO_VALIDO_PCT  PALAVRAS_VALIDAS_PCT  RESUMO_MEDIANA_CARACTERES  IDS_DUPLICADOS_AMOSTRA
2013            67431        400                   11              100.0              100.0                 100.0                     1894.5                       0
2014            70487        400                   13              100.0              100.0                 100.0                     2022.0                       0
2015            75666        400                    1              100.0              100.0             

In [8]:

# SCIENTOMETRICS 2026 — ETAPA 06
# Amostra-mestra probabilística estratificada por ano
# CPU | Leitura seletiva por row group | Sem embeddings

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import json

ROOT = Path("/content/gdrive/MyDrive")
BASE = ROOT / "CAPES_Epistemic_Change"
SOURCE = BASE / "05_transferencia_colab"
LAB = BASE / "08_epistemic_measurement_validation"

OUT = LAB / "03_ground_truth"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 20261020
N_PER_YEAR = 120
rng = np.random.default_rng(SEED)

TEXT_COLS = [
    "ID_TRABALHO",
    "TITULO",
    "RESUMO",
    "PALAVRAS_CHAVE"
]

all_samples = []
sampling_manifest = []

for year in range(2013, 2025):

    path = SOURCE / f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    pf = pq.ParquetFile(path)

    sizes = np.array([
        pf.metadata.row_group(i).num_rows
        for i in range(pf.num_row_groups)
    ], dtype=np.int64)

    boundaries = np.cumsum(sizes)
    total = int(boundaries[-1])

    # Seleção aleatória uniforme de linhas no ano.
    selected = np.sort(
        rng.choice(
            total,
            size=min(N_PER_YEAR, total),
            replace=False
        )
    )

    groups = np.searchsorted(
        boundaries,
        selected,
        side="right"
    )

    frames = []

    for group in np.unique(groups):

        mask = groups == group
        absolute_rows = selected[mask]

        start = (
            0 if group == 0
            else int(boundaries[group - 1])
        )

        local_rows = absolute_rows - start

        table = pf.read_row_group(
            int(group),
            columns=TEXT_COLS
        )

        subset = (
            table.to_pandas()
            .iloc[local_rows]
            .copy()
        )

        subset["ANO"] = year
        subset["SOURCE_ROW"] = absolute_rows

        frames.append(subset)

    year_df = pd.concat(frames, ignore_index=True)

    assert len(year_df) == len(selected)
    assert year_df["SOURCE_ROW"].is_unique

    year_df["SELECTION_PROBABILITY"] = len(selected) / total
    year_df["DESIGN_WEIGHT"] = total / len(selected)

    all_samples.append(year_df)

    sampling_manifest.append({
        "ANO": year,
        "N_UNIVERSO": total,
        "N_AMOSTRA": len(selected),
        "PROBABILIDADE_SELECAO": len(selected) / total,
        "SEED": SEED
    })

    print(
        year,
        "| População:", total,
        "| Amostra:", len(selected)
    )

sample = pd.concat(all_samples, ignore_index=True)

# Vincular metadados administrativos do índice.
index_path = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

META_COLS = [
    "ID_TRABALHO", "ANO", "AREA_HARMONIZADA",
    "NM_GRANDE_AREA_CONHECIMENTO",
    "NM_GRAU_ACADEMICO", "NM_IDIOMA",
    "CD_PROGRAMA", "SG_ENTIDADE_ENSINO",
    "NM_REGIAO"
]

meta = pq.read_table(
    index_path, columns=META_COLS
).to_pandas()

assert meta["ID_TRABALHO"].is_unique, \
    "IDs duplicados no índice harmonizado."

sample = sample.merge(
    meta,
    on="ID_TRABALHO",
    how="left",
    suffixes=("", "_INDICE"),
    indicator=True,
    validate="one_to_one"
)

assert (sample["_merge"] == "both").all(), \
    "Documentos sem correspondência no índice."

assert (sample["ANO"] == sample["ANO_INDICE"]).all(), \
    "Divergência entre anos da fonte e do índice."

sample = sample.drop(
    columns=["_merge", "ANO_INDICE"]
)

assert sample["ID_TRABALHO"].is_unique, \
    "ID duplicado na amostra-mestra."

# ID cego para avaliações independentes.
sample = sample.sort_values(
    ["ANO", "SOURCE_ROW"]
).reset_index(drop=True)

sample.insert(
    0, "CASE_ID",
    [f"ECV-{i:05d}" for i in range(1, len(sample) + 1)]
)

# Manifesto e amostra completa ficam em arquivos separados.
manifest = pd.DataFrame(sampling_manifest)

sample.to_parquet(
    OUT / "master_probability_sample_1440.parquet",
    index=False
)

manifest.to_csv(
    OUT / "sampling_design_manifest.csv",
    index=False
)

# Gerar tabelas de caracterização.
print("\n=== AMOSTRA POR ANO ===")
print(sample["ANO"].value_counts().sort_index())

print("\n=== GRANDES ÁREAS ===")
print(
    sample["NM_GRANDE_AREA_CONHECIMENTO"]
    .value_counts(dropna=False)
)

print("\n=== IDIOMAS ===")
print(
    sample["NM_IDIOMA"]
    .value_counts(dropna=False)
)

print("\n=== GRAUS ACADÊMICOS ===")
print(
    sample["NM_GRAU_ACADEMICO"]
    .value_counts(dropna=False)
)

print("\n=== INTEGRIDADE ===")
print("N documentos:", len(sample))
print("IDs únicos:", sample["ID_TRABALHO"].nunique())

print("\nAMOSTRA SALVA EM:", OUT)
print("ETAPA 06 CONCLUÍDA")


2013 | População: 67431 | Amostra: 120
2014 | População: 70487 | Amostra: 120
2015 | População: 75666 | Amostra: 120
2016 | População: 80075 | Amostra: 120
2017 | População: 85137 | Amostra: 120
2018 | População: 90295 | Amostra: 120
2019 | População: 94349 | Amostra: 120
2020 | População: 79989 | Amostra: 120
2021 | População: 82600 | Amostra: 120
2022 | População: 83981 | Amostra: 120
2023 | População: 91355 | Amostra: 120
2024 | População: 91721 | Amostra: 120

=== AMOSTRA POR ANO ===
ANO
2013    120
2014    120
2015    120
2016    120
2017    120
2018    120
2019    120
2020    120
2021    120
2022    120
2023    120
2024    120
Name: count, dtype: int64

=== GRANDES ÁREAS ===
NM_GRANDE_AREA_CONHECIMENTO
CIÊNCIAS HUMANAS               229
CIÊNCIAS SOCIAIS APLICADAS     218
CIÊNCIAS DA SAÚDE              218
MULTIDISCIPLINAR               203
CIÊNCIAS AGRÁRIAS              135
ENGENHARIAS                    126
CIÊNCIAS EXATAS E DA TERRA     125
LINGÜÍSTICA, LETRAS E ARTES    102
CI

In [9]:

# SCIENTOMETRICS 2026 — ETAPA 07
# Preparação do piloto de anotação
# CPU | Sem embeddings | Sem scores ETS

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SRC = LAB / "03_ground_truth"
OUT = SRC / "pilot_90"
OUT.mkdir(parents=True, exist_ok=True)

source_file = SRC / "master_probability_sample_1440.parquet"

assert source_file.is_file(), "Amostra-mestra não localizada."

master = pd.read_parquet(source_file)

assert len(master) == 1440
assert master["ID_TRABALHO"].is_unique

AREA = "NM_GRANDE_AREA_CONHECIMENTO"

areas = sorted(master[AREA].dropna().unique())

assert len(areas) == 9, \
    f"Esperadas 9 grandes áreas; encontradas {len(areas)}"

# Evitar sobrescrever um piloto já congelado.
pilot_file = OUT / "pilot_cases_blinded.csv"

if pilot_file.exists():
    raise FileExistsError(
        "O piloto já existe. Não sobrescrever: " +
        str(pilot_file)
    )

# Seleção aleatória independente dos indicadores.
pilot = (
    master.groupby(AREA, group_keys=False)
    .sample(n=10, random_state=20261020)
    .sample(frac=1, random_state=20261021)
    .reset_index(drop=True)
)

pilot.insert(
    0, "PILOT_ID",
    [f"PILOT-{i:03d}" for i in range(1, 91)]
)

assert len(pilot) == 90
assert pilot["CASE_ID"].is_unique

# Chave de reidentificação para uso exclusivo da pesquisa.
key = pilot[
    ["PILOT_ID", "CASE_ID", "ID_TRABALHO", "ANO", AREA]
].copy()

key.to_csv(
    OUT / "pilot_linkage_key_RESTRICTED.csv",
    index=False
)

# Material textual sem indicadores ETS,
# instituição, programa ou identificação interna.
blinded = pilot[
    ["PILOT_ID", "TITULO", "RESUMO", "PALAVRAS_CHAVE"]
].copy()

blinded.to_csv(
    pilot_file,
    index=False,
    encoding="utf-8-sig"
)

# Duas fichas independentes, inicialmente vazias.
for reviewer in ["A", "B"]:
    sheet = pd.DataFrame({
        "PILOT_ID": blinded["PILOT_ID"],
        "CONTRIBUICAO_DECLARADA": "",
        "PROBLEMA_CIENTIFICO": "",
        "CONCEITOS_CENTRAIS": "",
        "ABORDAGEM_METODOLOGICA": "",
        "EVIDENCIA_TEXTUAL_SUFICIENTE": "",
        "CONFIANCA_1_A_5": "",
        "JUSTIFICATIVA": "",
        "AVALIADOR": reviewer
    })

    sheet.to_csv(
        OUT / f"annotation_sheet_{reviewer}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("=== PILOTO CRIADO ===")
print("Documentos:", len(pilot))

print("\n=== DISTRIBUIÇÃO POR ÁREA ===")
print(pilot[AREA].value_counts().to_string())

print("\n=== DISTRIBUIÇÃO POR ANO ===")
print(pilot["ANO"].value_counts().sort_index().to_string())

print("\n=== DISTRIBUIÇÃO POR IDIOMA ===")
print(pilot["NM_IDIOMA"].value_counts(dropna=False).to_string())

print("\nArquivos:")
for f in sorted(OUT.iterdir()):
    print(f.name)

print("\nETAPA 07 CONCLUÍDA")


=== PILOTO CRIADO ===
Documentos: 90

=== DISTRIBUIÇÃO POR ÁREA ===
NM_GRANDE_AREA_CONHECIMENTO
CIÊNCIAS SOCIAIS APLICADAS     10
CIÊNCIAS BIOLÓGICAS            10
MULTIDISCIPLINAR               10
CIÊNCIAS AGRÁRIAS              10
CIÊNCIAS EXATAS E DA TERRA     10
CIÊNCIAS HUMANAS               10
CIÊNCIAS DA SAÚDE              10
ENGENHARIAS                    10
LINGÜÍSTICA, LETRAS E ARTES    10

=== DISTRIBUIÇÃO POR ANO ===
ANO
2013     8
2014     9
2015     6
2016     7
2017     5
2018     8
2019    10
2020    10
2021     4
2022     7
2023    10
2024     6

=== DISTRIBUIÇÃO POR IDIOMA ===
NM_IDIOMA
PORTUGUES    88
INGLES        2

Arquivos:
annotation_sheet_A.csv
annotation_sheet_B.csv
pilot_cases_blinded.csv
pilot_linkage_key_RESTRICTED.csv

ETAPA 07 CONCLUÍDA


In [10]:

# SCIENTOMETRICS 2026 — ETAPA 08
# Calibração do protocolo de anotação
# CPU | Sem embeddings | Sem indicadores ETS

from pathlib import Path
import pandas as pd
import json
import hashlib

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

PILOT_DIR = LAB / "03_ground_truth" / "pilot_90"
OUT = LAB / "03_ground_truth" / "calibration_18"
OUT.mkdir(parents=True, exist_ok=True)

PILOT_FILE = PILOT_DIR / "pilot_cases_blinded.csv"
KEY_FILE = PILOT_DIR / "pilot_linkage_key_RESTRICTED.csv"

assert PILOT_FILE.exists()
assert KEY_FILE.exists()

# Não sobrescrever uma rodada já iniciada.
if any(OUT.iterdir()):
    raise FileExistsError(
        "Diretório de calibração não está vazio. "
        "Preservar a execução anterior."
    )

pilot = pd.read_csv(PILOT_FILE, dtype=str)
key = pd.read_csv(KEY_FILE, dtype=str)

assert len(pilot) == 90
assert pilot["PILOT_ID"].is_unique
assert key["PILOT_ID"].is_unique

# A área fica restrita ao procedimento de amostragem.
selection = pilot.merge(
    key[["PILOT_ID", "NM_GRANDE_AREA_CONHECIMENTO"]],
    on="PILOT_ID",
    validate="one_to_one"
)

AREA = "NM_GRANDE_AREA_CONHECIMENTO"

assert selection[AREA].nunique() == 9

calibration = (
    selection.groupby(AREA, group_keys=False)
    .sample(n=2, random_state=20261020)
    .sample(frac=1, random_state=20261021)
    .reset_index(drop=True)
)

assert len(calibration) == 18

FIELDS = [
    "RESEARCH_OBJECT",
    "CLAIMED_CONTRIBUTION",
    "CLAIM_TYPE",
    "NOVELTY_CLAIM",
    "RECOMBINATION_CLAIM",
    "EVIDENCE_SUFFICIENCY",
    "CONFIDENCE",
    "EVIDENCE_EXCERPT",
    "RATIONALE"
]

TEXT_COLS = [
    "PILOT_ID", "TITULO", "RESUMO", "PALAVRAS_CHAVE"
]

# Material idêntico para os dois avaliadores.
review_cases = calibration[TEXT_COLS].copy()

for reviewer in ["A", "B"]:
    sheet = review_cases.copy()

    for field in FIELDS:
        sheet[field] = ""

    sheet["REVIEWER"] = reviewer

    sheet.to_csv(
        OUT / f"calibration_reviewer_{reviewer}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# Chave restrita: nunca compartilhar com avaliadores.
calibration[["PILOT_ID", AREA]].to_csv(
    OUT / "calibration_selection_RESTRICTED.csv",
    index=False
)

protocol = """# Epistemic Change Validation
## Annotation Protocol — Version 0.1

### Purpose
Calibrate document-level extraction of research claims.
This phase does not establish epistemic novelty or change.

### Material
Title, abstract and keywords only.
Do not consult ETS indicators or semantic scores.

### Variables

RESEARCH_OBJECT:
Briefly identify the scientific object or problem.

CLAIMED_CONTRIBUTION:
Record the contribution claimed by the authors.
Do not treat author claims as verified novelty.

CLAIM_TYPE:
Choose one or more:
theoretical_conceptual; methodological;
empirical; application; mixed;
not_identifiable.

NOVELTY_CLAIM:
explicit; implicit; absent; indeterminate.

RECOMBINATION_CLAIM:
explicit; implicit; absent; indeterminate.

EVIDENCE_SUFFICIENCY:
sufficient; partial; insufficient.

CONFIDENCE:
Integer from 1 (very low) to 5 (very high).

EVIDENCE_EXCERPT:
Short excerpt supporting the interpretation.

RATIONALE:
Brief justification of the decision.

### Independence
Reviewers A and B must code separately.
Do not discuss cases before submitting initial coding.

### Adjudication
Record disagreements before discussing them.
Preserve original coding and adjudicated decisions.

### Restrictions
Do not label actual epistemic change in this phase.
Do not infer citations, uptake, influence or disruption.
Do not use automated indicators as ground truth.
"""

(OUT / "annotation_protocol_v01.md").write_text(
    protocol, encoding="utf-8"
)

manifest = {
    "phase": "document_annotation_calibration",
    "protocol_version": "0.1",
    "selection_seed": 20261020,
    "shuffle_seed": 20261021,
    "n_documents": 18,
    "n_areas": 9,
    "documents_per_area": 2,
    "reviewers": ["A", "B"],
    "confirmatory_validation": False,
    "ground_truth_established": False,
    "pilot_ids": review_cases["PILOT_ID"].tolist()
}

(OUT / "calibration_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== ETAPA 08 CONCLUÍDA ===")
print("Casos:", len(review_cases))
print("Áreas representadas:", selection[AREA].nunique())
print("PILOT_ID únicos:", review_cases["PILOT_ID"].nunique())
print("\nArquivos:")
for p in sorted(OUT.iterdir()):
    print("-", p.name)

print("\nNão foram calculados indicadores.")
print("Não foi estabelecido ground truth.")


=== ETAPA 08 CONCLUÍDA ===
Casos: 18
Áreas representadas: 9
PILOT_ID únicos: 18

Arquivos:
- annotation_protocol_v01.md
- calibration_manifest.json
- calibration_reviewer_A.csv
- calibration_reviewer_B.csv
- calibration_selection_RESTRICTED.csv

Não foram calculados indicadores.
Não foi estabelecido ground truth.


In [11]:

# SCIENTOMETRICS 2026 — ETAPA 09
# Controle de integridade das fichas de calibração
# CPU | Nenhuma classificação automática

from pathlib import Path
import pandas as pd

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
DIR = BASE / "08_epistemic_measurement_validation" / "03_ground_truth" / "calibration_18"

FIELDS = [
    "RESEARCH_OBJECT",
    "CLAIMED_CONTRIBUTION",
    "CLAIM_TYPE",
    "NOVELTY_CLAIM",
    "RECOMBINATION_CLAIM",
    "EVIDENCE_SUFFICIENCY",
    "CONFIDENCE",
    "EVIDENCE_EXCERPT",
    "RATIONALE"
]

sheets = {}

for reviewer in ["A", "B"]:
    path = DIR / f"calibration_reviewer_{reviewer}.csv"
    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    assert len(df) == 18
    assert df["PILOT_ID"].is_unique
    assert all(field in df.columns for field in FIELDS)

    sheets[reviewer] = df

assert set(sheets["A"]["PILOT_ID"]) == set(
    sheets["B"]["PILOT_ID"]
)

print("=== INTEGRIDADE ===")
print("18 casos por avaliador: OK")
print("Identificadores coincidentes: OK")

for reviewer, df in sheets.items():
    print(f"\n=== AVALIADOR {reviewer} ===")

    for field in FIELDS:
        valid = df[field].str.strip().ne("")
        print(f"{field}: {valid.sum()}/18")

    completed = df[FIELDS].apply(
        lambda col: col.str.strip().ne("")
    ).all(axis=1)

    print("Casos integralmente preenchidos:",
          int(completed.sum()), "/18")

print("\nCONTROLE CONCLUÍDO")


=== INTEGRIDADE ===
18 casos por avaliador: OK
Identificadores coincidentes: OK

=== AVALIADOR A ===
RESEARCH_OBJECT: 0/18
CLAIMED_CONTRIBUTION: 0/18
CLAIM_TYPE: 0/18
NOVELTY_CLAIM: 0/18
RECOMBINATION_CLAIM: 0/18
EVIDENCE_SUFFICIENCY: 0/18
CONFIDENCE: 0/18
EVIDENCE_EXCERPT: 0/18
RATIONALE: 0/18
Casos integralmente preenchidos: 0 /18

=== AVALIADOR B ===
RESEARCH_OBJECT: 0/18
CLAIMED_CONTRIBUTION: 0/18
CLAIM_TYPE: 0/18
NOVELTY_CLAIM: 0/18
RECOMBINATION_CLAIM: 0/18
EVIDENCE_SUFFICIENCY: 0/18
CONFIDENCE: 0/18
EVIDENCE_EXCERPT: 0/18
RATIONALE: 0/18
Casos integralmente preenchidos: 0 /18

CONTROLE CONCLUÍDO


In [12]:

from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")

DIR = (
    BASE / "08_epistemic_measurement_validation"
    / "03_ground_truth" / "calibration_18"
)

df = pd.read_csv(
    DIR / "calibration_reviewer_A.csv",
    dtype=str,
    keep_default_na=False
)

for _, row in df.head(3).iterrows():

    display(Markdown(
        f"## Documento {row['PILOT_ID']}\n"
        f"**Título:** {row['TITULO']}\n\n"
        f"**Resumo:**\n\n{row['RESUMO']}\n\n"
        f"**Palavras-chave:** {row['PALAVRAS_CHAVE']}"
    ))

    print("=" * 75)


## Documento PILOT-008
**Título:** ENVOLVIMENTO DO ESTRESSE OXIDATIVO E DO INFILTRADO LINFOCÍTICO NO MICROAMBIENTE TUMORAL MAMÁRIO EM CADELAS

**Resumo:**

O CÂNCER DE MAMA É A PRINCIPAL NEOPLASIA DIAGNOSTICADA EM MULHERES E CADELAS. NO ENTANTO, A ETIOLOGIA DO TUMOR MAMÁRIO CANINO (TMC) AINDA NÃO ESTÁ ESCLARECIDA. ESTE TRABALHO TEVE COMO OBJETIVOS INVESTIGAR O NÍVEL SÉRICO DE OXIDANTES E ANTIOXIDANTES NÃO ENZIMÁTICOS, ESTUDAR O INFILTRADO CELULAR (IC) E INFILTRADO LINFOCÍTICO TUMORAL (TIL), ALÉM DA EXPRESSÃO DE BIOMARCADORES NO MICROAMBIENTE TUMORAL, BEM COMO A ASSOCIAÇÃO DE CARACTERÍSTICAS CLÍNICO-PATOLÓGICAS E TEMPO DE SOBREVIVÊNCIA DAS CADELAS. PARA TANTO, AS CADELAS FORAM DIVIDIDAS NO GRUPO COM TMC BENIGNO (BE), TMC MALIGNO (MA) E ANIMAIS CONTROLES (CO). O PROTOCOLO FOI APROVADO PELO COMITÊ DE ÉTICA PARA O USO DE ANIMAIS (CEUA-UECE), NÚMERO 12247080-2. OS NÍVEIS SÉRICOS DE PROTEÍNA TOTAL, GLOBULINA, BILIRRUBINA E MALONDIALDEÍDO (MDA) FORAM MAIORES (P < 0,05) NOS GRUPOS MA E BE COMPARADO AO GRUPO CO. TODAS AS CARACTERÍSTICAS MALIGNAS APRESENTARAM NÍVEIS ELEVADOS DE MDA E ESTAVAM ASSOCIADOS COM NECROSE E TIPO DE TUMOR (P < 0,05). NÃO FOI OBSERVADA ASSOCIAÇÃO ENTRE O MDA E O TEMPO DE SOBREVIDA GERAL. A IDADE MÉDIA DOS ANIMAIS FOI DE 9,3 ANOS, OS TUMORES ESTAVAM LOCALIZADOS NAS REGIÕES INGUINAIS E PREDOMÍNIO DE GRAU I-II DO TMC. IC ESTAVA DISTRIBUÍDO NAS REGIÕES PERI E INTRATUMORAL, TENDO LINFÓCITOS COMO PRINCIPAL POPULAÇÃO ENCONTRADA DENTRO DOS TUMORES. A CORRELAÇÃO DO IC COM O GRAU DO TMC DEMOSTROU QUE OS LINFÓCITOS (¿ = 0,28) E AS CÉLULAS PLASMÁTICAS (¿ = 0,22) APRESENTARAM UMA LIGEIRA CORRELAÇÃO POSITIVA, POR OUTRO LADO, NEUTRÓFILOS (¿ = -0,1) E MACRÓFAGOS (¿ = -0,38) APRESENTARAM CORRELAÇÃO NEGATIVA. CADELAS COM TIL MODERADO (< 800 LINFÓCITOS) TMC APRESENTARAM MAIOR TAXA DE SOBREVIVÊNCIA (P = 0,01) EM RELAÇÃO AOS ANIMAIS COM INFILTRADO LINFOCÍTICO INTENSO (= 800 LINFÓCITOS). FORAM OBSERVADOS LINFÓCITOS FOXP3 EM MENOR INTENSIDADE ENQUANTO QUE CD4 E CD8 ESTAVA EM MAIOR QUANTIDADE E CONCENTRADA NO TIL. HSP60 FOI OBSERVADO NAS CÉLULAS INFLAMATÓRIAS E TUMORAIS. NO ESTROMA TUMORAL PODE-SE OBSERVAR QUE 65,2% DOS TUMORES APRESENTAVAM BAIXO TIL E TUMORES COM ALTO TIL DIFERIU SIGNIFICATIVAMENTE (P < 0,001) EM RELAÇÃO AO BAIXO TIL. O TIL ESTROMAL FOI ASSOCIADO (P < 0,009) A EXPRESSÃO DE PD-L1 (39%) NO TMC. NAS METÁSTASES DOS LINFONODOS (N = 5), PD-L1 ESTAVA PRESENTE NO EPITÉLIO MALIGNO E ESTAVA ASSOCIADO (P < 0,034) AO ENVOLVIMENTO DE LINFONODOS REGIONAIS. AS CURVAS DE SOBREVIVÊNCIA DEMONSTRARAM MAIOR SOBREVIDA NOS ANIMAIS COM BAIXO TIL (P < 0,01) COMPARADO AO COM AUTO TIL. AS VARIÁVEIS CLÍNICO-PATOLÓGICAS CORRELACIONADAS COM SOBREVIDA GLOBAL POR ANÁLISE UNIVARIADA FORAM GRAU DE TUMOR (P < 0,009), ENVOLVIMENTO DOS LINFONODOS (P < 0,004), TIL (P < 0,016) E PD-L1+/AUTO TIL VS. PD -L1-/BAIXO TIL (P <0,01). A ANÁLISE MULTIVARIADA REVELOU QUE O GRUPO DE TUMORES COM GRAU II-III ERA UM FATOR INDEPENDENTE E PIOR PROGNÓSTICOS PARA A SOBREVIDA GLOBAL. CONCLUI-SE QUE OS TMC APRESENTAM INTENSA PEROXIDAÇÃO LIPÍDICA, PRESENÇA DE CD4, CD8 E FOXP3 NAS REGIÕES PERI E INTRATUMORAL E A EXPRESSÃO DE PD-L1 ESTÁ ASSOCIADA COM TIL ESTROMAL. ESTES MARCADORES PODEM ESTAR RELACIONADOS AO PROGNÓSTICO DOS TUMORES MAMÁRIOS CANINOS E PODEM SER UTILIZADOS COMO ALVOS PARA TERAPIAS ANTICÂNCER.

**Palavras-chave:** CÂNCER. TUMOR MAMÁRIO CANINO. ESTRESSE OXIDATIVO. LINFÓCITOS. PD-L1

## Documento PILOT-077
**Título:** UMA NOVA MEDICAÇÃO DE CANAL RADICULAR À BASE DE SILICATO DE CÁLCIO: PROPRIEDADES FÍSICO-QUÍMICAS, PENETRAÇÃO NOS TÚBULOS DENTINÁRIOS E CITOTOXICIDADE

**Resumo:**

OBJETIVO: O OBJETIVO DESTE ESTUDO FOI AVALIAR O PH, LIBERAÇÃO DE ÍONS CÁLCIO, RADIOPACIDADE E PENETRAÇÃO NOS TÚBULOS DENTINÁRIOS DE UMA NOVA MEDICAÇÃO INTRACANAL À BASE SILICATO DE CÁLCIO (BIO-C TEMP) EM COMPARAÇÃO À PASTA À BASE DE HIDRÓXIDO DE CÁLCIO (ULTRACAL XS) E VERIFICAR A CITOTOXICIDADE DO BIO-C TEMP. METODOLOGIA: PARA AVALIAÇÃO DO PH E LIBERAÇÃO DE ÍONS CÁLCIO (1, 24,72 E 168 HORAS), CINCO ESPÉCIMES PARA CADA TEMPO EXPERIMENTAL DE CADA GRUPO FORAM IMERSOS EM 10 ML DE ÁGUA DEIONIZADA E ARMAZENADOS À 37ºC PARA POSTERIOR AVALIAÇÃO EM UM PHMETRO DIGITAL E ESPECTROFOTÔMETRO, RESPECTIVAMENTE. PARA AVALIAÇÃO DA RADIOPACIDADE FOI SEGUIDA A RECOMENDAÇÃO ISO 6876:2012. A CITOTOXICIDADE FOI AVALIADA PELA PORCENTAGEM DE VIABILIDADE CELULAR, UTILIZANDO O ENSAIO MTT. IMAGENS DA PENETRAÇÃO DOS MATERIAIS NOS TÚBULOS DENTINÁRIOS FORAM OBTIDAS POR MICROSCOPIA CONFOCAL DE VARREDURA A LASER (MCVL). DADOS DE PH E LIBERAÇÃO DE ÍONS CÁLCIO E CITOTOXICIDADE FORAM ANALISADOS ESTATISTICAMENTE POR ANÁLISE DE VARIÂNCIA UMA VIA E TESTE POST HOC DE TUKEY. A RADIOPACIDADE FOI ANALISADA USANDO O TESTE T DE STUDENT. A PENETRAÇÃO NOS TÚBULOS DENTINÁRIOS FOI REALIZADA ATRAVÉS DE UMA ANÁLISE DESCRITIVA. RESULTADOS: OS MATERIAIS APRESENTARAM PH ALCALINO EM TODOS OS TEMPOS EXPERIMENTAIS. OS VALORES DE PH DA PASTA DE HIDRÓXIDO DE CÁLCIO FORAM MAIORES QUE A PASTA BIOCERÂMICA EM 1, 24 E 72 HORAS (P< 0,05). A LIBERAÇÃO DE CÁLCIO DO MATERIAL BIOCERÂMICO FOI MENOR QUE A DA PASTA DE HIDRÓXIDO DE CÁLCIO SOMENTE EM 24 HORAS (P< 0,05). A PASTA BIOCERÂMICA FOI MAIS RADIOPACA DO QUE A PASTA DE HIDRÓXIDO DE CÁLCIO (P<0,05). OS RESULTADOS DO ENSAIO MTT DEMONSTRARAM QUE A PASTA BIOCERÂMICA APRESENTOU DOSE E EFEITO CITOTÓXICO DEPENDENTE DO TEMPO. IMAGENS DE MCVL MOSTRARAM O BIO-C TEMP E O ULTRACAL XS EM TORNO DA PAREDE DO CANAL RADICULAR COM AUSÊNCIA DE PENETRAÇÃO TUBULAR. CONCLUSÕES: O NOVO MATERIAL À BASE DE SILICATO DE CÁLCIO APRESENTOU AS PROPRIEDADES FÍSICO-QUÍMICAS ESTUDADAS COMPATÍVEIS PARA SER EMPREGADA COMO MEDICAÇÃO INTRACANAL. O BIO-C TEMP APRESENTOU UMA DOSE E EFEITO CITOTÓXICO DEPENDENTE DO TEMPO, COM AUSÊNCIA DE PENETRAÇÃO NO TÚBULO DENTINÁRIO.

**Palavras-chave:** ENDODONTIA;BIOCERÂMICO;SILICATO DE CÁLCIO;CITOTOXICIDADE;PROPRIEDADES FÍSICO-QUÍMICAS;MEDICAÇÃO INTRACANAL

## Documento PILOT-090
**Título:** TESTES ADAPTATIVOS INFORMATIZADOS (TAI) E DESAFIOS DAS POLÍTICAS DE AVALIAÇÃO DE LEITURA NOS ANOS INICIAIS: UM MODELO DE ANÁLISE DE ALTERNATIVAS EM ITENS DE MÚLTIPLA ESCOLHA COMO CONTRIBUIÇÃO PARA O SUCESSO ESCOLAR.

**Resumo:**

ESTA DISSERTAÇÃO RESULTA DE UMA PESQUISA CUJO OBJETIVO PRINCIPAL FOI O DESENVOLVIMENTO DE UM TESTE ADAPTATIVO INFORMATIZADO (TAI), COM BASE NOS ITENS DA PROVINHA BRASIL (PB), PARA ESTIMAR A PROFICIÊNCIA EM LEITURA NOS ANOS INICIAIS DO ENSINO FUNDAMENTAL, CONSIDERANDO, EM SEUS ALGORITMOS, A DIFICULDADE DE CADA UMA DAS ALTERNATIVAS DOS ITENS PARA SUAS ANÁLISES E DESDOBRAMENTOS NO ENSINO DA LEITURA. COMO UM DOS GRANDES DESAFIOS DAS POLÍTICAS EDUCACIONAIS BRASILEIRAS, AINDA ENCONTRAMOS AMPLOS CONTINGENTES DE ALUNOS, ESPECIALMENTE NOS ANOS INICIAIS DO ENSINO FUNDAMENTAL, COM PROFUNDAS LIMITAÇÕES EM SUA COMPETÊNCIA LEITORA, ALGO QUE COMPROMETE A PRÓPRIA ESCOLARIZAÇÃO, ADEMAIS DE GRAVES CONSEQUÊNCIAS SOCIAIS. DAS VÁRIAS POSSIBILIDADES DE SUPERAR ESSE DESAFIO, CONSIDERA-SE A NECESSIDADE DE AVALIAR A APRENDIZAGEM EM LEITURA, EM PATAMARES CONSISTENTES, COM UM INSTRUMENTO QUE SEJA UM PONTO DE APOIO PARA PROCESSOS DE ENSINO. SENDO ASSIM, CONSIDERAMOS O POTENCIAL DOS TAI, QUE SE COLOCAM COMO UMA FERRAMENTA EXTREMAMENTE ÚTIL PARA O TRABALHO DOCENTE, SOBRETUDO, PORQUE, AO APOIAR A PRÁTICA AVALIATIVA, PODE FAVORECER O DESENVOLVIMENTO DO ENSINO MAIS AJUSTADO AOS ALUNOS E SE CONSTITUÍ NUM EXPEDIENTE POLÍTICO, EM FACE DO QUADRO NACIONAL DE DOMÍNIO DA LEITURA. EM UM TESTE ADAPTATIVO INFORMATIZADO (TAI) OS ITENS SÃO SELECIONADOS E APRESENTADOS AOS RESPONDENTES DE ACORDO COM AS RESPOSTAS DADAS ANTERIORMENTE E, NESSE SENTIDO, PROCURAMOS A RELAÇÃO QUE POSSA EXISTIR ENTRE A ESCOLHA DE CADA ALTERNATIVA E A APROXIMAÇÃO MAIS RÁPIDA DA ESTIMATIVA DA PROFICIÊNCIA, UMA DAS CONDIÇÕES DE FINALIZAÇÃO DE UMA SESSÃO DE APLICAÇÃO DO TESTE, ALÉM DE UMA INTERPRETAÇÃO PEDAGÓGICA MAIS CONSISTENTE, POIS CONSIDERA O DOMÍNIO DA LEITURA MESMO DIANTE DE EVENTUAIS ERROS, O QUE TORNA O TESTE E SEUS RESULTADOS UM FERRAMENTA MAIS SÓLIDA PARA DAR SUPORTE ÀS AÇÕES REALIZADAS NA SALA DE AULA.

**Palavras-chave:** AVALIAÇÃO DA APRENDIZAGEM;LEITURA;PROFICIÊNCIA EM LEITURA;TESTE ADAPTATIVO INFORMATIZADO (TAI);TEORIA DA RESPOSTA AO ITEM (TRI);MODELO ANINHADO;TEORIA CLÁSSICA DOS TESTES (TCT)

In [13]:

# SCIENTOMETRICS 2026 — ETAPA 11A
# Preparação da recuperação contextual
# CPU | Sem novos embeddings

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import json
from itertools import islice

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

PILOT = LAB / "03_ground_truth" / "pilot_90"
CAL = LAB / "03_ground_truth" / "calibration_18"
OUT = LAB / "04_antecedent_retrieval"

OUT.mkdir(parents=True, exist_ok=True)

# 1. Recuperar a seleção de calibração.
key = pd.read_csv(
    PILOT / "pilot_linkage_key_RESTRICTED.csv",
    dtype=str
)

selected = pd.read_csv(
    CAL / "calibration_selection_RESTRICTED.csv",
    dtype=str
)

cases = selected.merge(
    key[["PILOT_ID", "ID_TRABALHO", "ANO"]],
    on="PILOT_ID",
    validate="one_to_one"
)

assert len(cases) == 18
assert cases["PILOT_ID"].is_unique

cases["ANO"] = cases["ANO"].astype(int)
cases["HAS_PRIOR_CAPES"] = cases["ANO"] > 2013
cases["N_PRIOR_YEARS"] = cases["ANO"] - 2013

print("=== ELEGIBILIDADE TEMPORAL ===")
print(cases[
    ["PILOT_ID", "ANO", "HAS_PRIOR_CAPES", "N_PRIOR_YEARS"]
].to_string(index=False))

print("\nCasos com antecedentes possíveis:",
      int(cases["HAS_PRIOR_CAPES"].sum()))

print("Casos em 2013:",
      int((cases["ANO"] == 2013).sum()))

# 2. Inspecionar arquivos de vizinhos existentes.
SOURCES = {
    "historical_top50":
        BASE / "21_novidade_semantica_prior_only"
        / "01_vizinhos_top50",

    "frontier_top50":
        BASE / "22_frontier_novelty_3y"
        / "01_vizinhos_top50",
}

inventory = []

for name, folder in SOURCES.items():

    print(f"\n=== {name} ===")

    if not folder.is_dir():
        print("DIRETORIO AUSENTE")
        continue

    entries = list(islice(folder.iterdir(), 25))

    for path in entries:

        print("DIR " if path.is_dir() else "FILE",
              path.name)

        info = {
            "source": name,
            "path": str(path),
            "name": path.name,
            "type": "directory" if path.is_dir() else "file",
        }

        if path.is_file() and path.suffix == ".parquet":
            try:
                pf = pq.ParquetFile(path)
                info["rows"] = pf.metadata.num_rows
                info["columns"] = pf.schema_arrow.names

                print("  ROWS:", info["rows"])
                print("  COLUMNS:", info["columns"])

            except Exception as exc:
                info["read_error"] = str(exc)

        inventory.append(info)

# 3. Protocolo contextual.
protocol = {
    "version": "0.2",
    "study": "Bibliometric Measures of Epistemic Change",
    "phase": "antecedent_retrieval_design",
    "corpus_start": 2013,
    "corpus_end": 2024,
    "temporal_rule": "antecedent_year < focal_year",
    "retrieval_families": [
        "lexical_title_keywords_BM25",
        "disciplinary_prior_sampling",
        "semantic_prior_neighbors"
    ],
    "lexical_fields": [
        "TITULO",
        "PALAVRAS_CHAVE"
    ],
    "contextual_assessment": {
        "continuity": "Requires comparison with prior work",
        "novelty": "Requires identifiable difference from prior work",
        "recombination": "Requires documented integration of prior approaches",
        "undetermined": "Insufficient or conflicting evidence"
    },
    "rules": {
        "no_future_antecedents": True,
        "no_ETS_score_based_selection": True,
        "preserve_retrieval_provenance": True,
        "human_ground_truth_required": True,
        "2013_requires_external_pre2013_context": True,
        "pilot_is_not_confirmatory": True
    }
}

manifest_path = OUT / "retrieval_protocol_v02.json"
inventory_path = OUT / "neighbor_source_inventory.json"
cases_path = OUT / "calibration_temporal_eligibility.csv"

# Evitar substituição silenciosa de arquivos.
for p in [manifest_path, inventory_path, cases_path]:
    if p.exists():
        raise FileExistsError(
            f"Arquivo já existente: {p}. "
            "Preservar a versão anterior."
        )

manifest_path.write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

inventory_path.write_text(
    json.dumps(inventory, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

cases.to_csv(cases_path, index=False)

print("\n=== ETAPA 11A CONCLUÍDA ===")
print("Arquivos de vizinhos inspecionados:", len(inventory))
print("Protocolo:", manifest_path.name)
print("Inventário:", inventory_path.name)
print("Elegibilidade:", cases_path.name)


=== ELEGIBILIDADE TEMPORAL ===
 PILOT_ID  ANO  HAS_PRIOR_CAPES  N_PRIOR_YEARS
PILOT-008 2017             True              4
PILOT-077 2019             True              6
PILOT-090 2023             True             10
PILOT-054 2024             True             11
PILOT-002 2023             True             10
PILOT-040 2022             True              9
PILOT-078 2014             True              1
PILOT-070 2016             True              3
PILOT-010 2019             True              6
PILOT-043 2018             True              5
PILOT-022 2024             True             11
PILOT-071 2022             True              9
PILOT-020 2017             True              4
PILOT-012 2019             True              6
PILOT-067 2023             True             10
PILOT-066 2020             True              7
PILOT-005 2023             True             10
PILOT-082 2018             True              5

Casos com antecedentes possíveis: 18
Casos em 2013: 0

=== historical_top50

In [14]:

# SCIENTOMETRICS 2026 — ETAPA 11B
# Elegibilidade dos antecedentes e auditoria de índices
# CPU | Sem embeddings | Sem busca exaustiva

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
OUT = LAB / "04_antecedent_retrieval"
OUT.mkdir(parents=True, exist_ok=True)

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]

key = pd.read_csv(
    LAB / "03_ground_truth/pilot_90/"
    "pilot_linkage_key_RESTRICTED.csv",
    dtype=str
)

focals = key[
    key["PILOT_ID"].isin(FOCAL_IDS)
].copy()

assert len(focals) == 3
assert focals["PILOT_ID"].is_unique

focals["ANO"] = focals["ANO"].astype(int)

INDEX = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

# Somente colunas necessárias, sem carregar textos.
cols = [
    "ID_TRABALHO", "ANO", "AREA_HARMONIZADA",
    "NM_GRANDE_AREA_CONHECIMENTO", "NM_IDIOMA"
]

meta = pq.read_table(INDEX, columns=cols).to_pandas()

assert meta["ID_TRABALHO"].is_unique

focals = focals[["PILOT_ID", "ID_TRABALHO", "ANO"]].merge(
    meta.drop(columns=["ANO"]),
    on="ID_TRABALHO",
    validate="one_to_one",
    how="left"
)

assert focals["AREA_HARMONIZADA"].notna().all()

print("=== DOCUMENTOS FOCAIS ===")
print(focals.to_string(index=False))

counts = []

for r in focals.itertuples(index=False):
    prior = meta[meta["ANO"] < r.ANO]

    same = prior[
        prior["AREA_HARMONIZADA"] == r.AREA_HARMONIZADA
    ]

    counts.append({
        "PILOT_ID": r.PILOT_ID,
        "FOCAL_YEAR": r.ANO,
        "AREA": r.AREA_HARMONIZADA,
        "N_PRIOR_NATIONAL": len(prior),
        "N_PRIOR_SAME_AREA": len(same),
        "N_PRIOR_OTHER_AREAS": len(prior) - len(same),
        "N_PRIOR_LAST3_SAME_AREA": int(
            same["ANO"].ge(r.ANO - 3).sum()
        )
    })

coverage = pd.DataFrame(counts)

print("\n=== COBERTURA DOS ANTECEDENTES ===")
print(coverage.to_string(index=False))

# Auditoria leve: apenas metadados NPY
ranking_info = []

for r in focals.itertuples(index=False):

    for family, folder, pattern in [
        (
            "historical",
            "21_novidade_semantica_prior_only/01_vizinhos_top50",
            f"M4_{r.ANO}_prior_top50_indices_int32.npy"
        ),
        (
            "frontier3y",
            "22_frontier_novelty_3y/01_vizinhos_top50",
            f"M4_{r.ANO}_frontier3y_top50_indices_int32.npy"
        )
    ]:

        path = BASE / folder / pattern

        if not path.is_file():
            ranking_info.append({
                "PILOT_ID": r.PILOT_ID,
                "FAMILY": family,
                "FOUND": False
            })
            continue

        # Leitura somente do cabeçalho e metadados.
        arr = np.load(path, mmap_mode="r")

        ranking_info.append({
            "PILOT_ID": r.PILOT_ID,
            "FAMILY": family,
            "FOUND": True,
            "SHAPE": str(arr.shape),
            "DTYPE": str(arr.dtype),
            "FILE": str(path)
        })

rank_df = pd.DataFrame(ranking_info)

print("\n=== ESTRUTURA DOS RANKINGS ===")
print(rank_df[
    ["PILOT_ID", "FAMILY", "FOUND", "SHAPE", "DTYPE"]
].to_string(index=False))

# Inspecionar metadados de correspondência global.
ref_path = (
    BASE / "21_novidade_semantica_prior_only"
    / "global_reference_map_2013_2024.parquet"
)

if ref_path.is_file():
    pf = pq.ParquetFile(ref_path)

    print("\n=== GLOBAL REFERENCE MAP ===")
    print("ROWS:", pf.metadata.num_rows)
    print("COLUMNS:", pf.schema_arrow.names)

# Não sobrescrever resultados congelados.
for filename in [
    "focal_metadata_3.csv",
    "antecedent_coverage_3.csv",
    "ranking_structure_3.csv"
]:
    if (OUT / filename).exists():
        raise FileExistsError(
            f"Arquivo existente: {filename}. "
            "Preservar resultados anteriores."
        )

focals.to_csv(OUT / "focal_metadata_3.csv", index=False)
coverage.to_csv(OUT / "antecedent_coverage_3.csv", index=False)
rank_df.to_csv(OUT / "ranking_structure_3.csv", index=False)

print("\nETAPA 11B CONCLUÍDA")


=== DOCUMENTOS FOCAIS ===
 PILOT_ID ID_TRABALHO  ANO     AREA_HARMONIZADA NM_GRANDE_AREA_CONHECIMENTO NM_IDIOMA
PILOT-008    29591925 2017 MEDICINA VETERINARIA           CIÊNCIAS AGRÁRIAS PORTUGUES
PILOT-077    28076603 2019          ODONTOLOGIA           CIÊNCIAS DA SAÚDE PORTUGUES
PILOT-090    38635596 2023             EDUCACAO            CIÊNCIAS HUMANAS PORTUGUES

=== COBERTURA DOS ANTECEDENTES ===
 PILOT_ID  FOCAL_YEAR                 AREA  N_PRIOR_NATIONAL  N_PRIOR_SAME_AREA  N_PRIOR_OTHER_AREAS  N_PRIOR_LAST3_SAME_AREA
PILOT-008        2017 MEDICINA VETERINARIA            293659               5743               287916                     4337
PILOT-077        2019          ODONTOLOGIA            469091              12334               456757                     6608
PILOT-090        2023             EDUCACAO            810010              51785               758225                    16629

=== ESTRUTURA DOS RANKINGS ===
 PILOT_ID     FAMILY  FOUND       SHAPE DTYPE
PILOT-008 hi

In [17]:

# SCIENTOMETRICS 2026 — ETAPA 11B
# Correção do salvamento sem sobrescrever arquivos

from pathlib import Path
from datetime import datetime
import pandas as pd

OUT = (
    Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
    / "08_epistemic_measurement_validation"
    / "04_antecedent_retrieval"
)

OUT.mkdir(parents=True, exist_ok=True)

# Verificar se os resultados da célula anterior
# continuam disponíveis na sessão.
for name in ["focals", "coverage", "rank_df"]:
    assert name in globals(), (
        f"Variável {name} não encontrada. "
        "A sessão pode ter sido reiniciada."
    )

# Criar uma identificação única para esta execução.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

outputs = {
    "focal_metadata_3": focals,
    "antecedent_coverage_3": coverage,
    "ranking_structure_3": rank_df,
}

print("=== SALVAMENTO SEGURO ===")

for name, df in outputs.items():

    original = OUT / f"{name}.csv"

    if original.exists():
        target = OUT / f"{name}_{timestamp}.csv"
    else:
        target = original

    # Impede substituição silenciosa.
    df.to_csv(
        target,
        index=False,
        mode="x"
    )

    print("SALVO:", target.name)

print("\nETAPA 11B FINALIZADA.")
print("Nenhum arquivo anterior foi sobrescrito.")


=== SALVAMENTO SEGURO ===
SALVO: focal_metadata_3_20261020_172210.csv
SALVO: antecedent_coverage_3_20261020_172210.csv
SALVO: ranking_structure_3_20261020_172210.csv

ETAPA 11B FINALIZADA.
Nenhum arquivo anterior foi sobrescrito.


In [18]:

# SCIENTOMETRICS 2026 — ETAPA 12A
# BM25 disciplinar com antecedentes estritamente anteriores
# CPU | Sem GPU | Sem embeddings

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import unicodedata
import re
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import CountVectorizer

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
SOURCE = BASE / "05_transferencia_colab"
OUT_ROOT = LAB / "04_antecedent_retrieval"

RUN_ID = datetime.now(timezone.utc).strftime(
    "bm25_%Y%m%dT%H%M%SZ"
)
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]
TOP_K = 30
K1 = 1.5
B = 0.75

# ---------------------------------------------------
# 1. Recuperar documentos focais
# ---------------------------------------------------

key = pd.read_csv(
    LAB / "03_ground_truth/pilot_90/"
    "pilot_linkage_key_RESTRICTED.csv",
    dtype=str
)

texts = pd.read_csv(
    LAB / "03_ground_truth/pilot_90/"
    "pilot_cases_blinded.csv",
    dtype=str
)

focals = (
    key[key["PILOT_ID"].isin(FOCAL_IDS)]
    .merge(texts, on="PILOT_ID", validate="one_to_one")
)

assert len(focals) == 3
assert focals["PILOT_ID"].is_unique

focals["ANO"] = focals["ANO"].astype(int)

# ---------------------------------------------------
# 2. Metadados e elegibilidade temporal
# ---------------------------------------------------

index_path = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

meta = pq.read_table(
    index_path,
    columns=["ID_TRABALHO", "ANO", "AREA_HARMONIZADA"]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)

assert meta["ID_TRABALHO"].is_unique

eligible_parts = []

for focal in focals.itertuples(index=False):

    subset = meta[
        (meta["ANO"] < focal.ANO) &
        (meta["AREA_HARMONIZADA"] ==
         focal.NM_GRANDE_AREA_CONHECIMENTO)
    ]

    # Atenção: grande área e área harmonizada não
    # são equivalentes. Área específica virá da chave.
    # O filtro correto é definido abaixo.

focals_meta = focals.merge(
    meta[["ID_TRABALHO", "AREA_HARMONIZADA"]],
    on="ID_TRABALHO",
    validate="one_to_one"
)

for focal in focals_meta.itertuples(index=False):

    subset = meta[
        (meta["ANO"] < focal.ANO) &
        (meta["AREA_HARMONIZADA"] ==
         focal.AREA_HARMONIZADA)
    ].copy()

    subset["PILOT_ID"] = focal.PILOT_ID

    eligible_parts.append(subset)

eligible = pd.concat(eligible_parts, ignore_index=True)

assert eligible["ID_TRABALHO"].is_unique, (
    "As três áreas produziram IDs sobrepostos. "
    "Revisar a elegibilidade."
)

print("=== UNIVERSOS ELEGÍVEIS ===")
print(eligible.groupby("PILOT_ID").size())

# ---------------------------------------------------
# 3. Recuperar textos por ano, sem ler resumos
# ---------------------------------------------------

year_to_ids = {
    int(year): set(group["ID_TRABALHO"])
    for year, group in eligible.groupby("ANO")
}

collected = []

for year, selected_ids in sorted(year_to_ids.items()):

    path = SOURCE / (
        f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    )

    pf = pq.ParquetFile(path)

    frames = []

    for batch in pf.iter_batches(
        batch_size=16384,
        columns=[
            "ID_TRABALHO",
            "TITULO",
            "PALAVRAS_CHAVE"
        ]
    ):
        chunk = batch.to_pandas()
        chunk["ID_TRABALHO"] = (
            chunk["ID_TRABALHO"].astype(str)
        )

        chunk = chunk[
            chunk["ID_TRABALHO"].isin(selected_ids)
        ]

        if not chunk.empty:
            frames.append(chunk)

    if frames:
        year_data = pd.concat(
            frames, ignore_index=True
        )
        year_data["ANO"] = year
        collected.append(year_data)

    print(f"{year}: {sum(map(len, frames))} recuperados")

corpus = pd.concat(collected, ignore_index=True)

assert corpus["ID_TRABALHO"].is_unique

corpus = eligible.merge(
    corpus,
    on=["ID_TRABALHO", "ANO"],
    how="left",
    validate="one_to_one",
    indicator=True
)

assert (corpus["_merge"] == "both").all(), (
    "Nem todos os antecedentes foram encontrados."
)

corpus = corpus.drop(columns="_merge")

assert corpus["ID_TRABALHO"].notna().all()

# ---------------------------------------------------
# 4. Normalização lexical
# ---------------------------------------------------

STOP = {
    "a", "as", "o", "os", "um", "uma", "uns", "umas",
    "de", "da", "do", "das", "dos", "em", "no", "na",
    "nos", "nas", "por", "para", "com", "sem",
    "e", "ou", "que", "se", "ao", "aos", "sua",
    "suas", "seu", "seus", "sobre", "entre",
    "the", "and", "of", "in", "for", "on",
    "with", "to", "an", "by", "from"
}

def tokenize(text):
    text = unicodedata.normalize(
        "NFKD", str(text).lower()
    )
    text = "".join(
        c for c in text if not unicodedata.combining(c)
    )
    tokens = re.findall(r"[a-z0-9]+", text)
    return [
        t for t in tokens
        if len(t) > 1 and t not in STOP
    ]

def document_text(df):
    return (
        df["TITULO"].fillna("").astype(str)
        + " "
        + df["PALAVRAS_CHAVE"].fillna("").astype(str)
    )

# ---------------------------------------------------
# 5. BM25
# ---------------------------------------------------

def bm25_rank(documents, query, top_k=30):

    vec = CountVectorizer(
        tokenizer=tokenize,
        token_pattern=None,
        lowercase=False,
        dtype=np.float32
    )

    X = vec.fit_transform(documents).tocsc()
    N = X.shape[0]

    lengths = np.asarray(
        X.sum(axis=1)
    ).ravel()

    avgdl = float(lengths.mean())

    query_terms = set(tokenize(query))
    vocabulary = vec.vocabulary_

    scores = np.zeros(N, dtype=np.float64)

    for term in query_terms:

        if term not in vocabulary:
            continue

        col = X.getcol(vocabulary[term]).tocoo()

        df = len(col.data)

        idf = np.log(
            1 + (N - df + 0.5) / (df + 0.5)
        )

        tf = col.data.astype(np.float64)
        document_lengths = lengths[col.row]

        denom = (
            tf + K1 *
            (1 - B + B * document_lengths / avgdl)
        )

        scores[col.row] += (
            idf * tf * (K1 + 1) / denom
        )

    order = np.argsort(-scores, kind="stable")[:top_k]

    return order, scores[order]

rankings = []

for focal in focals_meta.itertuples(index=False):

    candidates = corpus[
        corpus["PILOT_ID"] == focal.PILOT_ID
    ].reset_index(drop=True)

    query = (
        str(focal.TITULO) + " " +
        str(focal.PALAVRAS_CHAVE)
    )

    positions, scores = bm25_rank(
        document_text(candidates).tolist(),
        query,
        TOP_K
    )

    result = candidates.iloc[positions].copy()

    result.insert(0, "RANK", range(1, len(result) + 1))
    result.insert(1, "BM25_SCORE", scores)

    result = result.rename(columns={
        "ID_TRABALHO": "ANTECEDENT_ID",
        "ANO": "ANTECEDENT_YEAR",
        "TITULO": "ANTECEDENT_TITLE",
        "PALAVRAS_CHAVE": "ANTECEDENT_KEYWORDS"
    })

    result["FOCAL_ID"] = focal.PILOT_ID
    result["FOCAL_YEAR"] = focal.ANO
    result["RETRIEVAL_METHOD"] = "BM25"
    result["RETRIEVAL_SCOPE"] = "prior_same_area"

    assert (
        result["ANTECEDENT_YEAR"] <
        result["FOCAL_YEAR"]
    ).all()

    rankings.append(result)

    print(
        "\n", focal.PILOT_ID,
        "| candidatos:", len(candidates),
        "| top:", len(result)
    )

ranking = pd.concat(rankings, ignore_index=True)

# ---------------------------------------------------
# 6. Salvar artefatos e parâmetros
# ---------------------------------------------------

ranking.to_csv(
    OUT / "bm25_top30_three_focals.csv",
    index=False,
    encoding="utf-8-sig"
)

ranking.to_parquet(
    OUT / "bm25_top30_three_focals.parquet",
    index=False
)

manifest = {
    "run_id": RUN_ID,
    "method": "BM25",
    "query_fields": ["TITULO", "PALAVRAS_CHAVE"],
    "document_fields": ["TITULO", "PALAVRAS_CHAVE"],
    "k1": K1,
    "b": B,
    "top_k": TOP_K,
    "scope": "same_harmonized_area_prior_years",
    "temporal_rule": "antecedent_year < focal_year",
    "retrieval_independent_of_ETS_scores": True,
    "ground_truth": False,
    "focal_ids": FOCAL_IDS,
    "n_candidates": {
        f: int(n)
        for f, n in eligible.groupby("PILOT_ID").size().items()
    }
}

(OUT / "bm25_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ETAPA 12A CONCLUÍDA ===")
print("Antecedentes classificados:", len(ranking))
print("Diretório:", OUT)


=== UNIVERSOS ELEGÍVEIS ===
PILOT_ID
PILOT-008     5743
PILOT-077    12334
PILOT-090    51785
dtype: int64
2013: 7331 recuperados
2014: 7504 recuperados
2015: 8132 recuperados
2016: 8682 recuperados
2017: 7702 recuperados
2018: 7869 recuperados
2019: 6013 recuperados
2020: 5363 recuperados
2021: 5664 recuperados
2022: 5602 recuperados

 PILOT-008 | candidatos: 5743 | top: 30

 PILOT-077 | candidatos: 12334 | top: 30

 PILOT-090 | candidatos: 51785 | top: 30

=== ETAPA 12A CONCLUÍDA ===
Antecedentes classificados: 90
Diretório: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/04_antecedent_retrieval/bm25_20261020T172459Z


In [19]:

# SCIENTOMETRICS 2026 — ETAPA 12B
# Auditoria dos rankings BM25
# CPU | Sem GPU | Sem embeddings

from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")

RUN = (
    BASE / "08_epistemic_measurement_validation"
    / "04_antecedent_retrieval"
    / "bm25_20261020T172459Z"
)

FILE = RUN / "bm25_top30_three_focals.parquet"

assert FILE.is_file(), "Ranking BM25 nao encontrado."

df = pd.read_parquet(FILE)

required = [
    "FOCAL_ID",
    "FOCAL_YEAR",
    "ANTECEDENT_ID",
    "ANTECEDENT_YEAR",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_KEYWORDS",
    "BM25_SCORE",
    "RANK"
]

assert all(c in df.columns for c in required)

df["BM25_SCORE"] = pd.to_numeric(
    df["BM25_SCORE"], errors="raise"
)

df["FOCAL_YEAR"] = pd.to_numeric(
    df["FOCAL_YEAR"], errors="raise"
)

df["ANTECEDENT_YEAR"] = pd.to_numeric(
    df["ANTECEDENT_YEAR"], errors="raise"
)

df["LAG_YEARS"] = (
    df["FOCAL_YEAR"] - df["ANTECEDENT_YEAR"]
)

# Auditoria de integridade
assert len(df) == 90
assert np.isfinite(df["BM25_SCORE"]).all()
assert (df["LAG_YEARS"] > 0).all()
assert not df.duplicated(
    ["FOCAL_ID", "ANTECEDENT_ID"]
).any()

print("=" * 70)
print("AUDITORIA GLOBAL")
print("=" * 70)

print("Documentos recuperados:", len(df))
print("IDs antecedentes distintos:", df["ANTECEDENT_ID"].nunique())
print("Pontuacoes nulas:", int((df["BM25_SCORE"] == 0).sum()))
print("Precedencia temporal:", "OK")

for focal_id, group in df.groupby("FOCAL_ID"):

    group = group.sort_values("RANK")

    print("\n" + "=" * 70)
    print("DOCUMENTO FOCAL:", focal_id)
    print("=" * 70)

    print("Antecedentes:", len(group))
    print("BM25 max:", round(group["BM25_SCORE"].max(), 4))
    print("BM25 min:", round(group["BM25_SCORE"].min(), 4))
    print("Lag mediano:", group["LAG_YEARS"].median())

    print("\nTOP 10 — ANTECEDENTES")

    for r in group.head(10).itertuples():
        print(f"\nRANK {r.RANK}")
        print("Ano:", r.ANTECEDENT_YEAR)
        print("BM25:", round(r.BM25_SCORE, 4))
        print("Titulo:", r.ANTECEDENT_TITLE)
        print("Palavras-chave:", r.ANTECEDENT_KEYWORDS)

    print("\n" + "-" * 70)

# Salvar auditoria em arquivo novo
audit = (
    df.groupby("FOCAL_ID")
    .agg(
        N_ANTECEDENTS=("ANTECEDENT_ID", "size"),
        BM25_MAX=("BM25_SCORE", "max"),
        BM25_MIN=("BM25_SCORE", "min"),
        BM25_MEDIAN=("BM25_SCORE", "median"),
        LAG_MEDIAN=("LAG_YEARS", "median"),
        LAG_MIN=("LAG_YEARS", "min"),
        LAG_MAX=("LAG_YEARS", "max")
    )
    .reset_index()
)

OUT = RUN / "bm25_diagnostic_summary_v1.csv"

if OUT.exists():
    print("\nAuditoria ja salva:", OUT.name)
else:
    audit.to_csv(OUT, index=False)
    print("\nAuditoria salva:", OUT.name)

print("\nETAPA 12B CONCLUIDA")


AUDITORIA GLOBAL
Documentos recuperados: 90
IDs antecedentes distintos: 90
Pontuacoes nulas: 0
Precedencia temporal: OK

DOCUMENTO FOCAL: PILOT-008
Antecedentes: 30
BM25 max: 32.5936
BM25 min: 11.0657
Lag mediano: 2.0

TOP 10 — ANTECEDENTES

RANK 1
Ano: 2016
BM25: 32.5936
Titulo: TUMOR VENÉREO TRANSMISSÍVEL CANINO: EXPRESSÃO DE GENES RELACIONADOS COM O COMPORTAMENTO BIOLÓGICO E MICROAMBIENTE TUMORAL
Palavras-chave: CÂNCER;MICROAMBIENTE TUMORAL;TVT;CÃO

RANK 2
Ano: 2014
BM25: 26.6162
Titulo: CINÉTICA DE MIGRAÇÃO DE CÉLULAS DO SISTEMA IMUNE APÓS DESAFIO COM CÉLULAS DE ADENOCARCINOMA MAMÁRIO 4T1
Palavras-chave: ADENOCARCINOMA MAMÁRIO 4T1. MICROAMBIENTE TUMORAL. LINFÓCITOS B-1.

RANK 3
Ano: 2013
BM25: 18.3665
Titulo: DETECÇÃO DE CONJUGADOS DE LECITINAS EM NEOPLASIAS MAMÁRIAS PRIMÁRIAS DE CADELAS E EM METÁSTASES
Palavras-chave: TUMOR MAMÁRIO CANINO, ONCOLOGIA VETERINÁRIA, UEA, PNA, HPA, DBA

RANK 4
Ano: 2016
BM25: 17.4741
Titulo: CD279 (PD-1) E SEUS LIGANTES CD273 (PD-L1) E CD274 (PD-L2) RE

In [20]:

# SCIENTOMETRICS 2026 — ETAPA 13
# Recuperação dos resumos e fichas de relevância
# CPU | Sem GPU | Sem inferência automática

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import pyarrow.parquet as pq
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
SRC = BASE / "05_transferencia_colab"

RUN = (
    LAB / "04_antecedent_retrieval"
    / "bm25_20261020T172459Z"
)

ranking = pd.read_parquet(
    RUN / "bm25_top30_three_focals.parquet"
)

assert len(ranking) == 90
assert not ranking.duplicated(
    ["FOCAL_ID", "ANTECEDENT_ID"]
).any()

ranking["ANTECEDENT_ID"] = (
    ranking["ANTECEDENT_ID"].astype(str)
)

# Nova execução versionada.
RUN_ID = datetime.now(timezone.utc).strftime(
    "context_%Y%m%dT%H%M%S_%fZ"
)

OUT = LAB / "04_antecedent_retrieval" / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

# Recuperar resumos por ano.
texts = []

for year, group in ranking.groupby("ANTECEDENT_YEAR"):

    year = int(year)
    wanted = set(group["ANTECEDENT_ID"])

    path = SRC / f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    assert path.is_file()

    pf = pq.ParquetFile(path)
    selected = []

    for batch in pf.iter_batches(
        batch_size=16384,
        columns=["ID_TRABALHO", "RESUMO"]
    ):
        chunk = batch.to_pandas()

        chunk["ID_TRABALHO"] = (
            chunk["ID_TRABALHO"].astype(str)
        )

        hit = chunk[
            chunk["ID_TRABALHO"].isin(wanted)
        ]

        if not hit.empty:
            selected.append(hit.copy())

    if selected:
        found = pd.concat(selected, ignore_index=True)
        found["ANTECEDENT_YEAR"] = year
        texts.append(found)

    print(year, "| solicitados:", len(wanted),
          "| recuperados:",
          sum(map(len, selected)))

text_df = pd.concat(texts, ignore_index=True)

assert not text_df.duplicated(
    ["ID_TRABALHO", "ANTECEDENT_YEAR"]
).any()

text_df = text_df.rename(columns={
    "ID_TRABALHO": "ANTECEDENT_ID",
    "RESUMO": "ANTECEDENT_ABSTRACT"
})

full = ranking.merge(
    text_df,
    on=["ANTECEDENT_ID", "ANTECEDENT_YEAR"],
    how="left",
    validate="many_to_one",
    indicator=True
)

assert (full["_merge"] == "both").all(), (
    "Existem antecedentes sem texto correspondente."
)

full = full.drop(columns="_merge")

assert (
    full["ANTECEDENT_YEAR"].astype(int)
    < full["FOCAL_YEAR"].astype(int)
).all()

# Recuperar textos dos três documentos focais.
pilot = pd.read_csv(
    LAB / "03_ground_truth/pilot_90/"
    "pilot_cases_blinded.csv",
    dtype=str,
    keep_default_na=False
)

focal_texts = pilot.rename(columns={
    "PILOT_ID": "FOCAL_ID",
    "TITULO": "FOCAL_TITLE",
    "RESUMO": "FOCAL_ABSTRACT",
    "PALAVRAS_CHAVE": "FOCAL_KEYWORDS"
})

full = full.merge(
    focal_texts,
    on="FOCAL_ID",
    how="left",
    validate="many_to_one"
)

assert full["FOCAL_ABSTRACT"].notna().all()

# Arquivo interno: preservar escores e proveniência.
full.to_parquet(
    OUT / "antecedent_context_INTERNAL.parquet",
    index=False
)

# Material para anotação, sem ranking nem escores.
# A ordem dos pares é embaralhada.
review = full[[
    "FOCAL_ID",
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]].copy()

review.insert(
    0, "PAIR_ID",
    [f"PAIR-{i:04d}" for i in range(1, len(review)+1)]
)

link = review[["PAIR_ID"]].copy()
link["FOCAL_ID"] = full["FOCAL_ID"].values
link["ANTECEDENT_ID"] = full["ANTECEDENT_ID"].values
link["ANTECEDENT_YEAR"] = full["ANTECEDENT_YEAR"].values
link["BM25_RANK"] = full["RANK"].values
link["BM25_SCORE"] = full["BM25_SCORE"].values

link.to_csv(
    OUT / "pair_linkage_RESTRICTED.csv",
    index=False
)

review = review.sample(
    frac=1, random_state=20261020
).reset_index(drop=True)

fields = [
    "RELEVANCE_0_1_2_3_U",
    "CONTRIBUTION_LOCUS",
    "IS_PLAUSIBLE_PRECEDENT",
    "EVIDENCE_SUFFICIENCY",
    "CONFIDENCE_1_TO_5",
    "EVIDENCE_EXCERPT",
    "RATIONALE"
]

for reviewer in ["A", "B"]:

    sheet = review.copy()

    for field in fields:
        sheet[field] = ""

    sheet["REVIEWER"] = reviewer

    sheet.to_csv(
        OUT / f"context_review_{reviewer}.csv",
        index=False,
        encoding="utf-8-sig"
    )

manifest = {
    "study": "Epistemic Change Validation",
    "stage": "contextual_relevance_calibration",
    "source_method": "BM25",
    "n_focal_documents": 3,
    "n_pairs": len(full),
    "reviewers": ["A", "B"],
    "scores_hidden_from_reviewers": True,
    "rank_hidden_from_reviewers": True,
    "confirmatory_ground_truth": False,
    "temporal_rule": "antecedent_year < focal_year",
    "relevance_scale": {
        "0": "No substantive relationship",
        "1": "Broad thematic relationship",
        "2": "Direct object/problem/method relationship",
        "3": "Plausible precedent for specific contribution",
        "U": "Insufficient evidence"
    }
}

(OUT / "context_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ETAPA 13 CONCLUÍDA ===")
print("Pares:", len(full))
print("Focais:", full["FOCAL_ID"].nunique())
print("Resumos ausentes:",
      int(full["ANTECEDENT_ABSTRACT"].isna().sum()))
print("Diretório:", OUT)


2013 | solicitados: 12 | recuperados: 12
2014 | solicitados: 15 | recuperados: 15
2015 | solicitados: 15 | recuperados: 15
2016 | solicitados: 14 | recuperados: 14
2017 | solicitados: 11 | recuperados: 11
2018 | solicitados: 11 | recuperados: 11
2019 | solicitados: 6 | recuperados: 6
2020 | solicitados: 2 | recuperados: 2
2021 | solicitados: 2 | recuperados: 2
2022 | solicitados: 2 | recuperados: 2

=== ETAPA 13 CONCLUÍDA ===
Pares: 90
Focais: 3
Resumos ausentes: 0
Diretório: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/04_antecedent_retrieval/context_20261020T173127_363143Z


In [21]:

# SCIENTOMETRICS 2026 — ETAPA 14
# Seleção de 15 pares para calibração contextual
# CPU | Sem embeddings | Sem classificação automática

from pathlib import Path
import pandas as pd
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SRC = (
    LAB / "04_antecedent_retrieval"
    / "context_20261020T173127_363143Z"
)

OUT = LAB / "03_ground_truth" / "context_calibration_15"
OUT.mkdir(parents=True, exist_ok=True)

# Impedir a substituição de um experimento existente.
if any(OUT.iterdir()):
    raise FileExistsError(
        "Diretório já contém arquivos. "
        "Preservar execução anterior."
    )

link = pd.read_csv(
    SRC / "pair_linkage_RESTRICTED.csv",
    dtype=str
)

review = pd.read_csv(
    SRC / "context_review_A.csv",
    dtype=str,
    keep_default_na=False
)

assert len(link) == 90
assert link["PAIR_ID"].is_unique
assert review["PAIR_ID"].is_unique

# Seleção de diferentes posições lexicais.
RANKS = {1, 5, 10, 20, 30}

selected = link[
    link["BM25_RANK"].astype(int).isin(RANKS)
].copy()

assert len(selected) == 15
assert selected.groupby("FOCAL_ID").size().eq(5).all()

# Recuperar os textos sem expor as pontuações.
cases = review.merge(
    selected[["PAIR_ID"]],
    on="PAIR_ID",
    validate="one_to_one"
)

assert len(cases) == 15

# Manter apenas textos e identificadores cegos.
text_cols = [
    "PAIR_ID",
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]

cases = (
    cases[text_cols]
    .sample(frac=1, random_state=20261020)
    .reset_index(drop=True)
)

fields = [
    "RELEVANCE_0_1_2_3_U",
    "CONTRIBUTION_LOCUS",
    "IS_PLAUSIBLE_PRECEDENT",
    "EVIDENCE_SUFFICIENCY",
    "CONFIDENCE_1_TO_5",
    "EVIDENCE_EXCERPT",
    "RATIONALE"
]

for reviewer in ["A", "B"]:
    sheet = cases.copy()

    for field in fields:
        sheet[field] = ""

    sheet["REVIEWER"] = reviewer

    sheet.to_csv(
        OUT / f"calibration_context_{reviewer}.csv",
        index=False,
        encoding="utf-8-sig"
    )

selected.to_csv(
    OUT / "selection_key_RESTRICTED.csv",
    index=False
)

manifest = {
    "phase": "contextual_relevance_calibration",
    "n_pairs": 15,
    "n_focal_documents": 3,
    "ranks_sampled": sorted(RANKS),
    "scores_hidden": True,
    "rank_hidden": True,
    "confirmatory": False,
    "ground_truth_established": False,
    "independent_human_review_required": True
}

(OUT / "calibration_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== ETAPA 14 CONCLUÍDA ===")
print("Pares:", len(cases))
print("Pares únicos:", cases["PAIR_ID"].nunique())
print("Arquivos preparados:")
for f in sorted(OUT.iterdir()):
    print("-", f.name)


=== ETAPA 14 CONCLUÍDA ===
Pares: 15
Pares únicos: 15
Arquivos preparados:
- calibration_context_A.csv
- calibration_context_B.csv
- calibration_manifest.json
- selection_key_RESTRICTED.csv


In [22]:

# SCIENTOMETRICS 2026 — ETAPA 15A
# Auditoria de correspondência dos rankings semânticos
# CPU | Somente leitura | Sem recalcular embeddings

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import json
from datetime import datetime, timezone

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
OUT = LAB / "04_antecedent_retrieval" / "semantic_mapping_audit"
OUT.mkdir(parents=True, exist_ok=True)

KEY = (
    LAB / "03_ground_truth/pilot_90/"
    "pilot_linkage_key_RESTRICTED.csv"
)

MAP = (
    BASE / "21_novidade_semantica_prior_only"
    / "global_reference_map_2013_2024.parquet"
)

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]

key = pd.read_csv(KEY, dtype=str)
focals = key[key["PILOT_ID"].isin(FOCAL_IDS)].copy()

assert len(focals) == 3
assert focals["PILOT_ID"].is_unique

focals["ANO"] = focals["ANO"].astype(int)

global_map = pq.read_table(
    MAP,
    columns=["GLOBAL_ROW", "ID_TRABALHO", "ANO"]
).to_pandas()

global_map["ID_TRABALHO"] = (
    global_map["ID_TRABALHO"].astype(str)
)

assert global_map["GLOBAL_ROW"].is_unique
assert global_map["ID_TRABALHO"].is_unique

N_GLOBAL = len(global_map)

# Testar se GLOBAL_ROW forma um intervalo contínuo.
global_zero_based = (
    global_map["GLOBAL_ROW"].min() == 0
    and global_map["GLOBAL_ROW"].max() == N_GLOBAL - 1
)

print("=== MAPA GLOBAL ===")
print("N:", N_GLOBAL)
print("GLOBAL_ROW mínimo:", global_map["GLOBAL_ROW"].min())
print("GLOBAL_ROW máximo:", global_map["GLOBAL_ROW"].max())
print("Indexação global 0-based contínua:", global_zero_based)

results = []

for focal in focals.itertuples(index=False):

    year = int(focal.ANO)

    id_file = (
        BASE / "07_embeddings_nacionais"
        / "M4" / f"M4_{year}_ids.parquet"
    )

    annual_ids = pd.read_parquet(
        id_file,
        columns=["ID_TRABALHO", "ANO"]
    )

    annual_ids["ID_TRABALHO"] = (
        annual_ids["ID_TRABALHO"].astype(str)
    )

    positions = np.flatnonzero(
        annual_ids["ID_TRABALHO"].to_numpy()
        == str(focal.ID_TRABALHO)
    )

    assert len(positions) == 1, (
        f"Documento focal não é único: {focal.PILOT_ID}"
    )

    local_position = int(positions[0])

    global_match = global_map.loc[
        global_map["ID_TRABALHO"] ==
        str(focal.ID_TRABALHO)
    ]

    assert len(global_match) == 1

    focal_global_row = int(
        global_match.iloc[0]["GLOBAL_ROW"]
    )

    for family, folder, suffix in [
        (
            "historical",
            "21_novidade_semantica_prior_only/01_vizinhos_top50",
            "prior"
        ),
        (
            "frontier3y",
            "22_frontier_novelty_3y/01_vizinhos_top50",
            "frontier3y"
        )
    ]:

        path = (
            BASE / folder /
            f"M4_{year}_{suffix}_top50_indices_int32.npy"
        )

        if not path.is_file():
            print("MISSING:", path.name)
            continue

        arr = np.load(path, mmap_mode="r")

        assert arr.shape[0] == len(annual_ids)
        assert arr.shape[1] == 50

        neighbors = np.asarray(
            arr[local_position], dtype=np.int64
        )

        valid = neighbors[neighbors >= 0]

        row = {
            "PILOT_ID": focal.PILOT_ID,
            "YEAR": year,
            "FAMILY": family,
            "FOCAL_LOCAL_POSITION": local_position,
            "FOCAL_GLOBAL_ROW": focal_global_row,
            "N_NEIGHBORS": len(valid),
            "MIN_INDEX": int(valid.min()) if len(valid) else None,
            "MAX_INDEX": int(valid.max()) if len(valid) else None,
            "N_NEGATIVE": int((neighbors < 0).sum()),
            "N_OUTSIDE_GLOBAL_RANGE": int(
                (valid >= N_GLOBAL).sum()
            ),
            "GLOBAL_MAPPING_CONFIRMED": False
        }

        results.append(row)

        print("\n=== ", focal.PILOT_ID, family, " ===")
        print("Ano:", year)
        print("Posição local:", local_position)
        print("GLOBAL_ROW focal:", focal_global_row)
        print("Shape da matriz:", arr.shape)
        print("Primeiros 10 índices:", neighbors[:10].tolist())
        print("Índice mínimo:", row["MIN_INDEX"])
        print("Índice máximo:", row["MAX_INDEX"])
        print("Índices negativos:", row["N_NEGATIVE"])
        print("Fora do intervalo global:",
              row["N_OUTSIDE_GLOBAL_RANGE"])

audit = pd.DataFrame(results)

# Arquivo versionado para impedir sobrescrita.
stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

outfile = OUT / f"mapping_audit_{stamp}.csv"

audit.to_csv(outfile, index=False)

print("\n=== ETAPA 15A CONCLUÍDA ===")
print("Arquivos auditados:", len(audit))
print("Auditoria salva:", outfile)
print(
    "ATENÇÃO: a semântica dos índices ainda "
    "não foi confirmada."
)


=== MAPA GLOBAL ===
N: 993086
GLOBAL_ROW mínimo: 0
GLOBAL_ROW máximo: 993085
Indexação global 0-based contínua: True

===  PILOT-008 historical  ===
Ano: 2017
Posição local: 60095
GLOBAL_ROW focal: 353754
Shape da matriz: (85137, 50)
Primeiros 10 índices: [224395, 155730, 174092, 42897, 141013, 210110, 164695, 251475, 178334, 41726]
Índice mínimo: 36
Índice máximo: 282143
Índices negativos: 0
Fora do intervalo global: 0

===  PILOT-008 frontier3y  ===
Ano: 2017
Posição local: 60095
GLOBAL_ROW focal: 353754
Shape da matriz: (85137, 50)
Primeiros 10 índices: [224395, 155730, 174092, 141013, 210110, 164695, 251475, 178334, 153518, 114113]
Índice mínimo: 70220
Índice máximo: 282143
Índices negativos: 0
Fora do intervalo global: 0

===  PILOT-077 historical  ===
Ano: 2019
Posição local: 27773
GLOBAL_ROW focal: 496864
Shape da matriz: (94349, 50)
Primeiros 10 índices: [246117, 277613, 382746, 24301, 230226, 460820, 299763, 350550, 171597, 310264]
Índice mínimo: 7187
Índice máximo: 460820
Índ

In [23]:

# SCIENTOMETRICS 2026 — ETAPA 15B
# Validacao independente do mapeamento semantico
# CPU | Leitura seletiva | Sem FAISS e sem inferencia

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

OUT = LAB / "04_antecedent_retrieval" / "semantic_mapping_audit"
OUT.mkdir(parents=True, exist_ok=True)

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]
N_CHECK = 3
TOLERANCE = 0.01

# 1. Mapa global
map_file = (
    BASE / "21_novidade_semantica_prior_only"
    / "global_reference_map_2013_2024.parquet"
)

gmap = pq.read_table(
    map_file,
    columns=["GLOBAL_ROW", "ID_TRABALHO", "ANO"]
).to_pandas()

gmap["ID_TRABALHO"] = gmap["ID_TRABALHO"].astype(str)
gmap = gmap.set_index("GLOBAL_ROW", verify_integrity=True)

assert len(gmap) == 993086

# 2. Identificadores focais
key = pd.read_csv(
    LAB / "03_ground_truth/pilot_90/"
    "pilot_linkage_key_RESTRICTED.csv",
    dtype=str
)

focals = key[key["PILOT_ID"].isin(FOCAL_IDS)].copy()
focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 3

# 3. Caches para nao reler arquivos
ids_cache = {}
emb_cache = {}

def get_ids(year):
    year = int(year)
    if year not in ids_cache:
        path = (
            BASE / "07_embeddings_nacionais" / "M4"
            / f"M4_{year}_ids.parquet"
        )
        ids = pd.read_parquet(path, columns=["ID_TRABALHO"])
        ids["ID_TRABALHO"] = ids["ID_TRABALHO"].astype(str)

        assert ids["ID_TRABALHO"].is_unique

        ids_cache[year] = pd.Series(
            np.arange(len(ids)),
            index=ids["ID_TRABALHO"]
        )
    return ids_cache[year]

def get_embeddings(year):
    year = int(year)
    if year not in emb_cache:
        path = (
            BASE / "07_embeddings_nacionais" / "M4"
            / f"M4_{year}_embeddings_float16.npy"
        )
        emb_cache[year] = np.load(path, mmap_mode="r")
    return emb_cache[year]

def get_vector(doc_id, year):
    pos = int(get_ids(year).loc[str(doc_id)])
    vec = np.asarray(
        get_embeddings(year)[pos],
        dtype=np.float32
    )
    norm = np.linalg.norm(vec)
    assert np.isfinite(norm) and norm > 0
    return vec / norm

records = []
temporal_records = []

for focal in focals.itertuples(index=False):

    year = int(focal.ANO)
    query_id = str(focal.ID_TRABALHO)

    query_pos = int(get_ids(year).loc[query_id])
    query_vec = get_vector(query_id, year)

    for family, folder, suffix in [
        (
            "historical",
            "21_novidade_semantica_prior_only/01_vizinhos_top50",
            "prior"
        ),
        (
            "frontier3y",
            "22_frontier_novelty_3y/01_vizinhos_top50",
            "frontier3y"
        )
    ]:

        prefix = f"M4_{year}_{suffix}_top50"

        idx = np.load(
            BASE / folder / f"{prefix}_indices_int32.npy",
            mmap_mode="r"
        )

        sims = np.load(
            BASE / folder / f"{prefix}_similarities_float32.npy",
            mmap_mode="r"
        )

        assert idx.shape == sims.shape
        assert idx.shape[0] == len(get_ids(year))
        assert idx.shape[1] == 50

        neighbor_rows = np.asarray(
            idx[query_pos], dtype=np.int64
        )
        saved_scores = np.asarray(
            sims[query_pos], dtype=np.float32
        )

        assert (neighbor_rows >= 0).all()
        assert np.isfinite(saved_scores).all()

        # Interpretacao hipotetica: indice = GLOBAL_ROW.
        antecedents = gmap.loc[neighbor_rows].reset_index()

        antecedent_years = antecedents["ANO"].astype(int)

        chronological = bool(
            (antecedent_years < year).all()
        )

        if family == "frontier3y":
            window_ok = bool(
                antecedent_years.between(year - 3, year - 1).all()
            )
        else:
            window_ok = chronological

        temporal_records.append({
            "PILOT_ID": focal.PILOT_ID,
            "FAMILY": family,
            "N": len(antecedents),
            "ALL_STRICTLY_PRIOR": chronological,
            "ALL_WITHIN_WINDOW": window_ok,
            "MIN_YEAR": int(antecedent_years.min()),
            "MAX_YEAR": int(antecedent_years.max()),
            "UNIQUE_NEIGHBORS": bool(
                antecedents["GLOBAL_ROW"].is_unique
            )
        })

        # Validacao direta das similaridades
        for rank in range(min(N_CHECK, len(antecedents))):
            ant = antecedents.iloc[rank]

            ant_id = str(ant["ID_TRABALHO"])
            ant_year = int(ant["ANO"])

            ant_vec = get_vector(ant_id, ant_year)

            cosine = float(
                np.dot(query_vec, ant_vec)
            )

            saved = float(saved_scores[rank])
            abs_error = abs(cosine - saved)

            records.append({
                "PILOT_ID": focal.PILOT_ID,
                "FAMILY": family,
                "RANK": rank + 1,
                "FOCAL_YEAR": year,
                "ANTECEDENT_ID": ant_id,
                "ANTECEDENT_YEAR": ant_year,
                "GLOBAL_ROW": int(ant["GLOBAL_ROW"]),
                "COSINE_RECOMPUTED": cosine,
                "COSINE_SAVED": saved,
                "ABS_ERROR": abs_error,
                "WITHIN_TOLERANCE": abs_error <= TOLERANCE
            })

temporal = pd.DataFrame(temporal_records)
validation = pd.DataFrame(records)

print("=== VALIDACAO TEMPORAL ===")
print(temporal.to_string(index=False))

print("\n=== VALIDACAO DAS SIMILARIDADES ===")
print(validation.to_string(index=False))

print("\n=== RESUMO ===")
print("Comparacoes:", len(validation))
print("Erro maximo:", validation["ABS_ERROR"].max())
print("Erro mediano:", validation["ABS_ERROR"].median())

temporal_ok = bool(
    temporal["ALL_STRICTLY_PRIOR"].all()
    and temporal["ALL_WITHIN_WINDOW"].all()
    and temporal["UNIQUE_NEIGHBORS"].all()
)

semantic_ok = bool(validation["WITHIN_TOLERANCE"].all())

print("Temporalidade:", temporal_ok)
print("Similaridades:", semantic_ok)

if temporal_ok and semantic_ok:
    print("RESULTADO: mapeamento GLOBAL_ROW validado")
else:
    print("ATENCAO: mapeamento nao validado.")
    print("Nao reconstruir os rankings antes de investigar.")

# 4. Preservar outputs com nomes versionados
stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

temporal.to_csv(
    OUT / f"temporal_validation_{stamp}.csv",
    index=False
)

validation.to_csv(
    OUT / f"cosine_validation_{stamp}.csv",
    index=False
)

print("\nETAPA 15B CONCLUIDA")


=== VALIDACAO TEMPORAL ===
 PILOT_ID     FAMILY  N  ALL_STRICTLY_PRIOR  ALL_WITHIN_WINDOW  MIN_YEAR  MAX_YEAR  UNIQUE_NEIGHBORS
PILOT-008 historical 50                True               True      2013      2016              True
PILOT-008 frontier3y 50                True               True      2014      2016              True
PILOT-077 historical 50                True               True      2013      2018              True
PILOT-077 frontier3y 50                True               True      2016      2018              True
PILOT-090 historical 50                True               True      2013      2022              True
PILOT-090 frontier3y 50                True               True      2020      2022              True

=== VALIDACAO DAS SIMILARIDADES ===
 PILOT_ID     FAMILY  RANK  FOCAL_YEAR ANTECEDENT_ID  ANTECEDENT_YEAR  GLOBAL_ROW  COSINE_RECOMPUTED  COSINE_SAVED    ABS_ERROR  WITHIN_TOLERANCE
PILOT-008 historical     1        2017      21011973             2016      224395  

In [24]:

# SCIENTOMETRICS 2026 — ETAPA 16
# Benchmark exploratorio BM25 x Historical x Frontier3Y
# CPU | Sem GPU | Sem recalcular embeddings

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import json
from itertools import combinations

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

BM25_RUN = (
    LAB / "04_antecedent_retrieval"
    / "bm25_20261020T172459Z"
)

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "04_antecedent_retrieval"
    / f"benchmark_multisystem_{STAMP}"
)
OUT.mkdir(parents=True, exist_ok=False)

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]

# 1. Mapa global, com area e ano.
MAP = (
    BASE / "21_novidade_semantica_prior_only"
    / "global_reference_map_2013_2024.parquet"
)

gmap = pq.read_table(
    MAP,
    columns=[
        "GLOBAL_ROW", "ID_TRABALHO",
        "ANO", "AREA_HARMONIZADA"
    ]
).to_pandas()

gmap["ID_TRABALHO"] = gmap["ID_TRABALHO"].astype(str)
assert gmap["GLOBAL_ROW"].is_unique
assert gmap["ID_TRABALHO"].is_unique

gmap = gmap.set_index("GLOBAL_ROW", verify_integrity=True)

# 2. Metadados dos focais.
key = pd.read_csv(
    LAB / "03_ground_truth/pilot_90/"
    "pilot_linkage_key_RESTRICTED.csv",
    dtype=str
)

focals = key[
    key["PILOT_ID"].isin(FOCAL_IDS)
].copy()

focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 3
assert focals["PILOT_ID"].is_unique

focal_area_map = (
    gmap.reset_index()
    .set_index("ID_TRABALHO")["AREA_HARMONIZADA"]
)

focals["FOCAL_AREA"] = (
    focals["ID_TRABALHO"].map(focal_area_map)
)
assert focals["FOCAL_AREA"].notna().all()

# 3. BM25 previamente calculado.
bm25 = pd.read_parquet(
    BM25_RUN / "bm25_top30_three_focals.parquet"
)

bm25["ANTECEDENT_ID"] = bm25["ANTECEDENT_ID"].astype(str)

bm = bm25[[
    "FOCAL_ID", "FOCAL_YEAR",
    "ANTECEDENT_ID", "ANTECEDENT_YEAR",
    "RANK", "BM25_SCORE"
]].copy()

bm = bm.rename(columns={
    "RANK": "METHOD_RANK",
    "BM25_SCORE": "METHOD_SCORE"
})

bm["METHOD"] = "BM25_same_area"

# Adicionar a area com base no mapa, por ID.
area_lookup = (
    gmap.reset_index()
    .set_index("ID_TRABALHO")["AREA_HARMONIZADA"]
)
bm["ANTECEDENT_AREA"] = bm["ANTECEDENT_ID"].map(area_lookup)

assert bm["ANTECEDENT_AREA"].notna().all()
assert bm["ANTECEDENT_YEAR"].lt(bm["FOCAL_YEAR"]).all()

# 4. Reconstruir rankings semanticos.
sem_records = []

for focal in focals.itertuples(index=False):

    year = int(focal.ANO)

    ids_path = (
        BASE / "07_embeddings_nacionais"
        / "M4" / f"M4_{year}_ids.parquet"
    )

    ids = pd.read_parquet(ids_path, columns=["ID_TRABALHO"])
    ids["ID_TRABALHO"] = ids["ID_TRABALHO"].astype(str)

    positions = np.flatnonzero(
        ids["ID_TRABALHO"].to_numpy()
        == str(focal.ID_TRABALHO)
    )
    assert len(positions) == 1
    position = int(positions[0])

    families = [
        (
            "Historical_M4",
            "21_novidade_semantica_prior_only/01_vizinhos_top50",
            "prior"
        ),
        (
            "Frontier3Y_M4",
            "22_frontier_novelty_3y/01_vizinhos_top50",
            "frontier3y"
        )
    ]

    for method, folder, suffix in families:

        prefix = f"M4_{year}_{suffix}_top50"

        rows = np.load(
            BASE / folder / f"{prefix}_indices_int32.npy",
            mmap_mode="r"
        )[position].astype(np.int64)

        scores = np.load(
            BASE / folder /
            f"{prefix}_similarities_float32.npy",
            mmap_mode="r"
        )[position].astype(float)

        assert len(rows) == len(scores) == 50
        assert (rows >= 0).all()

        ants = gmap.loc[rows]

        for rank, (global_row, score) in enumerate(
            zip(rows, scores), start=1
        ):
            item = ants.loc[int(global_row)]

            sem_records.append({
                "FOCAL_ID": focal.PILOT_ID,
                "FOCAL_YEAR": year,
                "ANTECEDENT_ID": str(item["ID_TRABALHO"]),
                "ANTECEDENT_YEAR": int(item["ANO"]),
                "ANTECEDENT_AREA": item["AREA_HARMONIZADA"],
                "METHOD": method,
                "METHOD_RANK": rank,
                "METHOD_SCORE": float(score)
            })

sem = pd.DataFrame(sem_records)

all_rankings = pd.concat([bm, sem], ignore_index=True)
all_rankings["FOCAL_YEAR"] = all_rankings["FOCAL_YEAR"].astype(int)
all_rankings["ANTECEDENT_YEAR"] = (
    all_rankings["ANTECEDENT_YEAR"].astype(int)
)
all_rankings["METHOD_RANK"] = (
    all_rankings["METHOD_RANK"].astype(int)
)

assert (all_rankings["ANTECEDENT_YEAR"]
        < all_rankings["FOCAL_YEAR"]).all()

assert not all_rankings.duplicated([
    "FOCAL_ID", "METHOD", "ANTECEDENT_ID"
]).any()

# 5. Metricas de sobreposicao.
def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else np.nan

comparison = []

for focal_id, group in all_rankings.groupby("FOCAL_ID"):

    methods = sorted(group["METHOD"].unique())

    for depth in [10, 30]:
        sets = {
            method: set(
                group.loc[
                    (group["METHOD"] == method) &
                    (group["METHOD_RANK"] <= depth),
                    "ANTECEDENT_ID"
                ]
            )
            for method in methods
        }

        for a, b in combinations(methods, 2):
            comparison.append({
                "FOCAL_ID": focal_id,
                "TOP_K": depth,
                "METHOD_A": a,
                "METHOD_B": b,
                "INTERSECTION": len(sets[a] & sets[b]),
                "JACCARD": jaccard(sets[a], sets[b]),
                "OVERLAP_AT_K": (
                    len(sets[a] & sets[b]) / depth
                )
            })

overlap = pd.DataFrame(comparison)

# 6. Cobertura por metodo.
coverage = (
    all_rankings
    .assign(
        SAME_AREA=lambda d: d["ANTECEDENT_AREA"].eq(
            d["FOCAL_ID"].map(
                focals.set_index("PILOT_ID")["FOCAL_AREA"]
            )
        ),
        LAG=lambda d: d["FOCAL_YEAR"] -
                      d["ANTECEDENT_YEAR"]
    )
    .groupby(["FOCAL_ID", "METHOD"])
    .agg(
        N=("ANTECEDENT_ID", "size"),
        SAME_AREA_SHARE=("SAME_AREA", "mean"),
        N_AREAS=("ANTECEDENT_AREA", "nunique"),
        MEDIAN_LAG=("LAG", "median"),
        MIN_LAG=("LAG", "min"),
        MAX_LAG=("LAG", "max")
    )
    .reset_index()
)

# 7. Pool multissistema (ainda sem anotacao).
pool = (
    all_rankings.groupby(["FOCAL_ID", "ANTECEDENT_ID"])
    .agg(
        N_METHODS=("METHOD", "nunique"),
        METHODS=("METHOD", lambda x: ";".join(sorted(set(x))))
    )
    .reset_index()
)

# 8. Salvamento.
all_rankings.to_parquet(
    OUT / "rankings_multisystem.parquet",
    index=False
)

overlap.to_csv(
    OUT / "ranking_overlap.csv", index=False
)

coverage.to_csv(
    OUT / "retrieval_coverage.csv", index=False
)

pool.to_parquet(
    OUT / "candidate_pool_INTERNAL.parquet",
    index=False
)

manifest = {
    "stage": "exploratory_multisystem_benchmark",
    "methods": [
        "BM25_same_area",
        "Historical_M4",
        "Frontier3Y_M4"
    ],
    "focal_ids": FOCAL_IDS,
    "bm25_depth": 30,
    "semantic_depth": 50,
    "overlap_depths": [10, 30],
    "retrieval_spaces_not_identical": True,
    "ground_truth": False,
    "confirmatory": False,
    "global_row_mapping": "validated_in_stage_15B"
}

(OUT / "benchmark_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== SOBREPOSICAO ===")
print(overlap.to_string(index=False))

print("\n=== COBERTURA ===")
print(coverage.to_string(index=False))

print("\n=== POOL UNIFICADO ===")
print(pool.groupby("FOCAL_ID").size())

print("\nDiretorio:", OUT)
print("ETAPA 16 CONCLUIDA")


=== SOBREPOSICAO ===
 FOCAL_ID  TOP_K       METHOD_A      METHOD_B  INTERSECTION  JACCARD  OVERLAP_AT_K
PILOT-008     10 BM25_same_area Frontier3Y_M4             0 0.000000      0.000000
PILOT-008     10 BM25_same_area Historical_M4             0 0.000000      0.000000
PILOT-008     10  Frontier3Y_M4 Historical_M4             8 0.666667      0.800000
PILOT-008     30 BM25_same_area Frontier3Y_M4             1 0.016949      0.033333
PILOT-008     30 BM25_same_area Historical_M4             1 0.016949      0.033333
PILOT-008     30  Frontier3Y_M4 Historical_M4            23 0.621622      0.766667
PILOT-077     10 BM25_same_area Frontier3Y_M4             2 0.111111      0.200000
PILOT-077     10 BM25_same_area Historical_M4             2 0.111111      0.200000
PILOT-077     10  Frontier3Y_M4 Historical_M4             8 0.666667      0.800000
PILOT-077     30 BM25_same_area Frontier3Y_M4             7 0.132075      0.233333
PILOT-077     30 BM25_same_area Historical_M4             8 0.1538

In [25]:

# SCIENTOMETRICS 2026 — ETAPA 17
# Diagnostico de divergencia entre recuperadores
# CPU | Sem embeddings | Somente rankings existentes

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

RUN = (
    LAB / "04_antecedent_retrieval"
    / "benchmark_multisystem_20261020T180159_900581Z"
)

assert RUN.is_dir()

r = pd.read_parquet(
    RUN / "rankings_multisystem.parquet"
)

required = [
    "FOCAL_ID", "FOCAL_YEAR", "ANTECEDENT_ID",
    "ANTECEDENT_YEAR", "ANTECEDENT_AREA",
    "METHOD", "METHOD_RANK", "METHOD_SCORE"
]

assert all(c in r.columns for c in required)

assert not r.duplicated(
    ["FOCAL_ID", "METHOD", "ANTECEDENT_ID"]
).any()

FOCAL_AREAS = {
    "PILOT-008": "MEDICINA VETERINARIA",
    "PILOT-077": "ODONTOLOGIA",
    "PILOT-090": "EDUCACAO"
}

r["SAME_AREA"] = [
    str(area) == FOCAL_AREAS[focal]
    for focal, area in zip(
        r["FOCAL_ID"], r["ANTECEDENT_AREA"]
    )
]

r["LAG_YEARS"] = (
    r["FOCAL_YEAR"].astype(int) -
    r["ANTECEDENT_YEAR"].astype(int)
)

assert (r["LAG_YEARS"] > 0).all()

# Presenca de cada documento nos tres sistemas.
presence = (
    r.assign(PRESENT=True)
    .pivot_table(
        index=["FOCAL_ID", "ANTECEDENT_ID"],
        columns="METHOD",
        values="PRESENT",
        aggfunc="max",
        fill_value=False
    )
    .reset_index()
)

methods = [
    "BM25_same_area",
    "Historical_M4",
    "Frontier3Y_M4"
]

for method in methods:
    assert method in presence.columns

presence["N_METHODS"] = (
    presence[methods].astype(int).sum(axis=1)
)

# Diagnostico de candidatos exclusivos.
exclusive = []

for method in methods:
    subset = presence[
        (presence[method]) &
        (presence["N_METHODS"] == 1)
    ].copy()

    subset["EXCLUSIVE_TO"] = method
    exclusive.append(subset[
        ["FOCAL_ID", "ANTECEDENT_ID", "EXCLUSIVE_TO"]
    ])

exclusive = pd.concat(exclusive, ignore_index=True)

print("=== CANDIDATOS EXCLUSIVOS ===")
print(
    exclusive.groupby(
        ["FOCAL_ID", "EXCLUSIVE_TO"]
    ).size().to_string()
)

# Perfil disciplinar dos rankings.
disciplinary = (
    r.groupby([
        "FOCAL_ID", "METHOD", "ANTECEDENT_AREA"
    ])
    .agg(
        N=("ANTECEDENT_ID", "size"),
        MEDIAN_LAG=("LAG_YEARS", "median")
    )
    .reset_index()
    .sort_values(
        ["FOCAL_ID", "METHOD", "N"],
        ascending=[True, True, False]
    )
)

print("\n=== AREAS MAIS REPRESENTADAS ===")

for (focal, method), group in disciplinary.groupby(
    ["FOCAL_ID", "METHOD"]
):
    print(f"\n{focal} | {method}")
    print(group.head(8).to_string(index=False))

# Sobreposicao observada no conjunto filtrado.
diagnostic = []

for focal, group in r.groupby("FOCAL_ID"):

    bm = set(
        group.loc[
            group["METHOD"] == "BM25_same_area",
            "ANTECEDENT_ID"
        ]
    )

    for method in ["Historical_M4", "Frontier3Y_M4"]:

        sem_all = group[
            group["METHOD"] == method
        ]

        sem_same = set(
            sem_all.loc[
                sem_all["SAME_AREA"],
                "ANTECEDENT_ID"
            ]
        )

        diagnostic.append({
            "FOCAL_ID": focal,
            "SEMANTIC_METHOD": method,
            "N_SEMANTIC_SAME_AREA_IN_TOP50":
                len(sem_same),
            "N_BM25_TOP30": len(bm),
            "INTERSECTION": len(bm & sem_same),
            "SEMANTIC_IN_BM25_SHARE":
                len(bm & sem_same) / len(sem_same)
                if sem_same else np.nan,
            "IS_CONTROLLED_COMPARISON": False
        })

diag = pd.DataFrame(diagnostic)

print("\n=== DIAGNOSTICO MESMA AREA ===")
print(diag.to_string(index=False))

# Preservar resultados em uma nova pasta.
stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "04_antecedent_retrieval"
    / f"diagnostic_stage17_{stamp}"
)

OUT.mkdir(parents=True, exist_ok=False)

disciplinary.to_csv(
    OUT / "disciplinary_profile.csv",
    index=False
)

exclusive.to_csv(
    OUT / "method_exclusive_candidates.csv",
    index=False
)

diag.to_csv(
    OUT / "same_area_top50_diagnostic.csv",
    index=False
)

manifest = {
    "stage": 17,
    "type": "exploratory_diagnostic",
    "source": str(RUN),
    "methods": methods,
    "semantic_top50_filtering_only": True,
    "controlled_comparison": False,
    "human_validation": False
}

(OUT / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nETAPA 17 CONCLUIDA")
print("Diretorio:", OUT)


=== CANDIDATOS EXCLUSIVOS ===
FOCAL_ID   EXCLUSIVE_TO  
PILOT-008  BM25_same_area    28
           Frontier3Y_M4     12
           Historical_M4     12
PILOT-077  BM25_same_area    21
           Frontier3Y_M4     20
           Historical_M4     20
PILOT-090  BM25_same_area    29
           Frontier3Y_M4     36
           Historical_M4     35

=== AREAS MAIS REPRESENTADAS ===

PILOT-008 | BM25_same_area
 FOCAL_ID         METHOD      ANTECEDENT_AREA  N  MEDIAN_LAG
PILOT-008 BM25_same_area MEDICINA VETERINARIA 30         2.0

PILOT-008 | Frontier3Y_M4
 FOCAL_ID        METHOD       ANTECEDENT_AREA  N  MEDIAN_LAG
PILOT-008 Frontier3Y_M4            MEDICINA I 18         2.0
PILOT-008 Frontier3Y_M4           MEDICINA II  9         3.0
PILOT-008 Frontier3Y_M4          MEDICINA III  5         3.0
PILOT-008 Frontier3Y_M4           ODONTOLOGIA  5         2.0
PILOT-008 Frontier3Y_M4  MEDICINA VETERINARIA  4         2.5
PILOT-008 Frontier3Y_M4        SAUDE COLETIVA  3         2.0
PILOT-008 Frontier

In [26]:

# SCIENTOMETRICS 2026 — ETAPA 18A
# Busca semantica EXAUSTIVA na mesma area CAPES
# CPU | Embeddings M4 congelados | Sem GPU

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

EMB_DIR = BASE / "07_embeddings_nacionais" / "M4"

BM25_FILE = (
    LAB / "04_antecedent_retrieval"
    / "bm25_20261020T172459Z"
    / "bm25_top30_three_focals.parquet"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

KEY_FILE = (
    LAB / "03_ground_truth/pilot_90"
    / "pilot_linkage_key_RESTRICTED.csv"
)

TOP_K = 30
FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "04_antecedent_retrieval"
    / f"controlled_semantic_stage18A_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

# --------------------------------------------------
# 1. Documentos focais e indice nacional
# --------------------------------------------------

key = pd.read_csv(KEY_FILE, dtype=str)

focals = key[
    key["PILOT_ID"].isin(FOCAL_IDS)
][["PILOT_ID", "ID_TRABALHO", "ANO"]].copy()

focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 3
assert focals["PILOT_ID"].is_unique

meta = pq.read_table(
    INDEX_FILE,
    columns=[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA"
    ]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
meta["ANO"] = meta["ANO"].astype(int)

assert meta["ID_TRABALHO"].is_unique

focals = focals.merge(
    meta[["ID_TRABALHO", "AREA_HARMONIZADA"]],
    on="ID_TRABALHO",
    validate="one_to_one",
    how="left"
)

assert focals["AREA_HARMONIZADA"].notna().all()

# --------------------------------------------------
# 2. Indexacao dos IDs dentro dos arquivos anuais
# --------------------------------------------------

ids_cache = {}
emb_cache = {}

def get_annual_index(year):

    year = int(year)

    if year not in ids_cache:

        path = EMB_DIR / f"M4_{year}_ids.parquet"

        ids = pd.read_parquet(
            path,
            columns=["ID_TRABALHO"]
        )

        ids["ID_TRABALHO"] = (
            ids["ID_TRABALHO"].astype(str)
        )

        assert ids["ID_TRABALHO"].is_unique

        ids_cache[year] = pd.Index(
            ids["ID_TRABALHO"]
        )

    return ids_cache[year]


def get_embeddings(year):

    year = int(year)

    if year not in emb_cache:

        path = (
            EMB_DIR /
            f"M4_{year}_embeddings_float16.npy"
        )

        arr = np.load(path, mmap_mode="r")

        assert len(arr) == len(get_annual_index(year))
        assert arr.ndim == 2

        emb_cache[year] = arr

    return emb_cache[year]


def normalized_vector(doc_id, year):

    annual_index = get_annual_index(year)

    pos = annual_index.get_loc(str(doc_id))
    assert isinstance(pos, (int, np.integer))

    v = np.asarray(
        get_embeddings(year)[pos],
        dtype=np.float32
    )

    n = float(np.linalg.norm(v))
    assert np.isfinite(n) and n > 0

    return v / n


# --------------------------------------------------
# 3. Busca exaustiva por blocos
# --------------------------------------------------

all_scores = []
diagnostics = []

for focal in focals.itertuples(index=False):

    focal_id = focal.PILOT_ID
    focal_year = int(focal.ANO)
    focal_area = focal.AREA_HARMONIZADA

    query_vec = normalized_vector(
        focal.ID_TRABALHO,
        focal_year
    )

    eligible = meta[
        (meta["ANO"] < focal_year) &
        (meta["AREA_HARMONIZADA"] == focal_area)
    ].copy()

    expected = len(eligible)

    print("\n" + "=" * 60)
    print("FOCAL:", focal_id)
    print("AREA:", focal_area)
    print("CANDIDATOS:", expected)

    parts = []

    for year, group in eligible.groupby("ANO"):

        year = int(year)

        annual_index = get_annual_index(year)
        arr = get_embeddings(year)

        doc_ids = group["ID_TRABALHO"].tolist()
        positions = annual_index.get_indexer(doc_ids)

        assert (positions >= 0).all(), (
            f"IDs sem embedding no ano {year}"
        )

        year_scores = np.empty(
            len(positions),
            dtype=np.float32
        )

        # Blocos pequenos para limitar memoria RAM.
        BATCH = 2048

        for start in range(0, len(positions), BATCH):

            end = min(start + BATCH, len(positions))

            block = np.asarray(
                arr[positions[start:end]],
                dtype=np.float32
            )

            norms = np.linalg.norm(
                block, axis=1
            )

            assert np.isfinite(norms).all()
            assert (norms > 0).all()

            block /= norms[:, None]

            year_scores[start:end] = (
                block @ query_vec
            )

        part = group[
            ["ID_TRABALHO", "ANO"]
        ].copy()

        part["COSINE_M4"] = year_scores

        parts.append(part)

    scored = pd.concat(
        parts,
        ignore_index=True
    )

    assert len(scored) == expected
    assert scored["ID_TRABALHO"].is_unique
    assert np.isfinite(scored["COSINE_M4"]).all()

    scored["FOCAL_ID"] = focal_id
    scored["FOCAL_YEAR"] = focal_year
    scored["ANTECEDENT_AREA"] = focal_area

    all_scores.append(scored)

    diagnostics.append({
        "FOCAL_ID": focal_id,
        "FOCAL_YEAR": focal_year,
        "FOCAL_AREA": focal_area,
        "N_SCORED": len(scored),
        "COSINE_MIN": float(
            scored["COSINE_M4"].min()
        ),
        "COSINE_MAX": float(
            scored["COSINE_M4"].max()
        )
    })

    print("DOCUMENTOS PONTUADOS:", len(scored))
    print(
        "COSINE MAX:",
        round(float(scored["COSINE_M4"].max()), 6)
    )

    # Descartar matrizes memmap do ano anterior
    # somente depois de terminar este focal.
    emb_cache.clear()

# --------------------------------------------------
# 4. Rankings controlados: historico e 3 anos
# --------------------------------------------------

scores = pd.concat(
    all_scores,
    ignore_index=True
)

scores = scores.rename(columns={
    "ID_TRABALHO": "ANTECEDENT_ID",
    "ANO": "ANTECEDENT_YEAR"
})

rankings = []

for focal_id, group in scores.groupby("FOCAL_ID"):

    focal_year = int(group["FOCAL_YEAR"].iloc[0])

    windows = [
        (
            "Historical_M4_same_area",
            group
        ),
        (
            "Frontier3Y_M4_same_area",
            group[
                group["ANTECEDENT_YEAR"] >=
                focal_year - 3
            ]
        )
    ]

    for method, candidates in windows:

        candidates = candidates.sort_values(
            ["COSINE_M4", "ANTECEDENT_ID"],
            ascending=[False, True],
            kind="mergesort"
        ).head(TOP_K).copy()

        candidates["METHOD"] = method

        candidates["METHOD_RANK"] = range(
            1, len(candidates) + 1
        )

        rankings.append(candidates)

ranking = pd.concat(
    rankings,
    ignore_index=True
)

assert not ranking.duplicated([
    "FOCAL_ID", "METHOD", "ANTECEDENT_ID"
]).any()

# --------------------------------------------------
# 5. Comparacao controlada com BM25 historico
# --------------------------------------------------

bm25 = pd.read_parquet(BM25_FILE)

bm25["ANTECEDENT_ID"] = (
    bm25["ANTECEDENT_ID"].astype(str)
)

comparisons = []

for focal_id in FOCAL_IDS:

    bm = set(
        bm25.loc[
            (bm25["FOCAL_ID"] == focal_id) &
            (bm25["RANK"] <= TOP_K),
            "ANTECEDENT_ID"
        ]
    )

    sem = set(
        ranking.loc[
            (ranking["FOCAL_ID"] == focal_id) &
            (ranking["METHOD"] ==
             "Historical_M4_same_area"),
            "ANTECEDENT_ID"
        ]
    )

    assert len(bm) == TOP_K
    assert len(sem) == TOP_K

    inter = len(bm & sem)
    union = len(bm | sem)

    comparisons.append({
        "FOCAL_ID": focal_id,
        "UNIVERSE": "same_area_prior_years",
        "TOP_K": TOP_K,
        "BM25_N": len(bm),
        "M4_N": len(sem),
        "INTERSECTION": inter,
        "OVERLAP_AT_K": inter / TOP_K,
        "JACCARD": inter / union
    })

comparison = pd.DataFrame(comparisons)

# --------------------------------------------------
# 6. Salvamento com rastreabilidade
# --------------------------------------------------

scores.to_parquet(
    OUT / "semantic_all_candidate_scores.parquet",
    index=False
)

ranking.to_parquet(
    OUT / "semantic_controlled_top30.parquet",
    index=False
)

comparison.to_csv(
    OUT / "bm25_vs_m4_same_area_historical.csv",
    index=False
)

pd.DataFrame(diagnostics).to_csv(
    OUT / "scoring_diagnostics.csv",
    index=False
)

manifest = {
    "stage": "18A",
    "study": "Epistemic Change Measurement Validation",
    "retrieval": "exhaustive_cosine_M4",
    "focal_ids": FOCAL_IDS,
    "candidate_universe": "same_area_prior_years",
    "semantic_input": "frozen_M4_embeddings",
    "windows": [
        "all_prior_years",
        "prior_3_calendar_years"
    ],
    "top_k": TOP_K,
    "bm25_comparison": "same_area_all_prior_years",
    "no_new_embeddings": True,
    "no_human_ground_truth": True,
    "confirmatory": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== COMPARACAO CONTROLADA ===")
print(comparison.to_string(index=False))

print("\n=== DIAGNOSTICO ===")
print(pd.DataFrame(diagnostics).to_string(index=False))

print("\nETAPA 18A CONCLUIDA")
print("Diretorio:", OUT)



FOCAL: PILOT-008
AREA: MEDICINA VETERINARIA
CANDIDATOS: 5743
DOCUMENTOS PONTUADOS: 5743
COSINE MAX: 0.97023

FOCAL: PILOT-077
AREA: ODONTOLOGIA
CANDIDATOS: 12334
DOCUMENTOS PONTUADOS: 12334
COSINE MAX: 0.961545

FOCAL: PILOT-090
AREA: EDUCACAO
CANDIDATOS: 51785
DOCUMENTOS PONTUADOS: 51785
COSINE MAX: 0.9817

=== COMPARACAO CONTROLADA ===
 FOCAL_ID              UNIVERSE  TOP_K  BM25_N  M4_N  INTERSECTION  OVERLAP_AT_K  JACCARD
PILOT-008 same_area_prior_years     30      30    30             3      0.100000 0.052632
PILOT-077 same_area_prior_years     30      30    30             8      0.266667 0.153846
PILOT-090 same_area_prior_years     30      30    30             1      0.033333 0.016949

=== DIAGNOSTICO ===
 FOCAL_ID  FOCAL_YEAR           FOCAL_AREA  N_SCORED  COSINE_MIN  COSINE_MAX
PILOT-008        2017 MEDICINA VETERINARIA      5743    0.857918    0.970230
PILOT-077        2019          ODONTOLOGIA     12334    0.840140    0.961545
PILOT-090        2023             EDUCACAO     

In [27]:

# SCIENTOMETRICS 2026 — ETAPA 18B
# BM25 x M4: mesma area, mesma janela de 3 anos
# CPU | Sem novos embeddings | Sem FAISS

from pathlib import Path
from datetime import datetime, timezone
import unicodedata
import re
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import CountVectorizer

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
SOURCE = BASE / "05_transferencia_colab"

STAGE18A = (
    LAB / "04_antecedent_retrieval"
    / "controlled_semantic_stage18A_20261020T181204_113099Z"
)

KEY = (
    LAB / "03_ground_truth/pilot_90"
    / "pilot_linkage_key_RESTRICTED.csv"
)

PILOT = (
    LAB / "03_ground_truth/pilot_90"
    / "pilot_cases_blinded.csv"
)

TOP_K = 30
K1 = 1.5
B = 0.75

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "04_antecedent_retrieval"
    / f"controlled_bm25_frontier_stage18B_{stamp}"
)

OUT.mkdir(parents=True, exist_ok=False)

# --------------------------------------------------
# 1. Dados focais
# --------------------------------------------------

key = pd.read_csv(KEY, dtype=str)

pilot = pd.read_csv(
    PILOT,
    dtype=str,
    keep_default_na=False
)

focals = key[
    key["PILOT_ID"].isin(FOCAL_IDS)
][["PILOT_ID", "ID_TRABALHO", "ANO"]].merge(
    pilot[["PILOT_ID", "TITULO", "PALAVRAS_CHAVE"]],
    on="PILOT_ID",
    how="left",
    validate="one_to_one"
)

assert len(focals) == 3
assert focals["TITULO"].notna().all()

focals["ANO"] = focals["ANO"].astype(int)

# --------------------------------------------------
# 2. Escores M4 exaustivos da etapa anterior
# --------------------------------------------------

scores = pd.read_parquet(
    STAGE18A / "semantic_all_candidate_scores.parquet"
)

scores["ANTECEDENT_ID"] = (
    scores["ANTECEDENT_ID"].astype(str)
)

scores["ANTECEDENT_YEAR"] = (
    scores["ANTECEDENT_YEAR"].astype(int)
)

scores["FOCAL_YEAR"] = scores["FOCAL_YEAR"].astype(int)

assert not scores.duplicated(
    ["FOCAL_ID", "ANTECEDENT_ID"]
).any()

eligible = scores[
    scores["ANTECEDENT_YEAR"].ge(
        scores["FOCAL_YEAR"] - 3
    ) &
    scores["ANTECEDENT_YEAR"].lt(
        scores["FOCAL_YEAR"]
    )
].copy()

assert eligible["COSINE_M4"].notna().all()

print("=== UNIVERSOS CONTROLADOS ===")
print(eligible.groupby("FOCAL_ID").size().to_string())

# --------------------------------------------------
# 3. Recuperar os mesmos textos candidatos
# --------------------------------------------------

wanted_by_year = {
    int(year): set(group["ANTECEDENT_ID"])
    for year, group in eligible.groupby("ANTECEDENT_YEAR")
}

parts = []

for year, wanted in sorted(wanted_by_year.items()):

    path = SOURCE / (
        f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    )

    assert path.is_file(), f"Arquivo ausente: {path}"

    pf = pq.ParquetFile(path)
    selected = []

    for batch in pf.iter_batches(
        batch_size=16384,
        columns=[
            "ID_TRABALHO",
            "TITULO",
            "PALAVRAS_CHAVE"
        ]
    ):
        chunk = batch.to_pandas()

        chunk["ID_TRABALHO"] = (
            chunk["ID_TRABALHO"].astype(str)
        )

        hit = chunk[
            chunk["ID_TRABALHO"].isin(wanted)
        ]

        if not hit.empty:
            selected.append(hit.copy())

    if selected:
        data = pd.concat(
            selected,
            ignore_index=True
        )
        data["ANTECEDENT_YEAR"] = year
        parts.append(data)

    found_n = sum(len(x) for x in selected)

    print(
        f"{year}: recuperados {found_n} "
        f"de {len(wanted)} IDs distintos"
    )

corpus = pd.concat(parts, ignore_index=True)

corpus = corpus.rename(columns={
    "ID_TRABALHO": "ANTECEDENT_ID"
})

assert not corpus.duplicated(
    ["ANTECEDENT_ID", "ANTECEDENT_YEAR"]
).any()

data = eligible.merge(
    corpus,
    on=["ANTECEDENT_ID", "ANTECEDENT_YEAR"],
    how="left",
    validate="many_to_one",
    indicator=True
)

assert (data["_merge"] == "both").all(), (
    "Falha no pareamento de candidatos."
)

data = data.drop(columns="_merge")

assert len(data) == len(eligible)

# --------------------------------------------------
# 4. Mesma tokenizacao e formula da etapa 12A
# --------------------------------------------------

STOP = {
    "a", "as", "o", "os", "um", "uma", "uns", "umas",
    "de", "da", "do", "das", "dos", "em", "no", "na",
    "nos", "nas", "por", "para", "com", "sem",
    "e", "ou", "que", "se", "ao", "aos", "sua",
    "suas", "seu", "seus", "sobre", "entre",
    "the", "and", "of", "in", "for", "on",
    "with", "to", "an", "by", "from"
}

def tokenize(text):
    text = unicodedata.normalize(
        "NFKD", str(text).lower()
    )
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    tokens = re.findall(r"[a-z0-9]+", text)
    return [
        t for t in tokens
        if len(t) > 1 and t not in STOP
    ]

def bm25_all(documents, query):
    vec = CountVectorizer(
        tokenizer=tokenize,
        token_pattern=None,
        lowercase=False,
        dtype=np.float32
    )

    X = vec.fit_transform(documents).tocsc()
    N = X.shape[0]

    lengths = np.asarray(X.sum(axis=1)).ravel()
    avgdl = float(lengths.mean())

    if avgdl <= 0:
        raise ValueError("Corpus lexical vazio.")

    scores = np.zeros(N, dtype=np.float64)

    for term in set(tokenize(query)):
        idx = vec.vocabulary_.get(term)

        if idx is None:
            continue

        col = X.getcol(idx).tocoo()
        df = len(col.data)

        idf = np.log(
            1 + (N - df + 0.5) / (df + 0.5)
        )

        tf = col.data.astype(np.float64)
        dl = lengths[col.row]

        denom = (
            tf + K1 * (1 - B + B * dl / avgdl)
        )

        scores[col.row] += (
            idf * tf * (K1 + 1) / denom
        )

    return scores

# --------------------------------------------------
# 5. Calcular BM25 e comparar ao M4
# --------------------------------------------------

bm25_results = []
m4_results = []
comparisons = []
diagnostics = []

for focal in focals.itertuples(index=False):

    focal_id = focal.PILOT_ID

    group = data[
        data["FOCAL_ID"] == focal_id
    ].copy().reset_index(drop=True)

    assert len(group) >= TOP_K

    documents = (
        group["TITULO"].fillna("").astype(str)
        + " "
        + group["PALAVRAS_CHAVE"].fillna("").astype(str)
    ).tolist()

    query = (
        str(focal.TITULO)
        + " "
        + str(focal.PALAVRAS_CHAVE)
    )

    bm_scores = bm25_all(documents, query)

    assert np.isfinite(bm_scores).all()

    group["BM25_SCORE_FRONTIER"] = bm_scores

    bm_top = group.sort_values(
        ["BM25_SCORE_FRONTIER", "ANTECEDENT_ID"],
        ascending=[False, True],
        kind="mergesort"
    ).head(TOP_K).copy()

    m4_top = group.sort_values(
        ["COSINE_M4", "ANTECEDENT_ID"],
        ascending=[False, True],
        kind="mergesort"
    ).head(TOP_K).copy()

    bm_top["RANK"] = range(1, TOP_K + 1)
    m4_top["RANK"] = range(1, TOP_K + 1)

    bm25_results.append(bm_top)
    m4_results.append(m4_top)

    bm_ids = set(bm_top["ANTECEDENT_ID"])
    m4_ids = set(m4_top["ANTECEDENT_ID"])

    common = bm_ids & m4_ids
    union = bm_ids | m4_ids

    comparisons.append({
        "FOCAL_ID": focal_id,
        "SCOPE": "same_area_frontier3y",
        "N_CANDIDATES": len(group),
        "TOP_K": TOP_K,
        "INTERSECTION": len(common),
        "OVERLAP_AT_K": len(common) / TOP_K,
        "JACCARD": len(common) / len(union)
    })

    diagnostics.append({
        "FOCAL_ID": focal_id,
        "N_CANDIDATES": len(group),
        "N_BM25_POSITIVE": int((bm_scores > 0).sum()),
        "BM25_MAX": float(bm_scores.max()),
        "M4_MAX": float(group["COSINE_M4"].max())
    })

bm_top_all = pd.concat(
    bm25_results, ignore_index=True
)

m4_top_all = pd.concat(
    m4_results, ignore_index=True
)

comparison = pd.DataFrame(comparisons)
diagnostic = pd.DataFrame(diagnostics)

# --------------------------------------------------
# 6. Auditar Top 30 M4 contra etapa 18A
# --------------------------------------------------

previous = pd.read_parquet(
    STAGE18A / "semantic_controlled_top30.parquet"
)

previous = previous[
    previous["METHOD"] == "Frontier3Y_M4_same_area"
].copy()

previous["ANTECEDENT_ID"] = (
    previous["ANTECEDENT_ID"].astype(str)
)

for focal_id in FOCAL_IDS:

    old_ids = set(
        previous.loc[
            previous["FOCAL_ID"] == focal_id,
            "ANTECEDENT_ID"
        ]
    )

    new_ids = set(
        m4_top_all.loc[
            m4_top_all["FOCAL_ID"] == focal_id,
            "ANTECEDENT_ID"
        ]
    )

    assert old_ids == new_ids, (
        f"Top 30 M4 não reproduzido: {focal_id}"
    )

# --------------------------------------------------
# 7. Salvar resultados e manifesto
# --------------------------------------------------

bm_top_all.to_parquet(
    OUT / "bm25_frontier_top30.parquet",
    index=False
)

m4_top_all.to_parquet(
    OUT / "m4_frontier_top30.parquet",
    index=False
)

comparison.to_csv(
    OUT / "controlled_frontier_comparison.csv",
    index=False
)

diagnostic.to_csv(
    OUT / "frontier_diagnostics.csv",
    index=False
)

manifest = {
    "stage": "18B",
    "purpose": "temporal_and_disciplinary_control",
    "candidate_rule": (
        "same_AREA_HARMONIZADA "
        "and focal_year-3 <= antecedent_year < focal_year"
    ),
    "lexical_fields": ["TITULO", "PALAVRAS_CHAVE"],
    "semantic_representation": "M4_frozen",
    "BM25_k1": K1,
    "BM25_b": B,
    "BM25_IDF_recomputed_within_frontier": True,
    "top_k": TOP_K,
    "semantic_rank_reproduced_from_18A": True,
    "human_relevance_assessed": False,
    "confirmatory": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== COMPARACAO CONTROLADA: FRONTEIRA 3Y ===")
print(comparison.to_string(index=False))

print("\n=== DIAGNOSTICOS ===")
print(diagnostic.to_string(index=False))

print("\nETAPA 18B CONCLUIDA")
print("Diretorio:", OUT)


=== UNIVERSOS CONTROLADOS ===
FOCAL_ID
PILOT-008     4337
PILOT-077     6608
PILOT-090    16629
2014: recuperados 1391 de 1391 IDs distintos
2015: recuperados 1457 de 1457 IDs distintos
2016: recuperados 3657 de 3657 IDs distintos
2017: recuperados 2119 de 2119 IDs distintos
2018: recuperados 2321 de 2321 IDs distintos
2020: recuperados 5363 de 5363 IDs distintos
2021: recuperados 5664 de 5664 IDs distintos
2022: recuperados 5602 de 5602 IDs distintos

=== COMPARACAO CONTROLADA: FRONTEIRA 3Y ===
 FOCAL_ID                SCOPE  N_CANDIDATES  TOP_K  INTERSECTION  OVERLAP_AT_K  JACCARD
PILOT-008 same_area_frontier3y          4337     30             4      0.133333 0.071429
PILOT-077 same_area_frontier3y          6608     30             8      0.266667 0.153846
PILOT-090 same_area_frontier3y         16629     30             0      0.000000 0.000000

=== DIAGNOSTICOS ===
 FOCAL_ID  N_CANDIDATES  N_BM25_POSITIVE  BM25_MAX   M4_MAX
PILOT-008          4337              387 31.738405 0.968994
P

In [28]:

# SCIENTOMETRICS 2026 — ETAPA 18C
# Comparacao controlada: BM25 vs M2 vs M4
# Historico completo e fronteira de 3 anos
# CPU | Embeddings congelados | Sem FAISS

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

EMB = BASE / "07_embeddings_nacionais"

RUN_BM25_H = (
    LAB / "04_antecedent_retrieval"
    / "bm25_20261020T172459Z"
)

RUN_18A = (
    LAB / "04_antecedent_retrieval"
    / "controlled_semantic_stage18A_20261020T181204_113099Z"
)

RUN_18B = (
    LAB / "04_antecedent_retrieval"
    / "controlled_bm25_frontier_stage18B_20261020T182726_242023Z"
)

KEY = (
    LAB / "03_ground_truth/pilot_90"
    / "pilot_linkage_key_RESTRICTED.csv"
)

FOCAL_IDS = ["PILOT-008", "PILOT-077", "PILOT-090"]
TOP_K = 30
BATCH_SIZE = 2048

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "04_antecedent_retrieval"
    / f"controlled_M2_M4_BM25_stage18C_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

# -------------------------------------------------
# 1. Carregar universos ja auditados
# -------------------------------------------------

scores = pd.read_parquet(
    RUN_18A / "semantic_all_candidate_scores.parquet"
)

scores["ANTECEDENT_ID"] = (
    scores["ANTECEDENT_ID"].astype(str)
)

scores["ANTECEDENT_YEAR"] = (
    scores["ANTECEDENT_YEAR"].astype(int)
)

scores["FOCAL_YEAR"] = (
    scores["FOCAL_YEAR"].astype(int)
)

assert scores["FOCAL_ID"].nunique() == 3
assert not scores.duplicated(
    ["FOCAL_ID", "ANTECEDENT_ID"]
).any()

assert (scores["ANTECEDENT_YEAR"]
        < scores["FOCAL_YEAR"]).all()

key = pd.read_csv(KEY, dtype=str)

focals = key[
    key["PILOT_ID"].isin(FOCAL_IDS)
][["PILOT_ID", "ID_TRABALHO", "ANO"]].copy()

focals["ANO"] = focals["ANO"].astype(int)
assert len(focals) == 3

bm25_h = pd.read_parquet(
    RUN_BM25_H / "bm25_top30_three_focals.parquet"
)

bm25_f = pd.read_parquet(
    RUN_18B / "bm25_frontier_top30.parquet"
)

for df in [bm25_h, bm25_f]:
    df["ANTECEDENT_ID"] = (
        df["ANTECEDENT_ID"].astype(str)
    )

# -------------------------------------------------
# 2. Ler IDs e vetores M2 por ano, sob demanda
# -------------------------------------------------

id_cache = {}
emb_cache = {}

def annual_index(year):
    year = int(year)

    if year not in id_cache:
        path = (
            EMB / "M2"
            / f"M2_{year}_ids.parquet"
        )

        ids = pd.read_parquet(
            path,
            columns=["ID_TRABALHO"]
        )

        ids["ID_TRABALHO"] = (
            ids["ID_TRABALHO"].astype(str)
        )

        assert ids["ID_TRABALHO"].is_unique

        id_cache[year] = pd.Index(
            ids["ID_TRABALHO"]
        )

    return id_cache[year]


def annual_vectors(year):
    year = int(year)

    if year not in emb_cache:
        path = (
            EMB / "M2"
            / f"M2_{year}_embeddings_float16.npy"
        )

        arr = np.load(path, mmap_mode="r")

        assert arr.ndim == 2
        assert len(arr) == len(annual_index(year))

        emb_cache[year] = arr

    return emb_cache[year]


def normalized_focal(doc_id, year):
    ids = annual_index(year)
    pos = ids.get_loc(str(doc_id))

    assert isinstance(pos, (int, np.integer))

    v = np.asarray(
        annual_vectors(year)[pos],
        dtype=np.float32
    )

    n = np.linalg.norm(v)
    assert np.isfinite(n) and n > 0

    return v / n

# -------------------------------------------------
# 3. Calcular cosseno M2 para os mesmos candidatos
# -------------------------------------------------

all_parts = []

for focal in focals.itertuples(index=False):

    focal_id = focal.PILOT_ID

    group = scores[
        scores["FOCAL_ID"] == focal_id
    ].copy()

    query = normalized_focal(
        focal.ID_TRABALHO,
        focal.ANO
    )

    results = []

    for year, part in group.groupby(
        "ANTECEDENT_YEAR"
    ):

        year = int(year)

        ids = annual_index(year)
        vectors = annual_vectors(year)

        positions = ids.get_indexer(
            part["ANTECEDENT_ID"].tolist()
        )

        assert (positions >= 0).all(), (
            f"IDs M2 ausentes no ano {year}"
        )

        sims = np.empty(
            len(part),
            dtype=np.float32
        )

        for start in range(
            0, len(positions), BATCH_SIZE
        ):

            end = min(
                start + BATCH_SIZE,
                len(positions)
            )

            block = np.asarray(
                vectors[positions[start:end]],
                dtype=np.float32
            )

            norms = np.linalg.norm(
                block, axis=1
            )

            assert np.isfinite(norms).all()
            assert (norms > 0).all()

            block /= norms[:, None]

            sims[start:end] = block @ query

        result = part.copy()
        result["COSINE_M2"] = sims

        results.append(result)

    focal_scores = pd.concat(
        results,
        ignore_index=True
    )

    assert len(focal_scores) == len(group)
    assert focal_scores["ANTECEDENT_ID"].is_unique
    assert np.isfinite(
        focal_scores["COSINE_M2"]
    ).all()

    all_parts.append(focal_scores)

    print(
        f"{focal_id}: "
        f"{len(focal_scores)} candidatos M2 pontuados"
    )

    # Os memmaps sao reabertos quando necessarios.
    emb_cache.clear()

all_scores = pd.concat(
    all_parts,
    ignore_index=True
)

assert len(all_scores) == len(scores)
assert all_scores["COSINE_M2"].notna().all()
assert all_scores["COSINE_M4"].notna().all()

# -------------------------------------------------
# 4. Construir Top 30 para cada janela
# -------------------------------------------------

rankings = []

for focal_id, group in all_scores.groupby("FOCAL_ID"):

    focal_year = int(group["FOCAL_YEAR"].iloc[0])

    windows = {
        "historical": group,
        "frontier3y": group[
            group["ANTECEDENT_YEAR"] >= focal_year - 3
        ]
    }

    for window, candidates in windows.items():

        assert len(candidates) >= TOP_K

        for method, score_col in [
            ("M2", "COSINE_M2"),
            ("M4", "COSINE_M4")
        ]:

            top = candidates.sort_values(
                [score_col, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K).copy()

            top["WINDOW"] = window
            top["METHOD"] = method
            top["METHOD_RANK"] = np.arange(
                1, len(top) + 1
            )

            rankings.append(top[[
                "FOCAL_ID",
                "FOCAL_YEAR",
                "ANTECEDENT_ID",
                "ANTECEDENT_YEAR",
                "WINDOW",
                "METHOD",
                "METHOD_RANK",
                score_col
            ]].rename(
                columns={score_col: "METHOD_SCORE"}
            ))

semantic_rankings = pd.concat(
    rankings,
    ignore_index=True
)

# -------------------------------------------------
# 5. Padronizar rankings BM25 existentes
# -------------------------------------------------

bm_h = bm25_h[[
    "FOCAL_ID",
    "FOCAL_YEAR",
    "ANTECEDENT_ID",
    "ANTECEDENT_YEAR",
    "RANK",
    "BM25_SCORE"
]].rename(columns={
    "RANK": "METHOD_RANK",
    "BM25_SCORE": "METHOD_SCORE"
}).copy()

bm_h["WINDOW"] = "historical"
bm_h["METHOD"] = "BM25"

bm_f = bm25_f[[
    "FOCAL_ID",
    "FOCAL_YEAR",
    "ANTECEDENT_ID",
    "ANTECEDENT_YEAR",
    "RANK",
    "BM25_SCORE_FRONTIER"
]].rename(columns={
    "RANK": "METHOD_RANK",
    "BM25_SCORE_FRONTIER": "METHOD_SCORE"
}).copy()

bm_f["WINDOW"] = "frontier3y"
bm_f["METHOD"] = "BM25"

all_rankings = pd.concat(
    [semantic_rankings, bm_h, bm_f],
    ignore_index=True
)

assert not all_rankings.duplicated([
    "FOCAL_ID", "WINDOW",
    "METHOD", "ANTECEDENT_ID"
]).any()

# -------------------------------------------------
# 6. Comparar metodos em condicoes iguais
# -------------------------------------------------

comparisons = []

pairs = [
    ("BM25", "M2"),
    ("BM25", "M4"),
    ("M2", "M4")
]

for focal_id in FOCAL_IDS:

    for window in ["historical", "frontier3y"]:

        subset = all_rankings[
            (all_rankings["FOCAL_ID"] == focal_id) &
            (all_rankings["WINDOW"] == window)
        ]

        sets = {
            method: set(
                subset.loc[
                    subset["METHOD"] == method,
                    "ANTECEDENT_ID"
                ]
            )
            for method in ["BM25", "M2", "M4"]
        }

        assert all(len(s) == TOP_K for s in sets.values())

        for a, b in pairs:

            shared = sets[a] & sets[b]
            union = sets[a] | sets[b]

            comparisons.append({
                "FOCAL_ID": focal_id,
                "WINDOW": window,
                "METHOD_A": a,
                "METHOD_B": b,
                "TOP_K": TOP_K,
                "INTERSECTION": len(shared),
                "OVERLAP_AT_K": len(shared) / TOP_K,
                "JACCARD": len(shared) / len(union)
            })

comparison = pd.DataFrame(comparisons)

# -------------------------------------------------
# 7. Auditoria: reproduzir resultados de 18A e 18B
# -------------------------------------------------

previous_h = pd.read_csv(
    RUN_18A / "bm25_vs_m4_same_area_historical.csv"
)

previous_f = pd.read_csv(
    RUN_18B / "controlled_frontier_comparison.csv"
)

for focal_id in FOCAL_IDS:

    new_h = comparison[
        (comparison["FOCAL_ID"] == focal_id) &
        (comparison["WINDOW"] == "historical") &
        (comparison["METHOD_A"] == "BM25") &
        (comparison["METHOD_B"] == "M4")
    ]["INTERSECTION"].iloc[0]

    old_h = previous_h.loc[
        previous_h["FOCAL_ID"] == focal_id,
        "INTERSECTION"
    ].iloc[0]

    assert int(new_h) == int(old_h)

    new_f = comparison[
        (comparison["FOCAL_ID"] == focal_id) &
        (comparison["WINDOW"] == "frontier3y") &
        (comparison["METHOD_A"] == "BM25") &
        (comparison["METHOD_B"] == "M4")
    ]["INTERSECTION"].iloc[0]

    old_f = previous_f.loc[
        previous_f["FOCAL_ID"] == focal_id,
        "INTERSECTION"
    ].iloc[0]

    assert int(new_f) == int(old_f)

# -------------------------------------------------
# 8. Salvar escores, rankings e manifesto
# -------------------------------------------------

all_scores.to_parquet(
    OUT / "all_candidate_scores_M2_M4.parquet",
    index=False
)

all_rankings.to_parquet(
    OUT / "controlled_rankings_3methods.parquet",
    index=False
)

comparison.to_csv(
    OUT / "controlled_overlap_M2_M4_BM25.csv",
    index=False
)

manifest = {
    "stage": "18C",
    "design": "three_methods_two_windows",
    "methods": {
        "BM25": "title_keywords",
        "M2": "title_keywords_embeddings",
        "M4": "title_abstract_keywords_embeddings"
    },
    "same_area": True,
    "same_candidate_universe_within_window": True,
    "windows": ["historical", "frontier3y"],
    "top_k": TOP_K,
    "focal_ids": FOCAL_IDS,
    "no_new_embeddings": True,
    "human_ground_truth_available": False,
    "confirmatory_validation": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== COMPARACAO CONTROLADA: BM25, M2, M4 ===")
print(comparison.to_string(index=False))

print("\n=== RESUMO POR CONTRASTE ===")
print(
    comparison.groupby(
        ["WINDOW", "METHOD_A", "METHOD_B"]
    )["OVERLAP_AT_K"].agg(
        ["mean", "min", "max"]
    ).to_string()
)

print("\nETAPA 18C CONCLUIDA")
print("Diretorio:", OUT)


PILOT-008: 5743 candidatos M2 pontuados
PILOT-077: 12334 candidatos M2 pontuados
PILOT-090: 51785 candidatos M2 pontuados

=== COMPARACAO CONTROLADA: BM25, M2, M4 ===
 FOCAL_ID     WINDOW METHOD_A METHOD_B  TOP_K  INTERSECTION  OVERLAP_AT_K  JACCARD
PILOT-008 historical     BM25       M2     30             7      0.233333 0.132075
PILOT-008 historical     BM25       M4     30             3      0.100000 0.052632
PILOT-008 historical       M2       M4     30             7      0.233333 0.132075
PILOT-008 frontier3y     BM25       M2     30             9      0.300000 0.176471
PILOT-008 frontier3y     BM25       M4     30             4      0.133333 0.071429
PILOT-008 frontier3y       M2       M4     30             7      0.233333 0.132075
PILOT-077 historical     BM25       M2     30            12      0.400000 0.250000
PILOT-077 historical     BM25       M4     30             8      0.266667 0.153846
PILOT-077 historical       M2       M4     30            10      0.333333 0.200000
PIL

In [29]:

# SCIENTOMETRICS 2026 — ETAPA 19A
# Benchmark multissistema de relevancia contextual
# 48 pares: 16 por documento focal
# CPU | Somente artefatos existentes

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

RUN18C = (
    LAB / "04_antecedent_retrieval"
    / "controlled_M2_M4_BM25_stage18C_20261020T183307_074733Z"
)

SRC = BASE / "05_transferencia_colab"
PILOT = LAB / "03_ground_truth/pilot_90/pilot_cases_blinded.csv"

SEED = 20261020
rng = np.random.default_rng(SEED)

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "03_ground_truth"
    / f"multisystem_context_48_{STAMP}"
)
OUT.mkdir(parents=True, exist_ok=False)

# 1. Rankings dos tres sistemas, somente fronteira 3Y.
r = pd.read_parquet(
    RUN18C / "controlled_rankings_3methods.parquet"
)

r = r[r["WINDOW"] == "frontier3y"].copy()
r["ANTECEDENT_ID"] = r["ANTECEDENT_ID"].astype(str)

METHODS = ["BM25", "M2", "M4"]
FOCALS = ["PILOT-008", "PILOT-077", "PILOT-090"]

assert set(r["METHOD"]) == set(METHODS)
assert not r.duplicated(
    ["FOCAL_ID", "METHOD", "ANTECEDENT_ID"]
).any()

# 2. Universo completo, identico para os metodos.
all_scores = pd.read_parquet(
    RUN18C / "all_candidate_scores_M2_M4.parquet"
)

all_scores["ANTECEDENT_ID"] = (
    all_scores["ANTECEDENT_ID"].astype(str)
)

all_scores = all_scores[
    all_scores["ANTECEDENT_YEAR"].astype(int)
    >= all_scores["FOCAL_YEAR"].astype(int) - 3
].copy()

assert not all_scores.duplicated(
    ["FOCAL_ID", "ANTECEDENT_ID"]
).any()

# 3. Presenca do antecedente nos tres Top 30.
presence = (
    r.assign(PRESENT=1)
    .pivot_table(
        index=["FOCAL_ID", "ANTECEDENT_ID"],
        columns="METHOD",
        values="PRESENT",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

for method in METHODS:
    presence[method] = presence[method].astype(int)

presence["N_METHODS"] = presence[METHODS].sum(axis=1)

selected_parts = []
selection_diag = []

# 4. Selecionar estratos sem usar julgamentos.
for focal in FOCALS:

    p = presence[presence["FOCAL_ID"] == focal].copy()
    universe = all_scores[
        all_scores["FOCAL_ID"] == focal
    ].copy()

    chosen = []

    for method in METHODS:

        exclusive = p[
            (p[method] == 1) &
            (p["N_METHODS"] == 1)
        ]

        assert len(exclusive) >= 4, (
            f"Exclusivos insuficientes: {focal}, {method}"
        )

        sample = exclusive.sample(
            n=4,
            random_state=SEED + FOCALS.index(focal)
            + METHODS.index(method)
        ).copy()

        sample["SELECTION_STRATUM"] = (
            f"exclusive_{method}"
        )
        chosen.append(sample)

    shared = p[p["N_METHODS"] >= 2].copy()

    assert len(shared) >= 2, (
        f"Compartilhados insuficientes: {focal}"
    )

    shared_sample = shared.sample(
        n=2,
        random_state=SEED + 100 + FOCALS.index(focal)
    ).copy()

    shared_sample["SELECTION_STRATUM"] = "multisystem"
    chosen.append(shared_sample)

    selected_ids = set(p["ANTECEDENT_ID"])

    controls = universe[
        ~universe["ANTECEDENT_ID"].isin(selected_ids)
    ][["FOCAL_ID", "ANTECEDENT_ID"]].copy()

    assert len(controls) >= 2

    control_sample = controls.sample(
        n=2,
        random_state=SEED + 200 + FOCALS.index(focal)
    ).copy()

    for method in METHODS:
        control_sample[method] = 0

    control_sample["N_METHODS"] = 0
    control_sample["SELECTION_STRATUM"] = "random_control"

    chosen.append(control_sample)

    selected = pd.concat(chosen, ignore_index=True)

    assert len(selected) == 16
    assert selected["ANTECEDENT_ID"].is_unique

    selected_parts.append(selected)

    print(f"\n{focal} | total selecionado: {len(selected)}")
    print(selected["SELECTION_STRATUM"].value_counts())

selection = pd.concat(selected_parts, ignore_index=True)

assert len(selection) == 48

# 5. Vincular metadados dos antecedentes.
meta = all_scores[[
    "FOCAL_ID",
    "ANTECEDENT_ID",
    "FOCAL_YEAR",
    "ANTECEDENT_YEAR",
    "COSINE_M2",
    "COSINE_M4"
]]

selection = selection.merge(
    meta,
    on=["FOCAL_ID", "ANTECEDENT_ID"],
    how="left",
    validate="one_to_one"
)

assert selection["ANTECEDENT_YEAR"].notna().all()

# 6. Recuperar textos somente dos 48 documentos.
wanted_by_year = {
    int(y): set(g["ANTECEDENT_ID"])
    for y, g in selection.groupby("ANTECEDENT_YEAR")
}

parts = []

for year, wanted in sorted(wanted_by_year.items()):

    path = SRC / f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    pf = pq.ParquetFile(path)

    matches = []

    for batch in pf.iter_batches(
        batch_size=16384,
        columns=[
            "ID_TRABALHO", "TITULO",
            "RESUMO", "PALAVRAS_CHAVE"
        ]
    ):
        chunk = batch.to_pandas()
        chunk["ID_TRABALHO"] = (
            chunk["ID_TRABALHO"].astype(str)
        )

        hit = chunk[
            chunk["ID_TRABALHO"].isin(wanted)
        ]

        if not hit.empty:
            matches.append(hit)

    assert matches, f"Sem documentos no ano {year}"

    year_df = pd.concat(matches, ignore_index=True)

    assert set(year_df["ID_TRABALHO"]) == wanted

    year_df["ANTECEDENT_YEAR"] = year
    parts.append(year_df)

    print(
        "Ano", year,
        "| solicitados:", len(wanted),
        "| recuperados:", len(year_df)
    )

texts = pd.concat(parts, ignore_index=True)

texts = texts.rename(columns={
    "ID_TRABALHO": "ANTECEDENT_ID",
    "TITULO": "ANTECEDENT_TITLE",
    "RESUMO": "ANTECEDENT_ABSTRACT",
    "PALAVRAS_CHAVE": "ANTECEDENT_KEYWORDS"
})

selection = selection.merge(
    texts,
    on=["ANTECEDENT_ID", "ANTECEDENT_YEAR"],
    validate="many_to_one",
    how="left"
)

assert selection["ANTECEDENT_TITLE"].notna().all()

# 7. Textos focais.
pilot = pd.read_csv(
    PILOT,
    dtype=str,
    keep_default_na=False
)

focal_text = pilot[[
    "PILOT_ID", "TITULO", "RESUMO", "PALAVRAS_CHAVE"
]].rename(columns={
    "PILOT_ID": "FOCAL_ID",
    "TITULO": "FOCAL_TITLE",
    "RESUMO": "FOCAL_ABSTRACT",
    "PALAVRAS_CHAVE": "FOCAL_KEYWORDS"
})

selection = selection.merge(
    focal_text,
    on="FOCAL_ID",
    validate="many_to_one",
    how="left"
)

assert selection["FOCAL_ABSTRACT"].notna().all()

# 8. Gerar IDs cegos e embaralhar.
selection = selection.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

selection.insert(
    0, "PAIR_ID",
    [f"MSPAIR-{i:04d}" for i in range(1, 49)]
)

# Chave restrita: metodo, escores e origem.
selection.to_parquet(
    OUT / "selection_linkage_RESTRICTED.parquet",
    index=False
)

# Fichas sem metodos nem pontuacoes.
review_cols = [
    "PAIR_ID",
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_YEAR",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]

annotation_fields = [
    "RELEVANCE_0_1_2_3_U",
    "CONTRIBUTION_LOCUS",
    "IS_PLAUSIBLE_PRECEDENT",
    "EVIDENCE_SUFFICIENCY",
    "CONFIDENCE_1_TO_5",
    "EVIDENCE_EXCERPT",
    "RATIONALE"
]

for reviewer in ["A", "B"]:

    sheet = selection[review_cols].copy()

    for field in annotation_fields:
        sheet[field] = ""

    sheet["REVIEWER"] = reviewer

    sheet.to_csv(
        OUT / f"multisystem_review_{reviewer}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# 9. Verificacoes e manifesto.
diagnostic = (
    selection.groupby([
        "FOCAL_ID", "SELECTION_STRATUM"
    ]).size().reset_index(name="N")
)

diagnostic.to_csv(
    OUT / "selection_diagnostics.csv",
    index=False
)

manifest = {
    "stage": "19A",
    "purpose": "multisystem_contextual_calibration",
    "n_pairs": 48,
    "focal_documents": FOCALS,
    "scope": "same_area_frontier3y",
    "methods": METHODS,
    "selection_per_focal": {
        "exclusive_BM25": 4,
        "exclusive_M2": 4,
        "exclusive_M4": 4,
        "multisystem": 2,
        "random_control": 2
    },
    "sampling": "stratified_intentional_not_probability",
    "reviewers": ["A", "B"],
    "blind_to_methods_and_scores": True,
    "ground_truth_established": False,
    "confirmatory": False,
    "random_seed": SEED
}

(OUT / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ETAPA 19A CONCLUIDA ===")
print("Pares:", len(selection))
print("Focais:", selection["FOCAL_ID"].nunique())
print("Resumo antecedente ausente:",
      selection["ANTECEDENT_ABSTRACT"].isna().sum())
print("\n=== ESTRATOS ===")
print(diagnostic.to_string(index=False))
print("\nDiretorio:", OUT)



PILOT-008 | total selecionado: 16
SELECTION_STRATUM
exclusive_BM25    4
exclusive_M2      4
exclusive_M4      4
multisystem       2
random_control    2
Name: count, dtype: int64

PILOT-077 | total selecionado: 16
SELECTION_STRATUM
exclusive_BM25    4
exclusive_M2      4
exclusive_M4      4
multisystem       2
random_control    2
Name: count, dtype: int64

PILOT-090 | total selecionado: 16
SELECTION_STRATUM
exclusive_BM25    4
exclusive_M2      4
exclusive_M4      4
multisystem       2
random_control    2
Name: count, dtype: int64
Ano 2014 | solicitados: 4 | recuperados: 4
Ano 2015 | solicitados: 8 | recuperados: 8
Ano 2016 | solicitados: 9 | recuperados: 9
Ano 2017 | solicitados: 3 | recuperados: 3
Ano 2018 | solicitados: 8 | recuperados: 8
Ano 2020 | solicitados: 4 | recuperados: 4
Ano 2021 | solicitados: 5 | recuperados: 5
Ano 2022 | solicitados: 7 | recuperados: 7

=== ETAPA 19A CONCLUIDA ===
Pares: 48
Focais: 3
Resumo antecedente ausente: 0

=== ESTRATOS ===
 FOCAL_ID SELECTION_ST

In [30]:

# SCIENTOMETRICS 2026 — ETAPA 19B
# Auditoria e congelamento do protocolo de anotacao
# CPU | Sem embeddings | Sem classificacao automatica

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

LAB = Path(
    "/content/gdrive/MyDrive/CAPES_Epistemic_Change"
    "/08_epistemic_measurement_validation"
)

SRC = (
    LAB / "03_ground_truth"
    / "multisystem_context_48_20261020T185341_194580Z"
)

assert SRC.is_dir()

a = pd.read_csv(
    SRC / "multisystem_review_A.csv",
    dtype=str,
    keep_default_na=False
)

b = pd.read_csv(
    SRC / "multisystem_review_B.csv",
    dtype=str,
    keep_default_na=False
)

link = pd.read_parquet(
    SRC / "selection_linkage_RESTRICTED.parquet"
)

link["PAIR_ID"] = link["PAIR_ID"].astype(str)

# -----------------------------------------------
# 1. Auditoria de integridade
# -----------------------------------------------

assert len(a) == len(b) == len(link) == 48
assert a["PAIR_ID"].is_unique
assert b["PAIR_ID"].is_unique
assert link["PAIR_ID"].is_unique

assert set(a["PAIR_ID"]) == set(b["PAIR_ID"])
assert set(a["PAIR_ID"]) == set(link["PAIR_ID"])

text_fields = [
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_YEAR",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]

paired = a[["PAIR_ID"] + text_fields].merge(
    b[["PAIR_ID"] + text_fields],
    on="PAIR_ID",
    validate="one_to_one",
    suffixes=("_A", "_B")
)

for field in text_fields:
    assert (
        paired[f"{field}_A"]
        == paired[f"{field}_B"]
    ).all(), f"Divergencia entre fichas: {field}"

annotation_fields = [
    "RELEVANCE_0_1_2_3_U",
    "CONTRIBUTION_LOCUS",
    "IS_PLAUSIBLE_PRECEDENT",
    "EVIDENCE_SUFFICIENCY",
    "CONFIDENCE_1_TO_5",
    "EVIDENCE_EXCERPT",
    "RATIONALE"
]

for df, name in [(a, "A"), (b, "B")]:
    assert set(annotation_fields).issubset(df.columns)

    for field in annotation_fields:
        assert df[field].eq("").all(), (
            f"Formulario {name} ja possui respostas "
            f"na coluna {field}. Preservar as anotacoes."
        )

assert (link["ANTECEDENT_YEAR"].astype(int)
        < link["FOCAL_YEAR"].astype(int)).all()

expected = {
    "exclusive_BM25": 4,
    "exclusive_M2": 4,
    "exclusive_M4": 4,
    "multisystem": 2,
    "random_control": 2
}

for focal, group in link.groupby("FOCAL_ID"):
    observed = group["SELECTION_STRATUM"].value_counts()
    for category, n in expected.items():
        assert int(observed.get(category, 0)) == n, (
            f"Estrato invalido: {focal}, {category}"
        )

# -----------------------------------------------
# 2. Protocolo metodologico
# -----------------------------------------------

protocol = """# Protocol v1.0 — Contextual Antecedent Relevance

## Objetivo

Avaliar a relacao cientifica entre um documento focal e
um antecedente cronologicamente anterior, com base nos
titulos, resumos e palavras-chave disponiveis.

A avaliacao nao equivale a uma verificacao exaustiva
da originalidade cientifica ou historica.

## Unidade de avaliacao

Par documento focal–documento antecedente.

O mesmo documento focal aparece em varios pares.
Cada par deve ser avaliado individualmente.

## Passo 1 — Identificar a contribuicao focal

Antes de atribuir relevancia, registrar mentalmente
ou nas justificativas:

- Qual problema cientifico o documento focal enfrenta?
- Qual contribuicao especifica declara?
- Ela envolve objeto, conceito, metodo, evidencia
  empirica, aplicacao ou combinacao desses elementos?

Distinguir reivindicacao de novidade de novidade
efetivamente demonstrada.

## Passo 2 — Classificar a relevancia do antecedente

0: Sem relacao substantiva identificavel.

1: Relacao tematica geral. Compartilha um assunto ou
campo, mas nao trata diretamente do problema,
objeto ou metodo focal.

2: Relacao direta com o problema, objeto ou metodo.
Pode fundamentar ou contextualizar a contribuicao,
sem evidencia suficiente de equivalencia.

3: Precedente plausivel da contribuicao especifica.
O resumo anterior apresenta evidencias de que
componente central reivindicado pelo trabalho focal
ja foi desenvolvido ou investigado de modo
substancialmente semelhante.

U: Evidencia insuficiente para classificacao.

Usar 3 com parcimonia. A mesma terminologia
nao demonstra equivalencia cientifica.

## Passo 3 — Identificar o locus da contribuicao

CONTRIBUTION_LOCUS:
OBJECT | CONCEPT | METHOD | EVIDENCE |
APPLICATION | MULTIPLE | UNCLEAR

Selecionar o elemento central da relacao.
MULTIPLE somente quando mais de um componente for
substantivamente relevante.

## Passo 4 — Julgar precedencia separadamente

IS_PLAUSIBLE_PRECEDENT:
YES | NO | UNCERTAIN

YES indica precedente plausivel da contribuicao
especifica, e nao prova de ausencia de originalidade.

Uma relacao direta (codigo 2) nao implica YES.

Para classificacao 3, registrar YES.
Se a evidencia for insuficiente, preferir U e
UNCERTAIN, justificando.

## Passo 5 — Avaliar suficiencia das evidencias

EVIDENCE_SUFFICIENCY:
SUFFICIENT | PARTIAL | INSUFFICIENT

SUFFICIENT: o resumo permite fundamentar a categoria
atribuida, dentro do escopo da avaliacao.

PARTIAL: ha indicios relevantes, mas faltam detalhes.

INSUFFICIENT: seria necessario consultar o texto
integral ou outra fonte para uma decisao fundamentada.

Nao interpretar SUFFICIENT como prova de originalidade
historica perante a literatura internacional.

## Passo 6 — Registrar confianca e justificativa

CONFIDENCE_1_TO_5:
1 = muito baixa; 5 = muito alta.

EVIDENCE_EXCERPT:
Registrar trecho curto de evidencias efetivamente
presente nos titulos ou resumos fornecidos.

RATIONALE:
Explicar a correspondencia ou divergencia entre a
contribuicao focal e o antecedente.

Nao inferir resultados que o resumo nao relata.

## Cegamento

Nao disponibilizar aos avaliadores:
- Metodo de recuperacao;
- Estrato de selecao;
- Posicao nos rankings;
- Escores BM25 ou cosseno;
- Chave restrita de identificacao.

Os avaliadores devem trabalhar independentemente,
sem trocar classificacoes ate o congelamento das
duas fichas.

## Restricoes metodologicas

- Corpus CAPES de teses e dissertacoes, nao toda a
  literatura cientifica internacional.
- Cobertura temporal iniciada em 2013.
- Ausencia de antecedente relevante no corpus
  nao demonstra originalidade.
- Classificacoes derivadas de resumos, nao da leitura
  sistematica de textos integrais.
- Amostra intencional de 48 pares, nao probabilistica.
- Tres documentos focais; sem generalizacao nacional.
- Nenhuma resposta automatica constitui julgamento
  humano independente.

## Pos-avaliacao

1. Congelar e preservar as duas fichas originais.
2. Auditar valores ausentes e categorias invalidas.
3. Calcular concordancia observada.
4. Estimar kappa ponderado com ressalvas amostrais.
5. Examinar divergencias qualitativamente.
6. Realizar adjudicacao documentada, preservando
   os julgamentos originais.
7. Vincular os julgamentos aos metodos somente
   apos o congelamento das avaliacoes.
"""

# -----------------------------------------------
# 3. Salvar protocolo e auditoria versionados
# -----------------------------------------------

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = SRC / f"annotation_protocol_v1_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

(OUT / "annotation_protocol_v1.md").write_text(
    protocol,
    encoding="utf-8"
)

audit = {
    "stage": "19B",
    "n_pairs": 48,
    "n_focals": int(link["FOCAL_ID"].nunique()),
    "two_forms_identical_in_content": True,
    "forms_unannotated_at_audit": True,
    "all_antecedents_prior": True,
    "selection_strata_verified": True,
    "ground_truth_established": False,
    "human_annotation_completed": False,
    "source_directory": str(SRC),
    "protocol_version": "1.0"
}

(OUT / "annotation_audit.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== ETAPA 19B CONCLUIDA ===")
print("Pares auditados:", len(link))
print("Documentos focais:", link["FOCAL_ID"].nunique())
print("Fichas A e B:", "INTEGRAS E EQUIVALENTES")
print("Precedencia temporal:", "OK")
print("Protocolo:", OUT / "annotation_protocol_v1.md")
print("Ground truth:", "AINDA NAO ESTABELECIDO")


=== ETAPA 19B CONCLUIDA ===
Pares auditados: 48
Documentos focais: 3
Fichas A e B: INTEGRAS E EQUIVALENTES
Precedencia temporal: OK
Protocolo: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/03_ground_truth/multisystem_context_48_20261020T185341_194580Z/annotation_protocol_v1_20261020T185838_624714Z/annotation_protocol_v1.md
Ground truth: AINDA NAO ESTABELECIDO


In [31]:

# SCIENTOMETRICS 2026 — ETAPA 20A
# Congelamento da amostra nacional balanceada
# 108 documentos focais | CPU | Sem embeddings

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import json
import hashlib

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

MASTER_FILE = (
    LAB / "03_ground_truth"
    / "master_probability_sample_1440.parquet"
)

PILOT_KEY = (
    LAB / "03_ground_truth/pilot_90"
    / "pilot_linkage_key_RESTRICTED.csv"
)

SEED = 20261020
N_PER_AREA = 12
MAX_PER_AREA_YEAR = 2

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "05_national_expansion"
    / f"focal_sample_108_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

# ---------------------------------------------------
# 1. Carregar amostra mestre
# ---------------------------------------------------

master = pd.read_parquet(MASTER_FILE)

print("=== ESTRUTURA DA AMOSTRA MESTRE ===")
print("Linhas:", len(master))
print("Colunas:", master.columns.tolist())

required = [
    "ID_TRABALHO",
    "ANO",
    "NM_GRANDE_AREA_CONHECIMENTO"
]

missing = [c for c in required if c not in master.columns]

if missing:
    raise KeyError(
        "Colunas ausentes na amostra mestre: "
        + ", ".join(missing)
    )

assert len(master) == 1440

master["ID_TRABALHO"] = (
    master["ID_TRABALHO"].astype(str)
)

master["ANO"] = pd.to_numeric(
    master["ANO"], errors="raise"
).astype(int)

assert master["ID_TRABALHO"].is_unique

# ---------------------------------------------------
# 2. Excluir documentos usados no piloto
# ---------------------------------------------------

pilot = pd.read_csv(PILOT_KEY, dtype=str)

assert "ID_TRABALHO" in pilot.columns

pilot_ids = set(
    pilot["ID_TRABALHO"].astype(str)
)

eligible = master[
    master["ANO"].between(2016, 2024)
    & ~master["ID_TRABALHO"].isin(pilot_ids)
].copy()

eligible = eligible.dropna(
    subset=["NM_GRANDE_AREA_CONHECIMENTO"]
)

print("\n=== ELEGIBILIDADE ===")
print("Amostra mestre:", len(master))
print("Documentos do piloto:", len(pilot_ids))
print("Elegíveis para expansão:", len(eligible))

print("\n=== GRANDES ÁREAS ELEGÍVEIS ===")
print(
    eligible["NM_GRANDE_AREA_CONHECIMENTO"]
    .value_counts()
    .to_string()
)

# ---------------------------------------------------
# 3. Amostragem estratificada balanceada
# ---------------------------------------------------

areas = sorted(
    eligible["NM_GRANDE_AREA_CONHECIMENTO"].unique()
)

assert len(areas) == 9, (
    f"Esperadas 9 grandes áreas; encontradas {len(areas)}"
)

rng = np.random.default_rng(SEED)

selected_rows = []

for area in areas:

    area_data = eligible[
        eligible["NM_GRANDE_AREA_CONHECIMENTO"] == area
    ].copy()

    # Embaralhar separadamente dentro de cada ano.
    year_groups = {}

    for year, group in area_data.groupby("ANO"):

        positions = rng.permutation(len(group))

        year_groups[int(year)] = (
            group.iloc[positions]
            .reset_index(drop=True)
        )

    # Rodadas: primeiro um por ano, depois o segundo.
    # A ordem anual varia por rodada para evitar
    # favorecer sistematicamente anos mais antigos.
    chosen = []

    for round_number in range(MAX_PER_AREA_YEAR):

        years = list(year_groups.keys())
        rng.shuffle(years)

        for year in years:

            group = year_groups[year]

            if len(group) <= round_number:
                continue

            row = group.iloc[[round_number]]
            chosen.append(row)

            if len(chosen) == N_PER_AREA:
                break

        if len(chosen) == N_PER_AREA:
            break

    assert len(chosen) == N_PER_AREA, (
        f"Não foi possível selecionar {N_PER_AREA} "
        f"documentos para a área {area} "
        f"com máximo de {MAX_PER_AREA_YEAR} por ano."
    )

    area_selection = pd.concat(
        chosen,
        ignore_index=True
    )

    counts = area_selection["ANO"].value_counts()

    assert counts.max() <= MAX_PER_AREA_YEAR

    selected_rows.append(area_selection)

selected = pd.concat(
    selected_rows,
    ignore_index=True
)

assert len(selected) == 108
assert selected["ID_TRABALHO"].is_unique

assert not selected["ID_TRABALHO"].isin(
    pilot_ids
).any()

assert selected["ANO"].between(
    2016, 2024
).all()

# ---------------------------------------------------
# 4. IDs de focal e chave restrita
# ---------------------------------------------------

selected = selected.sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

selected.insert(
    0,
    "FOCAL_ID",
    [f"NAT-{i:04d}" for i in range(1, 109)]
)

# Conservar dados originais e identificadores.
selected.to_parquet(
    OUT / "national_focals_108_RESTRICTED.parquet",
    index=False
)

# Metadados mínimos para processamento.
index_cols = [
    "FOCAL_ID",
    "ID_TRABALHO",
    "ANO",
    "NM_GRANDE_AREA_CONHECIMENTO"
]

if "AREA_HARMONIZADA" in selected.columns:
    index_cols.append("AREA_HARMONIZADA")

selected[index_cols].to_csv(
    OUT / "focal_processing_index.csv",
    index=False
)

# ---------------------------------------------------
# 5. Diagnósticos
# ---------------------------------------------------

area_counts = (
    selected.groupby(
        "NM_GRANDE_AREA_CONHECIMENTO"
    )
    .size()
    .reset_index(name="N")
)

area_year = (
    selected.groupby([
        "NM_GRANDE_AREA_CONHECIMENTO", "ANO"
    ])
    .size()
    .reset_index(name="N")
)

assert area_counts["N"].eq(12).all()
assert area_year["N"].le(2).all()

area_counts.to_csv(
    OUT / "area_balance.csv",
    index=False
)

area_year.to_csv(
    OUT / "area_year_balance.csv",
    index=False
)

year_distribution = (
    selected.groupby("ANO")
    .size()
    .reset_index(name="N")
)

year_distribution.to_csv(
    OUT / "year_distribution.csv",
    index=False
)

# Hash dos IDs na ordem final.
id_string = "\n".join(
    selected["ID_TRABALHO"].astype(str)
)

selection_hash = hashlib.sha256(
    id_string.encode("utf-8")
).hexdigest()

manifest = {
    "stage": "20A",
    "purpose": "national_computational_expansion",
    "source": str(MASTER_FILE),
    "source_n": len(master),
    "focal_years": [2016, 2024],
    "pilot_excluded": True,
    "pilot_n_excluded_by_id": int(
        master["ID_TRABALHO"].isin(pilot_ids).sum()
    ),
    "n_eligible_after_filters": len(eligible),
    "n_selected": len(selected),
    "n_per_grand_area": N_PER_AREA,
    "max_per_grand_area_year": MAX_PER_AREA_YEAR,
    "selection_seed": SEED,
    "selection_id_sha256": selection_hash,
    "design": "balanced_stratified_exploratory",
    "direct_national_representativeness": False,
    "no_indicator_based_selection": True,
    "no_new_embeddings": True,
    "human_ground_truth": False
}

(OUT / "selection_manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== AMOSTRA NACIONAL CONGELADA ===")
print("Documentos selecionados:", len(selected))
print("Grandes áreas:", len(areas))
print("IDs duplicados:", selected["ID_TRABALHO"].duplicated().sum())
print("Sobreposição com piloto:",
      selected["ID_TRABALHO"].isin(pilot_ids).sum())

print("\n=== DISTRIBUIÇÃO POR GRANDE ÁREA ===")
print(area_counts.to_string(index=False))

print("\n=== DISTRIBUIÇÃO POR ANO ===")
print(year_distribution.to_string(index=False))

print("\nSHA256:", selection_hash)
print("Diretório:", OUT)
print("\nETAPA 20A CONCLUÍDA")


=== ESTRUTURA DA AMOSTRA MESTRE ===
Linhas: 1440
Colunas: ['CASE_ID', 'ID_TRABALHO', 'TITULO', 'RESUMO', 'PALAVRAS_CHAVE', 'ANO', 'SOURCE_ROW', 'SELECTION_PROBABILITY', 'DESIGN_WEIGHT', 'AREA_HARMONIZADA', 'NM_GRANDE_AREA_CONHECIMENTO', 'NM_GRAU_ACADEMICO', 'NM_IDIOMA', 'CD_PROGRAMA', 'SG_ENTIDADE_ENSINO', 'NM_REGIAO']

=== ELEGIBILIDADE ===
Amostra mestre: 1440
Documentos do piloto: 90
Elegíveis para expansão: 1013

=== GRANDES ÁREAS ELEGÍVEIS ===
NM_GRANDE_AREA_CONHECIMENTO
CIÊNCIAS HUMANAS               167
CIÊNCIAS SOCIAIS APLICADAS     165
MULTIDISCIPLINAR               154
CIÊNCIAS DA SAÚDE              151
CIÊNCIAS AGRÁRIAS               96
ENGENHARIAS                     82
CIÊNCIAS EXATAS E DA TERRA      79
LINGÜÍSTICA, LETRAS E ARTES     66
CIÊNCIAS BIOLÓGICAS             53

=== AMOSTRA NACIONAL CONGELADA ===
Documentos selecionados: 108
Grandes áreas: 9
IDs duplicados: 0
Sobreposição com piloto: 0

=== DISTRIBUIÇÃO POR GRANDE ÁREA ===
NM_GRANDE_AREA_CONHECIMENTO  N
        

In [32]:

# SCIENTOMETRICS 2026 — ETAPA 20B
# Auditoria de cobertura M2/M4 e dimensionamento
# CPU | Sem busca vetorial | Sem gerar embeddings

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import hashlib
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SAMPLE_DIR = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

FOCAL_FILE = SAMPLE_DIR / "focal_processing_index.csv"

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

EMB = BASE / "07_embeddings_nacionais"

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "05_national_expansion"
    / f"coverage_plan_20B_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

# --------------------------------------------------
# 1. Amostra congelada
# --------------------------------------------------

focals = pd.read_csv(
    FOCAL_FILE,
    dtype={"ID_TRABALHO": str, "FOCAL_ID": str}
)

assert len(focals) == 108
assert focals["FOCAL_ID"].is_unique
assert focals["ID_TRABALHO"].is_unique

focals["ANO"] = focals["ANO"].astype(int)

# Conferir identificadores com a amostra original.
original = pd.read_parquet(
    SAMPLE_DIR / "national_focals_108_RESTRICTED.parquet"
)

original["ID_TRABALHO"] = (
    original["ID_TRABALHO"].astype(str)
)

assert set(focals["ID_TRABALHO"]) == set(
    original["ID_TRABALHO"]
)

# --------------------------------------------------
# 2. Metadados nacionais
# --------------------------------------------------

meta = pq.read_table(
    INDEX_FILE,
    columns=[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA"
    ]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
meta["ANO"] = meta["ANO"].astype(int)

assert meta["ID_TRABALHO"].is_unique
assert len(meta) == 993086

# Preferir área recuperada do índice nacional.
focals = focals.drop(
    columns=["AREA_HARMONIZADA"],
    errors="ignore"
)

focals = focals.merge(
    meta[[
        "ID_TRABALHO", "ANO", "AREA_HARMONIZADA"
    ]],
    on=["ID_TRABALHO", "ANO"],
    validate="one_to_one",
    how="left"
)

assert focals["AREA_HARMONIZADA"].notna().all()

# --------------------------------------------------
# 3. Auditoria dos embeddings anuais
# --------------------------------------------------

annual_records = []
coverage_records = []

for year in range(2013, 2025):

    annual = {}

    for modality in ["M2", "M4"]:

        ids_path = (
            EMB / modality
            / f"{modality}_{year}_ids.parquet"
        )

        vec_path = (
            EMB / modality
            / f"{modality}_{year}_embeddings_float16.npy"
        )

        assert ids_path.is_file(), str(ids_path)
        assert vec_path.is_file(), str(vec_path)

        ids = pd.read_parquet(
            ids_path,
            columns=["ID_TRABALHO"]
        )

        ids["ID_TRABALHO"] = (
            ids["ID_TRABALHO"].astype(str)
        )

        assert ids["ID_TRABALHO"].is_unique

        # Apenas cabeçalho e mapeamento de arquivo.
        arr = np.load(vec_path, mmap_mode="r")

        assert arr.ndim == 2
        assert arr.shape[0] == len(ids)
        assert arr.shape[1] == 768

        annual[modality] = {
            "ids": ids["ID_TRABALHO"].to_numpy(),
            "shape": tuple(arr.shape),
            "dtype": str(arr.dtype)
        }

        del arr

    # Os dois tipos de embedding devem representar
    # exatamente a mesma colecao documental.
    order_equal = np.array_equal(
        annual["M2"]["ids"],
        annual["M4"]["ids"]
    )

    assert order_equal, (
        f"Ordenacao M2/M4 diferente em {year}. "
        "Interromper antes de calcular similaridades."
    )

    indexed = set(
        meta.loc[
            meta["ANO"] == year,
            "ID_TRABALHO"
        ]
    )

    embedded = set(annual["M2"]["ids"])

    same_annual_population = indexed == embedded

    assert same_annual_population, (
        f"Populacao de IDs inconsistente em {year}."
    )

    annual_records.append({
        "ANO": year,
        "N_M2": len(annual["M2"]["ids"]),
        "N_M4": len(annual["M4"]["ids"]),
        "M2_M4_ORDER_EQUAL": order_equal,
        "INDEX_IDS_EQUAL": same_annual_population,
        "EMBEDDING_DIM": 768,
        "M2_DTYPE": annual["M2"]["dtype"],
        "M4_DTYPE": annual["M4"]["dtype"]
    })

    focal_year = focals[focals["ANO"] == year]

    id_to_position = pd.Index(
        annual["M2"]["ids"]
    )

    positions = id_to_position.get_indexer(
        focal_year["ID_TRABALHO"].tolist()
    )

    for focal_id, position in zip(
        focal_year["FOCAL_ID"], positions
    ):
        coverage_records.append({
            "FOCAL_ID": focal_id,
            "ANO": year,
            "M2_PRESENT": bool(position >= 0),
            "M4_PRESENT": bool(position >= 0),
            "ANNUAL_ROW": int(position)
        })

    print(
        f"{year}: {len(embedded):,} documentos; "
        f"ordem M2/M4 OK; "
        f"focais {len(focal_year)}"
    )

annual_df = pd.DataFrame(annual_records)
coverage_df = pd.DataFrame(coverage_records)

assert len(coverage_df) == 108
assert coverage_df["M2_PRESENT"].all()
assert coverage_df["M4_PRESENT"].all()

# --------------------------------------------------
# 4. Dimensionamento dos antecedentes por focal
# --------------------------------------------------

area_year_counts = (
    meta.groupby(
        ["AREA_HARMONIZADA", "ANO"]
    )
    .size()
    .rename("N")
    .reset_index()
)

lookup = {
    (str(r.AREA_HARMONIZADA), int(r.ANO)): int(r.N)
    for r in area_year_counts.itertuples(index=False)
}

planning = []

for focal in focals.itertuples(index=False):

    year = int(focal.ANO)
    area = str(focal.AREA_HARMONIZADA)

    historical = sum(
        lookup.get((area, y), 0)
        for y in range(2013, year)
    )

    frontier = sum(
        lookup.get((area, y), 0)
        for y in range(max(2013, year - 3), year)
    )

    planning.append({
        "FOCAL_ID": focal.FOCAL_ID,
        "ID_TRABALHO": focal.ID_TRABALHO,
        "ANO": year,
        "GRANDE_AREA": focal.NM_GRANDE_AREA_CONHECIMENTO,
        "AREA_HARMONIZADA": area,
        "N_HISTORICAL": historical,
        "N_FRONTIER3Y": frontier,
        "N_OLDER_THAN_FRONTIER": historical - frontier,
        "N_FULL_YEARS_AVAILABLE": year - 2013,
        "FRONTIER_SHARE": (
            frontier / historical if historical else np.nan
        )
    })

plan = pd.DataFrame(planning)

assert len(plan) == 108
assert (plan["N_HISTORICAL"] >= plan["N_FRONTIER3Y"]).all()
assert (plan["N_FRONTIER3Y"] > 0).all()

# --------------------------------------------------
# 5. Estimativa de trabalho
# --------------------------------------------------

# Pontuar todos os antecedentes historicos para
# M2 e M4 permite derivar o ranking da fronteira
# por filtragem dos escores, sem repetir os produtos.
total_pairs = int(plan["N_HISTORICAL"].sum())

embedding_methods = 2
vector_dim = 768

estimated_multiply_adds = (
    total_pairs * embedding_methods * vector_dim
)

# RAM de um bloco float32 de 2048 vetores.
block_size = 2048
block_memory_mib = (
    block_size * vector_dim * 4 / (1024 ** 2)
)

# Esta e estimativa aritmetica, nao tempo de GPU/CPU.
resource = {
    "n_focals": len(plan),
    "n_areas_harmonized": int(
        plan["AREA_HARMONIZADA"].nunique()
    ),
    "total_candidate_focal_pairs_historical": total_pairs,
    "total_candidate_focal_pairs_frontier": int(
        plan["N_FRONTIER3Y"].sum()
    ),
    "max_historical_candidates_per_focal": int(
        plan["N_HISTORICAL"].max()
    ),
    "median_historical_candidates_per_focal": float(
        plan["N_HISTORICAL"].median()
    ),
    "embedding_methods": ["M2", "M4"],
    "embedding_dimension": vector_dim,
    "estimated_scalar_multiplications": estimated_multiply_adds,
    "block_size": block_size,
    "approx_float32_block_memory_mib": block_memory_mib,
    "new_embedding_inference_required": False,
    "estimated_runtime_seconds": None
}

# --------------------------------------------------
# 6. Preservar plano auditavel
# --------------------------------------------------

annual_df.to_csv(
    OUT / "annual_embedding_integrity.csv",
    index=False
)

coverage_df.to_csv(
    OUT / "focal_embedding_coverage.csv",
    index=False
)

plan.to_csv(
    OUT / "focal_candidate_plan.csv",
    index=False
)

area_year_counts.to_csv(
    OUT / "area_year_population_counts.csv",
    index=False
)

(OUT / "resource_plan.json").write_text(
    json.dumps(
        resource,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

manifest = {
    "stage": "20B",
    "source_sample": str(SAMPLE_DIR),
    "n_focals": 108,
    "year_range": [2016, 2024],
    "candidate_rule_historical": (
        "same AREA_HARMONIZADA, antecedent_year < focal_year"
    ),
    "candidate_rule_frontier": (
        "same AREA_HARMONIZADA, "
        "focal_year-3 <= antecedent_year < focal_year"
    ),
    "M2_M4_document_order_audited": True,
    "national_index_coverage_audited": True,
    "rankings_computed": False,
    "human_ground_truth_established": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== AUDITORIA DOS FOCOS ===")
print("Focais:", len(plan))
print("Cobertura M2:", int(coverage_df["M2_PRESENT"].sum()))
print("Cobertura M4:", int(coverage_df["M4_PRESENT"].sum()))
print(
    "Áreas específicas:",
    plan["AREA_HARMONIZADA"].nunique()
)

print("\n=== VOLUME DE CANDIDATOS ===")
print(
    plan[[
        "N_HISTORICAL",
        "N_FRONTIER3Y"
    ]].describe().to_string()
)

print("\nTotal de pares focal-antecedente históricos:", total_pairs)
print(
    "Total de pares focal-antecedente fronteira:",
    int(plan["N_FRONTIER3Y"].sum())
)
print(
    "Multiplicações escalares estimadas M2 + M4:",
    f"{estimated_multiply_adds:,}"
)
print(
    "RAM aproximada por bloco float32:",
    round(block_memory_mib, 2), "MiB"
)

print("\n=== MAIORES UNIVERSOS ===")
print(
    plan.nlargest(
        10, "N_HISTORICAL"
    )[[
        "FOCAL_ID",
        "ANO",
        "AREA_HARMONIZADA",
        "N_HISTORICAL",
        "N_FRONTIER3Y"
    ]].to_string(index=False)
)

print("\nDiretório:", OUT)
print("ETAPA 20B CONCLUÍDA")


2013: 67,431 documentos; ordem M2/M4 OK; focais 0
2014: 70,487 documentos; ordem M2/M4 OK; focais 0
2015: 75,666 documentos; ordem M2/M4 OK; focais 0
2016: 80,075 documentos; ordem M2/M4 OK; focais 14
2017: 85,137 documentos; ordem M2/M4 OK; focais 12
2018: 90,295 documentos; ordem M2/M4 OK; focais 12
2019: 94,349 documentos; ordem M2/M4 OK; focais 13
2020: 79,989 documentos; ordem M2/M4 OK; focais 12
2021: 82,600 documentos; ordem M2/M4 OK; focais 13
2022: 83,981 documentos; ordem M2/M4 OK; focais 10
2023: 91,355 documentos; ordem M2/M4 OK; focais 11
2024: 91,721 documentos; ordem M2/M4 OK; focais 11

=== AUDITORIA DOS FOCOS ===
Focais: 108
Cobertura M2: 108
Cobertura M4: 108
Áreas específicas: 38

=== VOLUME DE CANDIDATOS ===
       N_HISTORICAL  N_FRONTIER3Y
count    108.000000    108.000000
mean   15187.694444   7009.064815
std    10590.177915   3972.841111
min     1237.000000   1237.000000
25%     6805.750000   3992.000000
50%    12224.500000   5726.000000
75%    19975.000000  101

In [33]:

# SCIENTOMETRICS 2026 — ETAPA 21A
# Expansao nacional, lote 01: 12 focais
# CPU | M2 e M4 congelados | Checkpoint por focal

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import json
import gc

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
EMB = BASE / "07_embeddings_nacionais"

SAMPLE = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

# Diretorio fixo para retomada dos lotes.
OUT = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CHECKPOINTS = OUT / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

BATCH_NUMBER = 1
BATCH_SIZE_FOCALS = 12
VECTOR_BLOCK = 2048
TOP_K = 30
MODALITIES = ["M2", "M4"]

# --------------------------------------------------
# 1. Amostra congelada e plano original
# --------------------------------------------------

focals = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

plan = pd.read_csv(
    PLAN,
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 108
assert focals["FOCAL_ID"].is_unique
assert focals["ID_TRABALHO"].is_unique
assert set(focals["FOCAL_ID"]) == set(plan["FOCAL_ID"])

focals = focals.sort_values("FOCAL_ID").reset_index(drop=True)

start = (BATCH_NUMBER - 1) * BATCH_SIZE_FOCALS
end = start + BATCH_SIZE_FOCALS

batch = focals.iloc[start:end].copy()

assert len(batch) == 12

# Criar/verificar configuracao imutavel da execucao.
config = {
    "stage": "21A",
    "sample": str(SAMPLE),
    "sample_n": 108,
    "modalities": MODALITIES,
    "candidate_universe": "same_area_all_prior_years",
    "frontier_years": 3,
    "vector_block": VECTOR_BLOCK,
    "top_k": TOP_K,
    "score": "cosine_float32_from_float16_embeddings",
    "checkpoint_per_focal": True
}

config_path = OUT / "run_configuration.json"

if config_path.exists():
    existing = json.loads(config_path.read_text(
        encoding="utf-8"
    ))
    assert existing == config, (
        "Configuracao anterior diferente. "
        "Nao misturar experimentos."
    )
else:
    config_path.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

# --------------------------------------------------
# 2. Metadados nacionais
# --------------------------------------------------

meta = pq.read_table(
    INDEX,
    columns=["ID_TRABALHO", "ANO", "AREA_HARMONIZADA"]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
meta["ANO"] = meta["ANO"].astype(int)

assert meta["ID_TRABALHO"].is_unique

area_lookup = meta.set_index(
    "ID_TRABALHO"
)["AREA_HARMONIZADA"]

focals["AREA_HARMONIZADA"] = (
    focals["ID_TRABALHO"].map(area_lookup)
)

assert focals["AREA_HARMONIZADA"].notna().all()

batch = focals.iloc[start:end].copy()

# --------------------------------------------------
# 3. Acesso seletivo aos embeddings anuais
# --------------------------------------------------

id_cache = {}
emb_cache = {}

def get_ids(year, modality):
    key = (int(year), modality)

    if key not in id_cache:
        path = (
            EMB / modality
            / f"{modality}_{year}_ids.parquet"
        )
        ids = pd.read_parquet(
            path, columns=["ID_TRABALHO"]
        )["ID_TRABALHO"].astype(str)

        assert ids.is_unique
        id_cache[key] = pd.Index(ids)

    return id_cache[key]


def get_vectors(year, modality):
    key = (int(year), modality)

    if key not in emb_cache:
        path = (
            EMB / modality
            / f"{modality}_{year}_embeddings_float16.npy"
        )

        arr = np.load(path, mmap_mode="r")

        assert arr.ndim == 2
        assert arr.shape[1] == 768
        assert len(arr) == len(get_ids(year, modality))

        emb_cache[key] = arr

    return emb_cache[key]


def focal_vector(doc_id, year, modality):
    idx = get_ids(year, modality)
    pos = idx.get_loc(str(doc_id))

    assert isinstance(pos, (int, np.integer))

    vec = np.array(
        get_vectors(year, modality)[pos],
        dtype=np.float32,
        copy=True
    )

    norm = np.linalg.norm(vec)
    assert np.isfinite(norm) and norm > 0

    return vec / norm


def score_antecedents(doc_ids, year, modality, query):
    ids = get_ids(year, modality)
    vectors = get_vectors(year, modality)

    positions = ids.get_indexer(
        [str(x) for x in doc_ids]
    )

    assert (positions >= 0).all(), (
        f"IDs nao localizados em {year}, {modality}"
    )

    output = np.empty(len(positions), dtype=np.float32)

    for first in range(0, len(positions), VECTOR_BLOCK):
        last = min(first + VECTOR_BLOCK, len(positions))

        block = np.array(
            vectors[positions[first:last]],
            dtype=np.float32,
            copy=True
        )

        norms = np.linalg.norm(block, axis=1)

        assert np.isfinite(norms).all()
        assert (norms > 0).all()

        block /= norms[:, None]

        output[first:last] = block @ query

    assert np.isfinite(output).all()

    return output


# --------------------------------------------------
# 4. Processamento com checkpoint por documento
# --------------------------------------------------

completed = []
skipped = []

for focal in batch.itertuples(index=False):

    fid = focal.FOCAL_ID
    checkpoint = CHECKPOINTS / f"{fid}_scores.parquet"
    ranking_file = CHECKPOINTS / f"{fid}_top30.parquet"

    expected = int(
        plan.loc[
            plan["FOCAL_ID"] == fid,
            "N_HISTORICAL"
        ].iloc[0]
    )

    if checkpoint.exists() and ranking_file.exists():

        # Verificar antes de aceitar checkpoint.
        pf = pq.ParquetFile(checkpoint)
        n_saved = pf.metadata.num_rows

        rt = pd.read_parquet(ranking_file)

        valid = (
            n_saved == expected
            and len(rt) == 120
            and set(rt["MODALITY"]) == {"M2", "M4"}
            and set(rt["WINDOW"]) == {"historical", "frontier3y"}
            and not rt.duplicated(
                ["FOCAL_ID", "MODALITY", "WINDOW", "RANK"]
            ).any()
        )

        if not valid:
            raise RuntimeError(
                f"Checkpoint inconsistente: {fid}"
            )

        skipped.append(fid)
        print(f"{fid}: checkpoint validado, ignorado")
        continue

    if checkpoint.exists() or ranking_file.exists():
        raise RuntimeError(
            f"Checkpoint parcial: {fid}. "
            "Examinar arquivos antes de continuar."
        )

    candidates = meta[
        (meta["ANO"] < int(focal.ANO)) &
        (meta["AREA_HARMONIZADA"] ==
         focal.AREA_HARMONIZADA)
    ][["ID_TRABALHO", "ANO"]].copy()

    assert len(candidates) == expected
    assert candidates["ID_TRABALHO"].is_unique

    candidates = candidates.rename(columns={
        "ID_TRABALHO": "ANTECEDENT_ID",
        "ANO": "ANTECEDENT_YEAR"
    })

    candidates["FOCAL_ID"] = fid
    candidates["FOCAL_YEAR"] = int(focal.ANO)

    for modality in MODALITIES:

        query = focal_vector(
            focal.ID_TRABALHO, focal.ANO, modality
        )

        score_col = f"COSINE_{modality}"
        candidates[score_col] = np.nan

        for year, indices in candidates.groupby(
            "ANTECEDENT_YEAR"
        ).groups.items():

            row_positions = np.asarray(
                list(indices), dtype=np.int64
            )

            ids = candidates.loc[
                row_positions, "ANTECEDENT_ID"
            ].tolist()

            sims = score_antecedents(
                ids, int(year), modality, query
            )

            candidates.loc[
                row_positions, score_col
            ] = sims

        assert candidates[score_col].notna().all()

    # ------------------------------------------------
    # 5. Rankings historico e fronteira
    # ------------------------------------------------

    ranking_parts = []

    for window, subset in [
        ("historical", candidates),
        (
            "frontier3y",
            candidates[
                candidates["ANTECEDENT_YEAR"] >=
                int(focal.ANO) - 3
            ]
        )
    ]:

        assert len(subset) >= TOP_K

        for modality in MODALITIES:

            score_col = f"COSINE_{modality}"

            top = subset.sort_values(
                [score_col, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K).copy()

            top["WINDOW"] = window
            top["MODALITY"] = modality
            top["RANK"] = range(1, TOP_K + 1)
            top["SCORE"] = top[score_col]

            ranking_parts.append(top[[
                "FOCAL_ID",
                "FOCAL_YEAR",
                "ANTECEDENT_ID",
                "ANTECEDENT_YEAR",
                "WINDOW",
                "MODALITY",
                "RANK",
                "SCORE"
            ]])

    ranking = pd.concat(
        ranking_parts, ignore_index=True
    )

    assert len(ranking) == 120
    assert (candidates["ANTECEDENT_YEAR"]
            < candidates["FOCAL_YEAR"]).all()

    # Salvar primeiro em arquivos temporarios.
    tmp_scores = CHECKPOINTS / f"{fid}_scores.partial.parquet"
    tmp_rank = CHECKPOINTS / f"{fid}_top30.partial.parquet"

    if tmp_scores.exists() or tmp_rank.exists():
        raise RuntimeError(
            f"Arquivos temporarios preexistentes: {fid}"
        )

    candidates.to_parquet(tmp_scores, index=False)
    ranking.to_parquet(tmp_rank, index=False)

    # Verificar antes de publicar os checkpoints.
    assert pq.ParquetFile(tmp_scores).metadata.num_rows == expected
    assert pq.ParquetFile(tmp_rank).metadata.num_rows == 120

    # Publicar apenas apos salvar ambos os arquivos.
    # A verificacao de retomada rejeita checkpoints parciais.
    tmp_scores.rename(checkpoint)
    tmp_rank.rename(ranking_file)

    completed.append(fid)

    print(
        f"{fid}: {expected:,} antecedentes "
        f"| M2/M4 OK | historico e 3Y OK"
    )

    del candidates, ranking
    emb_cache.clear()
    gc.collect()

# --------------------------------------------------
# 6. Resumo da execucao
# --------------------------------------------------

print("\n=== ETAPA 21A — LOTE 01 ===")
print("Focais esperados no lote:", len(batch))
print("Novos concluidos:", len(completed))
print("Checkpoints reutilizados:", len(skipped))
print("Total verificado:", len(completed) + len(skipped))

assert len(completed) + len(skipped) == len(batch)

print("\nPasta:", OUT)
print("ETAPA 21A CONCLUIDA")


NAT-0001: 21,627 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0002: 9,475 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0003: 7,430 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0004: 14,534 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0005: 5,726 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0006: 23,313 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0007: 15,281 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0008: 24,691 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0009: 16,626 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0010: 25,412 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0011: 5,829 antecedentes | M2/M4 OK | historico e 3Y OK
NAT-0012: 1,237 antecedentes | M2/M4 OK | historico e 3Y OK

=== ETAPA 21A — LOTE 01 ===
Focais esperados no lote: 12
Novos concluidos: 12
Checkpoints reutilizados: 0
Total verificado: 12

Pasta: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion/21_semantic_controlled_national_

In [34]:

# SCIENTOMETRICS 2026 — ETAPA 21B
# Auditoria integral do primeiro lote nacional
# CPU | Sem embeddings | Sem recalcular similaridades

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SAMPLE_DIR = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CKPT = RUN / "checkpoints"

expected_ids = [
    f"NAT-{i:04d}" for i in range(1, 13)
]

plan = pd.read_csv(
    PLAN_FILE,
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
).set_index("FOCAL_ID")

sample = pd.read_csv(
    SAMPLE_DIR / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
).set_index("FOCAL_ID")

meta = pq.read_table(
    INDEX_FILE,
    columns=["ID_TRABALHO", "ANO", "AREA_HARMONIZADA"]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
meta["ANO"] = meta["ANO"].astype(int)

assert meta["ID_TRABALHO"].is_unique

area_by_id = meta.set_index(
    "ID_TRABALHO"
)["AREA_HARMONIZADA"]

reports = []

for fid in expected_ids:

    score_path = CKPT / f"{fid}_scores.parquet"
    top_path = CKPT / f"{fid}_top30.parquet"

    assert score_path.is_file(), f"Ausente: {score_path}"
    assert top_path.is_file(), f"Ausente: {top_path}"

    scores = pd.read_parquet(score_path)
    ranks = pd.read_parquet(top_path)

    focal_year = int(sample.loc[fid, "ANO"])
    focal_doc_id = str(sample.loc[fid, "ID_TRABALHO"])
    focal_area = str(area_by_id.loc[focal_doc_id])

    n_hist = int(plan.loc[fid, "N_HISTORICAL"])
    n_front = int(plan.loc[fid, "N_FRONTIER3Y"])

    # 1. Integridade dos escores
    assert len(scores) == n_hist, fid
    assert len(ranks) == 120, fid

    assert scores["FOCAL_ID"].eq(fid).all()
    assert ranks["FOCAL_ID"].eq(fid).all()

    assert scores["FOCAL_YEAR"].astype(int).eq(
        focal_year
    ).all()

    assert scores["ANTECEDENT_ID"].is_unique

    assert scores["ANTECEDENT_YEAR"].astype(int).lt(
        focal_year
    ).all()

    # 2. Correspondencia da area
    antecedent_areas = scores[
        "ANTECEDENT_ID"
    ].astype(str).map(area_by_id)

    assert antecedent_areas.notna().all()
    assert antecedent_areas.eq(focal_area).all()

    # 3. Valores de cosseno
    for mod in ["M2", "M4"]:
        x = scores[f"COSINE_{mod}"].to_numpy(
            dtype=np.float64
        )

        assert np.isfinite(x).all()
        assert ((x >= -1.00001) & (x <= 1.00001)).all()

    # 4. Fronteira esperada
    frontier = scores[
        scores["ANTECEDENT_YEAR"].astype(int)
        .between(focal_year - 3, focal_year - 1)
    ]

    assert len(frontier) == n_front, (
        f"{fid}: fronteira {len(frontier)} "
        f"difere do plano {n_front}"
    )

    # 5. Reproduzir os quatro rankings exatamente
    # a partir dos escores salvos.
    for window, candidates in [
        ("historical", scores),
        ("frontier3y", frontier)
    ]:

        for mod in ["M2", "M4"]:

            expected = candidates.sort_values(
                [f"COSINE_{mod}", "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(30)

            stored = ranks[
                ranks["WINDOW"].eq(window) &
                ranks["MODALITY"].eq(mod)
            ].sort_values("RANK")

            assert len(stored) == 30

            assert stored["RANK"].tolist() == list(
                range(1, 31)
            )

            expected_ids_top = (
                expected["ANTECEDENT_ID"]
                .astype(str).tolist()
            )
            stored_ids_top = (
                stored["ANTECEDENT_ID"]
                .astype(str).tolist()
            )

            assert expected_ids_top == stored_ids_top, (
                f"Ranking divergente: {fid}, "
                f"{window}, {mod}"
            )

            assert np.allclose(
                expected[f"COSINE_{mod}"].to_numpy(
                    dtype=float
                ),
                stored["SCORE"].to_numpy(dtype=float),
                atol=1e-6,
                rtol=0
            ), f"Escores divergentes: {fid}, {window}, {mod}"

    reports.append({
        "FOCAL_ID": fid,
        "ANO": focal_year,
        "AREA_HARMONIZADA": focal_area,
        "N_HISTORICAL": len(scores),
        "N_FRONTIER3Y": len(frontier),
        "RANKING_ROWS": len(ranks),
        "RANKINGS_REPRODUCED": 4,
        "AUDIT_STATUS": "PASS"
    })

    print(
        f"{fid}: PASS | historico={len(scores):,}"
        f" | fronteira={len(frontier):,}"
        f" | 4 rankings reproduzidos"
    )

audit = pd.DataFrame(reports)

assert len(audit) == 12
assert audit["AUDIT_STATUS"].eq("PASS").all()

# Salvar auditoria fora da pasta de checkpoints.
stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

AUDIT_DIR = RUN / "audits"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

audit_file = AUDIT_DIR / f"batch01_audit_{stamp}.csv"
audit.to_csv(audit_file, index=False)

summary = {
    "stage": "21B",
    "batch": 1,
    "focals_audited": len(audit),
    "total_historical_pairs": int(
        audit["N_HISTORICAL"].sum()
    ),
    "total_frontier_pairs": int(
        audit["N_FRONTIER3Y"].sum()
    ),
    "rankings_reconstructed": int(
        audit["RANKINGS_REPRODUCED"].sum()
    ),
    "all_pass": True,
    "human_ground_truth_established": False
}

summary_file = AUDIT_DIR / f"batch01_summary_{stamp}.json"

summary_file.write_text(
    json.dumps(
        summary, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== AUDITORIA LOTE 01 ===")
print("Focais auditados:", len(audit))
print("Pares historicos:", summary["total_historical_pairs"])
print("Pares fronteira 3Y:", summary["total_frontier_pairs"])
print("Rankings reproduzidos:", summary["rankings_reconstructed"])
print("Status:", "PASS" if summary["all_pass"] else "FAIL")
print("Relatorio:", audit_file)
print("\nETAPA 21B CONCLUIDA")


NAT-0001: PASS | historico=21,627 | fronteira=14,523 | 4 rankings reproduzidos
NAT-0002: PASS | historico=9,475 | fronteira=3,074 | 4 rankings reproduzidos
NAT-0003: PASS | historico=7,430 | fronteira=7,430 | 4 rankings reproduzidos
NAT-0004: PASS | historico=14,534 | fronteira=11,294 | 4 rankings reproduzidos
NAT-0005: PASS | historico=5,726 | fronteira=5,726 | 4 rankings reproduzidos
NAT-0006: PASS | historico=23,313 | fronteira=7,229 | 4 rankings reproduzidos
NAT-0007: PASS | historico=15,281 | fronteira=4,985 | 4 rankings reproduzidos
NAT-0008: PASS | historico=24,691 | fronteira=10,843 | 4 rankings reproduzidos
NAT-0009: PASS | historico=16,626 | fronteira=12,670 | 4 rankings reproduzidos
NAT-0010: PASS | historico=25,412 | fronteira=6,515 | 4 rankings reproduzidos
NAT-0011: PASS | historico=5,829 | fronteira=4,531 | 4 rankings reproduzidos
NAT-0012: PASS | historico=1,237 | fronteira=1,237 | 4 rankings reproduzidos

=== AUDITORIA LOTE 01 ===
Focais auditados: 12
Pares historicos:

In [35]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 21C
# LOTE 02: PROCESSAMENTO + AUDITORIA INTEGRAL
#
# Documentos: NAT-0013 a NAT-0024
# Ambiente: CPU
# Embeddings: M2 e M4 congelados
# Saida: checkpoints + relatorio de auditoria
#
# NAO modifica os resultados anteriores.
# NAO recalcula embeddings.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import json
import gc

# ============================================================
# 1. CAMINHOS FIXOS
# ============================================================

BASE = Path(
    "/content/gdrive/MyDrive/CAPES_Epistemic_Change"
)

LAB = BASE / "08_epistemic_measurement_validation"

EMB = BASE / "07_embeddings_nacionais"

SAMPLE = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CHECKPOINTS = RUN / "checkpoints"
AUDITS = RUN / "audits"

CHECKPOINTS.mkdir(parents=True, exist_ok=True)
AUDITS.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. PARAMETROS DO EXPERIMENTO
# ============================================================

BATCH_NUMBER = 2
BATCH_SIZE_FOCALS = 12

START_FOCAL = 13
END_FOCAL = 24

TOP_K = 30
VECTOR_BLOCK = 2048

MODALITIES = ["M2", "M4"]

EXPECTED_FOCALS = [
    f"NAT-{i:04d}"
    for i in range(START_FOCAL, END_FOCAL + 1)
]

print("=" * 65)
print("SCIENTOMETRICS — ETAPA 21C")
print("LOTE 02: NAT-0013 A NAT-0024")
print("=" * 65)

# ============================================================
# 3. VERIFICAR CONFIGURACAO JA CONGELADA
# ============================================================

CONFIG_FILE = RUN / "run_configuration.json"

assert CONFIG_FILE.is_file(), (
    "Configuracao do experimento nacional nao encontrada."
)

existing_config = json.loads(
    CONFIG_FILE.read_text(encoding="utf-8")
)

expected_config = {
    "stage": "21A",
    "sample": str(SAMPLE),
    "sample_n": 108,
    "modalities": MODALITIES,
    "candidate_universe": "same_area_all_prior_years",
    "frontier_years": 3,
    "vector_block": VECTOR_BLOCK,
    "top_k": TOP_K,
    "score": "cosine_float32_from_float16_embeddings",
    "checkpoint_per_focal": True
}

assert existing_config == expected_config, (
    "A configuracao congelada difere dos parametros "
    "esperados. Processamento interrompido."
)

print("\nConfiguracao metodologica: VALIDADA")

# ============================================================
# 4. CARREGAR AMOSTRA E PLANO
# ============================================================

focals = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
)

plan = pd.read_csv(
    PLAN_FILE,
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
)

focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 108
assert focals["FOCAL_ID"].is_unique
assert focals["ID_TRABALHO"].is_unique
assert len(plan) == 108

assert set(focals["FOCAL_ID"]) == set(plan["FOCAL_ID"])

batch = (
    focals[
        focals["FOCAL_ID"].isin(EXPECTED_FOCALS)
    ]
    .sort_values("FOCAL_ID")
    .reset_index(drop=True)
)

assert batch["FOCAL_ID"].tolist() == EXPECTED_FOCALS

plan_lookup = plan.set_index(
    "FOCAL_ID",
    verify_integrity=True
)

print("\nDocumentos do lote:", len(batch))
print("Primeiro:", batch["FOCAL_ID"].iloc[0])
print("Ultimo:", batch["FOCAL_ID"].iloc[-1])

# ============================================================
# 5. CARREGAR METADADOS NACIONAIS
# ============================================================

meta = pq.read_table(
    INDEX_FILE,
    columns=[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA"
    ]
).to_pandas()

meta["ID_TRABALHO"] = (
    meta["ID_TRABALHO"].astype(str)
)

meta["ANO"] = meta["ANO"].astype(int)

assert len(meta) == 993086
assert meta["ID_TRABALHO"].is_unique

meta_lookup = meta.set_index(
    "ID_TRABALHO",
    verify_integrity=True
)

batch["AREA_HARMONIZADA"] = (
    batch["ID_TRABALHO"]
    .map(meta_lookup["AREA_HARMONIZADA"])
)

assert batch["AREA_HARMONIZADA"].notna().all()

# Verificar que ano e area concordam com o plano.
for focal in batch.itertuples(index=False):

    p = plan_lookup.loc[focal.FOCAL_ID]

    assert int(focal.ANO) == int(p["ANO"])

    assert (
        str(focal.AREA_HARMONIZADA)
        == str(p["AREA_HARMONIZADA"])
    )

print("Metadados e plano: VALIDADOS")

# ============================================================
# 6. ACESSO SELETIVO AOS EMBEDDINGS
# ============================================================

id_cache = {}
emb_cache = {}

def get_ids(year, modality):

    year = int(year)
    key = (year, modality)

    if key not in id_cache:

        path = (
            EMB / modality
            / f"{modality}_{year}_ids.parquet"
        )

        assert path.is_file()

        ids = pd.read_parquet(
            path,
            columns=["ID_TRABALHO"]
        )["ID_TRABALHO"].astype(str)

        assert ids.is_unique

        id_cache[key] = pd.Index(ids)

    return id_cache[key]


def get_vectors(year, modality):

    year = int(year)
    key = (year, modality)

    if key not in emb_cache:

        path = (
            EMB / modality
            / f"{modality}_{year}_embeddings_float16.npy"
        )

        assert path.is_file()

        arr = np.load(
            path,
            mmap_mode="r"
        )

        assert arr.ndim == 2
        assert arr.shape[1] == 768
        assert len(arr) == len(
            get_ids(year, modality)
        )

        emb_cache[key] = arr

    return emb_cache[key]


def normalized_focal_vector(doc_id, year, modality):

    ids = get_ids(year, modality)

    position = ids.get_loc(str(doc_id))

    assert isinstance(
        position,
        (int, np.integer)
    )

    vector = np.array(
        get_vectors(year, modality)[position],
        dtype=np.float32,
        copy=True
    )

    norm = np.linalg.norm(vector)

    assert np.isfinite(norm)
    assert norm > 0

    return vector / norm


def compute_cosine_scores(
    document_ids,
    year,
    modality,
    query_vector
):

    ids = get_ids(year, modality)
    vectors = get_vectors(year, modality)

    positions = ids.get_indexer(
        [str(x) for x in document_ids]
    )

    assert (positions >= 0).all(), (
        f"IDs nao encontrados: {year}, {modality}"
    )

    output = np.empty(
        len(positions),
        dtype=np.float32
    )

    for start in range(
        0,
        len(positions),
        VECTOR_BLOCK
    ):

        end = min(
            start + VECTOR_BLOCK,
            len(positions)
        )

        block = np.array(
            vectors[positions[start:end]],
            dtype=np.float32,
            copy=True
        )

        norms = np.linalg.norm(
            block,
            axis=1
        )

        assert np.isfinite(norms).all()
        assert (norms > 0).all()

        block /= norms[:, None]

        output[start:end] = (
            block @ query_vector
        )

    assert np.isfinite(output).all()

    return output


# ============================================================
# 7. FUNCAO DE AUDITORIA COMPLETA
# ============================================================

def audit_checkpoint(fid, focal_year, focal_area):

    score_file = (
        CHECKPOINTS / f"{fid}_scores.parquet"
    )

    rank_file = (
        CHECKPOINTS / f"{fid}_top30.parquet"
    )

    assert score_file.is_file(), score_file
    assert rank_file.is_file(), rank_file

    scores = pd.read_parquet(score_file)
    ranks = pd.read_parquet(rank_file)

    expected_h = int(
        plan_lookup.loc[fid, "N_HISTORICAL"]
    )

    expected_f = int(
        plan_lookup.loc[fid, "N_FRONTIER3Y"]
    )

    assert len(scores) == expected_h
    assert len(ranks) == 120

    assert scores["FOCAL_ID"].eq(fid).all()
    assert ranks["FOCAL_ID"].eq(fid).all()

    assert scores["FOCAL_YEAR"].astype(int).eq(
        focal_year
    ).all()

    assert scores["ANTECEDENT_ID"].is_unique

    antecedent_years = (
        scores["ANTECEDENT_YEAR"].astype(int)
    )

    assert antecedent_years.lt(
        focal_year
    ).all()

    # Verificar area disciplinar.
    ant_areas = (
        scores["ANTECEDENT_ID"]
        .astype(str)
        .map(meta_lookup["AREA_HARMONIZADA"])
    )

    assert ant_areas.notna().all()
    assert ant_areas.eq(focal_area).all()

    frontier = scores[
        antecedent_years.between(
            focal_year - 3,
            focal_year - 1
        )
    ]

    assert len(frontier) == expected_f

    assert not ranks.duplicated([
        "FOCAL_ID",
        "MODALITY",
        "WINDOW",
        "RANK"
    ]).any()

    reproduced = 0

    for modality in MODALITIES:

        column = f"COSINE_{modality}"

        values = scores[column].to_numpy(
            dtype=np.float64
        )

        assert np.isfinite(values).all()

        assert (
            (values >= -1.00001) &
            (values <= 1.00001)
        ).all()

        for window, candidates in [
            ("historical", scores),
            ("frontier3y", frontier)
        ]:

            expected = candidates.sort_values(
                [column, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K)

            stored = ranks[
                (ranks["MODALITY"] == modality) &
                (ranks["WINDOW"] == window)
            ].sort_values("RANK")

            assert len(stored) == TOP_K

            assert stored["RANK"].tolist() == list(
                range(1, TOP_K + 1)
            )

            expected_ids = (
                expected["ANTECEDENT_ID"]
                .astype(str)
                .tolist()
            )

            stored_ids = (
                stored["ANTECEDENT_ID"]
                .astype(str)
                .tolist()
            )

            assert expected_ids == stored_ids, (
                f"Ranking divergente: "
                f"{fid}, {modality}, {window}"
            )

            assert np.allclose(
                expected[column].to_numpy(
                    dtype=float
                ),
                stored["SCORE"].to_numpy(
                    dtype=float
                ),
                atol=1e-6,
                rtol=0
            )

            if window == "frontier3y":

                assert stored[
                    "ANTECEDENT_YEAR"
                ].astype(int).between(
                    focal_year - 3,
                    focal_year - 1
                ).all()

            reproduced += 1

    assert reproduced == 4

    return {
        "FOCAL_ID": fid,
        "ANO": focal_year,
        "AREA_HARMONIZADA": focal_area,
        "N_HISTORICAL": len(scores),
        "N_FRONTIER3Y": len(frontier),
        "RANKINGS_REPRODUCED": reproduced,
        "STATUS": "PASS"
    }


# ============================================================
# 8. PROCESSAMENTO DO SEGUNDO LOTE
# ============================================================

processed = []
reused = []

print("\n" + "=" * 65)
print("PROCESSAMENTO DO LOTE 02")
print("=" * 65)

for focal in batch.itertuples(index=False):

    fid = focal.FOCAL_ID
    year = int(focal.ANO)
    area = str(focal.AREA_HARMONIZADA)

    score_file = (
        CHECKPOINTS / f"{fid}_scores.parquet"
    )

    rank_file = (
        CHECKPOINTS / f"{fid}_top30.parquet"
    )

    # Retomada segura de checkpoints completos.
    if score_file.exists() and rank_file.exists():

        audit_checkpoint(
            fid,
            year,
            area
        )

        reused.append(fid)

        print(
            f"{fid}: checkpoint existente "
            f"e auditado — reutilizado"
        )

        continue

    # Nunca substituir arquivos parciais.
    if score_file.exists() or rank_file.exists():

        raise RuntimeError(
            f"Checkpoint incompleto para {fid}. "
            "Inspecionar antes de prosseguir."
        )

    tmp_scores = (
        CHECKPOINTS /
        f"{fid}_scores.partial.parquet"
    )

    tmp_rank = (
        CHECKPOINTS /
        f"{fid}_top30.partial.parquet"
    )

    if tmp_scores.exists() or tmp_rank.exists():

        raise RuntimeError(
            f"Arquivos temporarios ja existem "
            f"para {fid}. Inspecionar."
        )

    # Selecionar os antecedentes.
    candidates = meta[
        (meta["ANO"] < year) &
        (meta["AREA_HARMONIZADA"] == area)
    ][["ID_TRABALHO", "ANO"]].copy()

    candidates = candidates.rename(columns={
        "ID_TRABALHO": "ANTECEDENT_ID",
        "ANO": "ANTECEDENT_YEAR"
    }).reset_index(drop=True)

    expected_n = int(
        plan_lookup.loc[fid, "N_HISTORICAL"]
    )

    assert len(candidates) == expected_n
    assert candidates["ANTECEDENT_ID"].is_unique

    candidates["FOCAL_ID"] = fid
    candidates["FOCAL_YEAR"] = year

    # Calcular M2 e M4.
    for modality in MODALITIES:

        query = normalized_focal_vector(
            focal.ID_TRABALHO,
            year,
            modality
        )

        score_column = f"COSINE_{modality}"

        candidates[score_column] = np.nan

        for ant_year, group in candidates.groupby(
            "ANTECEDENT_YEAR",
            sort=True
        ):

            positions = group.index.to_numpy()

            doc_ids = group[
                "ANTECEDENT_ID"
            ].tolist()

            similarities = compute_cosine_scores(
                doc_ids,
                int(ant_year),
                modality,
                query
            )

            candidates.loc[
                positions,
                score_column
            ] = similarities

        assert candidates[
            score_column
        ].notna().all()

    # Rankings derivados dos escores completos.
    ranking_parts = []

    for window, subset in [
        ("historical", candidates),
        (
            "frontier3y",
            candidates[
                candidates["ANTECEDENT_YEAR"]
                >= year - 3
            ]
        )
    ]:

        assert len(subset) >= TOP_K

        for modality in MODALITIES:

            score_column = f"COSINE_{modality}"

            top = subset.sort_values(
                [score_column, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K).copy()

            top["WINDOW"] = window
            top["MODALITY"] = modality

            top["RANK"] = range(
                1,
                TOP_K + 1
            )

            top["SCORE"] = top[score_column]

            ranking_parts.append(top[[
                "FOCAL_ID",
                "FOCAL_YEAR",
                "ANTECEDENT_ID",
                "ANTECEDENT_YEAR",
                "WINDOW",
                "MODALITY",
                "RANK",
                "SCORE"
            ]])

    rankings = pd.concat(
        ranking_parts,
        ignore_index=True
    )

    assert len(rankings) == 120

    # Gravar primeiro em arquivos temporarios.
    candidates.to_parquet(
        tmp_scores,
        index=False
    )

    rankings.to_parquet(
        tmp_rank,
        index=False
    )

    assert (
        pq.ParquetFile(tmp_scores).metadata.num_rows
        == expected_n
    )

    assert (
        pq.ParquetFile(tmp_rank).metadata.num_rows
        == 120
    )

    # Publicar checkpoints.
    tmp_scores.rename(score_file)
    tmp_rank.rename(rank_file)

    # Auditoria imediata.
    audit_checkpoint(
        fid,
        year,
        area
    )

    processed.append(fid)

    print(
        f"{fid}: PASS | "
        f"historico={expected_n:,} | "
        f"M2/M4 | 4 rankings reproduzidos"
    )

    del candidates, rankings
    emb_cache.clear()
    gc.collect()

# ============================================================
# 9. AUDITORIA CONSOLIDADA DO LOTE 02
# ============================================================

print("\n" + "=" * 65)
print("AUDITORIA CONSOLIDADA")
print("=" * 65)

audit_rows = []

for focal in batch.itertuples(index=False):

    result = audit_checkpoint(
        focal.FOCAL_ID,
        int(focal.ANO),
        str(focal.AREA_HARMONIZADA)
    )

    audit_rows.append(result)

audit = pd.DataFrame(audit_rows)

assert len(audit) == 12
assert audit["STATUS"].eq("PASS").all()

assert audit[
    "RANKINGS_REPRODUCED"
].sum() == 48

# ============================================================
# 10. SALVAR RELATORIO VERSIONADO
# ============================================================

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

audit_file = (
    AUDITS /
    f"batch02_audit_{stamp}.csv"
)

summary_file = (
    AUDITS /
    f"batch02_summary_{stamp}.json"
)

audit.to_csv(
    audit_file,
    index=False,
    mode="x"
)

summary = {
    "stage": "21C",
    "batch": 2,
    "first_focal": EXPECTED_FOCALS[0],
    "last_focal": EXPECTED_FOCALS[-1],
    "focals_audited": len(audit),
    "newly_processed": len(processed),
    "checkpoints_reused": len(reused),
    "historical_pairs": int(
        audit["N_HISTORICAL"].sum()
    ),
    "frontier3y_pairs": int(
        audit["N_FRONTIER3Y"].sum()
    ),
    "rankings_reproduced": int(
        audit["RANKINGS_REPRODUCED"].sum()
    ),
    "audit_status": "PASS",
    "human_ground_truth_established": False
}

summary_file.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ============================================================
# 11. RESULTADO FINAL
# ============================================================

print("\n=== ETAPA 21C — RESULTADO FINAL ===")

print("Focais do lote:", len(audit))
print("Novos processados:", len(processed))
print("Checkpoints reutilizados:", len(reused))

print(
    "Pares historicos:",
    summary["historical_pairs"]
)

print(
    "Pares fronteira 3Y:",
    summary["frontier3y_pairs"]
)

print(
    "Rankings reproduzidos:",
    summary["rankings_reproduced"]
)

print("Status:", summary["audit_status"])

print("\nRelatorio:", audit_file)

print("\nProgresso nacional esperado: 24/108 focais")

print("\nETAPA 21C CONCLUIDA")


SCIENTOMETRICS — ETAPA 21C
LOTE 02: NAT-0013 A NAT-0024

Configuracao metodologica: VALIDADA

Documentos do lote: 12
Primeiro: NAT-0013
Ultimo: NAT-0024
Metadados e plano: VALIDADOS

PROCESSAMENTO DO LOTE 02
NAT-0013: PASS | historico=10,173 | M2/M4 | 4 rankings reproduzidos
NAT-0014: PASS | historico=9,146 | M2/M4 | 4 rankings reproduzidos
NAT-0015: PASS | historico=13,654 | M2/M4 | 4 rankings reproduzidos
NAT-0016: PASS | historico=22,092 | M2/M4 | 4 rankings reproduzidos
NAT-0017: PASS | historico=12,341 | M2/M4 | 4 rankings reproduzidos
NAT-0018: PASS | historico=11,869 | M2/M4 | 4 rankings reproduzidos
NAT-0019: PASS | historico=4,794 | M2/M4 | 4 rankings reproduzidos
NAT-0020: PASS | historico=21,665 | M2/M4 | 4 rankings reproduzidos
NAT-0021: PASS | historico=8,803 | M2/M4 | 4 rankings reproduzidos
NAT-0022: PASS | historico=10,013 | M2/M4 | 4 rankings reproduzidos
NAT-0023: PASS | historico=12,341 | M2/M4 | 4 rankings reproduzidos
NAT-0024: PASS | historico=7,115 | M2/M4 | 4 ra

In [36]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 21D
# LOTE 03: NAT-0025 A NAT-0036
# PROCESSAMENTO + AUDITORIA AUTOMATICA
#
# CPU | M2/M4 congelados | Checkpoints por focal
# Nao altera resultados dos lotes 01 e 02
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. Caminhos e parametros congelados
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
EMB = BASE / "07_embeddings_nacionais"

SAMPLE = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CKPT = RUN / "checkpoints"
AUDITS = RUN / "audits"

CKPT.mkdir(parents=True, exist_ok=True)
AUDITS.mkdir(parents=True, exist_ok=True)

BATCH = 3
START = 25
END = 36
TOP_K = 30
VECTOR_BLOCK = 2048
MODALITIES = ["M2", "M4"]

FOCAL_IDS = [f"NAT-{i:04d}" for i in range(START, END + 1)]

print("=" * 65)
print("SCIENTOMETRICS — ETAPA 21D | LOTE 03")
print("FOCAIS:", FOCAL_IDS[0], "a", FOCAL_IDS[-1])
print("=" * 65)

# ------------------------------------------------------------
# 2. Validar configuracao original
# ------------------------------------------------------------

CONFIG_FILE = RUN / "run_configuration.json"
assert CONFIG_FILE.is_file(), "Configuracao original ausente."

original_config = json.loads(
    CONFIG_FILE.read_text(encoding="utf-8")
)

expected_config = {
    "stage": "21A",
    "sample": str(SAMPLE),
    "sample_n": 108,
    "modalities": MODALITIES,
    "candidate_universe": "same_area_all_prior_years",
    "frontier_years": 3,
    "vector_block": VECTOR_BLOCK,
    "top_k": TOP_K,
    "score": "cosine_float32_from_float16_embeddings",
    "checkpoint_per_focal": True
}

assert original_config == expected_config, (
    "Configuracao incompatível com o experimento congelado."
)

# ------------------------------------------------------------
# 3. Amostra, plano e metadados
# ------------------------------------------------------------

sample = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)
sample["ANO"] = sample["ANO"].astype(int)

plan = pd.read_csv(
    PLAN_FILE,
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)
plan["ANO"] = plan["ANO"].astype(int)

assert len(sample) == len(plan) == 108
assert sample["FOCAL_ID"].is_unique
assert plan["FOCAL_ID"].is_unique

plan_idx = plan.set_index("FOCAL_ID", verify_integrity=True)

batch = (
    sample[sample["FOCAL_ID"].isin(FOCAL_IDS)]
    .sort_values("FOCAL_ID")
    .reset_index(drop=True)
)

assert batch["FOCAL_ID"].tolist() == FOCAL_IDS

meta = pq.read_table(
    INDEX_FILE,
    columns=["ID_TRABALHO", "ANO", "AREA_HARMONIZADA"]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
meta["ANO"] = meta["ANO"].astype(int)

assert len(meta) == 993086
assert meta["ID_TRABALHO"].is_unique

area_lookup = meta.set_index(
    "ID_TRABALHO", verify_integrity=True
)["AREA_HARMONIZADA"]

batch["AREA_HARMONIZADA"] = batch[
    "ID_TRABALHO"
].map(area_lookup)

assert batch["AREA_HARMONIZADA"].notna().all()

for f in batch.itertuples(index=False):
    p = plan_idx.loc[f.FOCAL_ID]
    assert int(f.ANO) == int(p["ANO"])
    assert str(f.ID_TRABALHO) == str(p["ID_TRABALHO"])
    assert str(f.AREA_HARMONIZADA) == str(p["AREA_HARMONIZADA"])

print("\nConfiguracao, amostra e metadados: OK")

# ------------------------------------------------------------
# 4. Ler vetores existentes sob demanda
# ------------------------------------------------------------

ids_cache = {}
vectors_cache = {}

def annual_ids(year, modality):
    key = (int(year), modality)

    if key not in ids_cache:
        file = EMB / modality / f"{modality}_{year}_ids.parquet"
        assert file.is_file()

        data = pd.read_parquet(
            file, columns=["ID_TRABALHO"]
        )["ID_TRABALHO"].astype(str)

        assert data.is_unique
        ids_cache[key] = pd.Index(data)

    return ids_cache[key]


def annual_vectors(year, modality):
    key = (int(year), modality)

    if key not in vectors_cache:
        file = (
            EMB / modality
            / f"{modality}_{year}_embeddings_float16.npy"
        )
        assert file.is_file()

        arr = np.load(file, mmap_mode="r")
        assert arr.ndim == 2 and arr.shape[1] == 768
        assert arr.shape[0] == len(annual_ids(year, modality))

        vectors_cache[key] = arr

    return vectors_cache[key]


def normalized_query(doc_id, year, modality):
    pos = annual_ids(year, modality).get_loc(str(doc_id))
    assert isinstance(pos, (int, np.integer))

    vec = np.array(
        annual_vectors(year, modality)[pos],
        dtype=np.float32,
        copy=True
    )

    norm = np.linalg.norm(vec)
    assert np.isfinite(norm) and norm > 0

    return vec / norm


def cosine_for_documents(doc_ids, year, modality, query):
    positions = annual_ids(year, modality).get_indexer(
        [str(x) for x in doc_ids]
    )
    assert (positions >= 0).all()

    arr = annual_vectors(year, modality)
    scores = np.empty(len(positions), dtype=np.float32)

    for start in range(0, len(positions), VECTOR_BLOCK):
        end = min(start + VECTOR_BLOCK, len(positions))

        block = np.array(
            arr[positions[start:end]],
            dtype=np.float32,
            copy=True
        )

        norms = np.linalg.norm(block, axis=1)
        assert np.isfinite(norms).all()
        assert (norms > 0).all()

        block /= norms[:, None]
        scores[start:end] = block @ query

    assert np.isfinite(scores).all()
    return scores

# ------------------------------------------------------------
# 5. Auditoria integral de um focal salvo
# ------------------------------------------------------------

def audit_focal(focal):
    fid = focal.FOCAL_ID
    year = int(focal.ANO)
    area = str(focal.AREA_HARMONIZADA)

    scores_file = CKPT / f"{fid}_scores.parquet"
    ranking_file = CKPT / f"{fid}_top30.parquet"

    assert scores_file.is_file()
    assert ranking_file.is_file()

    scores = pd.read_parquet(scores_file)
    ranks = pd.read_parquet(ranking_file)

    expected_h = int(plan_idx.loc[fid, "N_HISTORICAL"])
    expected_f = int(plan_idx.loc[fid, "N_FRONTIER3Y"])

    assert len(scores) == expected_h
    assert len(ranks) == 120
    assert scores["FOCAL_ID"].eq(fid).all()
    assert ranks["FOCAL_ID"].eq(fid).all()

    scores["ANTECEDENT_ID"] = scores["ANTECEDENT_ID"].astype(str)
    ranks["ANTECEDENT_ID"] = ranks["ANTECEDENT_ID"].astype(str)

    assert scores["ANTECEDENT_ID"].is_unique
    assert scores["FOCAL_YEAR"].astype(int).eq(year).all()
    assert ranks["FOCAL_YEAR"].astype(int).eq(year).all()

    years = scores["ANTECEDENT_YEAR"].astype(int)
    assert years.lt(year).all()

    areas = scores["ANTECEDENT_ID"].map(area_lookup)
    assert areas.notna().all()
    assert areas.eq(area).all()

    frontier = scores[years.between(year - 3, year - 1)]
    assert len(frontier) == expected_f

    assert not ranks.duplicated(
        ["FOCAL_ID", "WINDOW", "MODALITY", "RANK"]
    ).any()

    reproduced = 0

    for mod in MODALITIES:
        col = f"COSINE_{mod}"
        values = scores[col].to_numpy(dtype=float)

        assert np.isfinite(values).all()
        assert ((values >= -1.00001) &
                (values <= 1.00001)).all()

        for window, candidates in [
            ("historical", scores),
            ("frontier3y", frontier)
        ]:
            expected = candidates.sort_values(
                [col, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K)

            saved = ranks[
                (ranks["WINDOW"] == window) &
                (ranks["MODALITY"] == mod)
            ].sort_values("RANK")

            assert len(saved) == TOP_K
            assert saved["RANK"].tolist() == list(
                range(1, TOP_K + 1)
            )

            assert (
                expected["ANTECEDENT_ID"].tolist()
                == saved["ANTECEDENT_ID"].tolist()
            )

            assert np.allclose(
                expected[col].to_numpy(dtype=float),
                saved["SCORE"].to_numpy(dtype=float),
                atol=1e-6,
                rtol=0
            )

            if window == "frontier3y":
                assert saved[
                    "ANTECEDENT_YEAR"
                ].astype(int).between(year - 3, year - 1).all()

            reproduced += 1

    return {
        "FOCAL_ID": fid,
        "ANO": year,
        "AREA_HARMONIZADA": area,
        "N_HISTORICAL": expected_h,
        "N_FRONTIER3Y": expected_f,
        "RANKINGS_REPRODUCED": reproduced,
        "STATUS": "PASS"
    }

# ------------------------------------------------------------
# 6. Processar e publicar checkpoints novos
# ------------------------------------------------------------

new_focals = []
reused_focals = []

for focal in batch.itertuples(index=False):
    fid = focal.FOCAL_ID
    year = int(focal.ANO)
    area = str(focal.AREA_HARMONIZADA)

    scores_file = CKPT / f"{fid}_scores.parquet"
    ranking_file = CKPT / f"{fid}_top30.parquet"

    if scores_file.exists() and ranking_file.exists():
        audit_focal(focal)
        reused_focals.append(fid)
        print(f"{fid}: checkpoint existente, PASS")
        continue

    if scores_file.exists() or ranking_file.exists():
        raise RuntimeError(
            f"{fid}: checkpoint parcial encontrado; "
            "nenhum arquivo sera substituido."
        )

    temp_scores = CKPT / f"{fid}_scores.partial.parquet"
    temp_rank = CKPT / f"{fid}_top30.partial.parquet"

    if temp_scores.exists() or temp_rank.exists():
        raise RuntimeError(
            f"{fid}: arquivo temporario preexistente."
        )

    candidates = meta[
        (meta["ANO"] < year) &
        (meta["AREA_HARMONIZADA"] == area)
    ][["ID_TRABALHO", "ANO"]].rename(columns={
        "ID_TRABALHO": "ANTECEDENT_ID",
        "ANO": "ANTECEDENT_YEAR"
    }).reset_index(drop=True)

    expected_h = int(plan_idx.loc[fid, "N_HISTORICAL"])

    assert len(candidates) == expected_h
    assert candidates["ANTECEDENT_ID"].is_unique

    candidates["FOCAL_ID"] = fid
    candidates["FOCAL_YEAR"] = year

    for mod in MODALITIES:
        query = normalized_query(focal.ID_TRABALHO, year, mod)
        column = f"COSINE_{mod}"
        candidates[column] = np.nan

        for ant_year, group in candidates.groupby(
            "ANTECEDENT_YEAR", sort=True
        ):
            loc = group.index.to_numpy()

            candidates.loc[loc, column] = cosine_for_documents(
                group["ANTECEDENT_ID"].tolist(),
                int(ant_year),
                mod,
                query
            )

        assert candidates[column].notna().all()

    parts = []

    for window, subset in [
        ("historical", candidates),
        ("frontier3y", candidates[
            candidates["ANTECEDENT_YEAR"] >= year - 3
        ])
    ]:
        assert len(subset) >= TOP_K

        for mod in MODALITIES:
            col = f"COSINE_{mod}"

            top = subset.sort_values(
                [col, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K).copy()

            top["WINDOW"] = window
            top["MODALITY"] = mod
            top["RANK"] = np.arange(1, TOP_K + 1)
            top["SCORE"] = top[col]

            parts.append(top[[
                "FOCAL_ID", "FOCAL_YEAR",
                "ANTECEDENT_ID", "ANTECEDENT_YEAR",
                "WINDOW", "MODALITY", "RANK", "SCORE"
            ]])

    rankings = pd.concat(parts, ignore_index=True)

    assert len(rankings) == 120

    candidates.to_parquet(temp_scores, index=False)
    rankings.to_parquet(temp_rank, index=False)

    assert pq.ParquetFile(temp_scores).metadata.num_rows == expected_h
    assert pq.ParquetFile(temp_rank).metadata.num_rows == 120

    temp_scores.rename(scores_file)
    temp_rank.rename(ranking_file)

    audit_focal(focal)
    new_focals.append(fid)

    print(
        f"{fid}: PASS | {expected_h:,} antecedentes "
        "| M2/M4 | quatro rankings reproduzidos"
    )

    del candidates, rankings, parts
    vectors_cache.clear()
    gc.collect()

# ------------------------------------------------------------
# 7. Auditoria consolidada, sem sobrescritas
# ------------------------------------------------------------

audit = pd.DataFrame([
    audit_focal(focal)
    for focal in batch.itertuples(index=False)
])

assert len(audit) == 12
assert audit["STATUS"].eq("PASS").all()
assert audit["RANKINGS_REPRODUCED"].sum() == 48

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

audit_file = AUDITS / f"batch03_audit_{stamp}.csv"
summary_file = AUDITS / f"batch03_summary_{stamp}.json"

audit.to_csv(audit_file, index=False, mode="x")

summary = {
    "stage": "21D",
    "batch": BATCH,
    "first_focal": FOCAL_IDS[0],
    "last_focal": FOCAL_IDS[-1],
    "focals_audited": 12,
    "newly_processed": len(new_focals),
    "checkpoints_reused": len(reused_focals),
    "historical_pairs": int(audit["N_HISTORICAL"].sum()),
    "frontier3y_pairs": int(audit["N_FRONTIER3Y"].sum()),
    "rankings_reproduced": 48,
    "status": "PASS",
    "human_ground_truth_established": False
}

with summary_file.open("x", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 8. Resultado
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ETAPA 21D — RESULTADO FINAL")
print("=" * 65)

print("Focais auditados:", len(audit))
print("Novos processados:", len(new_focals))
print("Checkpoints reutilizados:", len(reused_focals))
print("Pares historicos:", summary["historical_pairs"])
print("Pares fronteira 3Y:", summary["frontier3y_pairs"])
print("Rankings reproduzidos:", summary["rankings_reproduced"])
print("Status:", summary["status"])
print("Relatorio:", audit_file)

# Conferir total acumulado de arquivos completos.
completed = [
    f"NAT-{i:04d}"
    for i in range(1, 109)
    if (CKPT / f"NAT-{i:04d}_scores.parquet").is_file()
    and (CKPT / f"NAT-{i:04d}_top30.parquet").is_file()
]

print("\nFocais com dois checkpoints:", len(completed), "/108")
print("Progresso esperado apos lote 03: 36/108")
print("\nETAPA 21D CONCLUIDA")


SCIENTOMETRICS — ETAPA 21D | LOTE 03
FOCAIS: NAT-0025 a NAT-0036

Configuracao, amostra e metadados: OK
NAT-0025: PASS | 10,578 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0026: PASS | 32,760 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0027: PASS | 3,742 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0028: PASS | 19,655 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0029: PASS | 35,156 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0030: PASS | 21,665 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0031: PASS | 15,542 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0032: PASS | 23,015 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0033: PASS | 14,141 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0034: PASS | 17,113 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0035: PASS | 12,987 antecedentes | M2/M4 | quatro rankings reproduzidos
NAT-0036: PASS | 3,742 antecedentes | M2/M4 | quatro rankings

In [38]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 21E
# EXECUTOR NACIONAL RETOMAVEL — LOTES 04 A 09
#
# Primeira execucao: NAT-0037 a NAT-0048
# Proximas execucoes: primeiro lote pendente
#
# CPU | Embeddings M2/M4 congelados
# Sem novos embeddings | Checkpoints por focal
# Sem sobrescrever artefatos anteriores
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. Caminhos e configuracao
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
EMB = BASE / "07_embeddings_nacionais"

SAMPLE = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CKPT = RUN / "checkpoints"
AUDITS = RUN / "audits"

CKPT.mkdir(parents=True, exist_ok=True)
AUDITS.mkdir(parents=True, exist_ok=True)

MODALITIES = ["M2", "M4"]
TOP_K = 30
VECTOR_BLOCK = 2048
BATCH_SIZE = 12

# ------------------------------------------------------------
# 2. Verificar configuracao congelada
# ------------------------------------------------------------

config_file = RUN / "run_configuration.json"
assert config_file.is_file()

config = json.loads(
    config_file.read_text(encoding="utf-8")
)

expected_config = {
    "stage": "21A",
    "sample": str(SAMPLE),
    "sample_n": 108,
    "modalities": MODALITIES,
    "candidate_universe": "same_area_all_prior_years",
    "frontier_years": 3,
    "vector_block": VECTOR_BLOCK,
    "top_k": TOP_K,
    "score": "cosine_float32_from_float16_embeddings",
    "checkpoint_per_focal": True
}

assert config == expected_config, (
    "Configuracao diferente da execucao original."
)

# ------------------------------------------------------------
# 3. Identificar proximo lote pendente
# ------------------------------------------------------------

def expected_batch_ids(batch_number):
    first = (batch_number - 1) * BATCH_SIZE + 1
    return [
        f"NAT-{i:04d}"
        for i in range(first, first + BATCH_SIZE)
    ]

def checkpoint_pair_exists(fid):
    return (
        (CKPT / f"{fid}_scores.parquet").is_file()
        and (CKPT / f"{fid}_top30.parquet").is_file()
    )

selected_batch = None

for candidate_batch in range(4, 10):
    ids = expected_batch_ids(candidate_batch)

    if not all(checkpoint_pair_exists(fid) for fid in ids):
        selected_batch = candidate_batch
        break

if selected_batch is None:
    print("TODOS OS LOTES 04 A 09 POSSUEM CHECKPOINTS.")
    print("Execute posteriormente a auditoria nacional consolidada.")
else:
    print("=" * 65)
    print("SCIENTOMETRICS — ETAPA 21E")
    print("LOTE SELECIONADO:", selected_batch)
    print(
        "FOCAIS:",
        expected_batch_ids(selected_batch)[0],
        "a",
        expected_batch_ids(selected_batch)[-1]
    )
    print("=" * 65)

# ------------------------------------------------------------
# 4. Executar somente se houver lote pendente
# ------------------------------------------------------------

if selected_batch is not None:

    FOCAL_IDS = expected_batch_ids(selected_batch)

    sample = pd.read_csv(
        SAMPLE / "focal_processing_index.csv",
        dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
    )

    plan = pd.read_csv(
        PLAN_FILE,
        dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
    )

    sample["ANO"] = sample["ANO"].astype(int)
    plan["ANO"] = plan["ANO"].astype(int)

    assert len(sample) == len(plan) == 108
    assert sample["FOCAL_ID"].is_unique
    assert plan["FOCAL_ID"].is_unique

    plan_idx = plan.set_index(
        "FOCAL_ID",
        verify_integrity=True
    )

    batch = (
        sample[sample["FOCAL_ID"].isin(FOCAL_IDS)]
        .sort_values("FOCAL_ID")
        .reset_index(drop=True)
    )

    assert batch["FOCAL_ID"].tolist() == FOCAL_IDS

    # --------------------------------------------------------
    # 5. Metadados
    # --------------------------------------------------------

    meta = pq.read_table(
        INDEX_FILE,
        columns=[
            "ID_TRABALHO",
            "ANO",
            "AREA_HARMONIZADA"
        ]
    ).to_pandas()

    meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
    meta["ANO"] = meta["ANO"].astype(int)

    assert len(meta) == 993086
    assert meta["ID_TRABALHO"].is_unique

    area_lookup = meta.set_index(
        "ID_TRABALHO",
        verify_integrity=True
    )["AREA_HARMONIZADA"]

    batch["AREA_HARMONIZADA"] = (
        batch["ID_TRABALHO"].map(area_lookup)
    )

    assert batch["AREA_HARMONIZADA"].notna().all()

    for f in batch.itertuples(index=False):
        p = plan_idx.loc[f.FOCAL_ID]

        assert int(f.ANO) == int(p["ANO"])
        assert str(f.ID_TRABALHO) == str(p["ID_TRABALHO"])
        assert str(f.AREA_HARMONIZADA) == str(
            p["AREA_HARMONIZADA"]
        )

    print("\nConfiguracao, amostra e metadados: OK")

    # --------------------------------------------------------
    # 6. Funcoes de leitura seletiva dos embeddings
    # --------------------------------------------------------

    id_cache = {}
    vector_cache = {}

    def annual_ids(year, modality):
        key = (int(year), modality)

        if key not in id_cache:
            path = (
                EMB / modality
                / f"{modality}_{year}_ids.parquet"
            )

            assert path.is_file()

            ids = pd.read_parquet(
                path,
                columns=["ID_TRABALHO"]
            )["ID_TRABALHO"].astype(str)

            assert ids.is_unique
            id_cache[key] = pd.Index(ids)

        return id_cache[key]

    def annual_vectors(year, modality):
        key = (int(year), modality)

        if key not in vector_cache:
            path = (
                EMB / modality
                / f"{modality}_{year}_embeddings_float16.npy"
            )

            assert path.is_file()

            arr = np.load(path, mmap_mode="r")

            assert arr.ndim == 2
            assert arr.shape[1] == 768
            assert len(arr) == len(
                annual_ids(year, modality)
            )

            vector_cache[key] = arr

        return vector_cache[key]

    def get_query(doc_id, year, modality):
        pos = annual_ids(year, modality).get_loc(
            str(doc_id)
        )

        assert isinstance(pos, (int, np.integer))

        v = np.array(
            annual_vectors(year, modality)[pos],
            dtype=np.float32,
            copy=True
        )

        norm = np.linalg.norm(v)
        assert np.isfinite(norm) and norm > 0

        return v / norm

    def score_documents(doc_ids, year, modality, query):
        positions = annual_ids(
            year, modality
        ).get_indexer(
            [str(x) for x in doc_ids]
        )

        assert (positions >= 0).all()

        arr = annual_vectors(year, modality)
        result = np.empty(
            len(positions),
            dtype=np.float32
        )

        for start in range(
            0, len(positions), VECTOR_BLOCK
        ):
            end = min(
                start + VECTOR_BLOCK,
                len(positions)
            )

            block = np.array(
                arr[positions[start:end]],
                dtype=np.float32,
                copy=True
            )

            norms = np.linalg.norm(
                block, axis=1
            )

            assert np.isfinite(norms).all()
            assert (norms > 0).all()

            block /= norms[:, None]
            result[start:end] = block @ query

        assert np.isfinite(result).all()

        return result

    # --------------------------------------------------------
    # 7. Funcao de auditoria
    # --------------------------------------------------------

    def audit_focal(focal):

        fid = focal.FOCAL_ID
        year = int(focal.ANO)
        area = str(focal.AREA_HARMONIZADA)

        sf = CKPT / f"{fid}_scores.parquet"
        rf = CKPT / f"{fid}_top30.parquet"

        assert sf.is_file() and rf.is_file()

        scores = pd.read_parquet(sf)
        ranks = pd.read_parquet(rf)

        nh = int(plan_idx.loc[fid, "N_HISTORICAL"])
        nf = int(plan_idx.loc[fid, "N_FRONTIER3Y"])

        assert len(scores) == nh
        assert len(ranks) == 120

        scores["ANTECEDENT_ID"] = (
            scores["ANTECEDENT_ID"].astype(str)
        )
        ranks["ANTECEDENT_ID"] = (
            ranks["ANTECEDENT_ID"].astype(str)
        )

        assert scores["FOCAL_ID"].eq(fid).all()
        assert ranks["FOCAL_ID"].eq(fid).all()
        assert scores["FOCAL_YEAR"].astype(int).eq(year).all()
        assert ranks["FOCAL_YEAR"].astype(int).eq(year).all()
        assert scores["ANTECEDENT_ID"].is_unique

        years = scores["ANTECEDENT_YEAR"].astype(int)

        assert years.lt(year).all()

        areas = scores["ANTECEDENT_ID"].map(
            area_lookup
        )

        assert areas.notna().all()
        assert areas.eq(area).all()

        frontier = scores[
            years.between(year - 3, year - 1)
        ]

        assert len(frontier) == nf

        assert not ranks.duplicated([
            "FOCAL_ID", "WINDOW", "MODALITY", "RANK"
        ]).any()

        for modality in MODALITIES:

            col = f"COSINE_{modality}"
            vals = scores[col].to_numpy(dtype=float)

            assert np.isfinite(vals).all()
            assert ((vals >= -1.00001) &
                    (vals <= 1.00001)).all()

            for window, candidates in [
                ("historical", scores),
                ("frontier3y", frontier)
            ]:

                expected = candidates.sort_values(
                    [col, "ANTECEDENT_ID"],
                    ascending=[False, True],
                    kind="mergesort"
                ).head(TOP_K)

                stored = ranks[
                    (ranks["WINDOW"] == window) &
                    (ranks["MODALITY"] == modality)
                ].sort_values("RANK")

                assert len(stored) == TOP_K
                assert stored["RANK"].tolist() == list(
                    range(1, TOP_K + 1)
                )

                assert (
                    expected["ANTECEDENT_ID"].tolist()
                    == stored["ANTECEDENT_ID"].tolist()
                )

                assert np.allclose(
                    expected[col].to_numpy(dtype=float),
                    stored["SCORE"].to_numpy(dtype=float),
                    atol=1e-6,
                    rtol=0
                )

                if window == "frontier3y":
                    assert stored[
                        "ANTECEDENT_YEAR"
                    ].astype(int).between(
                        year - 3, year - 1
                    ).all()

        return {
            "FOCAL_ID": fid,
            "ANO": year,
            "AREA_HARMONIZADA": area,
            "N_HISTORICAL": nh,
            "N_FRONTIER3Y": nf,
            "RANKINGS_REPRODUCED": 4,
            "STATUS": "PASS"
        }

    # --------------------------------------------------------
    # 8. Processar o lote com checkpoints
    # --------------------------------------------------------

    new_focals = []
    reused_focals = []

    for focal in batch.itertuples(index=False):

        fid = focal.FOCAL_ID
        year = int(focal.ANO)
        area = str(focal.AREA_HARMONIZADA)

        sf = CKPT / f"{fid}_scores.parquet"
        rf = CKPT / f"{fid}_top30.parquet"

        if sf.exists() and rf.exists():
            audit_focal(focal)
            reused_focals.append(fid)
            print(f"{fid}: checkpoint reutilizado | PASS")
            continue

        if sf.exists() or rf.exists():
            raise RuntimeError(
                f"{fid}: checkpoint incompleto. "
                "Nenhum arquivo foi substituido."
            )

        tmp_sf = (
            CKPT / f"{fid}_scores.partial.parquet"
        )
        tmp_rf = (
            CKPT / f"{fid}_top30.partial.parquet"
        )

        if tmp_sf.exists() or tmp_rf.exists():
            raise RuntimeError(
                f"{fid}: arquivo temporario preexistente."
            )

        candidates = meta[
            (meta["ANO"] < year) &
            (meta["AREA_HARMONIZADA"] == area)
        ][["ID_TRABALHO", "ANO"]].rename(columns={
            "ID_TRABALHO": "ANTECEDENT_ID",
            "ANO": "ANTECEDENT_YEAR"
        }).reset_index(drop=True)

        expected_n = int(
            plan_idx.loc[fid, "N_HISTORICAL"]
        )

        assert len(candidates) == expected_n
        assert candidates["ANTECEDENT_ID"].is_unique

        candidates["FOCAL_ID"] = fid
        candidates["FOCAL_YEAR"] = year

        for modality in MODALITIES:

            query = get_query(
                focal.ID_TRABALHO,
                year,
                modality
            )

            col = f"COSINE_{modality}"
            candidates[col] = np.nan

            for ant_year, group in candidates.groupby(
                "ANTECEDENT_YEAR", sort=True
            ):
                positions = group.index.to_numpy()

                candidates.loc[
                    positions, col
                ] = score_documents(
                    group["ANTECEDENT_ID"].tolist(),
                    int(ant_year),
                    modality,
                    query
                )

            assert candidates[col].notna().all()

        ranking_parts = []

        for window, subset in [
            ("historical", candidates),
            (
                "frontier3y",
                candidates[
                    candidates["ANTECEDENT_YEAR"] >= year - 3
                ]
            )
        ]:

            assert len(subset) >= TOP_K

            for modality in MODALITIES:

                col = f"COSINE_{modality}"

                top = subset.sort_values(
                    [col, "ANTECEDENT_ID"],
                    ascending=[False, True],
                    kind="mergesort"
                ).head(TOP_K).copy()

                top["WINDOW"] = window
                top["MODALITY"] = modality
                top["RANK"] = np.arange(1, TOP_K + 1)
                top["SCORE"] = top[col]

                ranking_parts.append(top[[
                    "FOCAL_ID",
                    "FOCAL_YEAR",
                    "ANTECEDENT_ID",
                    "ANTECEDENT_YEAR",
                    "WINDOW",
                    "MODALITY",
                    "RANK",
                    "SCORE"
                ]])

        ranks = pd.concat(
            ranking_parts,
            ignore_index=True
        )

        assert len(ranks) == 120

        candidates.to_parquet(
            tmp_sf, index=False
        )
        ranks.to_parquet(
            tmp_rf, index=False
        )

        assert (
            pq.ParquetFile(tmp_sf).metadata.num_rows
            == expected_n
        )
        assert (
            pq.ParquetFile(tmp_rf).metadata.num_rows
            == 120
        )

        tmp_sf.rename(sf)
        tmp_rf.rename(rf)

        audit_focal(focal)
        new_focals.append(fid)

        print(
            f"{fid}: PASS | {expected_n:,} antecedentes "
            "| M2/M4 | 4 rankings reproduzidos"
        )

        del candidates, ranks, ranking_parts
        vector_cache.clear()
        gc.collect()

    # --------------------------------------------------------
    # 9. Auditoria consolidada
    # --------------------------------------------------------

    audit = pd.DataFrame([
        audit_focal(focal)
        for focal in batch.itertuples(index=False)
    ])

    assert len(audit) == 12
    assert audit["STATUS"].eq("PASS").all()
    assert audit["RANKINGS_REPRODUCED"].sum() == 48

    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )

    audit_path = (
        AUDITS /
        f"batch{selected_batch:02d}_audit_{stamp}.csv"
    )
    summary_path = (
        AUDITS /
        f"batch{selected_batch:02d}_summary_{stamp}.json"
    )

    audit.to_csv(
        audit_path,
        index=False,
        mode="x"
    )

    summary = {
        "stage": "21E",
        "batch": selected_batch,
        "first_focal": FOCAL_IDS[0],
        "last_focal": FOCAL_IDS[-1],
        "n_focals": 12,
        "newly_processed": len(new_focals),
        "reused": len(reused_focals),
        "historical_pairs": int(
            audit["N_HISTORICAL"].sum()
        ),
        "frontier3y_pairs": int(
            audit["N_FRONTIER3Y"].sum()
        ),
        "rankings_reproduced": 48,
        "status": "PASS",
        "ground_truth_established": False
    }

    with summary_path.open(
        "x", encoding="utf-8"
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False
        )

    completed = [
        fid
        for i in range(1, 109)
        if checkpoint_pair_exists(
            fid := f"NAT-{i:04d}"
        )
    ]

    print("\n" + "=" * 65)
    print("ETAPA 21E — RESULTADO FINAL")
    print("=" * 65)

    print("Lote:", selected_batch)
    print("Focais:", len(audit))
    print("Novos processados:", len(new_focals))
    print("Checkpoints reutilizados:", len(reused_focals))
    print(
        "Pares historicos:",
        summary["historical_pairs"]
    )
    print(
        "Pares fronteira:",
        summary["frontier3y_pairs"]
    )
    print(
        "Rankings reproduzidos:",
        summary["rankings_reproduced"]
    )
    print("Status:", summary["status"])
    print(
        "Total com ambos os checkpoints:",
        len(completed), "/108"
    )
    print("Relatorio:", audit_path)
    print("\nETAPA 21E CONCLUIDA")


SCIENTOMETRICS — ETAPA 21E
LOTE SELECIONADO: 5
FOCAIS: NAT-0049 a NAT-0060

Configuracao, amostra e metadados: OK
NAT-0049: PASS | 31,148 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0050: PASS | 10,940 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0051: PASS | 12,108 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0052: PASS | 17,113 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0053: PASS | 6,030 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0054: PASS | 27,657 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0055: PASS | 27,332 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0056: PASS | 14,946 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0057: PASS | 39,011 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0058: PASS | 23,595 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0059: PASS | 20,935 antecedentes | M2/M4 | 4 rankings reproduzidos
NAT-0060: PASS | 11,972 antecedentes | M2/M4 | 4 rankings reproduzidos

ETAPA 21E — RESULTADO FINAL
Lote: 

In [39]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 21F
# FINALIZACAO NACIONAL — LOTES 06 A 09
#
# CPU | M2 e M4 congelados
# 48 focais restantes
# Checkpoint individual e auditoria integral
# Retomavel | Sem sobrescrever resultados existentes
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. CAMINHOS ORIGINAIS
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
EMB = BASE / "07_embeddings_nacionais"

SAMPLE = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CKPT = RUN / "checkpoints"
AUDITS = RUN / "audits"

CKPT.mkdir(parents=True, exist_ok=True)
AUDITS.mkdir(parents=True, exist_ok=True)

MODALITIES = ["M2", "M4"]
TOP_K = 30
VECTOR_BLOCK = 2048

PENDING_IDS = [
    f"NAT-{i:04d}" for i in range(61, 109)
]

ALL_IDS = [
    f"NAT-{i:04d}" for i in range(1, 109)
]

# ------------------------------------------------------------
# 2. VALIDAR CONFIGURACAO CONGELADA
# ------------------------------------------------------------

config_path = RUN / "run_configuration.json"
assert config_path.is_file()

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

expected_config = {
    "stage": "21A",
    "sample": str(SAMPLE),
    "sample_n": 108,
    "modalities": MODALITIES,
    "candidate_universe": "same_area_all_prior_years",
    "frontier_years": 3,
    "vector_block": VECTOR_BLOCK,
    "top_k": TOP_K,
    "score": "cosine_float32_from_float16_embeddings",
    "checkpoint_per_focal": True
}

assert config == expected_config, (
    "Configuracao nao coincide com a original."
)

sample = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

plan = pd.read_csv(
    PLAN_FILE,
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

sample["ANO"] = sample["ANO"].astype(int)
plan["ANO"] = plan["ANO"].astype(int)

assert len(sample) == len(plan) == 108
assert sample["FOCAL_ID"].is_unique
assert plan["FOCAL_ID"].is_unique
assert set(sample["FOCAL_ID"]) == set(ALL_IDS)

plan_idx = plan.set_index(
    "FOCAL_ID", verify_integrity=True
)

# Verificar os 60 checkpoints previamente concluidos.
for i in range(1, 61):
    fid = f"NAT-{i:04d}"

    sf = CKPT / f"{fid}_scores.parquet"
    rf = CKPT / f"{fid}_top30.parquet"

    assert sf.is_file() and rf.is_file(), (
        f"Checkpoint anterior ausente: {fid}"
    )

print("=" * 68)
print("SCIENTOMETRICS — ETAPA 21F")
print("EXPANSAO NACIONAL: NAT-0061 A NAT-0108")
print("=" * 68)
print("Checkpoints anteriores presentes: 60/60")

# ------------------------------------------------------------
# 3. CARREGAR INDICE NACIONAL
# ------------------------------------------------------------

meta = pq.read_table(
    INDEX_FILE,
    columns=["ID_TRABALHO", "ANO", "AREA_HARMONIZADA"]
).to_pandas()

meta["ID_TRABALHO"] = (
    meta["ID_TRABALHO"].astype(str)
)

meta["ANO"] = meta["ANO"].astype(int)

assert len(meta) == 993086
assert meta["ID_TRABALHO"].is_unique

area_lookup = meta.set_index(
    "ID_TRABALHO", verify_integrity=True
)["AREA_HARMONIZADA"]

sample["AREA_HARMONIZADA"] = (
    sample["ID_TRABALHO"].map(area_lookup)
)

assert sample["AREA_HARMONIZADA"].notna().all()

focals = (
    sample[sample["FOCAL_ID"].isin(PENDING_IDS)]
    .sort_values("FOCAL_ID")
    .reset_index(drop=True)
)

assert focals["FOCAL_ID"].tolist() == PENDING_IDS

for f in focals.itertuples(index=False):
    p = plan_idx.loc[f.FOCAL_ID]

    assert str(f.ID_TRABALHO) == str(p["ID_TRABALHO"])
    assert int(f.ANO) == int(p["ANO"])
    assert str(f.AREA_HARMONIZADA) == str(
        p["AREA_HARMONIZADA"]
    )

# ------------------------------------------------------------
# 4. CACHE DE EMBEDDINGS
# ------------------------------------------------------------

id_cache = {}
vec_cache = {}

def annual_ids(year, modality):
    key = (int(year), modality)

    if key not in id_cache:
        file = (
            EMB / modality
            / f"{modality}_{year}_ids.parquet"
        )

        assert file.is_file()

        values = pd.read_parquet(
            file, columns=["ID_TRABALHO"]
        )["ID_TRABALHO"].astype(str)

        assert values.is_unique
        id_cache[key] = pd.Index(values)

    return id_cache[key]


def annual_vectors(year, modality):
    key = (int(year), modality)

    if key not in vec_cache:
        file = (
            EMB / modality
            / f"{modality}_{year}_embeddings_float16.npy"
        )

        assert file.is_file()

        arr = np.load(file, mmap_mode="r")

        assert arr.ndim == 2
        assert arr.shape[1] == 768
        assert len(arr) == len(annual_ids(year, modality))

        vec_cache[key] = arr

    return vec_cache[key]


def normalized_query(doc_id, year, modality):
    pos = annual_ids(year, modality).get_loc(
        str(doc_id)
    )

    assert isinstance(pos, (int, np.integer))

    vector = np.array(
        annual_vectors(year, modality)[pos],
        dtype=np.float32,
        copy=True
    )

    norm = np.linalg.norm(vector)

    assert np.isfinite(norm) and norm > 0

    return vector / norm


def cosine_scores(doc_ids, year, modality, query):
    positions = annual_ids(
        year, modality
    ).get_indexer([str(x) for x in doc_ids])

    assert (positions >= 0).all()

    arr = annual_vectors(year, modality)

    output = np.empty(
        len(positions),
        dtype=np.float32
    )

    for start in range(
        0, len(positions), VECTOR_BLOCK
    ):
        stop = min(
            start + VECTOR_BLOCK, len(positions)
        )

        block = np.array(
            arr[positions[start:stop]],
            dtype=np.float32,
            copy=True
        )

        norms = np.linalg.norm(block, axis=1)

        assert np.isfinite(norms).all()
        assert (norms > 0).all()

        block /= norms[:, None]
        output[start:stop] = block @ query

    assert np.isfinite(output).all()

    return output

# ------------------------------------------------------------
# 5. AUDITORIA REUTILIZAVEL
# ------------------------------------------------------------

def audit_focal(focal):
    fid = focal.FOCAL_ID
    year = int(focal.ANO)
    area = str(focal.AREA_HARMONIZADA)

    scores = pd.read_parquet(
        CKPT / f"{fid}_scores.parquet"
    )

    ranks = pd.read_parquet(
        CKPT / f"{fid}_top30.parquet"
    )

    expected_h = int(
        plan_idx.loc[fid, "N_HISTORICAL"]
    )

    expected_f = int(
        plan_idx.loc[fid, "N_FRONTIER3Y"]
    )

    assert len(scores) == expected_h
    assert len(ranks) == 120

    scores["ANTECEDENT_ID"] = (
        scores["ANTECEDENT_ID"].astype(str)
    )
    ranks["ANTECEDENT_ID"] = (
        ranks["ANTECEDENT_ID"].astype(str)
    )

    assert scores["ANTECEDENT_ID"].is_unique
    assert scores["FOCAL_ID"].eq(fid).all()
    assert ranks["FOCAL_ID"].eq(fid).all()

    assert scores["FOCAL_YEAR"].astype(int).eq(year).all()
    assert ranks["FOCAL_YEAR"].astype(int).eq(year).all()

    years = scores["ANTECEDENT_YEAR"].astype(int)

    assert years.lt(year).all()

    areas = scores["ANTECEDENT_ID"].map(
        area_lookup
    )

    assert areas.notna().all()
    assert areas.eq(area).all()

    frontier = scores[
        years.between(year - 3, year - 1)
    ]

    assert len(frontier) == expected_f

    assert not ranks.duplicated([
        "FOCAL_ID", "WINDOW", "MODALITY", "RANK"
    ]).any()

    for modality in MODALITIES:
        col = f"COSINE_{modality}"
        vals = scores[col].to_numpy(dtype=float)

        assert np.isfinite(vals).all()
        assert (
            (vals >= -1.00001)
            & (vals <= 1.00001)
        ).all()

        for window, subset in [
            ("historical", scores),
            ("frontier3y", frontier)
        ]:

            expected = subset.sort_values(
                [col, "ANTECEDENT_ID"],
                ascending=[False, True],
                kind="mergesort"
            ).head(TOP_K)

            stored = ranks[
                (ranks["WINDOW"] == window)
                & (ranks["MODALITY"] == modality)
            ].sort_values("RANK")

            assert len(stored) == TOP_K

            assert stored["RANK"].tolist() == list(
                range(1, TOP_K + 1)
            )

            assert (
                expected["ANTECEDENT_ID"].tolist()
                == stored["ANTECEDENT_ID"].tolist()
            )

            assert np.allclose(
                expected[col].to_numpy(dtype=float),
                stored["SCORE"].to_numpy(dtype=float),
                atol=1e-6,
                rtol=0
            )

            if window == "frontier3y":
                assert stored[
                    "ANTECEDENT_YEAR"
                ].astype(int).between(
                    year - 3, year - 1
                ).all()

    return {
        "FOCAL_ID": fid,
        "ANO": year,
        "AREA_HARMONIZADA": area,
        "N_HISTORICAL": expected_h,
        "N_FRONTIER3Y": expected_f,
        "RANKINGS_REPRODUCED": 4,
        "STATUS": "PASS"
    }

# ------------------------------------------------------------
# 6. PROCESSAR TODOS OS LOTES RESTANTES
# ------------------------------------------------------------

all_audits = []

for batch_number in range(6, 10):

    first = (batch_number - 1) * 12 + 1
    last = batch_number * 12

    batch = focals[
        focals["FOCAL_ID"].isin(
            [f"NAT-{i:04d}" for i in range(first, last + 1)]
        )
    ].sort_values("FOCAL_ID")

    assert len(batch) == 12

    print("\n" + "=" * 68)
    print(
        f"LOTE {batch_number:02d}: "
        f"NAT-{first:04d} A NAT-{last:04d}"
    )
    print("=" * 68)

    newly_processed = 0
    reused = 0
    batch_audit = []

    for focal in batch.itertuples(index=False):

        fid = focal.FOCAL_ID
        year = int(focal.ANO)
        area = str(focal.AREA_HARMONIZADA)

        sf = CKPT / f"{fid}_scores.parquet"
        rf = CKPT / f"{fid}_top30.parquet"

        # Aproveitar checkpoint ja completo.
        if sf.exists() and rf.exists():
            result = audit_focal(focal)
            batch_audit.append(result)
            reused += 1
            print(f"{fid}: REUTILIZADO | PASS")
            continue

        if sf.exists() or rf.exists():
            raise RuntimeError(
                f"{fid}: checkpoint parcial. "
                "Interrompido para preservar arquivos."
            )

        tmp_sf = CKPT / f"{fid}_scores.partial.parquet"
        tmp_rf = CKPT / f"{fid}_top30.partial.parquet"

        if tmp_sf.exists() or tmp_rf.exists():
            raise RuntimeError(
                f"{fid}: arquivo temporario preexistente."
            )

        # Antecedentes estritamente anteriores,
        # pertencentes a mesma area harmonizada.
        candidates = meta[
            (meta["ANO"] < year)
            & (meta["AREA_HARMONIZADA"] == area)
        ][["ID_TRABALHO", "ANO"]].rename(columns={
            "ID_TRABALHO": "ANTECEDENT_ID",
            "ANO": "ANTECEDENT_YEAR"
        }).reset_index(drop=True)

        expected_n = int(
            plan_idx.loc[fid, "N_HISTORICAL"]
        )

        assert len(candidates) == expected_n
        assert candidates["ANTECEDENT_ID"].is_unique

        candidates["FOCAL_ID"] = fid
        candidates["FOCAL_YEAR"] = year

        # Escores completos M2 e M4.
        for modality in MODALITIES:
            query = normalized_query(
                focal.ID_TRABALHO,
                year,
                modality
            )

            col = f"COSINE_{modality}"
            candidates[col] = np.nan

            for ant_year, group in candidates.groupby(
                "ANTECEDENT_YEAR",
                sort=True
            ):

                rows = group.index.to_numpy()

                candidates.loc[rows, col] = cosine_scores(
                    group["ANTECEDENT_ID"].tolist(),
                    int(ant_year),
                    modality,
                    query
                )

            assert candidates[col].notna().all()

        # Derivar quatro rankings por focal.
        rank_parts = []

        for window, subset in [
            ("historical", candidates),
            (
                "frontier3y",
                candidates[
                    candidates["ANTECEDENT_YEAR"] >= year - 3
                ]
            )
        ]:
            assert len(subset) >= TOP_K

            for modality in MODALITIES:
                col = f"COSINE_{modality}"

                top = subset.sort_values(
                    [col, "ANTECEDENT_ID"],
                    ascending=[False, True],
                    kind="mergesort"
                ).head(TOP_K).copy()

                top["WINDOW"] = window
                top["MODALITY"] = modality
                top["RANK"] = np.arange(1, TOP_K + 1)
                top["SCORE"] = top[col]

                rank_parts.append(top[[
                    "FOCAL_ID",
                    "FOCAL_YEAR",
                    "ANTECEDENT_ID",
                    "ANTECEDENT_YEAR",
                    "WINDOW",
                    "MODALITY",
                    "RANK",
                    "SCORE"
                ]])

        ranks = pd.concat(
            rank_parts, ignore_index=True
        )

        assert len(ranks) == 120

        # Publicar somente depois de salvar ambos.
        candidates.to_parquet(tmp_sf, index=False)
        ranks.to_parquet(tmp_rf, index=False)

        assert (
            pq.ParquetFile(tmp_sf).metadata.num_rows
            == expected_n
        )

        assert (
            pq.ParquetFile(tmp_rf).metadata.num_rows
            == 120
        )

        tmp_sf.rename(sf)
        tmp_rf.rename(rf)

        result = audit_focal(focal)
        batch_audit.append(result)
        newly_processed += 1

        print(
            f"{fid}: PASS | "
            f"{expected_n:,} antecedentes | "
            "4 rankings reproduzidos"
        )

        del candidates, ranks, rank_parts
        vec_cache.clear()
        gc.collect()

    # --------------------------------------------------------
    # 7. RELATORIO POR LOTE
    # --------------------------------------------------------

    audit_df = pd.DataFrame(batch_audit)

    assert len(audit_df) == 12
    assert audit_df["STATUS"].eq("PASS").all()
    assert audit_df["RANKINGS_REPRODUCED"].sum() == 48

    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )

    audit_file = (
        AUDITS /
        f"batch{batch_number:02d}_audit_{stamp}.csv"
    )

    summary_file = (
        AUDITS /
        f"batch{batch_number:02d}_summary_{stamp}.json"
    )

    audit_df.to_csv(
        audit_file, index=False, mode="x"
    )

    summary = {
        "stage": "21F",
        "batch": batch_number,
        "newly_processed": newly_processed,
        "reused": reused,
        "focals_audited": len(audit_df),
        "historical_pairs": int(
            audit_df["N_HISTORICAL"].sum()
        ),
        "frontier3y_pairs": int(
            audit_df["N_FRONTIER3Y"].sum()
        ),
        "rankings_reproduced": 48,
        "status": "PASS"
    }

    with summary_file.open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2
        )

    all_audits.append(audit_df)

    print(
        f"\nLOTE {batch_number:02d} APROVADO | "
        f"Novos: {newly_processed} | "
        f"Reutilizados: {reused}"
    )

    print(
        "Pares historicos:",
        summary["historical_pairs"]
    )

    print(
        "Pares fronteira:",
        summary["frontier3y_pairs"]
    )

    print("Rankings reproduzidos: 48")

# ------------------------------------------------------------
# 8. AUDITORIA FINAL DE COBERTURA NACIONAL
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("AUDITORIA FINAL DE COBERTURA")
print("=" * 68)

complete = []

for fid in ALL_IDS:

    sf = CKPT / f"{fid}_scores.parquet"
    rf = CKPT / f"{fid}_top30.parquet"

    if sf.is_file() and rf.is_file():

        expected_n = int(
            plan_idx.loc[fid, "N_HISTORICAL"]
        )

        assert (
            pq.ParquetFile(sf).metadata.num_rows
            == expected_n
        )

        assert (
            pq.ParquetFile(rf).metadata.num_rows
            == 120
        )

        complete.append(fid)

assert len(complete) == 108

total_h = int(plan["N_HISTORICAL"].sum())
total_f = int(plan["N_FRONTIER3Y"].sum())

assert total_h == 1640271
assert total_f == 756979

print("\n=== RESULTADO FINAL — ETAPA 21F ===")
print("Focais com checkpoints:", len(complete), "/108")
print("Pares historicos previstos:", total_h)
print("Pares fronteira 3Y previstos:", total_f)
print("Rankings armazenados:", len(complete) * 4)
print("Registros dos rankings:", len(complete) * 120)
print("Modalidades:", ", ".join(MODALITIES))
print("Janelas: historical, frontier3y")
print("Status de cobertura: PASS")

print("\nETAPA 21F CONCLUIDA")


SCIENTOMETRICS — ETAPA 21F
EXPANSAO NACIONAL: NAT-0061 A NAT-0108
Checkpoints anteriores presentes: 60/60

LOTE 06: NAT-0061 A NAT-0072
NAT-0061: PASS | 5,455 antecedentes | 4 rankings reproduzidos
NAT-0062: PASS | 16,963 antecedentes | 4 rankings reproduzidos
NAT-0063: PASS | 6,188 antecedentes | 4 rankings reproduzidos
NAT-0064: PASS | 3,970 antecedentes | 4 rankings reproduzidos
NAT-0065: PASS | 15,620 antecedentes | 4 rankings reproduzidos
NAT-0066: PASS | 35,398 antecedentes | 4 rankings reproduzidos
NAT-0067: PASS | 27,234 antecedentes | 4 rankings reproduzidos
NAT-0068: PASS | 10,817 antecedentes | 4 rankings reproduzidos
NAT-0069: PASS | 4,295 antecedentes | 4 rankings reproduzidos
NAT-0070: PASS | 17,064 antecedentes | 4 rankings reproduzidos
NAT-0071: PASS | 10,511 antecedentes | 4 rankings reproduzidos
NAT-0072: PASS | 6,870 antecedentes | 4 rankings reproduzidos

LOTE 06 APROVADO | Novos: 12 | Reutilizados: 0
Pares historicos: 160385
Pares fronteira: 76270
Rankings reproduz

In [40]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 22A
# MATRIZ NACIONAL DE ESTABILIDADE DE ANTECEDENTES
#
# CPU | 108 documentos focais
# M2 x M4 | Historico x Fronteira 3 anos
# Top 10, Top 30 e Top 50
#
# Apenas leitura dos checkpoints da Etapa 21
# Sem recalcular embeddings ou similaridades
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. CAMINHOS
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SAMPLE_DIR = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CKPT = RUN / "checkpoints"

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "05_national_expansion"
    / f"22A_national_stability_{STAMP}"
)

OUT.mkdir(parents=True, exist_ok=False)

K_VALUES = [10, 30, 50]
MODALITIES = ["M2", "M4"]
WINDOWS = ["historical", "frontier3y"]

# ------------------------------------------------------------
# 2. AMOSTRA E PLANO CONGELADOS
# ------------------------------------------------------------

sample = pd.read_parquet(
    SAMPLE_DIR / "national_focals_108_RESTRICTED.parquet"
)

sample["ID_TRABALHO"] = (
    sample["ID_TRABALHO"].astype(str)
)

sample["ANO"] = sample["ANO"].astype(int)

index = pd.read_csv(
    SAMPLE_DIR / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

plan = pd.read_csv(
    PLAN_FILE,
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

assert len(sample) == len(index) == len(plan) == 108

assert sample["ID_TRABALHO"].is_unique
assert index["FOCAL_ID"].is_unique
assert index["ID_TRABALHO"].is_unique
assert plan["FOCAL_ID"].is_unique

index = index[["FOCAL_ID", "ID_TRABALHO"]].merge(
    sample[[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA",
        "NM_GRANDE_AREA_CONHECIMENTO"
    ]],
    on="ID_TRABALHO",
    how="left",
    validate="one_to_one"
)

assert index["ANO"].notna().all()
assert index["AREA_HARMONIZADA"].notna().all()
assert index["NM_GRANDE_AREA_CONHECIMENTO"].notna().all()

plan = plan.set_index(
    "FOCAL_ID",
    verify_integrity=True
)

index = index.sort_values(
    "FOCAL_ID"
).reset_index(drop=True)

expected_ids = [
    f"NAT-{i:04d}" for i in range(1, 109)
]

assert index["FOCAL_ID"].tolist() == expected_ids

# ------------------------------------------------------------
# 3. INDICADORES
# ------------------------------------------------------------

def overlap_statistics(list_a, list_b, k):

    a = set(list_a[:k])
    b = set(list_b[:k])

    assert len(a) == len(b) == k

    shared = len(a.intersection(b))
    union = len(a.union(b))

    return {
        "N_SHARED": shared,
        "OVERLAP_RATE": shared / k,
        "JACCARD": shared / union
    }


def get_ranking(scores, modality, window, focal_year, k):

    if window == "historical":
        data = scores
    elif window == "frontier3y":
        data = scores[
            scores["ANTECEDENT_YEAR"].astype(int).between(
                focal_year - 3,
                focal_year - 1
            )
        ]
    else:
        raise ValueError(window)

    column = f"COSINE_{modality}"

    assert len(data) >= k

    ranked = data.sort_values(
        [column, "ANTECEDENT_ID"],
        ascending=[False, True],
        kind="mergesort"
    ).head(k)

    return ranked[[
        "ANTECEDENT_ID",
        "ANTECEDENT_YEAR",
        column
    ]].copy()


def summarize_group(df, group_cols):

    return (
        df.groupby(
            group_cols,
            dropna=False
        )
        .agg(
            N_FOCALS=("FOCAL_ID", "nunique"),
            MEAN_OVERLAP=("OVERLAP_RATE", "mean"),
            MEDIAN_OVERLAP=("OVERLAP_RATE", "median"),
            SD_OVERLAP=("OVERLAP_RATE", "std"),
            MIN_OVERLAP=("OVERLAP_RATE", "min"),
            MAX_OVERLAP=("OVERLAP_RATE", "max"),
            MEAN_JACCARD=("JACCARD", "mean")
        )
        .reset_index()
    )

# ------------------------------------------------------------
# 4. PROCESSAR OS 108 FOCAIS
# ------------------------------------------------------------

comparisons = []
score_summaries = []
audited_rankings = 0

print("=" * 65)
print("ETAPA 22A — ESTABILIDADE NACIONAL")
print("=" * 65)

for focal in index.itertuples(index=False):

    fid = focal.FOCAL_ID
    year = int(focal.ANO)

    score_file = CKPT / f"{fid}_scores.parquet"
    rank_file = CKPT / f"{fid}_top30.parquet"

    assert score_file.is_file(), score_file
    assert rank_file.is_file(), rank_file

    scores = pd.read_parquet(score_file)
    saved_ranks = pd.read_parquet(rank_file)

    scores["ANTECEDENT_ID"] = (
        scores["ANTECEDENT_ID"].astype(str)
    )
    saved_ranks["ANTECEDENT_ID"] = (
        saved_ranks["ANTECEDENT_ID"].astype(str)
    )

    expected_h = int(
        plan.loc[fid, "N_HISTORICAL"]
    )

    expected_f = int(
        plan.loc[fid, "N_FRONTIER3Y"]
    )

    assert len(scores) == expected_h
    assert len(saved_ranks) == 120

    assert scores["ANTECEDENT_ID"].is_unique
    assert scores["FOCAL_ID"].eq(fid).all()

    assert scores["ANTECEDENT_YEAR"].astype(int).lt(
        year
    ).all()

    frontier = scores[
        scores["ANTECEDENT_YEAR"].astype(int).between(
            year - 3,
            year - 1
        )
    ]

    assert len(frontier) == expected_f

    # Calcular rankings ate Top 50 a partir
    # dos escores previamente persistidos.
    top50 = {}

    for window in WINDOWS:
        for modality in MODALITIES:

            ranking = get_ranking(
                scores, modality, window, year, 50
            )

            top50[(window, modality)] = ranking

            # Auditoria dos Top 30 originalmente salvos.
            saved = saved_ranks[
                (saved_ranks["WINDOW"] == window) &
                (saved_ranks["MODALITY"] == modality)
            ].sort_values("RANK")

            assert len(saved) == 30
            assert saved["RANK"].tolist() == list(
                range(1, 31)
            )

            assert (
                ranking["ANTECEDENT_ID"].head(30).tolist()
                == saved["ANTECEDENT_ID"].tolist()
            ), (
                f"Top30 divergente: {fid}, "
                f"{window}, {modality}"
            )

            assert np.allclose(
                ranking[
                    f"COSINE_{modality}"
                ].head(30).to_numpy(dtype=float),
                saved["SCORE"].to_numpy(dtype=float),
                atol=1e-6,
                rtol=0
            )

            audited_rankings += 1

            # Descritivas de similaridade.
            universe = (
                scores if window == "historical"
                else frontier
            )

            values = universe[
                f"COSINE_{modality}"
            ].to_numpy(dtype=float)

            assert np.isfinite(values).all()
            assert (
                (values >= -1.00001) &
                (values <= 1.00001)
            ).all()

            score_summaries.append({
                "FOCAL_ID": fid,
                "ANO": year,
                "GRANDE_AREA": (
                    focal.NM_GRANDE_AREA_CONHECIMENTO
                ),
                "AREA_HARMONIZADA": focal.AREA_HARMONIZADA,
                "WINDOW": window,
                "MODALITY": modality,
                "N_CANDIDATES": len(values),
                "COSINE_MAX": float(values.max()),
                "COSINE_MEAN": float(values.mean()),
                "COSINE_MEDIAN": float(np.median(values)),
                "COSINE_P95": float(
                    np.percentile(values, 95)
                ),
                "COSINE_TOP10_MEAN": float(
                    ranking[
                        f"COSINE_{modality}"
                    ].head(10).mean()
                ),
                "DISTANCE_TO_NEAREST": float(
                    1.0 - values.max()
                )
            })

    # --------------------------------------------------------
    # 5. Comparacoes de rankings por K
    # --------------------------------------------------------

    for k in K_VALUES:

        tests = [
            (
                "representation",
                "historical",
                "M2_vs_M4",
                ("historical", "M2"),
                ("historical", "M4")
            ),
            (
                "representation",
                "frontier3y",
                "M2_vs_M4",
                ("frontier3y", "M2"),
                ("frontier3y", "M4")
            ),
            (
                "temporal_window",
                "M2",
                "historical_vs_frontier3y",
                ("historical", "M2"),
                ("frontier3y", "M2")
            ),
            (
                "temporal_window",
                "M4",
                "historical_vs_frontier3y",
                ("historical", "M4"),
                ("frontier3y", "M4")
            )
        ]

        for (
            comparison_type,
            conditioning,
            comparison_label,
            key_a,
            key_b
        ) in tests:

            list_a = top50[key_a][
                "ANTECEDENT_ID"
            ].tolist()

            list_b = top50[key_b][
                "ANTECEDENT_ID"
            ].tolist()

            metrics = overlap_statistics(
                list_a, list_b, k
            )

            comparisons.append({
                "FOCAL_ID": fid,
                "ANO": year,
                "GRANDE_AREA": (
                    focal.NM_GRANDE_AREA_CONHECIMENTO
                ),
                "AREA_HARMONIZADA": focal.AREA_HARMONIZADA,
                "COMPARISON_TYPE": comparison_type,
                "CONDITION": conditioning,
                "COMPARISON": comparison_label,
                "K": k,
                "N_HISTORICAL": expected_h,
                "N_FRONTIER3Y": expected_f,
                **metrics
            })

    if int(fid[-4:]) % 12 == 0:
        print(
            f"{fid}: auditado | "
            f"rankings acumulados={audited_rankings}"
        )

# ------------------------------------------------------------
# 6. BASE NACIONAL
# ------------------------------------------------------------

results = pd.DataFrame(comparisons)
score_stats = pd.DataFrame(score_summaries)

assert results["FOCAL_ID"].nunique() == 108
assert len(results) == 108 * 3 * 4

assert len(score_stats) == 108 * 2 * 2
assert audited_rankings == 432

assert results["OVERLAP_RATE"].between(0, 1).all()
assert results["JACCARD"].between(0, 1).all()

# ------------------------------------------------------------
# 7. SINTESES NACIONAIS E DISCIPLINARES
# ------------------------------------------------------------

grouping = [
    "COMPARISON_TYPE",
    "CONDITION",
    "K"
]

national = summarize_group(
    results,
    grouping
)

by_area = summarize_group(
    results,
    ["GRANDE_AREA"] + grouping
)

by_year = summarize_group(
    results,
    ["ANO"] + grouping
)

# Comparacoes pareadas por focal
# sem inferencia estatistica prematura.
wide = results.pivot(
    index=["FOCAL_ID", "ANO", "GRANDE_AREA"],
    columns=[
        "COMPARISON_TYPE",
        "CONDITION",
        "K"
    ],
    values="OVERLAP_RATE"
)

wide.columns = [
    f"{a}__{b}__TOP{k}"
    for a, b, k in wide.columns
]

wide = wide.reset_index()

# ------------------------------------------------------------
# 8. SALVAR ARTEFATOS
# ------------------------------------------------------------

results.to_parquet(
    OUT / "focal_level_stability_1296.parquet",
    index=False
)

results.to_csv(
    OUT / "focal_level_stability_1296.csv",
    index=False
)

score_stats.to_parquet(
    OUT / "focal_score_distributions_432.parquet",
    index=False
)

national.to_csv(
    OUT / "national_overlap_summary.csv",
    index=False
)

by_area.to_csv(
    OUT / "overlap_by_grand_area.csv",
    index=False
)

by_year.to_csv(
    OUT / "overlap_by_year.csv",
    index=False
)

wide.to_csv(
    OUT / "focal_stability_wide.csv",
    index=False
)

manifest = {
    "stage": "22A",
    "n_focals": 108,
    "n_areas": int(
        results["GRANDE_AREA"].nunique()
    ),
    "comparisons_per_focal": 12,
    "n_comparisons": len(results),
    "n_score_summaries": len(score_stats),
    "saved_top30_rankings_reaudited": audited_rankings,
    "k_values": K_VALUES,
    "modalities": MODALITIES,
    "windows": WINDOWS,
    "ranking_tie_break": "ANTECEDENT_ID_ascending",
    "embeddings_recomputed": False,
    "similarities_recomputed": False,
    "human_validity_labels_available": False,
    "sampling_design_note": (
        "Balanced second-phase sample by nine grand areas; "
        "descriptive summaries are not design-weighted "
        "national population estimates."
    ),
    "historical_coverage_note": (
        "Antecedents before 2013 unavailable; "
        "historical window left-truncated."
    )
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 9. RESULTADOS PARA INTERPRETACAO
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ETAPA 22A — RESULTADO FINAL")
print("=" * 65)

print("Focais analisados:", results["FOCAL_ID"].nunique())
print("Comparacoes calculadas:", len(results))
print("Rankings Top30 reauditados:", audited_rankings)
print("Grandes areas:", results["GRANDE_AREA"].nunique())

print("\n=== CONCORDANCIA NACIONAL POR K ===")
print(
    national[[
        "COMPARISON_TYPE",
        "CONDITION",
        "K",
        "N_FOCALS",
        "MEAN_OVERLAP",
        "MEDIAN_OVERLAP",
        "SD_OVERLAP",
        "MIN_OVERLAP",
        "MAX_OVERLAP"
    ]].round(4).to_string(index=False)
)

print("\n=== REPRESENTACAO M2 vs M4 — TOP30 POR GRANDE AREA ===")

area_top30 = by_area[
    (by_area["COMPARISON_TYPE"] == "representation") &
    (by_area["K"] == 30)
]

print(
    area_top30[[
        "GRANDE_AREA",
        "CONDITION",
        "N_FOCALS",
        "MEAN_OVERLAP",
        "MEDIAN_OVERLAP"
    ]]
    .sort_values(["CONDITION", "MEAN_OVERLAP"])
    .round(4)
    .to_string(index=False)
)

print("\n=== SIMILARIDADE DO ANTECEDENTE MAIS PROXIMO ===")

print(
    score_stats.groupby(
        ["WINDOW", "MODALITY"]
    )["COSINE_MAX"]
    .agg(["mean", "median", "std", "min", "max"])
    .round(4)
    .to_string()
)

print("\nDiretorio:", OUT)
print("\nETAPA 22A CONCLUIDA")


ETAPA 22A — ESTABILIDADE NACIONAL
NAT-0012: auditado | rankings acumulados=48
NAT-0024: auditado | rankings acumulados=96
NAT-0036: auditado | rankings acumulados=144
NAT-0048: auditado | rankings acumulados=192
NAT-0060: auditado | rankings acumulados=240
NAT-0072: auditado | rankings acumulados=288
NAT-0084: auditado | rankings acumulados=336
NAT-0096: auditado | rankings acumulados=384
NAT-0108: auditado | rankings acumulados=432

ETAPA 22A — RESULTADO FINAL
Focais analisados: 108
Comparacoes calculadas: 1296
Rankings Top30 reauditados: 432
Grandes areas: 9

=== CONCORDANCIA NACIONAL POR K ===
COMPARISON_TYPE  CONDITION  K  N_FOCALS  MEAN_OVERLAP  MEDIAN_OVERLAP  SD_OVERLAP  MIN_OVERLAP  MAX_OVERLAP
 representation frontier3y 10       108        0.1963          0.2000      0.1713       0.0000         0.60
 representation frontier3y 30       108        0.2120          0.2000      0.1264       0.0000         0.50
 representation frontier3y 50       108        0.2261          0.2000   

In [41]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 22B
# ROBUSTEZ, TRUNCAMENTO TEMPORAL E CONCORDANCIA AO ACASO
#
# Usa exclusivamente os resultados salvos da Etapa 22A
# Sem recalcular embeddings, cossenos ou rankings
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import json

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SOURCE = (
    LAB / "05_national_expansion"
    / "22A_national_stability_20261020T203226_948841Z"
)

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "05_national_expansion"
    / f"22B_robustness_{STAMP}"
)
OUT.mkdir(parents=True, exist_ok=False)

# ------------------------------------------------------------
# 1. Carregamento e verificacoes
# ------------------------------------------------------------

df = pd.read_parquet(
    SOURCE / "focal_level_stability_1296.parquet"
)

assert len(df) == 1296
assert df["FOCAL_ID"].nunique() == 108
assert df["GRANDE_AREA"].nunique() == 9
assert set(df["K"]) == {10, 30, 50}

assert not df.duplicated([
    "FOCAL_ID",
    "COMPARISON_TYPE",
    "CONDITION",
    "K"
]).any()

df["ANO"] = df["ANO"].astype(int)

# ------------------------------------------------------------
# 2. Concordancia acima do acaso: representacao
# ------------------------------------------------------------

representation = df[
    df["COMPARISON_TYPE"] == "representation"
].copy()

# Para dois conjuntos aleatorios de tamanho k
# selecionados de N candidatos sem reposicao:
# E[|intersecao| / k] = k / N.
representation["N_UNIVERSE"] = np.where(
    representation["CONDITION"] == "historical",
    representation["N_HISTORICAL"],
    representation["N_FRONTIER3Y"]
)

representation["RANDOM_EXPECTED_OVERLAP"] = (
    representation["K"] /
    representation["N_UNIVERSE"]
)

representation["EXCESS_OVER_RANDOM"] = (
    representation["OVERLAP_RATE"] -
    representation["RANDOM_EXPECTED_OVERLAP"]
)

representation["CHANCE_ADJUSTED_OVERLAP"] = (
    representation["EXCESS_OVER_RANDOM"] /
    (1 - representation["RANDOM_EXPECTED_OVERLAP"])
)

assert (
    representation["N_UNIVERSE"] >=
    representation["K"]
).all()

assert representation[
    "RANDOM_EXPECTED_OVERLAP"
].between(0, 1).all()

# ------------------------------------------------------------
# 3. Comparacao de janelas com e sem 2016
# ------------------------------------------------------------

temporal = df[
    df["COMPARISON_TYPE"] == "temporal_window"
].copy()

temporal["FRONTIER_SHARE"] = (
    temporal["N_FRONTIER3Y"] /
    temporal["N_HISTORICAL"]
)

temporal["TRUNCATED_AT_2013"] = (
    temporal["ANO"] == 2016
)

# 2016 possui historico disponivel igual a fronteira.
checks_2016 = temporal[
    temporal["ANO"] == 2016
]

assert len(checks_2016) == 14 * 2 * 3

assert np.allclose(
    checks_2016["OVERLAP_RATE"].to_numpy(),
    1.0
)

assert np.allclose(
    checks_2016["FRONTIER_SHARE"].to_numpy(),
    1.0
)

def descriptive(data, grouping):
    return (
        data.groupby(grouping)
        .agg(
            N_FOCALS=("FOCAL_ID", "nunique"),
            MEAN=("OVERLAP_RATE", "mean"),
            MEDIAN=("OVERLAP_RATE", "median"),
            SD=("OVERLAP_RATE", "std")
        )
        .reset_index()
    )

temporal_all = descriptive(
    temporal,
    ["CONDITION", "K"]
)

temporal_post2016 = descriptive(
    temporal[temporal["ANO"] >= 2017],
    ["CONDITION", "K"]
)

temporal_all["SUBSAMPLE"] = "2016-2024"
temporal_post2016["SUBSAMPLE"] = "2017-2024"

temporal_sensitivity = pd.concat(
    [temporal_all, temporal_post2016],
    ignore_index=True
)

# ------------------------------------------------------------
# 4. Bootstrap estratificado por grande area
# ------------------------------------------------------------

# Reamostragem independente dos 12 focais
# dentro de cada uma das 9 grandes areas.
# Os mesmos IDs reamostrados sao utilizados
# em todas as condicoes de comparacao.

SEED = 20261020
B = 3000

rng = np.random.default_rng(SEED)

areas = sorted(df["GRANDE_AREA"].unique())

area_focals = {}

for area in areas:
    focal_ids = sorted(
        df.loc[
            df["GRANDE_AREA"] == area,
            "FOCAL_ID"
        ].unique()
    )
    assert len(focal_ids) == 12
    area_focals[area] = focal_ids

# Uma linha por focal e condicao de comparacao.
wide = df.pivot(
    index="FOCAL_ID",
    columns=[
        "COMPARISON_TYPE",
        "CONDITION",
        "K"
    ],
    values="OVERLAP_RATE"
)

assert wide.shape == (108, 12)
assert not wide.isna().any().any()

columns = wide.columns

# Matrix na ordem das grandes areas.
ordered_ids = [
    fid
    for area in areas
    for fid in area_focals[area]
]

values = wide.loc[ordered_ids].to_numpy(
    dtype=float
)

point_estimates = values.mean(axis=0)

bootstrap_estimates = np.empty(
    (B, values.shape[1]),
    dtype=np.float64
)

for b in range(B):

    chosen = np.concatenate([
        rng.choice(
            np.arange(a * 12, (a + 1) * 12),
            size=12,
            replace=True
        )
        for a in range(len(areas))
    ])

    bootstrap_estimates[b] = (
        values[chosen].mean(axis=0)
    )

ci_low = np.percentile(
    bootstrap_estimates, 2.5, axis=0
)

ci_high = np.percentile(
    bootstrap_estimates, 97.5, axis=0
)

bootstrap_results = pd.DataFrame([
    {
        "COMPARISON_TYPE": col[0],
        "CONDITION": col[1],
        "K": int(col[2]),
        "N_FOCALS": 108,
        "MEAN_OVERLAP": float(point_estimates[j]),
        "BOOT_CI_LOW": float(ci_low[j]),
        "BOOT_CI_HIGH": float(ci_high[j])
    }
    for j, col in enumerate(columns)
])

# ------------------------------------------------------------
# 5. Contrastes pareados de interesse
# ------------------------------------------------------------

def column_position(kind, condition, k):
    target = (kind, condition, k)
    return list(columns).index(target)

contrasts = []

for k in [10, 30, 50]:

    # M2 vs M4: fronteira menos historico
    j_f = column_position(
        "representation", "frontier3y", k
    )
    j_h = column_position(
        "representation", "historical", k
    )

    diff = values[:, j_f] - values[:, j_h]
    boot_diff = (
        bootstrap_estimates[:, j_f] -
        bootstrap_estimates[:, j_h]
    )

    contrasts.append({
        "CONTRAST": "representation_frontier_minus_historical",
        "K": k,
        "MEAN_PAIRED_DIFFERENCE": float(diff.mean()),
        "CI_LOW": float(np.percentile(boot_diff, 2.5)),
        "CI_HIGH": float(np.percentile(boot_diff, 97.5))
    })

    # Estabilidade temporal: M4 menos M2
    j_m4 = column_position(
        "temporal_window", "M4", k
    )
    j_m2 = column_position(
        "temporal_window", "M2", k
    )

    diff = values[:, j_m4] - values[:, j_m2]
    boot_diff = (
        bootstrap_estimates[:, j_m4] -
        bootstrap_estimates[:, j_m2]
    )

    contrasts.append({
        "CONTRAST": "temporal_M4_minus_M2",
        "K": k,
        "MEAN_PAIRED_DIFFERENCE": float(diff.mean()),
        "CI_LOW": float(np.percentile(
            bootstrap_estimates[:, j_m4] -
            bootstrap_estimates[:, j_m2],
            2.5
        )),
        "CI_HIGH": float(np.percentile(
            bootstrap_estimates[:, j_m4] -
            bootstrap_estimates[:, j_m2],
            97.5
        ))
    })

contrasts_df = pd.DataFrame(contrasts)

# ------------------------------------------------------------
# 6. Descritivas de sobreposicao acima do acaso
# ------------------------------------------------------------

chance_summary = (
    representation.groupby(["CONDITION", "K"])
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        OBSERVED_OVERLAP=("OVERLAP_RATE", "mean"),
        RANDOM_EXPECTED=("RANDOM_EXPECTED_OVERLAP", "mean"),
        EXCESS_OVER_RANDOM=("EXCESS_OVER_RANDOM", "mean"),
        CHANCE_ADJUSTED=("CHANCE_ADJUSTED_OVERLAP", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Salvar resultados
# ------------------------------------------------------------

representation.to_parquet(
    OUT / "representation_chance_adjusted.parquet",
    index=False
)

temporal_sensitivity.to_csv(
    OUT / "temporal_2016_sensitivity.csv",
    index=False
)

bootstrap_results.to_csv(
    OUT / "stratified_bootstrap_3000.csv",
    index=False
)

contrasts_df.to_csv(
    OUT / "paired_bootstrap_contrasts.csv",
    index=False
)

chance_summary.to_csv(
    OUT / "chance_adjusted_summary.csv",
    index=False
)

manifest = {
    "stage": "22B",
    "source": str(SOURCE),
    "n_focals": 108,
    "n_grand_areas": 9,
    "bootstrap_iterations": B,
    "bootstrap_seed": SEED,
    "bootstrap_method": (
        "Resample 12 focal documents within "
        "each of nine grand areas"
    ),
    "random_overlap_reference": "k/N",
    "temporal_truncation_exclusion": "ANO=2016",
    "inferential_caution": (
        "Intervals describe variability under the "
        "balanced second-phase sample; they are "
        "not design-based population confidence intervals."
    ),
    "human_validation_completed": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. Saida
# ------------------------------------------------------------

print("\n=== ETAPA 22B — RESULTADOS ===")

print("\n1. CONCORDANCIA M2/M4 VERSUS ACASO")
print(
    chance_summary.round(5).to_string(index=False)
)

print("\n2. ESTABILIDADE TEMPORAL COM E SEM 2016")
print(
    temporal_sensitivity[
        ["SUBSAMPLE", "CONDITION", "K",
         "N_FOCALS", "MEAN", "MEDIAN", "SD"]
    ].round(4).to_string(index=False)
)

print("\n3. BOOTSTRAP ESTRATIFICADO — 108 FOCAIS")
print(
    bootstrap_results.round(4).to_string(index=False)
)

print("\n4. CONTRASTES PAREADOS")
print(
    contrasts_df.round(4).to_string(index=False)
)

print("\nDiretorio:", OUT)
print("ETAPA 22B CONCLUIDA")



=== ETAPA 22B — RESULTADOS ===

1. CONCORDANCIA M2/M4 VERSUS ACASO
 CONDITION  K  N_FOCALS  OBSERVED_OVERLAP  RANDOM_EXPECTED  EXCESS_OVER_RANDOM  CHANCE_ADJUSTED
frontier3y 10       108           0.19630          0.00194             0.19435          0.19478
frontier3y 30       108           0.21204          0.00583             0.20621          0.20751
frontier3y 50       108           0.22611          0.00971             0.21640          0.21867
historical 10       108           0.19630          0.00108             0.19521          0.19545
historical 30       108           0.20247          0.00325             0.19922          0.19993
historical 50       108           0.21222          0.00541             0.20681          0.20803

2. ESTABILIDADE TEMPORAL COM E SEM 2016
SUBSAMPLE CONDITION  K  N_FOCALS   MEAN  MEDIAN     SD
2016-2024        M2 10       108 0.6213  0.6000 0.2605
2016-2024        M2 30       108 0.6083  0.6000 0.2362
2016-2024        M2 50       108 0.6030  0.6000 0.2306

In [42]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 22C
# DIAGNOSTICO DA GEOMETRIA DOS ESCORES SEMANTICOS
#
# 108 focais | M2 e M4 | Historico e fronteira 3Y
# CPU | Leitura de checkpoints congelados
# Nao gera embeddings nem recalcula similaridades
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"

SAMPLE_DIR = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

PLAN_FILE = (
    LAB / "05_national_expansion"
    / "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

RUN = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
)

CKPT = RUN / "checkpoints"

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    LAB / "05_national_expansion"
    / f"22C_score_geometry_{STAMP}"
)

OUT.mkdir(parents=True, exist_ok=False)

# ------------------------------------------------------------
# 1. Amostra congelada
# ------------------------------------------------------------

focals = pd.read_parquet(
    SAMPLE_DIR / "national_focals_108_RESTRICTED.parquet"
)

focals["ID_TRABALHO"] = (
    focals["ID_TRABALHO"].astype(str)
)

focals["ANO"] = focals["ANO"].astype(int)

mapping = pd.read_csv(
    SAMPLE_DIR / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

plan = pd.read_csv(
    PLAN_FILE,
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

assert len(focals) == len(mapping) == len(plan) == 108

assert focals["ID_TRABALHO"].is_unique
assert mapping["FOCAL_ID"].is_unique
assert mapping["ID_TRABALHO"].is_unique

focals = mapping[[
    "FOCAL_ID", "ID_TRABALHO"
]].merge(
    focals[[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA",
        "NM_GRANDE_AREA_CONHECIMENTO"
    ]],
    on="ID_TRABALHO",
    validate="one_to_one",
    how="left"
)

assert focals["ANO"].notna().all()
assert focals["AREA_HARMONIZADA"].notna().all()
assert focals["NM_GRANDE_AREA_CONHECIMENTO"].notna().all()

focals = focals.sort_values(
    "FOCAL_ID"
).reset_index(drop=True)

assert focals["FOCAL_ID"].tolist() == [
    f"NAT-{i:04d}" for i in range(1, 109)
]

plan = plan.set_index(
    "FOCAL_ID",
    verify_integrity=True
)

# ------------------------------------------------------------
# 2. Estatisticas de distribuicao por focal
# ------------------------------------------------------------

def distribution_metrics(values):

    x = np.asarray(values, dtype=np.float64)

    assert len(x) >= 50
    assert np.isfinite(x).all()

    assert ((x >= -1.00001) &
            (x <= 1.00001)).all()

    sorted_x = np.sort(x)

    top1 = float(sorted_x[-1])
    top2 = float(sorted_x[-2])

    p = np.percentile(
        x, [1, 5, 25, 50, 75, 95, 99]
    )

    p01, p05, p25, p50, p75, p95, p99 = map(
        float, p
    )

    mean = float(x.mean())
    sd = float(x.std(ddof=1))

    top10_mean = float(
        sorted_x[-10:].mean()
    )

    top30_mean = float(
        sorted_x[-30:].mean()
    )

    # Proporcoes sob limiares descritivos.
    # Nao correspondem a categorias de relevancia.
    above_090 = float(np.mean(x >= 0.90))
    above_095 = float(np.mean(x >= 0.95))

    return {
        "N_CANDIDATES": len(x),
        "COSINE_MEAN": mean,
        "COSINE_SD": sd,
        "COSINE_MIN": float(sorted_x[0]),
        "COSINE_P01": p01,
        "COSINE_P05": p05,
        "COSINE_P25": p25,
        "COSINE_MEDIAN": p50,
        "COSINE_P75": p75,
        "COSINE_P95": p95,
        "COSINE_P99": p99,
        "COSINE_MAX": top1,
        "COSINE_TOP2": top2,
        "TOP10_MEAN": top10_mean,
        "TOP30_MEAN": top30_mean,
        "GAP_TOP1_TOP2": top1 - top2,
        "GAP_TOP1_P99": top1 - p99,
        "GAP_TOP1_MEDIAN": top1 - p50,
        "IQR": p75 - p25,
        "P95_MINUS_P05": p95 - p05,
        "STANDARDIZED_TOP1": (
            (top1 - mean) / sd
            if sd > 0 else np.nan
        ),
        "SHARE_GE_090": above_090,
        "SHARE_GE_095": above_095
    }

# ------------------------------------------------------------
# 3. Ler scores congelados
# ------------------------------------------------------------

records = []
pair_counts = {}

print("=" * 66)
print("ETAPA 22C — DIAGNOSTICO DAS SIMILARIDADES")
print("=" * 66)

for focal in focals.itertuples(index=False):

    fid = focal.FOCAL_ID
    year = int(focal.ANO)

    path = CKPT / f"{fid}_scores.parquet"

    assert path.is_file(), path

    scores = pd.read_parquet(
        path,
        columns=[
            "FOCAL_ID",
            "FOCAL_YEAR",
            "ANTECEDENT_ID",
            "ANTECEDENT_YEAR",
            "COSINE_M2",
            "COSINE_M4"
        ]
    )

    assert scores["FOCAL_ID"].eq(fid).all()
    assert scores["FOCAL_YEAR"].astype(int).eq(year).all()
    assert scores["ANTECEDENT_ID"].is_unique

    ant_years = (
        scores["ANTECEDENT_YEAR"].astype(int)
    )

    assert ant_years.lt(year).all()

    historical_n = int(
        plan.loc[fid, "N_HISTORICAL"]
    )

    frontier_n = int(
        plan.loc[fid, "N_FRONTIER3Y"]
    )

    assert len(scores) == historical_n

    frontier_mask = ant_years.between(
        year - 3, year - 1
    )

    assert int(frontier_mask.sum()) == frontier_n

    pair_counts[fid] = historical_n

    for window, subset in [
        ("historical", scores),
        (
            "frontier3y",
            scores.loc[frontier_mask]
        )
    ]:

        for modality in ["M2", "M4"]:

            values = subset[
                f"COSINE_{modality}"
            ].to_numpy(dtype=np.float64)

            metrics = distribution_metrics(values)

            records.append({
                "FOCAL_ID": fid,
                "ANO": year,
                "GRANDE_AREA": (
                    focal.NM_GRANDE_AREA_CONHECIMENTO
                ),
                "AREA_HARMONIZADA": (
                    focal.AREA_HARMONIZADA
                ),
                "WINDOW": window,
                "MODALITY": modality,
                **metrics
            })

    if int(fid[-4:]) % 12 == 0:
        print(
            f"{fid}: processado | "
            f"{len(records)} distribuicoes"
        )

# ------------------------------------------------------------
# 4. Consolidar indicadores
# ------------------------------------------------------------

result = pd.DataFrame(records)

assert len(result) == 108 * 2 * 2
assert result["FOCAL_ID"].nunique() == 108

assert sum(pair_counts.values()) == 1640271

assert result["COSINE_MAX"].ge(
    result["COSINE_P99"]
).all()

assert result["COSINE_P99"].ge(
    result["COSINE_MEDIAN"]
).all()

assert result["GAP_TOP1_TOP2"].ge(-1e-7).all()
assert result["IQR"].ge(0).all()

# ------------------------------------------------------------
# 5. Estatisticas nacionais por janela e modalidade
# ------------------------------------------------------------

metrics_of_interest = [
    "COSINE_MEAN",
    "COSINE_SD",
    "COSINE_MEDIAN",
    "COSINE_P95",
    "COSINE_P99",
    "COSINE_MAX",
    "GAP_TOP1_TOP2",
    "GAP_TOP1_P99",
    "GAP_TOP1_MEDIAN",
    "STANDARDIZED_TOP1",
    "SHARE_GE_090",
    "SHARE_GE_095"
]

national = (
    result.groupby(
        ["WINDOW", "MODALITY"]
    )[metrics_of_interest]
    .agg(["mean", "median", "std"])
)

national.columns = [
    f"{metric}_{stat}".upper()
    for metric, stat in national.columns
]

national = national.reset_index()

# ------------------------------------------------------------
# 6. Diferencas pareadas de escores M4 - M2
# ------------------------------------------------------------

paired = result.pivot(
    index=[
        "FOCAL_ID",
        "ANO",
        "GRANDE_AREA",
        "WINDOW"
    ],
    columns="MODALITY",
    values=[
        "COSINE_MEAN",
        "COSINE_SD",
        "COSINE_MEDIAN",
        "COSINE_MAX",
        "GAP_TOP1_TOP2",
        "GAP_TOP1_P99",
        "GAP_TOP1_MEDIAN",
        "STANDARDIZED_TOP1"
    ]
)

paired.columns = [
    f"{metric}_{modality}"
    for metric, modality in paired.columns
]

paired = paired.reset_index()

for metric in [
    "COSINE_MEAN",
    "COSINE_SD",
    "COSINE_MEDIAN",
    "COSINE_MAX",
    "GAP_TOP1_TOP2",
    "GAP_TOP1_P99",
    "GAP_TOP1_MEDIAN",
    "STANDARDIZED_TOP1"
]:
    paired[f"DELTA_{metric}_M4_MINUS_M2"] = (
        paired[f"{metric}_M4"]
        - paired[f"{metric}_M2"]
    )

# ------------------------------------------------------------
# 7. Heterogeneidade disciplinar
# ------------------------------------------------------------

by_area = (
    result.groupby(
        ["GRANDE_AREA", "WINDOW", "MODALITY"]
    )
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_COSINE_MEDIAN=("COSINE_MEDIAN", "mean"),
        MEAN_COSINE_MAX=("COSINE_MAX", "mean"),
        MEAN_COSINE_SD=("COSINE_SD", "mean"),
        MEAN_GAP_TOP1_P99=("GAP_TOP1_P99", "mean"),
        MEAN_GAP_TOP1_MEDIAN=(
            "GAP_TOP1_MEDIAN", "mean"
        ),
        MEAN_STANDARDIZED_TOP1=(
            "STANDARDIZED_TOP1", "mean"
        )
    )
    .reset_index()
)

assert by_area["N_FOCALS"].eq(12).all()

# ------------------------------------------------------------
# 8. Relacionar geometria com estabilidade M2/M4
# ------------------------------------------------------------

STABILITY_SOURCE = (
    LAB / "05_national_expansion"
    / "22A_national_stability_20261020T203226_948841Z"
    / "focal_level_stability_1296.parquet"
)

stability = pd.read_parquet(STABILITY_SOURCE)

top30 = stability[
    (stability["COMPARISON_TYPE"] == "representation")
    & (stability["K"] == 30)
][[
    "FOCAL_ID",
    "CONDITION",
    "OVERLAP_RATE"
]].rename(columns={
    "CONDITION": "WINDOW",
    "OVERLAP_RATE": "M2_M4_OVERLAP_TOP30"
})

geometry_overlap = paired.merge(
    top30,
    on=["FOCAL_ID", "WINDOW"],
    how="inner",
    validate="one_to_one"
)

assert len(geometry_overlap) == 216

# Correlacoes exploratorias.
# Nao sao estimativas causais.
corr_rows = []

for window, subset in geometry_overlap.groupby("WINDOW"):

    for variable in [
        "COSINE_SD_M2",
        "COSINE_SD_M4",
        "GAP_TOP1_P99_M2",
        "GAP_TOP1_P99_M4",
        "GAP_TOP1_MEDIAN_M2",
        "GAP_TOP1_MEDIAN_M4",
        "DELTA_COSINE_MAX_M4_MINUS_M2"
    ]:

        rho = subset[
            [variable, "M2_M4_OVERLAP_TOP30"]
        ].corr(method="spearman").iloc[0, 1]

        corr_rows.append({
            "WINDOW": window,
            "VARIABLE": variable,
            "SPEARMAN_RHO": float(rho),
            "N_FOCALS": len(subset)
        })

correlations = pd.DataFrame(corr_rows)

# ------------------------------------------------------------
# 9. Salvar artefatos
# ------------------------------------------------------------

result.to_parquet(
    OUT / "focal_geometry_432.parquet",
    index=False
)

result.to_csv(
    OUT / "focal_geometry_432.csv",
    index=False
)

national.to_csv(
    OUT / "national_geometry_summary.csv",
    index=False
)

paired.to_csv(
    OUT / "paired_M2_M4_geometry.csv",
    index=False
)

by_area.to_csv(
    OUT / "geometry_by_grand_area.csv",
    index=False
)

geometry_overlap.to_csv(
    OUT / "geometry_vs_overlap_top30.csv",
    index=False
)

correlations.to_csv(
    OUT / "geometry_overlap_spearman.csv",
    index=False
)

manifest = {
    "stage": "22C",
    "n_focals": 108,
    "n_historical_pairs": 1640271,
    "n_metric_rows": len(result),
    "modalities": ["M2", "M4"],
    "windows": ["historical", "frontier3y"],
    "recomputed_embeddings": False,
    "recomputed_cosine_scores": False,
    "diagnostic_scope": (
        "Distributions of focal-to-eligible-antecedent "
        "cosines; not an independent estimate of "
        "global embedding anisotropy."
    ),
    "threshold_warning": (
        "Cosine cutoffs 0.90 and 0.95 are descriptive, "
        "not validated relevance thresholds."
    ),
    "human_validation_completed": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 10. Saidas para interpretacao
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("ETAPA 22C — RESULTADOS")
print("=" * 68)

print("\n1. DISTRIBUICAO DOS ESCORES POR MODALIDADE")
print(
    result.groupby(
        ["WINDOW", "MODALITY"]
    )[[
        "COSINE_MEAN",
        "COSINE_MEDIAN",
        "COSINE_SD",
        "COSINE_P99",
        "COSINE_MAX"
    ]]
    .mean()
    .round(5)
    .to_string()
)

print("\n2. SEPARACAO DOS ANTECEDENTES MAIS PROXIMOS")
print(
    result.groupby(
        ["WINDOW", "MODALITY"]
    )[[
        "GAP_TOP1_TOP2",
        "GAP_TOP1_P99",
        "GAP_TOP1_MEDIAN",
        "STANDARDIZED_TOP1"
    ]]
    .mean()
    .round(5)
    .to_string()
)

print("\n3. DIFERENCAS PAREADAS M4 - M2")
delta_cols = [
    "DELTA_COSINE_MEAN_M4_MINUS_M2",
    "DELTA_COSINE_MAX_M4_MINUS_M2",
    "DELTA_COSINE_SD_M4_MINUS_M2",
    "DELTA_GAP_TOP1_MEDIAN_M4_MINUS_M2"
]

print(
    paired.groupby("WINDOW")[delta_cols]
    .mean()
    .round(5)
    .to_string()
)

print("\n4. GEOMETRIA E CONCORDANCIA TOP30")
print(
    correlations.sort_values(
        ["WINDOW", "VARIABLE"]
    )
    .round(4)
    .to_string(index=False)
)

print("\n5. CONTROLE DE INTEGRIDADE")
print("Focais:", result["FOCAL_ID"].nunique())
print("Distribuicoes:", len(result))
print("Pares historicos:", sum(pair_counts.values()))
print("Status: PASS")

print("\nDiretorio:", OUT)
print("\nETAPA 22C CONCLUIDA")


ETAPA 22C — DIAGNOSTICO DAS SIMILARIDADES
NAT-0012: processado | 48 distribuicoes
NAT-0024: processado | 96 distribuicoes
NAT-0036: processado | 144 distribuicoes
NAT-0048: processado | 192 distribuicoes
NAT-0060: processado | 240 distribuicoes
NAT-0072: processado | 288 distribuicoes
NAT-0084: processado | 336 distribuicoes
NAT-0096: processado | 384 distribuicoes
NAT-0108: processado | 432 distribuicoes

ETAPA 22C — RESULTADOS

1. DISTRIBUICAO DOS ESCORES POR MODALIDADE
                     COSINE_MEAN  COSINE_MEDIAN  COSINE_SD  COSINE_P99  COSINE_MAX
WINDOW     MODALITY                                                               
frontier3y M2            0.88032        0.88066    0.01360     0.91145     0.93705
           M4            0.92102        0.92217    0.01265     0.94555     0.95971
historical M2            0.87967        0.88003    0.01372     0.91090     0.94002
           M4            0.92037        0.92159    0.01288     0.94513     0.96174

2. SEPARACAO DOS ANTECED

In [43]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 22D
# REFERENCIA SEMANTICA COM CONTROLE TEMPORAL
#
# 108 focais
# 200 antecedentes da mesma area por focal
# 200 antecedentes de outras areas por focal
# Anos pareados exatamente
#
# M2 e M4 | CPU | Embeddings congelados
# Checkpoints individuais e retomada segura
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = Path("/content/gdrive/MyDrive/CAPES_Epistemic_Change")
LAB = BASE / "08_epistemic_measurement_validation"
EMB = BASE / "07_embeddings_nacionais"

SAMPLE_DIR = (
    LAB / "05_national_expansion"
    / "focal_sample_108_20261020T190807_559853Z"
)

INDEX_FILE = (
    BASE / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

SCORES_DIR = (
    LAB / "05_national_expansion"
    / "21_semantic_controlled_national_v1"
    / "checkpoints"
)

OUT = (
    LAB / "05_national_expansion"
    / "22D_temporally_matched_reference_v1"
)

PARTS = OUT / "focal_checkpoints"
PARTS.mkdir(parents=True, exist_ok=True)

SEED = 20261020
N_PER_GROUP = 200
BLOCK = 2048

CONFIG = {
    "stage": "22D",
    "sample_dir": str(SAMPLE_DIR),
    "seed": SEED,
    "n_per_group": N_PER_GROUP,
    "matching": "exact_antecedent_year",
    "same_area_scores_reused": True,
    "cross_area_rule": "AREA_HARMONIZADA differs",
    "modalities": ["M2", "M4"],
    "embedding_dimension": 768
}

config_path = OUT / "configuration.json"

if config_path.exists():
    assert json.loads(
        config_path.read_text(encoding="utf-8")
    ) == CONFIG
else:
    config_path.write_text(
        json.dumps(CONFIG, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 1. Focais e metadados
# ------------------------------------------------------------

focals = pd.read_csv(
    SAMPLE_DIR / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 108
assert focals["FOCAL_ID"].is_unique

focals = focals.sort_values(
    "FOCAL_ID"
).reset_index(drop=True)

assert focals["FOCAL_ID"].tolist() == [
    f"NAT-{i:04d}" for i in range(1, 109)
]

meta = pq.read_table(
    INDEX_FILE,
    columns=[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA"
    ]
).to_pandas()

meta["ID_TRABALHO"] = meta["ID_TRABALHO"].astype(str)
meta["ANO"] = meta["ANO"].astype(int)

assert len(meta) == 993086
assert meta["ID_TRABALHO"].is_unique

area_lookup = meta.set_index(
    "ID_TRABALHO",
    verify_integrity=True
)["AREA_HARMONIZADA"]

focals["AREA_HARMONIZADA"] = (
    focals["ID_TRABALHO"].map(area_lookup)
)

assert focals["AREA_HARMONIZADA"].notna().all()

year_pools = {
    int(year): group[[
        "ID_TRABALHO",
        "AREA_HARMONIZADA"
    ]].reset_index(drop=True)
    for year, group in meta.groupby("ANO")
}

# ------------------------------------------------------------
# 2. Acesso aos embeddings congelados
# ------------------------------------------------------------

id_cache = {}
vector_cache = {}

def annual_ids(year, modality):

    key = (int(year), modality)

    if key not in id_cache:

        path = (
            EMB / modality
            / f"{modality}_{year}_ids.parquet"
        )

        ids = pd.read_parquet(
            path,
            columns=["ID_TRABALHO"]
        )["ID_TRABALHO"].astype(str)

        assert ids.is_unique
        id_cache[key] = pd.Index(ids)

    return id_cache[key]


def annual_vectors(year, modality):

    key = (int(year), modality)

    if key not in vector_cache:

        path = (
            EMB / modality
            / f"{modality}_{year}_embeddings_float16.npy"
        )

        arr = np.load(path, mmap_mode="r")

        assert arr.ndim == 2
        assert arr.shape[1] == 768
        assert len(arr) == len(
            annual_ids(year, modality)
        )

        vector_cache[key] = arr

    return vector_cache[key]


def get_query(doc_id, year, modality):

    position = annual_ids(
        year, modality
    ).get_loc(str(doc_id))

    vector = np.array(
        annual_vectors(year, modality)[position],
        dtype=np.float32,
        copy=True
    )

    norm = np.linalg.norm(vector)

    assert np.isfinite(norm) and norm > 0

    return vector / norm


def calculate_scores(ids, year, modality, query):

    positions = annual_ids(
        year, modality
    ).get_indexer([str(x) for x in ids])

    assert (positions >= 0).all()

    vectors = annual_vectors(year, modality)

    output = np.empty(
        len(positions),
        dtype=np.float32
    )

    for first in range(0, len(positions), BLOCK):

        last = min(
            first + BLOCK,
            len(positions)
        )

        block = np.array(
            vectors[positions[first:last]],
            dtype=np.float32,
            copy=True
        )

        norms = np.linalg.norm(block, axis=1)

        assert np.isfinite(norms).all()
        assert (norms > 0).all()

        block /= norms[:, None]
        output[first:last] = block @ query

    assert np.isfinite(output).all()

    return output

# ------------------------------------------------------------
# 3. Selecao pareada por focal
# ------------------------------------------------------------

def focal_rng(fid):

    digest = hashlib.sha256(
        f"{SEED}:{fid}".encode("utf-8")
    ).digest()

    seed_value = int.from_bytes(
        digest[:8], "little"
    )

    return np.random.default_rng(seed_value)


def make_focal_reference(focal):

    fid = focal.FOCAL_ID
    year = int(focal.ANO)
    area = str(focal.AREA_HARMONIZADA)

    rng = focal_rng(fid)

    original = pd.read_parquet(
        SCORES_DIR / f"{fid}_scores.parquet",
        columns=[
            "ANTECEDENT_ID",
            "ANTECEDENT_YEAR",
            "COSINE_M2",
            "COSINE_M4"
        ]
    )

    original["ANTECEDENT_ID"] = (
        original["ANTECEDENT_ID"].astype(str)
    )

    assert original["ANTECEDENT_ID"].is_unique
    assert len(original) >= N_PER_GROUP

    # Amostra da mesma area: recuperar
    # pontuacoes ja calculadas na Etapa 21.
    indices = rng.choice(
        len(original),
        size=N_PER_GROUP,
        replace=False
    )

    same = original.iloc[indices].copy()

    same = same.rename(columns={
        "ANTECEDENT_ID": "DOCUMENT_ID",
        "ANTECEDENT_YEAR": "DOCUMENT_YEAR"
    })

    same["GROUP"] = "same_area"

    cross_parts = []

    # Para cada ano, igualar exatamente
    # o numero de observacoes do grupo same_area.
    for ant_year, group in same.groupby(
        "DOCUMENT_YEAR", sort=True
    ):

        count = len(group)

        eligible = year_pools[int(ant_year)]

        eligible = eligible[
            eligible["AREA_HARMONIZADA"] != area
        ]

        assert len(eligible) >= count

        chosen = rng.choice(
            len(eligible),
            size=count,
            replace=False
        )

        cross_part = eligible.iloc[
            chosen
        ][["ID_TRABALHO"]].copy()

        cross_part = cross_part.rename(columns={
            "ID_TRABALHO": "DOCUMENT_ID"
        })

        cross_part["DOCUMENT_YEAR"] = int(ant_year)
        cross_part["GROUP"] = "other_area"

        cross_parts.append(cross_part)

    cross = pd.concat(
        cross_parts,
        ignore_index=True
    )

    assert len(same) == N_PER_GROUP
    assert len(cross) == N_PER_GROUP

    assert same["DOCUMENT_ID"].is_unique
    assert cross["DOCUMENT_ID"].is_unique

    assert set(same["DOCUMENT_ID"]).isdisjoint(
        set(cross["DOCUMENT_ID"])
    )

    same_year_counts = same[
        "DOCUMENT_YEAR"
    ].value_counts().sort_index()

    cross_year_counts = cross[
        "DOCUMENT_YEAR"
    ].value_counts().sort_index()

    assert same_year_counts.equals(
        cross_year_counts
    )

    # Pontuar apenas os documentos de outras areas.
    for modality in ["M2", "M4"]:

        query = get_query(
            focal.ID_TRABALHO,
            year,
            modality
        )

        col = f"COSINE_{modality}"
        cross[col] = np.nan

        for ant_year, group in cross.groupby(
            "DOCUMENT_YEAR", sort=True
        ):

            positions = group.index.to_numpy()

            cross.loc[
                positions, col
            ] = calculate_scores(
                group["DOCUMENT_ID"].tolist(),
                int(ant_year),
                modality,
                query
            )

        assert cross[col].notna().all()

    result = pd.concat(
        [
            same[[
                "DOCUMENT_ID",
                "DOCUMENT_YEAR",
                "GROUP",
                "COSINE_M2",
                "COSINE_M4"
            ]],
            cross[[
                "DOCUMENT_ID",
                "DOCUMENT_YEAR",
                "GROUP",
                "COSINE_M2",
                "COSINE_M4"
            ]]
        ],
        ignore_index=True
    )

    result["FOCAL_ID"] = fid
    result["FOCAL_YEAR"] = year
    result["FOCAL_AREA"] = area

    assert len(result) == 400
    assert result["DOCUMENT_ID"].is_unique

    assert result["DOCUMENT_YEAR"].astype(int).lt(
        year
    ).all()

    for col in ["COSINE_M2", "COSINE_M4"]:
        values = result[col].to_numpy(dtype=float)

        assert np.isfinite(values).all()
        assert ((values >= -1.00001) &
                (values <= 1.00001)).all()

    return result

# ------------------------------------------------------------
# 4. Processamento retomavel
# ------------------------------------------------------------

print("=" * 65)
print("ETAPA 22D — REFERENCIA SEMANTICA TEMPORALMENTE PAREADA")
print("=" * 65)

new = 0
reused = 0

for focal in focals.itertuples(index=False):

    fid = focal.FOCAL_ID

    output = PARTS / f"{fid}_reference.parquet"
    temporary = PARTS / f"{fid}_reference.partial.parquet"

    if output.is_file():

        existing = pd.read_parquet(output)

        assert len(existing) == 400
        assert existing["FOCAL_ID"].eq(fid).all()
        assert existing["DOCUMENT_ID"].is_unique

        assert existing["GROUP"].value_counts().to_dict() == {
            "same_area": 200,
            "other_area": 200
        }

        reused += 1
        continue

    if temporary.exists():
        raise RuntimeError(
            f"Arquivo temporario preexistente: {fid}"
        )

    reference = make_focal_reference(focal)

    reference.to_parquet(
        temporary,
        index=False
    )

    assert (
        pq.ParquetFile(temporary).metadata.num_rows
        == 400
    )

    temporary.rename(output)

    new += 1

    if int(fid[-4:]) % 12 == 0:
        print(
            f"{fid}: checkpoint concluido "
            f"| novos={new} | reutilizados={reused}"
        )

    vector_cache.clear()
    gc.collect()

# ------------------------------------------------------------
# 5. Consolidar amostra e auditar pareamento
# ------------------------------------------------------------

pieces = []

for fid in focals["FOCAL_ID"]:

    path = PARTS / f"{fid}_reference.parquet"

    assert path.is_file()

    part = pd.read_parquet(path)
    assert len(part) == 400

    assert part["GROUP"].value_counts().to_dict() == {
        "same_area": 200,
        "other_area": 200
    }

    year_table = part.groupby(
        ["DOCUMENT_YEAR", "GROUP"]
    ).size().unstack(fill_value=0)

    assert (
        year_table["same_area"]
        == year_table["other_area"]
    ).all()

    part_areas = part[
        "DOCUMENT_ID"
    ].astype(str).map(area_lookup)

    assert part_areas.notna().all()

    assert (
        part_areas[
            part["GROUP"] == "same_area"
        ] == part.loc[
            part["GROUP"] == "same_area",
            "FOCAL_AREA"
        ]
    ).all()

    assert (
        part_areas[
            part["GROUP"] == "other_area"
        ] != part.loc[
            part["GROUP"] == "other_area",
            "FOCAL_AREA"
        ]
    ).all()

    pieces.append(part)

all_pairs = pd.concat(
    pieces,
    ignore_index=True
)

assert len(all_pairs) == 108 * 400
assert all_pairs["FOCAL_ID"].nunique() == 108

# ------------------------------------------------------------
# 6. Estatisticas pareadas
# ------------------------------------------------------------

long = all_pairs.melt(
    id_vars=[
        "FOCAL_ID",
        "FOCAL_YEAR",
        "FOCAL_AREA",
        "DOCUMENT_ID",
        "DOCUMENT_YEAR",
        "GROUP"
    ],
    value_vars=["COSINE_M2", "COSINE_M4"],
    var_name="MODALITY",
    value_name="COSINE"
)

long["MODALITY"] = long[
    "MODALITY"
].str.replace("COSINE_", "", regex=False)

focal_summary = (
    long.groupby(
        ["FOCAL_ID", "MODALITY", "GROUP"]
    )
    .agg(
        N=("COSINE", "size"),
        MEAN=("COSINE", "mean"),
        MEDIAN=("COSINE", "median"),
        SD=("COSINE", "std"),
        P95=("COSINE", lambda x: x.quantile(0.95))
    )
    .reset_index()
)

assert focal_summary["N"].eq(200).all()

wide = focal_summary.pivot(
    index=["FOCAL_ID", "MODALITY"],
    columns="GROUP",
    values=["MEAN", "MEDIAN", "SD", "P95"]
)

wide.columns = [
    f"{metric}_{group}"
    for metric, group in wide.columns
]

wide = wide.reset_index()

for metric in ["MEAN", "MEDIAN", "SD", "P95"]:
    wide[f"DELTA_{metric}_SAME_MINUS_OTHER"] = (
        wide[f"{metric}_same_area"]
        - wide[f"{metric}_other_area"]
    )

assert len(wide) == 216

summary = (
    focal_summary.groupby(
        ["MODALITY", "GROUP"]
    )
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_COSINE=("MEAN", "mean"),
        MEAN_WITHIN_FOCAL_SD=("SD", "mean"),
        MEAN_P95=("P95", "mean")
    )
    .reset_index()
)

paired_summary = (
    wide.groupby("MODALITY")
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_DELTA=(
            "DELTA_MEAN_SAME_MINUS_OTHER", "mean"
        ),
        MEDIAN_DELTA=(
            "DELTA_MEAN_SAME_MINUS_OTHER", "median"
        ),
        SD_DELTA=(
            "DELTA_MEAN_SAME_MINUS_OTHER", "std"
        ),
        SHARE_POSITIVE_DELTA=(
            "DELTA_MEAN_SAME_MINUS_OTHER",
            lambda x: float(np.mean(x > 0))
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Salvar produtos imutaveis
# ------------------------------------------------------------

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

REPORT = OUT / f"analysis_{stamp}"
REPORT.mkdir(parents=True, exist_ok=False)

all_pairs.to_parquet(
    REPORT / "matched_reference_pairs_43200.parquet",
    index=False
)

focal_summary.to_csv(
    REPORT / "focal_group_statistics.csv",
    index=False
)

wide.to_csv(
    REPORT / "paired_focal_contrasts.csv",
    index=False
)

summary.to_csv(
    REPORT / "national_reference_summary.csv",
    index=False
)

paired_summary.to_csv(
    REPORT / "national_paired_contrasts.csv",
    index=False
)

manifest = {
    "stage": "22D",
    "n_focals": 108,
    "n_per_group_per_focal": 200,
    "n_pairs": len(all_pairs),
    "modalities": ["M2", "M4"],
    "random_seed": SEED,
    "temporal_matching": "exact antecedent year",
    "same_area_score_source": str(SCORES_DIR),
    "same_area_scores_recomputed": False,
    "cross_area_scores_newly_computed": True,
    "new_embedding_inference": False,
    "interpretation_limit": (
        "Other-area documents are not human-validated "
        "irrelevant documents. The experiment provides "
        "a temporally matched empirical baseline, "
        "not proof of global embedding anisotropy."
    )
}

(REPORT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. Resultados
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ETAPA 22D — RESULTADO FINAL")
print("=" * 65)

print("Focais:", all_pairs["FOCAL_ID"].nunique())
print("Pares:", len(all_pairs))
print("Novos checkpoints:", new)
print("Checkpoints reutilizados:", reused)

print("\n1. SIMILARIDADE POR GRUPO")
print(
    summary.round(5).to_string(index=False)
)

print("\n2. CONTRASTES PAREADOS")
print(
    paired_summary.round(5).to_string(index=False)
)

print("\n3. DISTRIBUICAO DOS CONTRASTES")
print(
    wide.groupby("MODALITY")[
        "DELTA_MEAN_SAME_MINUS_OTHER"
    ]
    .agg(["min", "median", "max"])
    .round(5)
    .to_string()
)

print("\nPareamento temporal: PASS")
print("Integridade documental: PASS")
print("Diretorio:", REPORT)
print("\nETAPA 22D CONCLUIDA")


ETAPA 22D — REFERENCIA SEMANTICA TEMPORALMENTE PAREADA
NAT-0012: checkpoint concluido | novos=12 | reutilizados=0
NAT-0024: checkpoint concluido | novos=24 | reutilizados=0
NAT-0036: checkpoint concluido | novos=36 | reutilizados=0
NAT-0048: checkpoint concluido | novos=48 | reutilizados=0
NAT-0060: checkpoint concluido | novos=60 | reutilizados=0
NAT-0072: checkpoint concluido | novos=72 | reutilizados=0
NAT-0084: checkpoint concluido | novos=84 | reutilizados=0
NAT-0096: checkpoint concluido | novos=96 | reutilizados=0
NAT-0108: checkpoint concluido | novos=108 | reutilizados=0

ETAPA 22D — RESULTADO FINAL
Focais: 108
Pares: 43200
Novos checkpoints: 108
Checkpoints reutilizados: 0

1. SIMILARIDADE POR GRUPO
MODALITY      GROUP  N_FOCALS  MEAN_COSINE  MEAN_WITHIN_FOCAL_SD  MEAN_P95
      M2 other_area       108      0.86768               0.01450   0.89056
      M2  same_area       108      0.87964               0.01370   0.90119
      M4 other_area       108      0.90611              

In [7]:

# ============================================================
# RECUPERACAO SEGURA — GOOGLE DRIVE / CAPES
#
# Nao exclui arquivos
# Nao desmonta o Drive atual
# Nao recalcula resultados
# Localiza os dados da Etapa 22D
# ============================================================

from pathlib import Path
from google.colab import drive

PROJECT = "CAPES_Epistemic_Change"
LAB_NAME = "08_epistemic_measurement_validation"

MOUNTS = [
    Path("/content/gdrive"),
    Path("/content/drive"),
    Path("/content/drive_capes_recovery")
]

def find_project(mount):
    candidates = [
        mount / "MyDrive" / PROJECT,
        mount / "My Drive" / PROJECT
    ]
    for candidate in candidates:
        if (
            candidate.is_dir()
            and (candidate / LAB_NAME).is_dir()
        ):
            return candidate
    return None

print("=== 1. PROCURANDO MONTAGENS EXISTENTES ===")

root = None

for mount in MOUNTS:
    print(
        "Montagem:",
        mount,
        "| Existe:",
        mount.exists()
    )

    candidate = find_project(mount)

    if candidate is not None:
        root = candidate
        print("Projeto encontrado:", root)
        break

# Montar em local novo, somente se necessario.
if root is None:

    new_mount = Path("/content/drive_capes_recovery")

    print("\n=== 2. NOVA MONTAGEM SEGURA ===")

    if new_mount.exists() and any(new_mount.iterdir()):
        raise RuntimeError(
            "A montagem alternativa ja contem arquivos. "
            "Nenhum arquivo foi apagado. "
            "Reinicie o ambiente de execucao do Colab "
            "ou examine esse caminho antes de prosseguir."
        )

    drive.mount(str(new_mount))

    root = find_project(new_mount)

if root is None:
    print("\nPROJETO NAO LOCALIZADO.")
    print(
        "Verifique a conta Google conectada "
        "e a pasta CAPES_Epistemic_Change."
    )

else:
    print("\n=== 3. PROJETO CAPES LOCALIZADO ===")
    print("Raiz:", root)

    lab = root / LAB_NAME

    expansion = lab / "05_national_expansion"

    print("Laboratorio existe:", lab.is_dir())
    print("Expansao existe:", expansion.is_dir())

    if expansion.is_dir():

        print("\n=== 4. PROCURAR ETAPA 22D ===")

        dirs_22d = sorted(
            p for p in expansion.iterdir()
            if p.is_dir() and "22D" in p.name
        )

        for p in dirs_22d:
            print("Diretorio:", p)

        print("\n=== 5. CHECKPOINTS ===")

        folders = list(
            expansion.glob(
                "**/focal_checkpoints"
            )
        )

        for folder in folders:

            files = list(
                folder.glob(
                    "NAT-*_reference.parquet"
                )
            )

            print("Pasta:", folder)
            print("Arquivos:", len(files))

        print("\n=== 6. CONSOLIDADO ===")

        consolidated = list(
            expansion.glob(
                "**/matched_reference_pairs_43200.parquet"
            )
        )

        for file in consolidated:
            print("Arquivo:", file)

        if not dirs_22d:
            print(
                "Nenhum diretorio 22D encontrado "
                "nesta expansao."
            )

        if not folders:
            print(
                "Nenhuma pasta focal_checkpoints "
                "encontrada."
            )

        if not consolidated:
            print(
                "Arquivo consolidado nao encontrado. "
                "Pode ser reconstruido se existirem "
                "os 108 checkpoints."
            )

print("\nDIAGNOSTICO FINALIZADO")


=== 1. PROCURANDO MONTAGENS EXISTENTES ===
Montagem: /content/gdrive | Existe: True
Projeto encontrado: /content/gdrive/MyDrive/CAPES_Epistemic_Change

=== 3. PROJETO CAPES LOCALIZADO ===
Raiz: /content/gdrive/MyDrive/CAPES_Epistemic_Change
Laboratorio existe: True
Expansao existe: True

=== 4. PROCURAR ETAPA 22D ===

=== 5. CHECKPOINTS ===

=== 6. CONSOLIDADO ===
Nenhum diretorio 22D encontrado nesta expansao.
Nenhuma pasta focal_checkpoints encontrada.
Arquivo consolidado nao encontrado. Pode ser reconstruido se existirem os 108 checkpoints.

DIAGNOSTICO FINALIZADO


In [8]:

# ============================================================
# CAPES — DIAGNOSTICO DE PERSISTENCIA DOS RESULTADOS
# ETAPAS 21, 22A, 22B, 22C, 22D
#
# SOMENTE LEITURA
# Nao apaga, move, modifica ou recalcula arquivos
# ============================================================

from pathlib import Path
import os

ROOT = Path(
    "/content/gdrive/MyDrive/CAPES_Epistemic_Change"
)

LAB = ROOT / "08_epistemic_measurement_validation"
EXP = LAB / "05_national_expansion"

print("=" * 65)
print("1. IDENTIDADE DO DIRETORIO")
print("=" * 65)

for path in [ROOT, LAB, EXP]:
    print("\nCaminho:", path)
    print("Existe:", path.exists())
    print("Diretorio:", path.is_dir())
    print("Caminho real:", os.path.realpath(path))

print("\n" + "=" * 65)
print("2. INVENTARIO DAS ETAPAS")
print("=" * 65)

if EXP.is_dir():

    for path in sorted(EXP.iterdir()):

        if path.is_dir():
            print("PASTA:", path.name)

        else:
            print("ARQUIVO:", path.name)

print("\n" + "=" * 65)
print("3. VERIFICAR ARTEFATOS DAS ETAPAS ANTERIORES")
print("=" * 65)

prefixes = [
    "focal_sample_108",
    "coverage_plan_20B",
    "21_semantic_controlled",
    "22A_national_stability",
    "22B_robustness",
    "22C_score_geometry",
    "22D_temporally_matched_reference"
]

if EXP.is_dir():

    for prefix in prefixes:

        found = sorted(
            p for p in EXP.iterdir()
            if p.name.startswith(prefix)
        )

        print(
            f"{prefix}:",
            len(found),
            "encontrado(s)"
        )

        for path in found:
            print("  ", path.name)

print("\n" + "=" * 65)
print("4. BUSCAR ETAPA 22D EM OUTRAS PASTAS")
print("=" * 65)

search_roots = [
    LAB,
    ROOT,
    Path("/content/gdrive/MyDrive"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive_capes_recovery/MyDrive")
]

seen = set()
found_22d = []

for search_root in search_roots:

    if not search_root.is_dir():
        continue

    real = str(search_root.resolve())

    if real in seen:
        continue

    seen.add(real)

    # Procurar apenas ate tres niveis de subpastas.
    patterns = [
        "22D*",
        "*/22D*",
        "*/*/22D*",
        "*/*/*/22D*"
    ]

    for pattern in patterns:

        try:
            for path in search_root.glob(pattern):

                if path.is_dir() and path not in found_22d:
                    found_22d.append(path)
                    print("ENCONTRADO:", path)

        except (OSError, PermissionError) as exc:
            print("Busca interrompida:", type(exc).__name__)

print("\n" + "=" * 65)
print("5. VERIFICAR CHECKPOINTS SEMANTICOS ANTERIORES")
print("=" * 65)

ckpt = (
    EXP / "21_semantic_controlled_national_v1"
    / "checkpoints"
)

if ckpt.is_dir():

    score_files = list(
        ckpt.glob("NAT-*_scores.parquet")
    )

    rank_files = list(
        ckpt.glob("NAT-*_top30.parquet")
    )

    print("Scores Etapa 21:", len(score_files))
    print("Rankings Etapa 21:", len(rank_files))

else:
    print("Pasta da Etapa 21 nao localizada.")

print("\n" + "=" * 65)
print("6. RESULTADO DO DIAGNOSTICO")
print("=" * 65)

if found_22d:

    print("RESULTADO: ETAPA 22D LOCALIZADA.")
    print("Podemos corrigir o caminho da Etapa 22E.")

else:

    print("RESULTADO: ETAPA 22D NAO LOCALIZADA.")
    print(
        "Os dados nao podem ser reconstruidos "
        "apenas a partir dos logs de execucao."
    )
    print(
        "Verificar no Google Drive web: "
        "Pesquisa, Atividade e Lixeira."
    )

print("\nDIAGNOSTICO CONCLUIDO")


1. IDENTIDADE DO DIRETORIO

Caminho: /content/gdrive/MyDrive/CAPES_Epistemic_Change
Existe: True
Diretorio: True
Caminho real: /content/gdrive/MyDrive/CAPES_Epistemic_Change

Caminho: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation
Existe: True
Diretorio: True
Caminho real: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation

Caminho: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion
Existe: True
Diretorio: True
Caminho real: /content/gdrive/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion

2. INVENTARIO DAS ETAPAS
PASTA: 22E_paired_geometry_inference_20261021T014003_578360Z
PASTA: 22E_paired_geometry_inference_20261021T014040_632890Z
PASTA: 22E_paired_geometry_inference_20261021T015323_030886Z

3. VERIFICAR ARTEFATOS DAS ETAPAS ANTERIORES
focal_sample_108: 0 encontrado(s)
coverage_plan_20B: 0 encontrado(s)
21_semantic_controll

In [11]:

# ============================================================
# CAPES — RECUPERACAO SEGURA DA MONTAGEM GOOGLE DRIVE
# ETAPA 22D
#
# Nao remove arquivos
# Nao altera checkpoints
# Nao executa analises novamente
# ============================================================

from pathlib import Path
import pandas as pd

from google.colab import drive

# Novo ponto de montagem independente.
MOUNT = Path("/content/capes_drive_clean_20261020")

if MOUNT.exists():
    assert MOUNT.is_dir(), "O ponto de montagem nao e uma pasta."

    # Uma montagem existente pode ser reutilizada
    # somente se o projeto estiver visivel.
    existing_root = (
        MOUNT / "MyDrive" / "CAPES_Epistemic_Change"
    )

    if not existing_root.is_dir():
        assert not any(MOUNT.iterdir()), (
            "O diretorio alternativo contem arquivos, "
            "mas o projeto nao esta visivel. "
            "Nenhum arquivo sera excluido."
        )

        drive.mount(str(MOUNT))
else:
    drive.mount(str(MOUNT))

ROOT = (
    MOUNT / "MyDrive"
    / "CAPES_Epistemic_Change"
)

assert ROOT.is_dir(), (
    "Projeto nao encontrado na nova montagem. "
    "Confirme a conta Google autorizada."
)

SOURCE = (
    ROOT
    / "08_epistemic_measurement_validation"
    / "05_national_expansion"
    / "22D_temporally_matched_reference_v1"
)

ANALYSIS = (
    SOURCE
    / "analysis_20261020T211405_678524Z"
)

DATA_FILE = (
    ANALYSIS
    / "matched_reference_pairs_43200.parquet"
)

CHECKPOINTS = SOURCE / "focal_checkpoints"

assert SOURCE.is_dir(), (
    "A Etapa 22D nao apareceu na nova montagem."
)

assert DATA_FILE.is_file(), (
    "O arquivo consolidado nao apareceu "
    "na nova montagem."
)

assert CHECKPOINTS.is_dir()

checkpoint_files = list(
    CHECKPOINTS.glob("NAT-*_reference.parquet")
)

assert len(checkpoint_files) == 108

# Leitura apenas das colunas essenciais.
pairs = pd.read_parquet(
    DATA_FILE,
    columns=[
        "FOCAL_ID",
        "DOCUMENT_ID",
        "GROUP",
        "COSINE_M2",
        "COSINE_M4"
    ]
)

assert len(pairs) == 43200
assert pairs["FOCAL_ID"].nunique() == 108

print("\n=== RECUPERACAO CONCLUIDA ===")
print("Projeto:", ROOT)
print("Etapa 22D: LOCALIZADA")
print("Checkpoints:", len(checkpoint_files))
print("Pares consolidados:", len(pairs))
print("Focais:", pairs["FOCAL_ID"].nunique())
print("Status: PASS")
print("\nArquivo recuperado:", DATA_FILE)


Mounted at /content/capes_drive_clean_20261020

=== RECUPERACAO CONCLUIDA ===
Projeto: /content/capes_drive_clean_20261020/MyDrive/CAPES_Epistemic_Change
Etapa 22D: LOCALIZADA
Checkpoints: 108
Pares consolidados: 43200
Focais: 108
Status: PASS

Arquivo recuperado: /content/capes_drive_clean_20261020/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion/22D_temporally_matched_reference_v1/analysis_20261020T211405_678524Z/matched_reference_pairs_43200.parquet


In [12]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 22E
# INFERENCIA PAREADA DA REFERENCIA SEMANTICA
#
# DRIVE: montagem recuperada e validada
# 108 focais | 43.200 pares | 9 grandes areas
# 3.000 replicacoes bootstrap
#
# CPU | Sem novos embeddings ou cossenos
# Nao altera arquivos das etapas anteriores
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CAMINHOS — MONTAGEM RECUPERADA
# ------------------------------------------------------------

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

LAB = ROOT / "08_epistemic_measurement_validation"
EXP = LAB / "05_national_expansion"

SOURCE = (
    EXP / "22D_temporally_matched_reference_v1"
    / "analysis_20261020T211405_678524Z"
)

PAIRS_FILE = (
    SOURCE / "matched_reference_pairs_43200.parquet"
)

SAMPLE = (
    EXP / "focal_sample_108_20261020T190807_559853Z"
)

assert ROOT.is_dir(), "Montagem recuperada indisponivel."
assert PAIRS_FILE.is_file(), "Dados da Etapa 22D ausentes."
assert SAMPLE.is_dir(), "Amostra congelada ausente."

# ------------------------------------------------------------
# 2. CARREGAR E VALIDAR DADOS
# ------------------------------------------------------------

pairs = pd.read_parquet(PAIRS_FILE)

mapping = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str, "ID_TRABALHO": str}
)

sample = pd.read_parquet(
    SAMPLE / "national_focals_108_RESTRICTED.parquet"
)

sample["ID_TRABALHO"] = (
    sample["ID_TRABALHO"].astype(str)
)

assert len(pairs) == 43200
assert pairs["FOCAL_ID"].nunique() == 108
assert len(mapping) == 108
assert len(sample) == 108

assert not pairs.duplicated(
    ["FOCAL_ID", "DOCUMENT_ID"]
).any()

counts = (
    pairs.groupby(["FOCAL_ID", "GROUP"])
    .size()
    .unstack(fill_value=0)
)

assert counts.shape == (108, 2)
assert counts["same_area"].eq(200).all()
assert counts["other_area"].eq(200).all()

year_counts = (
    pairs.groupby([
        "FOCAL_ID", "DOCUMENT_YEAR", "GROUP"
    ])
    .size()
    .unstack(fill_value=0)
)

assert (
    year_counts["same_area"]
    == year_counts["other_area"]
).all()

if "FOCAL_YEAR" in pairs.columns:
    assert (
        pairs["DOCUMENT_YEAR"].astype(int)
        < pairs["FOCAL_YEAR"].astype(int)
    ).all()

for mod in ["M2", "M4"]:
    scores = pairs[f"COSINE_{mod}"].to_numpy(
        dtype=float
    )
    assert np.isfinite(scores).all()
    assert (
        (scores >= -1.00001)
        & (scores <= 1.00001)
    ).all()

areas = mapping[[
    "FOCAL_ID", "ID_TRABALHO"
]].merge(
    sample[[
        "ID_TRABALHO",
        "NM_GRANDE_AREA_CONHECIMENTO"
    ]],
    on="ID_TRABALHO",
    validate="one_to_one"
).rename(columns={
    "NM_GRANDE_AREA_CONHECIMENTO": "GRANDE_AREA"
})

assert areas["GRANDE_AREA"].notna().all()
assert areas["GRANDE_AREA"].nunique() == 9
assert areas.groupby("GRANDE_AREA").size().eq(12).all()

print("=== INTEGRIDADE DA BASE ===")
print("Focais:", pairs["FOCAL_ID"].nunique())
print("Pares:", len(pairs))
print("Grandes areas:", areas["GRANDE_AREA"].nunique())
print("Pareamento anual: PASS")

# ------------------------------------------------------------
# 3. ESTATISTICAS POR FOCAL E MODALIDADE
# ------------------------------------------------------------

records = []

for fid, part in pairs.groupby("FOCAL_ID", sort=True):

    for modality in ["M2", "M4"]:

        col = f"COSINE_{modality}"

        same = part.loc[
            part["GROUP"] == "same_area", col
        ].to_numpy(dtype=np.float64)

        other = part.loc[
            part["GROUP"] == "other_area", col
        ].to_numpy(dtype=np.float64)

        assert len(same) == len(other) == 200

        delta = float(same.mean() - other.mean())

        pooled_sd = float(np.sqrt(
            (
                same.var(ddof=1)
                + other.var(ddof=1)
            ) / 2
        ))

        assert pooled_sd > 0

        records.append({
            "FOCAL_ID": fid,
            "MODALITY": modality,
            "MEAN_SAME": float(same.mean()),
            "MEAN_OTHER": float(other.mean()),
            "DELTA": delta,
            "POOLED_SD": pooled_sd,
            "STANDARDIZED_DELTA": delta / pooled_sd
        })

focal_results = pd.DataFrame(records).merge(
    areas[["FOCAL_ID", "GRANDE_AREA"]],
    on="FOCAL_ID",
    validate="many_to_one"
)

assert len(focal_results) == 216

# ------------------------------------------------------------
# 4. CONTRASTES PAREADOS M2/M4
# ------------------------------------------------------------

wide = focal_results.pivot(
    index=["FOCAL_ID", "GRANDE_AREA"],
    columns="MODALITY",
    values=["DELTA", "STANDARDIZED_DELTA"]
)

wide.columns = [
    f"{metric}_{mod}"
    for metric, mod in wide.columns
]

wide = wide.reset_index()

wide["DELTA_M4_MINUS_M2"] = (
    wide["DELTA_M4"] - wide["DELTA_M2"]
)

wide["STANDARDIZED_M4_MINUS_M2"] = (
    wide["STANDARDIZED_DELTA_M4"]
    - wide["STANDARDIZED_DELTA_M2"]
)

wide = wide.sort_values(
    ["GRANDE_AREA", "FOCAL_ID"]
).reset_index(drop=True)

assert len(wide) == 108
assert wide.groupby("GRANDE_AREA").size().eq(12).all()

# ------------------------------------------------------------
# 5. BOOTSTRAP ESTRATIFICADO
# ------------------------------------------------------------

B = 3000
SEED = 20261020

variables = [
    "DELTA_M2",
    "DELTA_M4",
    "DELTA_M4_MINUS_M2",
    "STANDARDIZED_DELTA_M2",
    "STANDARDIZED_DELTA_M4",
    "STANDARDIZED_M4_MINUS_M2"
]

matrix = wide[variables].to_numpy(dtype=float)

groups = [
    group.index.to_numpy()
    for _, group in wide.groupby(
        "GRANDE_AREA", sort=True
    )
]

rng = np.random.default_rng(SEED)

boot = np.empty((B, len(variables)), dtype=float)

for b in range(B):

    idx = np.concatenate([
        rng.choice(g, size=len(g), replace=True)
        for g in groups
    ])

    boot[b, :] = matrix[idx, :].mean(axis=0)

summary = pd.DataFrame([
    {
        "ESTIMAND": name,
        "N_FOCALS": 108,
        "MEAN": float(matrix[:, j].mean()),
        "MEDIAN": float(np.median(matrix[:, j])),
        "BOOT_CI_LOW": float(
            np.percentile(boot[:, j], 2.5)
        ),
        "BOOT_CI_HIGH": float(
            np.percentile(boot[:, j], 97.5)
        )
    }
    for j, name in enumerate(variables)
])

# ------------------------------------------------------------
# 6. HETEROGENEIDADE E DIRECAO DOS EFEITOS
# ------------------------------------------------------------

by_area = (
    wide.groupby("GRANDE_AREA")
    .agg(
        N_FOCALS=("FOCAL_ID", "size"),
        DELTA_M2_MEAN=("DELTA_M2", "mean"),
        DELTA_M4_MEAN=("DELTA_M4", "mean"),
        M4_MINUS_M2_MEAN=(
            "DELTA_M4_MINUS_M2", "mean"
        )
    )
    .reset_index()
)

direction = pd.DataFrame([
    {
        "MODALITY": mod,
        "N_POSITIVE": int(
            (wide[f"DELTA_{mod}"] > 0).sum()
        ),
        "N_NEGATIVE": int(
            (wide[f"DELTA_{mod}"] < 0).sum()
        ),
        "N_ZERO": int(
            (wide[f"DELTA_{mod}"] == 0).sum()
        )
    }
    for mod in ["M2", "M4"]
])

# ------------------------------------------------------------
# 7. VERIFICAR COERENCIA COM ETAPA 22D
# ------------------------------------------------------------

means = (
    wide[["DELTA_M2", "DELTA_M4"]]
    .mean()
)

# Os valores sao aproximados aos cinco decimais
# informados na Etapa 22D.
assert abs(means["DELTA_M2"] - 0.01196) < 0.0001
assert abs(means["DELTA_M4"] - 0.01425) < 0.0001

# ------------------------------------------------------------
# 8. SALVAR RESULTADOS
# ------------------------------------------------------------

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

OUT = (
    EXP / f"22E_paired_geometry_final_{STAMP}"
)

OUT.mkdir(parents=True, exist_ok=False)

focal_results.to_parquet(
    OUT / "focal_effects_216.parquet",
    index=False
)

wide.to_csv(
    OUT / "paired_focal_effects_108.csv",
    index=False
)

summary.to_csv(
    OUT / "bootstrap_stratified_summary.csv",
    index=False
)

by_area.to_csv(
    OUT / "grand_area_effects.csv",
    index=False
)

direction.to_csv(
    OUT / "effect_direction.csv",
    index=False
)

manifest = {
    "stage": "22E",
    "source": str(PAIRS_FILE),
    "n_focals": 108,
    "n_matched_pairs": 43200,
    "n_grand_areas": 9,
    "bootstrap_iterations": B,
    "bootstrap_seed": SEED,
    "resampling_unit": "focal_document",
    "stratification": "nine_grand_areas",
    "matching": "exact_antecedent_year",
    "effect_definition": "same_area_minus_other_area",
    "interpretation": (
        "Balanced-sample descriptive effects with "
        "stratified bootstrap uncertainty, not "
        "design-based national population inference."
    ),
    "recomputed_embeddings": False,
    "recomputed_cosine_scores": False,
    "human_ground_truth_completed": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest, ensure_ascii=False, indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 9. RESULTADOS FINAIS
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("ETAPA 22E — RESULTADOS FINAIS")
print("=" * 68)

print("\n1. CONTRASTES E INTERVALOS BOOTSTRAP")
print(
    summary.round(5).to_string(index=False)
)

print("\n2. DIRECAO DOS EFEITOS")
print(
    direction.to_string(index=False)
)

print("\n3. CONTRASTES POR GRANDE AREA")
print(
    by_area.round(5).to_string(index=False)
)

print("\n4. AUDITORIA")
print("Focais:", wide["FOCAL_ID"].nunique())
print("Registros originais:", len(pairs))
print("Bootstrap:", B)
print("Status: PASS")

print("\nDiretorio:", OUT)
print("ETAPA 22E CONCLUIDA")


=== INTEGRIDADE DA BASE ===
Focais: 108
Pares: 43200
Grandes areas: 9
Pareamento anual: PASS

ETAPA 22E — RESULTADOS FINAIS

1. CONTRASTES E INTERVALOS BOOTSTRAP
                ESTIMAND  N_FOCALS    MEAN  MEDIAN  BOOT_CI_LOW  BOOT_CI_HIGH
                DELTA_M2       108 0.01196 0.01234      0.01087       0.01306
                DELTA_M4       108 0.01425 0.01510      0.01292       0.01561
       DELTA_M4_MINUS_M2       108 0.00229 0.00265      0.00138       0.00322
   STANDARDIZED_DELTA_M2       108 0.87439 0.93295      0.79136       0.95646
   STANDARDIZED_DELTA_M4       108 1.02402 1.13931      0.92685       1.11917
STANDARDIZED_M4_MINUS_M2       108 0.14963 0.17772      0.07774       0.22158

2. DIRECAO DOS EFEITOS
MODALITY  N_POSITIVE  N_NEGATIVE  N_ZERO
      M2         104           4       0
      M4         104           4       0

3. CONTRASTES POR GRANDE AREA
                GRANDE_AREA  N_FOCALS  DELTA_M2_MEAN  DELTA_M4_MEAN  M4_MINUS_M2_MEAN
          CIÊNCIAS AGRÁRIAS 

In [13]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 23A
# PREPARACAO DO CORPUS LEXICAL NACIONAL
#
# Objetivo:
# - Auditar as fontes textuais CAPES 2013-2024
# - Conferir cobertura dos 108 documentos focais
# - Identificar todos os antecedentes elegiveis
# - Salvar corpus lexical anual reutilizavel
#
# CPU | Sem embeddings | Sem calculo BM25 nesta etapa
# Montagem do Google Drive recuperada
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. CAMINHOS
# ------------------------------------------------------------

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

LAB = ROOT / "08_epistemic_measurement_validation"
EXP = LAB / "05_national_expansion"

TEXT_SOURCE = ROOT / "05_transferencia_colab"

SAMPLE = (
    EXP /
    "focal_sample_108_20261020T190807_559853Z"
)

PLAN = (
    EXP /
    "coverage_plan_20B_20261020T191031_648431Z"
    / "focal_candidate_plan.csv"
)

INDEX_FILE = (
    ROOT / "12_indice_analitico_harmonizado"
    / "CAPES_INDICE_ANALITICO_HARMONIZADO_2013_2024.parquet"
)

OUT = EXP / "23A_lexical_corpus_national_v1"
YEAR_DIR = OUT / "annual_corpus"

YEAR_DIR.mkdir(parents=True, exist_ok=True)

assert ROOT.is_dir()
assert TEXT_SOURCE.is_dir()
assert SAMPLE.is_dir()
assert PLAN.is_file()
assert INDEX_FILE.is_file()

# ------------------------------------------------------------
# 2. CONFIGURACAO CONGELADA
# ------------------------------------------------------------

CONFIG = {
    "stage": "23A",
    "years": [2013, 2024],
    "n_focals": 108,
    "candidate_rule": (
        "same_AREA_HARMONIZADA_and_strictly_prior_year"
    ),
    "lexical_source": str(TEXT_SOURCE),
    "fields": [
        "TITULO",
        "RESUMO",
        "PALAVRAS_CHAVE"
    ],
    "tokenization": (
        "unicode_casefold_alphanumeric_tokens"
    ),
    "remove_stopwords": False,
    "remove_diacritics": False,
    "stemming": False,
    "compute_bm25": False
}

config_file = OUT / "configuration.json"

if config_file.exists():
    assert json.loads(
        config_file.read_text(encoding="utf-8")
    ) == CONFIG
else:
    config_file.write_text(
        json.dumps(
            CONFIG,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 3. CARREGAR INDICE E AMOSTRA
# ------------------------------------------------------------

meta = pq.read_table(
    INDEX_FILE,
    columns=[
        "ID_TRABALHO",
        "ANO",
        "AREA_HARMONIZADA"
    ]
).to_pandas()

meta["ID_TRABALHO"] = meta[
    "ID_TRABALHO"
].astype(str)

meta["ANO"] = meta["ANO"].astype(int)

assert len(meta) == 993086
assert meta["ID_TRABALHO"].is_unique

focals = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
)

focals["ANO"] = focals["ANO"].astype(int)

assert len(focals) == 108
assert focals["FOCAL_ID"].is_unique
assert focals["ID_TRABALHO"].is_unique

area_lookup = meta.set_index(
    "ID_TRABALHO"
)["AREA_HARMONIZADA"]

focals["AREA_HARMONIZADA"] = (
    focals["ID_TRABALHO"].map(area_lookup)
)

assert focals["AREA_HARMONIZADA"].notna().all()

plan = pd.read_csv(
    PLAN,
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
).set_index("FOCAL_ID")

assert len(plan) == 108

# ------------------------------------------------------------
# 4. IDENTIFICAR TODOS OS DOCUMENTOS NECESSARIOS
# ------------------------------------------------------------

# Para cada area, basta incluir documentos anteriores
# ao ultimo ano focal existente nessa area.
max_focal_year_by_area = (
    focals.groupby("AREA_HARMONIZADA")["ANO"]
    .max()
    .to_dict()
)

required = meta[
    meta.apply(
        lambda r: (
            r["AREA_HARMONIZADA"]
            in max_focal_year_by_area
            and r["ANO"] <
            max_focal_year_by_area.get(
                r["AREA_HARMONIZADA"], -1
            )
        ),
        axis=1
    )
][["ID_TRABALHO", "ANO"]].copy()

# Incluir tambem os documentos focais.
focal_required = focals[[
    "ID_TRABALHO", "ANO"
]]

required = pd.concat(
    [required, focal_required],
    ignore_index=True
).drop_duplicates("ID_TRABALHO")

assert required["ID_TRABALHO"].is_unique

required_ids = set(
    required["ID_TRABALHO"]
)

print("=== UNIVERSO LEXICAL NECESSARIO ===")
print("Documentos distintos:", len(required))
print("Focais:", len(focals))

# ------------------------------------------------------------
# 5. NORMALIZACAO MINIMA E REPRODUZIVEL
# ------------------------------------------------------------

def normalize_text(value):

    if pd.isna(value):
        return ""

    text = unicodedata.normalize(
        "NFKC", str(value)
    ).casefold()

    # Mantem palavras com caracteres Unicode.
    # Pontuacao e separadores sao removidos.
    tokens = re.findall(
        r"[^\W_]+",
        text,
        flags=re.UNICODE
    )

    return " ".join(tokens)


def join_fields(row):

    title = normalize_text(row["TITULO"])
    abstract = normalize_text(row["RESUMO"])
    keywords = normalize_text(row["PALAVRAS_CHAVE"])

    return title, abstract, keywords

# ------------------------------------------------------------
# 6. PROCESSAR TEXTOS ANUAIS
# ------------------------------------------------------------

annual_reports = []

for year in range(2013, 2025):

    source_file = (
        TEXT_SOURCE /
        f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    )

    assert source_file.is_file(), (
        f"Arquivo anual ausente: {source_file}"
    )

    columns = pq.ParquetFile(
        source_file
    ).schema_arrow.names

    needed_columns = [
        "ID_TRABALHO",
        "TITULO",
        "RESUMO",
        "PALAVRAS_CHAVE"
    ]

    assert set(needed_columns).issubset(
        set(columns)
    ), (
        f"Colunas ausentes em {year}: "
        f"{set(needed_columns) - set(columns)}"
    )

    print(f"\nProcessando {year}...")

    output_file = (
        YEAR_DIR /
        f"lexical_corpus_{year}.parquet"
    )

    # Conferir arquivo existente sem sobrescrever.
    if output_file.exists():

        data = pd.read_parquet(output_file)

        expected_ids = set(
            required.loc[
                required["ANO"] == year,
                "ID_TRABALHO"
            ]
        )

        assert set(data["ID_TRABALHO"]) == expected_ids

        print(
            f"{year}: arquivo existente reutilizado "
            f"({len(data):,} documentos)"
        )

    else:

        raw = pq.read_table(
            source_file,
            columns=needed_columns
        ).to_pandas()

        raw["ID_TRABALHO"] = (
            raw["ID_TRABALHO"].astype(str)
        )

        assert raw["ID_TRABALHO"].is_unique

        year_required = set(
            required.loc[
                required["ANO"] == year,
                "ID_TRABALHO"
            ]
        )

        data = raw[
            raw["ID_TRABALHO"].isin(
                year_required
            )
        ].copy()

        assert set(
            data["ID_TRABALHO"]
        ) == year_required, (
            f"Documentos necessarios ausentes: {year}"
        )

        del raw

        normalized = [
            join_fields(row)
            for _, row in data.iterrows()
        ]

        data["TITLE_NORM"] = [
            x[0] for x in normalized
        ]

        data["ABSTRACT_NORM"] = [
            x[1] for x in normalized
        ]

        data["KEYWORDS_NORM"] = [
            x[2] for x in normalized
        ]

        # Campo lexical para a comparacao BM25.
        # A ponderacao dos campos sera definida
        # explicitamente na proxima etapa.
        data["TEXT_FULL_NORM"] = (
            data["TITLE_NORM"] + " " +
            data["ABSTRACT_NORM"] + " " +
            data["KEYWORDS_NORM"]
        ).str.strip()

        data["TEXT_SHORT_NORM"] = (
            data["TITLE_NORM"] + " " +
            data["KEYWORDS_NORM"]
        ).str.strip()

        data = data[[
            "ID_TRABALHO",
            "TITLE_NORM",
            "ABSTRACT_NORM",
            "KEYWORDS_NORM",
            "TEXT_SHORT_NORM",
            "TEXT_FULL_NORM"
        ]].reset_index(drop=True)

        # Salvar somente apos verificacoes.
        temporary = (
            YEAR_DIR /
            f"lexical_corpus_{year}.partial.parquet"
        )

        assert not temporary.exists(), (
            f"Arquivo parcial encontrado: {temporary}"
        )

        data.to_parquet(
            temporary,
            index=False
        )

        assert (
            pq.ParquetFile(
                temporary
            ).metadata.num_rows == len(data)
        )

        temporary.rename(output_file)

        print(
            f"{year}: {len(data):,} documentos salvos"
        )

    # --------------------------------------------------------
    # 7. AUDITORIA ANUAL DE COBERTURA
    # --------------------------------------------------------

    title_empty = int(
        data["TITLE_NORM"].fillna("").eq("").sum()
    )

    abstract_empty = int(
        data["ABSTRACT_NORM"].fillna("").eq("").sum()
    )

    keywords_empty = int(
        data["KEYWORDS_NORM"].fillna("").eq("").sum()
    )

    full_empty = int(
        data["TEXT_FULL_NORM"].fillna("").eq("").sum()
    )

    annual_reports.append({
        "ANO": year,
        "N_DOCUMENTS": len(data),
        "N_TITLE_EMPTY": title_empty,
        "N_ABSTRACT_EMPTY": abstract_empty,
        "N_KEYWORDS_EMPTY": keywords_empty,
        "N_FULL_TEXT_EMPTY": full_empty,
        "SHARE_ABSTRACT_AVAILABLE": (
            1 - abstract_empty / len(data)
            if len(data) else np.nan
        )
    })

# ------------------------------------------------------------
# 8. VALIDAR COBERTURA DOS FOCAIS
# ------------------------------------------------------------

corpus_ids = set()

for year in range(2013, 2025):

    file = (
        YEAR_DIR /
        f"lexical_corpus_{year}.parquet"
    )

    ids = pd.read_parquet(
        file,
        columns=["ID_TRABALHO"]
    )

    corpus_ids.update(
        ids["ID_TRABALHO"].astype(str)
    )

assert corpus_ids == required_ids

assert set(
    focals["ID_TRABALHO"]
).issubset(corpus_ids)

# ------------------------------------------------------------
# 9. SALVAR RELATORIOS
# ------------------------------------------------------------

report = pd.DataFrame(annual_reports)

report.to_csv(
    OUT / "annual_text_coverage.csv",
    index=False
)

required.to_parquet(
    OUT / "required_document_ids.parquet",
    index=False
)

focals.to_csv(
    OUT / "focal_index_for_bm25.csv",
    index=False
)

manifest = {
    "stage": "23A",
    "n_focals": 108,
    "n_documents_in_lexical_cache": len(corpus_ids),
    "years": [2013, 2024],
    "candidate_rule": CONFIG["candidate_rule"],
    "lexical_normalization": (
        "NFKC, casefold, Unicode alphanumeric tokens"
    ),
    "stopword_removal": False,
    "stemming": False,
    "bm25_scores_calculated": False,
    "embedding_inference": False,
    "source_embedding_scores_preserved": True
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 10. RESULTADO FINAL
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ETAPA 23A — RESULTADO FINAL")
print("=" * 65)

print("Focais:", len(focals))
print("Documentos lexicais distintos:", len(corpus_ids))

print("\n=== COBERTURA ANUAL ===")

print(
    report.round(4).to_string(index=False)
)

print("\nTotal de titulos vazios:", report["N_TITLE_EMPTY"].sum())
print("Total de resumos vazios:", report["N_ABSTRACT_EMPTY"].sum())
print(
    "Total de palavras-chave vazias:",
    report["N_KEYWORDS_EMPTY"].sum()
)
print(
    "Documentos sem qualquer texto:",
    report["N_FULL_TEXT_EMPTY"].sum()
)

print("\nCobertura dos 108 focais: PASS")
print("Diretorio:", OUT)
print("\nETAPA 23A CONCLUIDA")


=== UNIVERSO LEXICAL NECESSARIO ===
Documentos distintos: 642779
Focais: 108

Processando 2013...
2013: 59,658 documentos salvos

Processando 2014...
2014: 62,130 documentos salvos

Processando 2015...
2015: 67,057 documentos salvos

Processando 2016...
2016: 70,749 documentos salvos

Processando 2017...
2017: 70,210 documentos salvos

Processando 2018...
2018: 69,601 documentos salvos

Processando 2019...
2019: 72,022 documentos salvos

Processando 2020...
2020: 57,472 documentos salvos

Processando 2021...
2021: 48,976 documentos salvos

Processando 2022...
2022: 44,131 documentos salvos

Processando 2023...
2023: 20,762 documentos salvos

Processando 2024...
2024: 11 documentos salvos

ETAPA 23A — RESULTADO FINAL
Focais: 108
Documentos lexicais distintos: 642779

=== COBERTURA ANUAL ===
 ANO  N_DOCUMENTS  N_TITLE_EMPTY  N_ABSTRACT_EMPTY  N_KEYWORDS_EMPTY  N_FULL_TEXT_EMPTY  SHARE_ABSTRACT_AVAILABLE
2013        59658              0                23                36                 

In [14]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 23B
# BENCHMARK NACIONAL BM25
#
# 108 focais | 4 rankings por focal
# BM25 curto: titulo + palavras-chave
# BM25 completo: titulo + resumo + palavras-chave
# Janelas: historical e frontier3y
#
# Mesmos antecedentes da Etapa 21
# k1 = 1.5 | b = 0.75
# IDF BM25 positivo (Robertson/Sparck Jones suavizado)
# CPU | Sem inferencia de embeddings
# Checkpoints individuais, com retomada
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import gc

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

# ------------------------------------------------------------
# 1. CAMINHOS
# ------------------------------------------------------------

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

LAB = ROOT / "08_epistemic_measurement_validation"
EXP = LAB / "05_national_expansion"

LEXICAL_DIR = (
    EXP / "23A_lexical_corpus_national_v1"
)

ANNUAL = LEXICAL_DIR / "annual_corpus"

SAMPLE = (
    EXP /
    "focal_sample_108_20261020T190807_559853Z"
)

SEMANTIC_CHECKPOINTS = (
    EXP / "21_semantic_controlled_national_v1"
    / "checkpoints"
)

OUT = EXP / "23B_BM25_national_v1"
CHECKPOINTS = OUT / "checkpoints"

assert ROOT.is_dir()
assert ANNUAL.is_dir()
assert SAMPLE.is_dir()
assert SEMANTIC_CHECKPOINTS.is_dir()

CHECKPOINTS.mkdir(parents=True, exist_ok=True)

K1 = 1.5
B = 0.75
TOP_K = 50

CONFIG = {
    "stage": "23B",
    "n_focals": 108,
    "bm25_k1": K1,
    "bm25_b": B,
    "top_k": TOP_K,
    "conditions": [
        "short_historical",
        "full_historical",
        "short_frontier3y",
        "full_frontier3y"
    ],
    "idf": "log(1+(N-df+0.5)/(df+0.5))",
    "query_term_weighting": "binary_unique_terms",
    "tokenization": "whitespace_on_23A_normalized_text",
    "stopwords": False,
    "stemming": False,
    "tie_breaking": "score_desc_document_id_asc",
    "same_candidate_universe_as_stage21": True,
    "parameters_reestimated_per_window": True,
    "embedding_inference": False
}

config_path = OUT / "configuration.json"

if config_path.exists():
    saved_config = json.loads(
        config_path.read_text(encoding="utf-8")
    )
    assert saved_config == CONFIG
else:
    config_path.write_text(
        json.dumps(
            CONFIG, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 2. AMOSTRA CONGELADA
# ------------------------------------------------------------

focals = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
)

focals["ANO"] = focals["ANO"].astype(int)

focals = focals.sort_values(
    "FOCAL_ID"
).reset_index(drop=True)

assert len(focals) == 108
assert focals["FOCAL_ID"].is_unique

assert focals["FOCAL_ID"].tolist() == [
    f"NAT-{i:04d}" for i in range(1, 109)
]

# ------------------------------------------------------------
# 3. CARREGAR CORPUS LEXICAL CONGELADO
# ------------------------------------------------------------

print("=" * 68)
print("ETAPA 23B — BM25 NACIONAL")
print("=" * 68)

print("\nCarregando corpus lexical 2013-2024...")

parts = []

for year in range(2013, 2025):

    file = ANNUAL / f"lexical_corpus_{year}.parquet"

    assert file.is_file(), file

    frame = pd.read_parquet(
        file,
        columns=[
            "ID_TRABALHO",
            "TEXT_SHORT_NORM",
            "TEXT_FULL_NORM"
        ]
    )

    frame["ID_TRABALHO"] = (
        frame["ID_TRABALHO"].astype(str)
    )

    assert frame["ID_TRABALHO"].is_unique

    parts.append(frame)

    print(year, ":", len(frame), "documentos")

lexical = pd.concat(
    parts,
    ignore_index=True
)

del parts
gc.collect()

assert len(lexical) == 642779
assert lexical["ID_TRABALHO"].is_unique

lexical = lexical.set_index(
    "ID_TRABALHO",
    verify_integrity=True
)

for col in [
    "TEXT_SHORT_NORM",
    "TEXT_FULL_NORM"
]:
    lexical[col] = (
        lexical[col]
        .fillna("")
        .astype(str)
    )

assert set(
    focals["ID_TRABALHO"]
).issubset(lexical.index)

print("\nCorpus lexical validado:", len(lexical))

# ------------------------------------------------------------
# 4. IMPLEMENTACAO BM25
# ------------------------------------------------------------

def bm25_ranking(
    candidate_ids,
    candidate_texts,
    query_text,
    focal_id,
    window,
    representation
):
    """
    Calcula BM25 para todos os candidatos elegiveis.

    Documentos sem correspondencia lexical recebem
    score zero. Empates sao resolvidos por ID.

    O IDF e estimado somente a partir do universo
    de candidatos fornecido nesta chamada.
    """

    candidate_ids = np.asarray(
        candidate_ids,
        dtype=str
    )

    texts = [
        "" if pd.isna(t) else str(t)
        for t in candidate_texts
    ]

    query_tokens = sorted(
        set(str(query_text).split())
    )

    assert len(candidate_ids) == len(texts)
    assert len(candidate_ids) >= TOP_K

    n_docs = len(candidate_ids)

    if not query_tokens:

        scores = np.zeros(
            n_docs, dtype=np.float64
        )

        n_query_terms = 0
        n_terms_in_vocabulary = 0

    else:

        # O 23A ja normalizou os tokens.
        # split() preserva exatamente essa
        # tokenizacao, inclusive tokens curtos.
        vectorizer = CountVectorizer(
            analyzer=str.split,
            lowercase=False,
            dtype=np.int32
        )

        # Tratar universo com todos os textos vazios.
        if not any(texts):

            scores = np.zeros(
                n_docs, dtype=np.float64
            )

            n_query_terms = len(query_tokens)
            n_terms_in_vocabulary = 0

        else:

            X = vectorizer.fit_transform(texts)

            n_docs, vocab_size = X.shape

            lengths = np.asarray(
                X.sum(axis=1)
            ).ravel().astype(np.float64)

            avgdl = float(lengths.mean())

            scores = np.zeros(
                n_docs, dtype=np.float64
            )

            vocab = vectorizer.vocabulary_

            query_columns = [
                vocab[token]
                for token in query_tokens
                if token in vocab
            ]

            n_query_terms = len(query_tokens)
            n_terms_in_vocabulary = len(
                query_columns
            )

            if query_columns and avgdl > 0:

                # Matriz CSC: acesso eficiente
                # aos postings de cada termo.
                Xc = X.tocsc()

                norm_factor = (
                    K1 *
                    (1 - B + B * lengths / avgdl)
                )

                for col in query_columns:

                    first = Xc.indptr[col]
                    last = Xc.indptr[col + 1]

                    doc_positions = (
                        Xc.indices[first:last]
                    )

                    tf = (
                        Xc.data[first:last]
                        .astype(np.float64)
                    )

                    df = len(doc_positions)

                    idf = np.log1p(
                        (
                            n_docs - df + 0.5
                        ) / (df + 0.5)
                    )

                    contribution = (
                        idf * (
                            tf * (K1 + 1.0)
                        ) / (
                            tf + norm_factor[
                                doc_positions
                            ]
                        )
                    )

                    scores[doc_positions] += (
                        contribution
                    )

                del Xc

            del X, vectorizer

    assert np.isfinite(scores).all()
    assert (scores >= -1e-10).all()

    n_positive = int(
        np.count_nonzero(scores > 0)
    )

    # Desempate reprodutivel:
    # primeiro score decrescente; depois ID crescente.
    order = np.lexsort(
        (candidate_ids, -scores)
    )

    positions = order[:TOP_K]

    ranking = pd.DataFrame({
        "FOCAL_ID": focal_id,
        "WINDOW": window,
        "REPRESENTATION": representation,
        "RANK": np.arange(
            1, TOP_K + 1
        ),
        "ANTECEDENT_ID": candidate_ids[
            positions
        ],
        "BM25_SCORE": scores[
            positions
        ]
    })

    audit = {
        "FOCAL_ID": focal_id,
        "WINDOW": window,
        "REPRESENTATION": representation,
        "N_CANDIDATES": n_docs,
        "N_QUERY_TERMS": n_query_terms,
        "N_QUERY_TERMS_IN_VOCAB": (
            n_terms_in_vocabulary
        ),
        "N_POSITIVE_SCORES": n_positive,
        "SHARE_POSITIVE_SCORES": (
            n_positive / n_docs
        ),
        "N_ZERO_TOP50": int(
            np.count_nonzero(
                ranking["BM25_SCORE"].to_numpy() == 0
            )
        ),
        "TOP1_SCORE": float(
            ranking["BM25_SCORE"].iloc[0]
        ),
        "TOP50_SCORE": float(
            ranking["BM25_SCORE"].iloc[-1]
        )
    }

    return ranking, audit

# ------------------------------------------------------------
# 5. PROCESSAR 108 FOCAIS
# ------------------------------------------------------------

new = 0
reused = 0

for focal in focals.itertuples(index=False):

    fid = focal.FOCAL_ID
    focal_year = int(focal.ANO)

    rank_file = (
        CHECKPOINTS / f"{fid}_bm25_top50.parquet"
    )

    audit_file = (
        CHECKPOINTS / f"{fid}_bm25_audit.json"
    )

    # Retomar somente checkpoints completos.
    if rank_file.is_file() and audit_file.is_file():

        saved = pd.read_parquet(rank_file)

        with open(
            audit_file,
            "r",
            encoding="utf-8"
        ) as stream:
            saved_audit = json.load(stream)

        assert len(saved) == 4 * TOP_K
        assert saved["FOCAL_ID"].eq(fid).all()
        assert len(saved_audit) == 4
        assert set(
            zip(
                saved["WINDOW"],
                saved["REPRESENTATION"]
            )
        ) == {
            ("historical", "short"),
            ("historical", "full"),
            ("frontier3y", "short"),
            ("frontier3y", "full")
        }

        reused += 1
        continue

    if rank_file.exists() or audit_file.exists():
        raise RuntimeError(
            f"Checkpoint incompleto de {fid}. "
            "Inspecionar antes de prosseguir: "
            f"{rank_file} | {audit_file}"
        )

    # Universo identico ao da Etapa 21.
    score_file = (
        SEMANTIC_CHECKPOINTS
        / f"{fid}_scores.parquet"
    )

    assert score_file.is_file(), score_file

    antecedents = pd.read_parquet(
        score_file,
        columns=[
            "ANTECEDENT_ID",
            "ANTECEDENT_YEAR"
        ]
    )

    antecedents["ANTECEDENT_ID"] = (
        antecedents["ANTECEDENT_ID"].astype(str)
    )

    antecedents["ANTECEDENT_YEAR"] = (
        antecedents["ANTECEDENT_YEAR"].astype(int)
    )

    assert antecedents["ANTECEDENT_ID"].is_unique
    assert antecedents["ANTECEDENT_YEAR"].lt(
        focal_year
    ).all()

    assert set(
        antecedents["ANTECEDENT_ID"]
    ).issubset(lexical.index), (
        f"Faltam antecedentes lexicais: {fid}"
    )

    focal_id = str(focal.ID_TRABALHO)

    assert focal_id in lexical.index

    focal_text = lexical.loc[focal_id]

    ranking_parts = []
    audit_parts = []

    for window in [
        "historical",
        "frontier3y"
    ]:

        if window == "historical":
            eligible = antecedents
        else:
            eligible = antecedents[
                antecedents[
                    "ANTECEDENT_YEAR"
                ].between(
                    focal_year - 3,
                    focal_year - 1
                )
            ]

        assert len(eligible) >= TOP_K

        ids = eligible[
            "ANTECEDENT_ID"
        ].to_numpy(dtype=str)

        for representation, text_col in [
            ("short", "TEXT_SHORT_NORM"),
            ("full", "TEXT_FULL_NORM")
        ]:

            candidate_texts = (
                lexical.loc[ids, text_col]
                .to_numpy()
            )

            query_text = focal_text[
                text_col
            ]

            ranking, audit = bm25_ranking(
                candidate_ids=ids,
                candidate_texts=candidate_texts,
                query_text=query_text,
                focal_id=fid,
                window=window,
                representation=representation
            )

            ranking_parts.append(ranking)
            audit_parts.append(audit)

    rankings = pd.concat(
        ranking_parts,
        ignore_index=True
    )

    assert len(rankings) == 4 * TOP_K

    for _, part in rankings.groupby(
        ["WINDOW", "REPRESENTATION"]
    ):
        assert len(part) == TOP_K
        assert part["RANK"].tolist() == list(
            range(1, TOP_K + 1)
        )
        assert part["ANTECEDENT_ID"].is_unique

    # Gravacao atomica por arquivo.
    temporary_rank = (
        CHECKPOINTS /
        f"{fid}_bm25_top50.partial.parquet"
    )

    temporary_audit = (
        CHECKPOINTS /
        f"{fid}_bm25_audit.partial.json"
    )

    assert not temporary_rank.exists()
    assert not temporary_audit.exists()

    rankings.to_parquet(
        temporary_rank,
        index=False
    )

    temporary_audit.write_text(
        json.dumps(
            audit_parts,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    temporary_rank.rename(rank_file)
    temporary_audit.rename(audit_file)

    new += 1

    if int(fid[-4:]) % 12 == 0:
        print(
            f"{fid}: concluido "
            f"| novos={new} "
            f"| reutilizados={reused}"
        )

    del antecedents
    gc.collect()

# ------------------------------------------------------------
# 6. AUDITORIA NACIONAL
# ------------------------------------------------------------

all_ranks = []
all_audits = []

for fid in focals["FOCAL_ID"]:

    rank_file = (
        CHECKPOINTS / f"{fid}_bm25_top50.parquet"
    )

    audit_file = (
        CHECKPOINTS / f"{fid}_bm25_audit.json"
    )

    assert rank_file.is_file()
    assert audit_file.is_file()

    ranks = pd.read_parquet(rank_file)

    assert len(ranks) == 4 * TOP_K
    assert ranks["FOCAL_ID"].eq(fid).all()

    all_ranks.append(ranks)

    all_audits.extend(
        json.loads(
            audit_file.read_text(
                encoding="utf-8"
            )
        )
    )

national_ranks = pd.concat(
    all_ranks,
    ignore_index=True
)

national_audit = pd.DataFrame(
    all_audits
)

assert len(national_ranks) == 108 * 4 * TOP_K
assert len(national_audit) == 108 * 4

assert national_ranks[
    "FOCAL_ID"
].nunique() == 108

# Verificar ausencia de candidatos repetidos.
assert not national_ranks.duplicated([
    "FOCAL_ID",
    "WINDOW",
    "REPRESENTATION",
    "ANTECEDENT_ID"
]).any()

# ------------------------------------------------------------
# 7. RESUMO DAS CONDICOES
# ------------------------------------------------------------

summary = (
    national_audit.groupby([
        "WINDOW",
        "REPRESENTATION"
    ])
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_CANDIDATES=(
            "N_CANDIDATES", "mean"
        ),
        MEAN_QUERY_TERMS=(
            "N_QUERY_TERMS", "mean"
        ),
        MEAN_QUERY_TERMS_IN_VOCAB=(
            "N_QUERY_TERMS_IN_VOCAB", "mean"
        ),
        MEAN_POSITIVE_SHARE=(
            "SHARE_POSITIVE_SCORES", "mean"
        ),
        FOCALS_WITH_ZERO_TOP50=(
            "N_ZERO_TOP50",
            lambda x: int((x > 0).sum())
        ),
        MEAN_TOP1_SCORE=(
            "TOP1_SCORE", "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 8. SALVAR RESULTADOS
# ------------------------------------------------------------

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S_%fZ")

REPORT = OUT / f"analysis_{STAMP}"

REPORT.mkdir(
    parents=True,
    exist_ok=False
)

national_ranks.to_parquet(
    REPORT / "bm25_national_top50_21600.parquet",
    index=False
)

national_audit.to_csv(
    REPORT / "bm25_focal_audit_432.csv",
    index=False
)

summary.to_csv(
    REPORT / "bm25_condition_summary.csv",
    index=False
)

manifest = {
    **CONFIG,
    "n_completed_focals": 108,
    "n_rankings": 432,
    "n_top50_rows": len(national_ranks),
    "new_focals_this_run": new,
    "reused_focals_this_run": reused,
    "source_lexical_corpus": str(ANNUAL),
    "source_candidate_checkpoints": (
        str(SEMANTIC_CHECKPOINTS)
    ),
    "important_limit": (
        "Zero-scored documents may enter Top50 via "
        "deterministic ID ties; such rankings must "
        "be flagged in concordance analyses."
    )
}

(REPORT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 9. RESULTADOS
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("ETAPA 23B — RESULTADOS FINAIS")
print("=" * 68)

print("\n1. COBERTURA DO BENCHMARK")
print("Focais:", national_audit["FOCAL_ID"].nunique())
print("Rankings:", len(national_audit))
print("Linhas Top50:", len(national_ranks))
print("Novos checkpoints:", new)
print("Checkpoints reutilizados:", reused)

print("\n2. AUDITORIA LEXICAL")
print(
    summary.round(4).to_string(index=False)
)

print("\n3. RANKINGS COM PONTUACOES ZERO")
print(
    national_audit.groupby(
        ["WINDOW", "REPRESENTATION"]
    )["N_ZERO_TOP50"]
    .agg(
        N_RANKINGS_WITH_ZERO=lambda x: int(
            (x > 0).sum()
        ),
        MAX_ZERO_IN_TOP50="max"
    )
    .to_string()
)

print("\n4. INTEGRIDADE")
print("Todos os focais: PASS")
print("Universos de candidatos: Etapa 21")
print("Rankings completos: PASS")

print("\nDiretorio:", REPORT)
print("ETAPA 23B CONCLUIDA")


ETAPA 23B — BM25 NACIONAL

Carregando corpus lexical 2013-2024...
2013 : 59658 documentos
2014 : 62130 documentos
2015 : 67057 documentos
2016 : 70749 documentos
2017 : 70210 documentos
2018 : 69601 documentos
2019 : 72022 documentos
2020 : 57472 documentos
2021 : 48976 documentos
2022 : 44131 documentos
2023 : 20762 documentos
2024 : 11 documentos

Corpus lexical validado: 642779
NAT-0012: concluido | novos=12 | reutilizados=0
NAT-0024: concluido | novos=24 | reutilizados=0
NAT-0036: concluido | novos=36 | reutilizados=0
NAT-0048: concluido | novos=48 | reutilizados=0
NAT-0060: concluido | novos=60 | reutilizados=0
NAT-0072: concluido | novos=72 | reutilizados=0
NAT-0084: concluido | novos=84 | reutilizados=0
NAT-0096: concluido | novos=96 | reutilizados=0
NAT-0108: concluido | novos=108 | reutilizados=0

ETAPA 23B — RESULTADOS FINAIS

1. COBERTURA DO BENCHMARK
Focais: 108
Rankings: 432
Linhas Top50: 21600
Novos checkpoints: 108
Checkpoints reutilizados: 0

2. AUDITORIA LEXICAL
    WI

In [15]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 23C
# CONCORDANCIA LEXICAL-SEMANTICA NACIONAL
#
# 108 focais | 9 grandes areas
# BM25 curto, BM25 completo, M2, M4
# Historico completo e fronteira 3Y
# Top 10, Top 30, Top 50
#
# Auditoria cruzada com Etapa 22A
# Bootstrap estratificado: 3000 replicacoes
# CPU | Somente leitura dos resultados originais
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. DIRETORIOS — MONTAGEM RECUPERADA
# ------------------------------------------------------------

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

LAB = ROOT / "08_epistemic_measurement_validation"
EXP = LAB / "05_national_expansion"

SEMANTIC = (
    EXP / "21_semantic_controlled_national_v1"
    / "checkpoints"
)

BM25 = (
    EXP / "23B_BM25_national_v1"
    / "checkpoints"
)

SAMPLE = (
    EXP / "focal_sample_108_20261020T190807_559853Z"
)

STABILITY_22A = (
    EXP
    / "22A_national_stability_20261020T203226_948841Z"
    / "focal_level_stability_1296.parquet"
)

for folder in [ROOT, SEMANTIC, BM25, SAMPLE]:
    assert folder.is_dir(), (
        f"Diretorio nao localizado: {folder}"
    )

assert STABILITY_22A.is_file(), (
    f"Auditoria 22A nao localizada: {STABILITY_22A}"
)

TOP_K = [10, 30, 50]
METHODS = ["BM25_short", "BM25_full", "M2", "M4"]
WINDOWS = ["historical", "frontier3y"]

COMPARISONS = [
    ("BM25_short", "M2", "LEXICAL_SEMANTIC_MATCHED_SHORT"),
    ("BM25_full", "M4", "LEXICAL_SEMANTIC_MATCHED_FULL"),
    ("BM25_short", "BM25_full", "LEXICAL_TEXT_EXPANSION"),
    ("M2", "M4", "SEMANTIC_TEXT_EXPANSION"),
    ("BM25_short", "M4", "CROSS_SHORT_FULL"),
    ("BM25_full", "M2", "CROSS_FULL_SHORT")
]

B = 3000
SEED = 20261020

# ------------------------------------------------------------
# 2. AMOSTRA E GRANDES AREAS
# ------------------------------------------------------------

mapping = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
)

sample = pd.read_parquet(
    SAMPLE / "national_focals_108_RESTRICTED.parquet"
)

sample["ID_TRABALHO"] = (
    sample["ID_TRABALHO"].astype(str)
)

areas = mapping[[
    "FOCAL_ID", "ID_TRABALHO"
]].merge(
    sample[[
        "ID_TRABALHO",
        "NM_GRANDE_AREA_CONHECIMENTO"
    ]],
    on="ID_TRABALHO",
    validate="one_to_one"
).rename(columns={
    "NM_GRANDE_AREA_CONHECIMENTO": "GRANDE_AREA"
})

assert len(areas) == 108
assert areas["FOCAL_ID"].is_unique
assert areas["GRANDE_AREA"].nunique() == 9
assert areas.groupby("GRANDE_AREA").size().eq(12).all()

mapping["ANO"] = mapping["ANO"].astype(int)

focal_year = mapping.set_index("FOCAL_ID")["ANO"].to_dict()

focal_ids = sorted(mapping["FOCAL_ID"])

assert focal_ids == [
    f"NAT-{i:04d}" for i in range(1, 109)
]

# ------------------------------------------------------------
# 3. FUNCOES DE RANKING E SOBREPOSICAO
# ------------------------------------------------------------

def semantic_top50(scores, modality):
    """
    Ranking por cosseno descendente,
    com desempate por ID ascendente.
    """

    ids = scores[
        "ANTECEDENT_ID"
    ].astype(str).to_numpy()

    values = scores[
        f"COSINE_{modality}"
    ].to_numpy(dtype=np.float64)

    assert np.isfinite(values).all()
    assert len(ids) >= 50

    order = np.lexsort(
        (ids, -values)
    )[:50]

    return ids[order].tolist()


def overlap_rate(a, b, k):

    aa = set(a[:k])
    bb = set(b[:k])

    assert len(aa) == k
    assert len(bb) == k

    return len(aa & bb) / k


def jaccard_rate(a, b, k):

    aa = set(a[:k])
    bb = set(b[:k])

    return len(aa & bb) / len(aa | bb)


def extract_bm25_rank(
    data, window, representation
):

    selected = data[
        (data["WINDOW"] == window)
        & (data["REPRESENTATION"] == representation)
    ].sort_values("RANK")

    assert len(selected) == 50

    assert selected["RANK"].tolist() == list(
        range(1, 51)
    )

    assert selected["BM25_SCORE"].gt(0).all()

    ids = selected[
        "ANTECEDENT_ID"
    ].astype(str).tolist()

    assert len(set(ids)) == 50

    return ids

# ------------------------------------------------------------
# 4. RECONSTRUIR RANKINGS E CALCULAR CONCORDANCIA
# ------------------------------------------------------------

records = []
audit_rows = []

print("=" * 68)
print("ETAPA 23C — CONCORDANCIA LEXICAL-SEMANTICA")
print("=" * 68)

for fid in focal_ids:

    year = focal_year[fid]

    score_file = (
        SEMANTIC / f"{fid}_scores.parquet"
    )

    bm25_file = (
        BM25 / f"{fid}_bm25_top50.parquet"
    )

    assert score_file.is_file(), score_file
    assert bm25_file.is_file(), bm25_file

    scores = pd.read_parquet(
        score_file,
        columns=[
            "ANTECEDENT_ID",
            "ANTECEDENT_YEAR",
            "COSINE_M2",
            "COSINE_M4"
        ]
    )

    lexical = pd.read_parquet(
        bm25_file
    )

    scores["ANTECEDENT_ID"] = (
        scores["ANTECEDENT_ID"].astype(str)
    )

    scores["ANTECEDENT_YEAR"] = (
        scores["ANTECEDENT_YEAR"].astype(int)
    )

    assert scores["ANTECEDENT_ID"].is_unique
    assert scores["ANTECEDENT_YEAR"].lt(year).all()
    assert len(lexical) == 200

    candidate_universe = set(
        scores["ANTECEDENT_ID"]
    )

    rankings = {}

    for window in WINDOWS:

        if window == "historical":
            eligible = scores
        else:
            eligible = scores[
                scores["ANTECEDENT_YEAR"].between(
                    year - 3, year - 1
                )
            ]

        assert len(eligible) >= 50

        rankings[window] = {
            "M2": semantic_top50(eligible, "M2"),
            "M4": semantic_top50(eligible, "M4"),
            "BM25_short": extract_bm25_rank(
                lexical, window, "short"
            ),
            "BM25_full": extract_bm25_rank(
                lexical, window, "full"
            )
        }

        eligible_ids = set(
            eligible["ANTECEDENT_ID"]
        )

        for method, ids in rankings[window].items():

            assert len(ids) == 50
            assert len(set(ids)) == 50
            assert set(ids).issubset(eligible_ids)

        audit_rows.append({
            "FOCAL_ID": fid,
            "WINDOW": window,
            "N_CANDIDATES": len(eligible),
            "RANKINGS_VALID": 4
        })

        # Seis comparacoes entre representacoes.
        for left, right, label in COMPARISONS:

            a = rankings[window][left]
            b = rankings[window][right]

            for k in TOP_K:

                records.append({
                    "FOCAL_ID": fid,
                    "COMPARISON_TYPE": "method_pair",
                    "CONDITION": window,
                    "COMPARISON": label,
                    "METHOD_A": left,
                    "METHOD_B": right,
                    "K": k,
                    "OVERLAP_RATE": overlap_rate(
                        a, b, k
                    ),
                    "JACCARD": jaccard_rate(
                        a, b, k
                    )
                })

    # Estabilidade da janela temporal por metodo.
    for method in METHODS:

        a = rankings["historical"][method]
        b = rankings["frontier3y"][method]

        for k in TOP_K:

            records.append({
                "FOCAL_ID": fid,
                "COMPARISON_TYPE": "temporal_window",
                "CONDITION": method,
                "COMPARISON": "HISTORICAL_VS_FRONTIER3Y",
                "METHOD_A": method,
                "METHOD_B": method,
                "K": k,
                "OVERLAP_RATE": overlap_rate(
                    a, b, k
                ),
                "JACCARD": jaccard_rate(
                    a, b, k
                )
            })

    if int(fid[-4:]) % 12 == 0:
        print(
            f"{fid}: concluido | "
            f"{len(records)} comparacoes"
        )

# ------------------------------------------------------------
# 5. CONSOLIDAR
# ------------------------------------------------------------

df = pd.DataFrame(records)

df = df.merge(
    areas[["FOCAL_ID", "GRANDE_AREA"]],
    on="FOCAL_ID",
    how="left",
    validate="many_to_one"
)

audit = pd.DataFrame(audit_rows)

assert len(df) == 108 * 3 * (2 * 6 + 4)
assert len(df) == 5184

assert len(audit) == 216
assert df["FOCAL_ID"].nunique() == 108

assert df["OVERLAP_RATE"].between(0, 1).all()
assert df["JACCARD"].between(0, 1).all()

# ------------------------------------------------------------
# 6. AUDITORIA CRUZADA COM ETAPA 22A
# ------------------------------------------------------------

previous = pd.read_parquet(STABILITY_22A)

# Converter as comparacoes da 23C para
# a nomenclatura da Etapa 22A.

semantic_representations = df[
    (df["COMPARISON_TYPE"] == "method_pair")
    & (
        df["COMPARISON"]
        == "SEMANTIC_TEXT_EXPANSION"
    )
][[
    "FOCAL_ID", "CONDITION", "K", "OVERLAP_RATE"
]].copy()

semantic_representations[
    "COMPARISON_TYPE"
] = "representation"

semantic_temporal = df[
    (df["COMPARISON_TYPE"] == "temporal_window")
    & (df["CONDITION"].isin(["M2", "M4"]))
][[
    "FOCAL_ID", "CONDITION", "K", "OVERLAP_RATE"
]].copy()

semantic_temporal[
    "COMPARISON_TYPE"
] = "temporal_window"

current_semantic = pd.concat(
    [
        semantic_representations,
        semantic_temporal
    ],
    ignore_index=True
)

keys = [
    "FOCAL_ID",
    "COMPARISON_TYPE",
    "CONDITION",
    "K"
]

previous_semantic = previous[
    keys + ["OVERLAP_RATE"]
].copy()

assert len(previous_semantic) == 1296
assert len(current_semantic) == 1296

comparison = current_semantic.merge(
    previous_semantic,
    on=keys,
    how="outer",
    suffixes=("_23C", "_22A"),
    validate="one_to_one",
    indicator=True
)

assert comparison["_merge"].eq("both").all()

max_deviation = float(
    np.max(
        np.abs(
            comparison["OVERLAP_RATE_23C"]
            - comparison["OVERLAP_RATE_22A"]
        )
    )
)

print("\nAUDITORIA ETAPA 22A")
print("Linhas verificadas:", len(comparison))
print("Maior diferenca:", max_deviation)

assert max_deviation < 1e-10, (
    "A reconstrucao dos rankings semanticos "
    "nao reproduziu a Etapa 22A. "
    "A analise foi interrompida para auditoria."
)

print("Reprodutibilidade semantica: PASS")

# ------------------------------------------------------------
# 7. ESTIMATIVAS DESCRITIVAS
# ------------------------------------------------------------

group_keys = [
    "COMPARISON_TYPE",
    "CONDITION",
    "COMPARISON",
    "METHOD_A",
    "METHOD_B",
    "K"
]

summary = (
    df.groupby(group_keys)
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_OVERLAP=("OVERLAP_RATE", "mean"),
        MEDIAN_OVERLAP=("OVERLAP_RATE", "median"),
        SD_OVERLAP=("OVERLAP_RATE", "std"),
        MEAN_JACCARD=("JACCARD", "mean")
    )
    .reset_index()
)

assert len(summary) == 48
assert summary["N_FOCALS"].eq(108).all()

# ------------------------------------------------------------
# 8. BOOTSTRAP ESTRATIFICADO
# ------------------------------------------------------------

# Uma linha por focal, com os 48 resultados
# preservando a dependencia entre metodos.

wide = df.pivot(
    index=["FOCAL_ID", "GRANDE_AREA"],
    columns=[
        "COMPARISON_TYPE",
        "CONDITION",
        "COMPARISON",
        "K"
    ],
    values="OVERLAP_RATE"
)

wide = wide.sort_index()

assert wide.shape == (108, 48)
assert not wide.isna().any().any()

bootstrap_cols = list(wide.columns)
values = wide.to_numpy(dtype=np.float64)

wide_index = wide.index.to_frame(
    index=False
)

strata = [
    np.flatnonzero(
        wide_index["GRANDE_AREA"].eq(area)
    )
    for area in sorted(
        wide_index["GRANDE_AREA"].unique()
    )
]

assert len(strata) == 9
assert all(len(ix) == 12 for ix in strata)

rng = np.random.default_rng(SEED)

boot = np.empty(
    (B, values.shape[1]),
    dtype=np.float64
)

for b in range(B):

    selected = np.concatenate([
        rng.choice(ix, size=12, replace=True)
        for ix in strata
    ])

    boot[b] = values[selected].mean(axis=0)

ci_rows = []

for j, col in enumerate(bootstrap_cols):

    ctype, condition, label, k = col

    ci_rows.append({
        "COMPARISON_TYPE": ctype,
        "CONDITION": condition,
        "COMPARISON": label,
        "K": int(k),
        "BOOT_CI_LOW": float(
            np.quantile(boot[:, j], 0.025)
        ),
        "BOOT_CI_HIGH": float(
            np.quantile(boot[:, j], 0.975)
        )
    })

ci = pd.DataFrame(ci_rows)

summary = summary.merge(
    ci,
    on=[
        "COMPARISON_TYPE",
        "CONDITION",
        "COMPARISON",
        "K"
    ],
    how="left",
    validate="one_to_one"
)

assert summary["BOOT_CI_LOW"].notna().all()

# ------------------------------------------------------------
# 9. CONTRASTES PAREADOS ENTRE TIPOS DE CONCORDANCIA
# ------------------------------------------------------------

contrast_rows = []

for window in WINDOWS:

    for k in TOP_K:

        part = df[
            (df["COMPARISON_TYPE"] == "method_pair")
            & (df["CONDITION"] == window)
            & (df["K"] == k)
        ]

        matrix = part.pivot(
            index="FOCAL_ID",
            columns="COMPARISON",
            values="OVERLAP_RATE"
        )

        # O mesmo focal entra em cada contraste.
        contrasts = {
            "BM25_SHORT_VS_M2_MINUS_M2_VS_M4":
                matrix["LEXICAL_SEMANTIC_MATCHED_SHORT"]
                - matrix["SEMANTIC_TEXT_EXPANSION"],

            "BM25_FULL_VS_M4_MINUS_M2_VS_M4":
                matrix["LEXICAL_SEMANTIC_MATCHED_FULL"]
                - matrix["SEMANTIC_TEXT_EXPANSION"],

            "BM25_SHORT_VS_FULL_MINUS_M2_VS_M4":
                matrix["LEXICAL_TEXT_EXPANSION"]
                - matrix["SEMANTIC_TEXT_EXPANSION"]
        }

        area_order = wide_index[[
            "FOCAL_ID", "GRANDE_AREA"
        ]]

        for label, series in contrasts.items():

            ordered = area_order.merge(
                series.rename("DIFFERENCE").reset_index(),
                on="FOCAL_ID",
                validate="one_to_one"
            )

            x = ordered[
                "DIFFERENCE"
            ].to_numpy(dtype=np.float64)

            draws = np.empty(B, dtype=np.float64)

            for b in range(B):

                indices = np.concatenate([
                    rng.choice(
                        ix, size=12, replace=True
                    )
                    for ix in strata
                ])

                draws[b] = x[indices].mean()

            contrast_rows.append({
                "WINDOW": window,
                "K": k,
                "CONTRAST": label,
                "MEAN_DIFFERENCE": float(
                    x.mean()
                ),
                "CI_LOW": float(
                    np.quantile(draws, 0.025)
                ),
                "CI_HIGH": float(
                    np.quantile(draws, 0.975)
                )
            })

contrasts = pd.DataFrame(contrast_rows)

# ------------------------------------------------------------
# 10. HETEROGENEIDADE DISCIPLINAR
# ------------------------------------------------------------

by_area = (
    df.groupby(
        group_keys + ["GRANDE_AREA"]
    )
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_OVERLAP=("OVERLAP_RATE", "mean")
    )
    .reset_index()
)

assert by_area["N_FOCALS"].eq(12).all()

# ------------------------------------------------------------
# 11. SALVAR PRODUTOS
# ------------------------------------------------------------

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S_%fZ")

OUT = (
    EXP / f"23C_lexical_semantic_agreement_{STAMP}"
)

OUT.mkdir(parents=True, exist_ok=False)

df.to_parquet(
    OUT / "focal_level_agreement_5184.parquet",
    index=False
)

summary.to_csv(
    OUT / "national_agreement_summary.csv",
    index=False
)

contrasts.to_csv(
    OUT / "paired_agreement_contrasts.csv",
    index=False
)

by_area.to_csv(
    OUT / "agreement_by_grand_area.csv",
    index=False
)

audit.to_csv(
    OUT / "ranking_candidate_audit.csv",
    index=False
)

comparison.to_csv(
    OUT / "cross_audit_stage22A.csv",
    index=False
)

manifest = {
    "stage": "23C",
    "n_focals": 108,
    "n_methods": 4,
    "n_windows": 2,
    "top_k": TOP_K,
    "n_focal_level_comparisons": len(df),
    "n_summary_rows": len(summary),
    "n_bootstrap": B,
    "bootstrap_seed": SEED,
    "bootstrap_strata": 9,
    "cross_audit_22A_max_deviation": max_deviation,
    "agreement_metric": (
        "intersection_of_top_k_divided_by_k"
    ),
    "ranking_reconstruction": (
        "cosine_desc_then_antecedent_id_asc"
    ),
    "inference_scope": (
        "balanced_second_phase_sample; "
        "not national_population_inference"
    ),
    "human_ground_truth": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 12. RESULTADOS PARA INTERPRETACAO
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("ETAPA 23C — RESULTADOS")
print("=" * 72)

print("\n1. CONCORDANCIA ENTRE METODOS — TOP30")

table30 = summary[
    (summary["COMPARISON_TYPE"] == "method_pair")
    & (summary["K"] == 30)
][[
    "CONDITION",
    "METHOD_A",
    "METHOD_B",
    "MEAN_OVERLAP",
    "BOOT_CI_LOW",
    "BOOT_CI_HIGH"
]]

print(
    table30.round(4).to_string(index=False)
)

print("\n2. ESTABILIDADE TEMPORAL — TOP30")

temporal = summary[
    (summary["COMPARISON_TYPE"] == "temporal_window")
    & (summary["K"] == 30)
][[
    "CONDITION",
    "MEAN_OVERLAP",
    "BOOT_CI_LOW",
    "BOOT_CI_HIGH"
]]

print(
    temporal.round(4).to_string(index=False)
)

print("\n3. CONTRASTES PAREADOS — TOP30")

print(
    contrasts[
        contrasts["K"] == 30
    ].round(4).to_string(index=False)
)

print("\n4. AUDITORIA FINAL")
print("Focais:", df["FOCAL_ID"].nunique())
print("Comparacoes:", len(df))
print("Estimativas agregadas:", len(summary))
print("Bootstrap:", B)
print("Etapa 22A reproduzida:", max_deviation < 1e-10)
print("Status: PASS")

print("\nDiretorio:", OUT)
print("ETAPA 23C CONCLUIDA")


ETAPA 23C — CONCORDANCIA LEXICAL-SEMANTICA
NAT-0012: concluido | 576 comparacoes
NAT-0024: concluido | 1152 comparacoes
NAT-0036: concluido | 1728 comparacoes
NAT-0048: concluido | 2304 comparacoes
NAT-0060: concluido | 2880 comparacoes
NAT-0072: concluido | 3456 comparacoes
NAT-0084: concluido | 4032 comparacoes
NAT-0096: concluido | 4608 comparacoes
NAT-0108: concluido | 5184 comparacoes

AUDITORIA ETAPA 22A
Linhas verificadas: 1296
Maior diferenca: 0.0
Reprodutibilidade semantica: PASS

ETAPA 23C — RESULTADOS

1. CONCORDANCIA ENTRE METODOS — TOP30
 CONDITION   METHOD_A  METHOD_B  MEAN_OVERLAP  BOOT_CI_LOW  BOOT_CI_HIGH
frontier3y  BM25_full        M2        0.1701       0.1503        0.1907
frontier3y BM25_short        M4        0.1651       0.1444        0.1864
frontier3y  BM25_full        M4        0.2648       0.2423        0.2886
frontier3y BM25_short        M2        0.2932       0.2660        0.3216
frontier3y BM25_short BM25_full        0.2321       0.2086        0.2574
front

In [16]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 23D
# ROBUSTEZ FINAL DO BENCHMARK LEXICAL-SEMANTICO
#
# 108 focais | 9 grandes areas
# Top 10, 30 e 50
# Sensibilidade temporal: excluir 2016
# Ajuste descritivo para sobreposicao aleatoria k/N
# Contrastes pareados com bootstrap estratificado
#
# CPU | Nao recalcula embeddings, BM25 ou rankings
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

EXP = (
    ROOT / "08_epistemic_measurement_validation"
    / "05_national_expansion"
)

SOURCE = (
    EXP /
    "23C_lexical_semantic_agreement_20261021T024747_361222Z"
)

SAMPLE = (
    EXP /
    "focal_sample_108_20261020T190807_559853Z"
)

assert SOURCE.is_dir()
assert SAMPLE.is_dir()

B = 3000
SEED = 20261020

df = pd.read_parquet(
    SOURCE / "focal_level_agreement_5184.parquet"
)

audit = pd.read_csv(
    SOURCE / "ranking_candidate_audit.csv",
    dtype={"FOCAL_ID": str}
)

mapping = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={"FOCAL_ID": str}
)

mapping["ANO"] = mapping["ANO"].astype(int)

assert len(df) == 5184
assert df["FOCAL_ID"].nunique() == 108
assert len(audit) == 216

df = df.merge(
    mapping[["FOCAL_ID", "ANO"]],
    on="FOCAL_ID",
    how="left",
    validate="many_to_one"
)

assert df["ANO"].notna().all()
assert df["GRANDE_AREA"].nunique() == 9

# ------------------------------------------------------------
# 1. AJUSTE DESCRITIVO AO ACASO
# ------------------------------------------------------------

method_rows = df[
    df["COMPARISON_TYPE"] == "method_pair"
].merge(
    audit[[
        "FOCAL_ID", "WINDOW", "N_CANDIDATES"
    ]].rename(columns={"WINDOW": "CONDITION"}),
    on=["FOCAL_ID", "CONDITION"],
    how="left",
    validate="many_to_one"
)

assert method_rows["N_CANDIDATES"].notna().all()
assert (
    method_rows["N_CANDIDATES"] >= method_rows["K"]
).all()

method_rows["RANDOM_EXPECTED"] = (
    method_rows["K"] / method_rows["N_CANDIDATES"]
)

method_rows["EXCESS_OVER_RANDOM"] = (
    method_rows["OVERLAP_RATE"]
    - method_rows["RANDOM_EXPECTED"]
)

method_rows["CHANCE_ADJUSTED"] = (
    method_rows["EXCESS_OVER_RANDOM"]
    / (1 - method_rows["RANDOM_EXPECTED"])
)

chance = (
    method_rows.groupby([
        "CONDITION", "COMPARISON",
        "METHOD_A", "METHOD_B", "K"
    ])
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        OBSERVED=("OVERLAP_RATE", "mean"),
        RANDOM_EXPECTED=("RANDOM_EXPECTED", "mean"),
        EXCESS=("EXCESS_OVER_RANDOM", "mean"),
        CHANCE_ADJUSTED=("CHANCE_ADJUSTED", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. FUNCAO BOOTSTRAP ESTRATIFICADO
# ------------------------------------------------------------

def paired_bootstrap(frame, value_col, b=B, seed=SEED):

    assert frame["FOCAL_ID"].is_unique
    assert frame["GRANDE_AREA"].notna().all()

    frame = frame.sort_values([
        "GRANDE_AREA", "FOCAL_ID"
    ]).reset_index(drop=True)

    values = frame[value_col].to_numpy(
        dtype=np.float64
    )

    strata = [
        group.index.to_numpy()
        for _, group in frame.groupby(
            "GRANDE_AREA", sort=True
        )
    ]

    rng = np.random.default_rng(seed)

    draws = np.empty(b, dtype=np.float64)

    for i in range(b):
        selected = np.concatenate([
            rng.choice(
                ix, size=len(ix), replace=True
            )
            for ix in strata
        ])

        draws[i] = values[selected].mean()

    return {
        "N_FOCALS": len(values),
        "MEAN": float(values.mean()),
        "CI_LOW": float(np.quantile(draws, 0.025)),
        "CI_HIGH": float(np.quantile(draws, 0.975))
    }

# ------------------------------------------------------------
# 3. ESTABILIDADE TEMPORAL — COM E SEM 2016
# ------------------------------------------------------------

temporal = df[
    df["COMPARISON_TYPE"] == "temporal_window"
].copy()

assert len(temporal) == 108 * 4 * 3

# Em 2016, historico e fronteira
# precisam coincidir integralmente.
in_2016 = temporal[temporal["ANO"] == 2016]

assert in_2016["FOCAL_ID"].nunique() == 14
assert np.allclose(
    in_2016["OVERLAP_RATE"], 1.0
)

temporal_rows = []

for period, subset in [
    ("2016-2024", temporal),
    ("2017-2024", temporal[temporal["ANO"] >= 2017])
]:

    for (method, k), group in subset.groupby([
        "CONDITION", "K"
    ]):

        estimate = paired_bootstrap(
            group[[
                "FOCAL_ID",
                "GRANDE_AREA",
                "OVERLAP_RATE"
            ]],
            "OVERLAP_RATE"
        )

        temporal_rows.append({
            "PERIOD": period,
            "METHOD": method,
            "K": int(k),
            **estimate
        })

temporal_summary = pd.DataFrame(temporal_rows)

# ------------------------------------------------------------
# 4. CONTRASTES TEMPORAIS PAREADOS — SEM 2016
# ------------------------------------------------------------

temporal_2017 = temporal[
    temporal["ANO"] >= 2017
]

temporal_wide = temporal_2017.pivot(
    index=["FOCAL_ID", "GRANDE_AREA"],
    columns=["CONDITION", "K"],
    values="OVERLAP_RATE"
)

temporal_wide.columns = [
    f"{method}_K{k}"
    for method, k in temporal_wide.columns
]

temporal_wide = temporal_wide.reset_index()

assert len(temporal_wide) == 94

temporal_contrasts = []

for k in [10, 30, 50]:

    definitions = {
        "M2_MINUS_BM25_SHORT": (
            f"M2_K{k}", f"BM25_short_K{k}"
        ),
        "M4_MINUS_BM25_FULL": (
            f"M4_K{k}", f"BM25_full_K{k}"
        ),
        "M4_MINUS_M2": (
            f"M4_K{k}", f"M2_K{k}"
        ),
        "BM25_FULL_MINUS_SHORT": (
            f"BM25_full_K{k}", f"BM25_short_K{k}"
        )
    }

    for label, (left, right) in definitions.items():

        tmp = temporal_wide[[
            "FOCAL_ID", "GRANDE_AREA"
        ]].copy()

        tmp["DIFFERENCE"] = (
            temporal_wide[left]
            - temporal_wide[right]
        )

        estimate = paired_bootstrap(
            tmp, "DIFFERENCE"
        )

        temporal_contrasts.append({
            "K": k,
            "CONTRAST": label,
            **estimate
        })

temporal_contrasts = pd.DataFrame(
    temporal_contrasts
)

# ------------------------------------------------------------
# 5. CONTRASTES DE ALINHAMENTO TEXTUAL
# ------------------------------------------------------------

# Cada contraste usa os mesmos focais:
# matched = mesma composicao textual
# crossed = composicao textual diferente.

alignment_rows = []

for window in ["historical", "frontier3y"]:

    for k in [10, 30, 50]:

        subset = df[
            (df["COMPARISON_TYPE"] == "method_pair")
            & (df["CONDITION"] == window)
            & (df["K"] == k)
        ]

        wide = subset.pivot(
            index=["FOCAL_ID", "GRANDE_AREA"],
            columns="COMPARISON",
            values="OVERLAP_RATE"
        ).reset_index()

        definitions = {
            "BM25_SHORT_M2_MINUS_BM25_FULL_M2": (
                "LEXICAL_SEMANTIC_MATCHED_SHORT",
                "CROSS_FULL_SHORT"
            ),
            "BM25_FULL_M4_MINUS_BM25_SHORT_M4": (
                "LEXICAL_SEMANTIC_MATCHED_FULL",
                "CROSS_SHORT_FULL"
            ),
            "MATCHED_AVERAGE_MINUS_CROSSED_AVERAGE": None
        }

        for label, pair in definitions.items():

            tmp = wide[[
                "FOCAL_ID", "GRANDE_AREA"
            ]].copy()

            if pair is not None:
                left, right = pair
                tmp["DIFFERENCE"] = (
                    wide[left] - wide[right]
                )
            else:
                matched = (
                    wide["LEXICAL_SEMANTIC_MATCHED_SHORT"]
                    + wide["LEXICAL_SEMANTIC_MATCHED_FULL"]
                ) / 2

                crossed = (
                    wide["CROSS_FULL_SHORT"]
                    + wide["CROSS_SHORT_FULL"]
                ) / 2

                tmp["DIFFERENCE"] = (
                    matched - crossed
                )

            estimate = paired_bootstrap(
                tmp, "DIFFERENCE"
            )

            alignment_rows.append({
                "WINDOW": window,
                "K": k,
                "CONTRAST": label,
                **estimate
            })

alignment_summary = pd.DataFrame(
    alignment_rows
)

# ------------------------------------------------------------
# 6. HETEROGENEIDADE DISCIPLINAR — TOP30
# ------------------------------------------------------------

area_summary = (
    df[
        (df["COMPARISON_TYPE"] == "method_pair")
        & (df["K"] == 30)
    ]
    .groupby([
        "GRANDE_AREA",
        "CONDITION",
        "COMPARISON"
    ])
    .agg(
        N_FOCALS=("FOCAL_ID", "nunique"),
        MEAN_OVERLAP=("OVERLAP_RATE", "mean")
    )
    .reset_index()
)

assert area_summary["N_FOCALS"].eq(12).all()

# ------------------------------------------------------------
# 7. SALVAR
# ------------------------------------------------------------

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S_%fZ")

OUT = EXP / f"23D_final_robustness_{STAMP}"
OUT.mkdir(parents=True, exist_ok=False)

chance.to_csv(
    OUT / "chance_adjusted_agreement.csv",
    index=False
)

temporal_summary.to_csv(
    OUT / "temporal_sensitivity_2016.csv",
    index=False
)

temporal_contrasts.to_csv(
    OUT / "paired_temporal_contrasts_excluding_2016.csv",
    index=False
)

alignment_summary.to_csv(
    OUT / "matched_vs_crossed_text_contrasts.csv",
    index=False
)

area_summary.to_csv(
    OUT / "agreement_by_grand_area_top30.csv",
    index=False
)

manifest = {
    "stage": "23D",
    "n_focals_full": 108,
    "n_focals_excluding_2016": 94,
    "n_bootstrap": B,
    "bootstrap_seed": SEED,
    "bootstrapping": (
        "Resample focal documents within nine "
        "grand-area strata, retaining original "
        "stratum size in each subset."
    ),
    "chance_reference": (
        "Uniform independent selection "
        "of k documents from the same N candidates."
    ),
    "national_design_based_inference": False,
    "new_bm25_or_embedding_calculations": False
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. RESULTADOS
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("ETAPA 23D — RESULTADOS")
print("=" * 68)

print("\n1. CONCORDANCIA ACIMA DO ACASO — TOP30")
print(
    chance[
        chance["K"] == 30
    ][[
        "CONDITION", "METHOD_A", "METHOD_B",
        "OBSERVED", "RANDOM_EXPECTED",
        "CHANCE_ADJUSTED"
    ]]
    .round(5)
    .to_string(index=False)
)

print("\n2. ESTABILIDADE TEMPORAL — TOP30")
print(
    temporal_summary[
        temporal_summary["K"] == 30
    ]
    .round(4)
    .to_string(index=False)
)

print("\n3. CONTRASTES TEMPORAIS — SEM 2016, TOP30")
print(
    temporal_contrasts[
        temporal_contrasts["K"] == 30
    ]
    .round(4)
    .to_string(index=False)
)

print("\n4. ALINHAMENTO TEXTUAL — TOP30")
print(
    alignment_summary[
        alignment_summary["K"] == 30
    ]
    .round(4)
    .to_string(index=False)
)

print("\n5. AUDITORIA")
print("Focais totais:", df["FOCAL_ID"].nunique())
print("Focais 2017-2024:", len(temporal_wide))
print("Referencias de acaso: calculadas")
print("Bootstrap:", B)
print("Status: PASS")

print("\nDiretorio:", OUT)
print("ETAPA 23D CONCLUIDA")



ETAPA 23D — RESULTADOS

1. CONCORDANCIA ACIMA DO ACASO — TOP30
 CONDITION   METHOD_A  METHOD_B  OBSERVED  RANDOM_EXPECTED  CHANCE_ADJUSTED
frontier3y  BM25_full        M2   0.17006          0.00583          0.16525
frontier3y BM25_short        M4   0.16512          0.00583          0.16033
frontier3y  BM25_full        M4   0.26481          0.00583          0.26057
frontier3y BM25_short        M2   0.29321          0.00583          0.28915
frontier3y BM25_short BM25_full   0.23210          0.00583          0.22764
frontier3y         M2        M4   0.21204          0.00583          0.20751
historical  BM25_full        M2   0.17068          0.00325          0.16800
historical BM25_short        M4   0.16204          0.00325          0.15937
historical  BM25_full        M4   0.26975          0.00325          0.26738
historical BM25_short        M2   0.29167          0.00325          0.28942
historical BM25_short BM25_full   0.22407          0.00325          0.22157
historical         M2   

In [18]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 24A CORRIGIDA
# AMOSTRAGEM PARA VALIDACAO HUMANA CEGA
#
# 18 focais | 9 grandes areas | 108 pares
# 2 avaliadores independentes
#
# CORRECOES:
# 1. Evita GRANDE_AREA_x / GRANDE_AREA_y no merge.
# 2. Verifica explicitamente o esquema das tabelas.
# 3. Registra estratos efetivamente obtidos.
# 4. Preserva qualquer relaxamento de elegibilidade.
# 5. So grava a amostra apos auditoria completa.
#
# CPU | Sem novos embeddings ou BM25
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# 1. CONFIGURACAO E DIRETORIOS
# ============================================================

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

EXP = (
    ROOT / "08_epistemic_measurement_validation"
    / "05_national_expansion"
)

SAMPLE = (
    EXP / "focal_sample_108_20261020T190807_559853Z"
)

SEMANTIC = (
    EXP / "21_semantic_controlled_national_v1"
    / "checkpoints"
)

BM25 = (
    EXP / "23B_BM25_national_v1"
    / "checkpoints"
)

RAW = ROOT / "05_transferencia_colab"

SEED = 20261020

METHODS = [
    "BM25_short",
    "BM25_full",
    "M2",
    "M4"
]

SLOTS = [
    "BM25_short",
    "BM25_full",
    "M2",
    "M4",
    "CONSENSUS",
    "RANK31_50"
]

for path in [ROOT, SAMPLE, SEMANTIC, BM25, RAW]:
    assert path.is_dir(), f"Diretorio ausente: {path}"

def stable_hash(value):
    return hashlib.sha256(
        f"{SEED}:{value}".encode("utf-8")
    ).hexdigest()

def check_columns(frame, columns, label):
    missing = set(columns) - set(frame.columns)
    assert not missing, (
        f"{label}: colunas ausentes: {sorted(missing)}. "
        f"Disponiveis: {frame.columns.tolist()}"
    )

# ============================================================
# 2. AMOSTRA FOCAL — MERGE SEM COLUNAS DUPLICADAS
# ============================================================

mapping = pd.read_csv(
    SAMPLE / "focal_processing_index.csv",
    dtype={
        "FOCAL_ID": str,
        "ID_TRABALHO": str
    }
)

sample = pd.read_parquet(
    SAMPLE / "national_focals_108_RESTRICTED.parquet"
)

check_columns(
    mapping,
    ["FOCAL_ID", "ID_TRABALHO"],
    "Indice de focais"
)

check_columns(
    sample,
    [
        "ID_TRABALHO",
        "ANO",
        "NM_GRANDE_AREA_CONHECIMENTO",
        "AREA_HARMONIZADA"
    ],
    "Amostra nacional congelada"
)

mapping["ID_TRABALHO"] = (
    mapping["ID_TRABALHO"].astype(str)
)

sample["ID_TRABALHO"] = (
    sample["ID_TRABALHO"].astype(str)
)

assert mapping["FOCAL_ID"].is_unique
assert mapping["ID_TRABALHO"].is_unique
assert sample["ID_TRABALHO"].is_unique

# Importante: selecionar APENAS identificadores do mapping.
# ANO e GRANDE_AREA serao derivados de uma unica fonte.
# Isto evita sufixos _x e _y.
focals = mapping[[
    "FOCAL_ID",
    "ID_TRABALHO"
]].merge(
    sample[[
        "ID_TRABALHO",
        "ANO",
        "NM_GRANDE_AREA_CONHECIMENTO",
        "AREA_HARMONIZADA"
    ]],
    on="ID_TRABALHO",
    how="inner",
    validate="one_to_one"
)

focals = focals.rename(columns={
    "NM_GRANDE_AREA_CONHECIMENTO": "GRANDE_AREA"
})

focals["ANO"] = focals["ANO"].astype(int)

check_columns(
    focals,
    [
        "FOCAL_ID",
        "ID_TRABALHO",
        "ANO",
        "GRANDE_AREA",
        "AREA_HARMONIZADA"
    ],
    "Focais harmonizados"
)

assert len(focals) == 108
assert focals["GRANDE_AREA"].notna().all()
assert focals["GRANDE_AREA"].nunique() == 9

assert (
    focals.groupby("GRANDE_AREA").size()
    .eq(12).all()
)

assert set(focals["FOCAL_ID"]) == {
    f"NAT-{i:04d}" for i in range(1, 109)
}

print("=" * 66)
print("ETAPA 24A — PREPARACAO CORRIGIDA")
print("=" * 66)

print("\n1. HARMONIZACAO DOS FOCAIS")
print("Documentos focais:", len(focals))
print("Grandes areas:", focals["GRANDE_AREA"].nunique())
print("Colunas:", focals.columns.tolist())
print("Status: PASS")

# ============================================================
# 3. SELECIONAR DOIS FOCAIS POR GRANDE AREA
# ============================================================

eligible_focals = focals[
    focals["ANO"] >= 2017
].copy()

selected_parts = []

for area, group in eligible_focals.groupby(
    "GRANDE_AREA", sort=True
):
    assert len(group) >= 2, (
        f"Menos de dois focais elegiveis: {area}"
    )

    group = group.copy()

    group["_HASH"] = group[
        "FOCAL_ID"
    ].map(stable_hash)

    group = group.sort_values(
        ["_HASH", "FOCAL_ID"]
    )

    first = group.iloc[0]

    distinct_year = group[
        group["ANO"] != int(first["ANO"])
    ]

    second = (
        distinct_year.iloc[0]
        if len(distinct_year)
        else group.iloc[1]
    )

    selected_parts.append(
        pd.DataFrame([first, second])
    )

selected = pd.concat(
    selected_parts,
    ignore_index=True
)

selected = selected.drop(
    columns=["_HASH"]
).sort_values(
    ["GRANDE_AREA", "FOCAL_ID"]
).reset_index(drop=True)

assert len(selected) == 18
assert selected["FOCAL_ID"].is_unique

assert (
    selected.groupby("GRANDE_AREA").size()
    .eq(2).all()
)

print("\n2. AMOSTRA ESTRATIFICADA")
print("Focais selecionados:", len(selected))
print("Focais de 2016:", (selected["ANO"] == 2016).sum())

# ============================================================
# 4. RECONSTRUIR RANKINGS CONGELADOS
# ============================================================

def semantic_top50(scores, mod):

    ids = scores[
        "ANTECEDENT_ID"
    ].astype(str).to_numpy()

    vals = scores[
        f"COSINE_{mod}"
    ].to_numpy(dtype=float)

    assert len(ids) >= 50
    assert np.isfinite(vals).all()

    order = np.lexsort(
        (ids, -vals)
    )[:50]

    result = ids[order].tolist()

    assert len(result) == 50
    assert len(set(result)) == 50

    return result

def bm25_top50(frame, representation):

    x = frame[
        (frame["WINDOW"] == "historical")
        & (frame["REPRESENTATION"] == representation)
    ].sort_values("RANK")

    assert len(x) == 50
    assert x["RANK"].tolist() == list(range(1, 51))
    assert x["BM25_SCORE"].gt(0).all()

    result = x[
        "ANTECEDENT_ID"
    ].astype(str).tolist()

    assert len(set(result)) == 50

    return result

# ============================================================
# 5. SELECAO DE SEIS PARES POR FOCAL
# ============================================================

pair_records = []
candidate_records = []
relaxation_records = []
antecedent_years = {}

for focal in selected.itertuples(index=False):

    fid = focal.FOCAL_ID
    year = int(focal.ANO)

    semantic_file = (
        SEMANTIC / f"{fid}_scores.parquet"
    )

    bm_file = (
        BM25 / f"{fid}_bm25_top50.parquet"
    )

    assert semantic_file.is_file(), semantic_file
    assert bm_file.is_file(), bm_file

    scores = pd.read_parquet(
        semantic_file,
        columns=[
            "ANTECEDENT_ID",
            "ANTECEDENT_YEAR",
            "COSINE_M2",
            "COSINE_M4"
        ]
    )

    lexical = pd.read_parquet(bm_file)

    scores["ANTECEDENT_ID"] = (
        scores["ANTECEDENT_ID"].astype(str)
    )

    scores["ANTECEDENT_YEAR"] = (
        scores["ANTECEDENT_YEAR"].astype(int)
    )

    assert scores["ANTECEDENT_ID"].is_unique
    assert scores["ANTECEDENT_YEAR"].lt(year).all()

    year_map = scores.set_index(
        "ANTECEDENT_ID"
    )["ANTECEDENT_YEAR"]

    rankings = {
        "BM25_short": bm25_top50(lexical, "short"),
        "BM25_full": bm25_top50(lexical, "full"),
        "M2": semantic_top50(scores, "M2"),
        "M4": semantic_top50(scores, "M4")
    }

    rank_maps = {
        method: {
            document: rank
            for rank, document in enumerate(
                documents, start=1
            )
        }
        for method, documents in rankings.items()
    }

    all_candidates = set().union(
        *[set(ids) for ids in rankings.values()]
    )

    candidate_rows = []

    for doc in sorted(all_candidates):

        ranks = {
            method: rank_maps[method].get(doc)
            for method in METHODS
        }

        n_top30 = sum(
            rank is not None and rank <= 30
            for rank in ranks.values()
        )

        n_top50 = sum(
            rank is not None
            for rank in ranks.values()
        )

        candidate_rows.append({
            "ANTECEDENT_ID": doc,
            "N_TOP30_METHODS": n_top30,
            "N_TOP50_METHODS": n_top50,
            **{
                f"RANK_{m}": ranks[m]
                for m in METHODS
            }
        })

    candidates = pd.DataFrame(candidate_rows)

    picked = set()

    def choose(pool, sort_columns):

        pool = pool[
            ~pool["ANTECEDENT_ID"].isin(picked)
        ].copy()

        if pool.empty:
            return None

        pool["_TIE"] = pool[
            "ANTECEDENT_ID"
        ].map(stable_hash)

        pool = pool.sort_values(
            sort_columns + ["_TIE"]
        )

        return pool.iloc[0]

    def register(slot, row, rule, relaxed):

        doc = str(row["ANTECEDENT_ID"])

        assert doc not in picked
        picked.add(doc)

        ant_year = int(year_map.loc[doc])

        antecedent_years[doc] = ant_year

        pair_records.append({
            "FOCAL_ID": fid,
            "FOCAL_DOCUMENT_ID": str(
                focal.ID_TRABALHO
            ),
            "FOCAL_YEAR": year,
            "GRANDE_AREA": focal.GRANDE_AREA,
            "SLOT": slot,
            "SELECTION_RULE": rule,
            "RELAXED": bool(relaxed),
            "ANTECEDENT_ID": doc,
            "ANTECEDENT_YEAR": ant_year
        })

        if relaxed:
            relaxation_records.append({
                "FOCAL_ID": fid,
                "SLOT": slot,
                "SELECTION_RULE": rule,
                "ANTECEDENT_ID": doc
            })

    # Quatro estratos associados aos metodos.
    for method in METHODS:

        col = f"RANK_{method}"

        # Preferencia: exclusivo do Top30.
        primary = candidates[
            (candidates["N_TOP30_METHODS"] == 1)
            & candidates[col].between(1, 30)
        ]

        chosen = choose(primary, [col])

        if chosen is not None:

            register(
                method, chosen,
                "exclusive_top30",
                False
            )

        else:

            # Alternativa auditada:
            # presente no Top30 do metodo, mas
            # compartilhado com algum outro metodo.
            fallback = candidates[
                candidates[col].between(1, 30)
            ].copy()

            fallback = fallback.sort_values(
                ["N_TOP30_METHODS", col]
            )

            chosen = choose(
                fallback,
                ["N_TOP30_METHODS", col]
            )

            assert chosen is not None, (
                f"{fid}: nenhum candidato "
                f"disponivel no Top30 de {method}"
            )

            register(
                method, chosen,
                "method_top30_nonexclusive",
                True
            )

    # Consenso: pelo menos dois metodos Top30.
    consensus = candidates[
        candidates["N_TOP30_METHODS"] >= 2
    ].copy()

    # Priorizar consenso mais amplo.
    consensus["_NEG_CONSENSUS"] = (
        -consensus["N_TOP30_METHODS"]
    )

    chosen = choose(
        consensus, ["_NEG_CONSENSUS"]
    )

    if chosen is None:
        raise RuntimeError(
            f"{fid}: nenhum candidato de consenso "
            "disponivel apos os quatro estratos. "
            "A amostra ainda nao foi gravada."
        )

    register(
        "CONSENSUS",
        chosen,
        "top30_in_two_or_more_methods",
        False
    )

    # Antecedente Top31-50, nao presente
    # no Top30 dos demais metodos.
    boundary = candidates[
        candidates["N_TOP30_METHODS"] == 0
    ].copy()

    rank_cols = [
        f"RANK_{m}" for m in METHODS
    ]

    boundary["BEST_RANK"] = boundary[
        rank_cols
    ].min(axis=1)

    boundary = boundary[
        boundary["BEST_RANK"].between(31, 50)
    ]

    chosen = choose(
        boundary, ["BEST_RANK"]
    )

    assert chosen is not None, (
        f"{fid}: nao foi encontrado candidato "
        "Top31-50 fora dos demais Top30. "
        "A amostra ainda nao foi gravada."
    )

    register(
        "RANK31_50",
        chosen,
        "rank31_50_outside_all_top30",
        False
    )

    candidates["FOCAL_ID"] = fid

    candidate_records.append(candidates)

    assert len(picked) == 6

    print(
        f"{fid}: 6 pares | "
        f"ano={year} | "
        f"relaxamentos acumulados="
        f"{len(relaxation_records)}"
    )

pairs = pd.DataFrame(pair_records)

candidate_audit = pd.concat(
    candidate_records,
    ignore_index=True
)

assert len(pairs) == 108

assert pairs.groupby(
    "FOCAL_ID"
).size().eq(6).all()

assert not pairs.duplicated([
    "FOCAL_ID",
    "ANTECEDENT_ID"
]).any()

assert (
    pairs.groupby("SLOT").size()
    .reindex(SLOTS).eq(18).all()
)

assert pairs["ANTECEDENT_YEAR"].lt(
    pairs["FOCAL_YEAR"]
).all()

print("\n3. PARES SELECIONADOS")
print("Pares:", len(pairs))
print("Relaxamentos:", len(relaxation_records))
print("Status: PASS")

# ============================================================
# 6. DOCUMENTOS COM TEXTOS ORIGINAIS
# ============================================================

document_years = dict(antecedent_years)

for focal in selected.itertuples(index=False):

    doc = str(focal.ID_TRABALHO)
    year = int(focal.ANO)

    if doc in document_years:
        assert document_years[doc] == year

    document_years[doc] = year

required = pd.DataFrame([
    {
        "ID_TRABALHO": doc,
        "ANO": year
    }
    for doc, year in document_years.items()
])

assert required["ID_TRABALHO"].is_unique

text_parts = []

for year, group in required.groupby("ANO"):

    path = (
        RAW /
        f"CAPES_BTD_EMBED_INPUT_{year}.parquet"
    )

    assert path.is_file(), path

    wanted = set(
        group["ID_TRABALHO"].astype(str)
    )

    columns = [
        "ID_TRABALHO",
        "TITULO",
        "RESUMO",
        "PALAVRAS_CHAVE"
    ]

    # Leitura apenas das colunas necessarias.
    source = pd.read_parquet(
        path,
        columns=columns
    )

    source["ID_TRABALHO"] = (
        source["ID_TRABALHO"].astype(str)
    )

    source = source[
        source["ID_TRABALHO"].isin(wanted)
    ].copy()

    assert source["ID_TRABALHO"].is_unique

    assert set(
        source["ID_TRABALHO"]
    ) == wanted, (
        f"Metadados incompletos em {year}"
    )

    for col in [
        "TITULO",
        "RESUMO",
        "PALAVRAS_CHAVE"
    ]:
        source[col] = (
            source[col].fillna("").astype(str)
        )

    source["ANO"] = int(year)

    text_parts.append(source)

texts = pd.concat(
    text_parts,
    ignore_index=True
)

assert texts["ID_TRABALHO"].is_unique
assert len(texts) == len(required)

lookup = texts.set_index(
    "ID_TRABALHO",
    verify_integrity=True
)

print("\n4. METADADOS TEXTUAIS")
print("Documentos distintos:", len(texts))
print("Cobertura:", "PASS")

# ============================================================
# 7. CONTEXTOS PARA OS AVALIADORES
# ============================================================

pairs = pairs.sort_values(
    ["GRANDE_AREA", "FOCAL_ID", "SLOT"]
).reset_index(drop=True)

contexts = []

for i, row in enumerate(
    pairs.itertuples(index=False),
    start=1
):

    focal_text = lookup.loc[
        row.FOCAL_DOCUMENT_ID
    ]

    antecedent_text = lookup.loc[
        row.ANTECEDENT_ID
    ]

    contexts.append({
        "PAIR_ID": f"VAL-{i:04d}",
        "FOCAL_ID": row.FOCAL_ID,
        "GRANDE_AREA": row.GRANDE_AREA,
        "SLOT": row.SLOT,
        "SELECTION_RULE": row.SELECTION_RULE,
        "RELAXED": row.RELAXED,
        "FOCAL_DOCUMENT_ID": row.FOCAL_DOCUMENT_ID,
        "ANTECEDENT_ID": row.ANTECEDENT_ID,
        "FOCAL_YEAR": row.FOCAL_YEAR,
        "ANTECEDENT_YEAR": row.ANTECEDENT_YEAR,
        "FOCAL_TITLE": focal_text["TITULO"],
        "FOCAL_ABSTRACT": focal_text["RESUMO"],
        "FOCAL_KEYWORDS": focal_text["PALAVRAS_CHAVE"],
        "ANTECEDENT_TITLE": antecedent_text["TITULO"],
        "ANTECEDENT_ABSTRACT": antecedent_text["RESUMO"],
        "ANTECEDENT_KEYWORDS": antecedent_text["PALAVRAS_CHAVE"]
    })

contexts = pd.DataFrame(contexts)

assert len(contexts) == 108
assert contexts["PAIR_ID"].is_unique

provenance = contexts[[
    "PAIR_ID",
    "FOCAL_ID",
    "SLOT",
    "SELECTION_RULE",
    "RELAXED",
    "FOCAL_DOCUMENT_ID",
    "ANTECEDENT_ID"
]].merge(
    candidate_audit,
    on=["FOCAL_ID", "ANTECEDENT_ID"],
    how="left",
    validate="one_to_one"
)

assert len(provenance) == 108
assert provenance["N_TOP30_METHODS"].notna().all()

# ============================================================
# 8. FORMULARIOS CEGOS
# ============================================================

blind_columns = [
    "PAIR_ID",
    "GRANDE_AREA",
    "FOCAL_YEAR",
    "ANTECEDENT_YEAR",
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]

rating_columns = [
    "RELEVANCIA_TEMATICA_0_3",
    "RELACAO_CONCEITUAL_0_3",
    "RELACAO_METODOLOGICA_0_3",
    "PLAUSIBILIDADE_ANTECEDENTE_0_3",
    "EVIDENCIA_INSUFICIENTE_SIM_NAO",
    "JUSTIFICATIVA",
    "OBSERVACOES"
]

forms = {}

for evaluator, offset in [("A", 17), ("B", 31)]:

    form = contexts[blind_columns].copy()

    for col in rating_columns:
        form[col] = ""

    form = form.sample(
        frac=1,
        random_state=SEED + offset
    ).reset_index(drop=True)

    form.insert(0, "EVALUATOR", evaluator)

    assert len(form) == 108
    assert form["PAIR_ID"].is_unique

    forbidden = {
        "SLOT",
        "SELECTION_RULE",
        "RELAXED",
        "ANTECEDENT_ID",
        "FOCAL_DOCUMENT_ID",
        "BM25_SCORE"
    }

    assert not (
        forbidden & set(form.columns)
    )

    forms[evaluator] = form

# ============================================================
# 9. PROTOCOLO
# ============================================================

protocol = """
# Protocolo — Avaliacao humana cega de antecedentes

## Objetivo

Julgar a plausibilidade cientifica de um documento
anterior como antecedente de uma tese ou dissertacao.

Nao se deve inferir uma citacao efetiva ou influencia
historica apenas pela similaridade textual.

## Escala

Para cada criterio, utilizar:

0 = Ausente ou nao identificavel
1 = Relacao fraca ou generica
2 = Relacao substantiva
3 = Relacao direta e central

## Criterios

RELEVANCIA_TEMATICA:
Proximidade entre problema, objeto e tema de pesquisa.

RELACAO_CONCEITUAL:
Articulacao entre conceitos, teorias ou perspectivas.

RELACAO_METODOLOGICA:
Proximidade entre desenhos, metodos ou procedimentos.

PLAUSIBILIDADE_ANTECEDENTE:
Possibilidade de o documento anterior constituir
referencia cientificamente pertinente ao focal.

## Evidencia insuficiente

Quando titulo, resumo e palavras-chave nao permitirem
avaliar um criterio, marcar SIM para evidencia
insuficiente e deixar o criterio sem pontuacao.

Ausencia de informacao nao equivale a nota zero.

## Justificativa

Registrar evidencia textual concreta que sustenta
a avaliacao.

## Cegamento

Os avaliadores nao conhecem o algoritmo utilizado,
a posicao no ranking ou as pontuacoes computacionais.

Avaliadores A e B devem trabalhar independentemente.

Nao alterar PAIR_ID.

## Restricoes

A amostra e exploratoria e estratificada.

O julgamento descreve plausibilidade a partir
dos metadados fornecidos, nao influencia comprovada.

Arquivos RESTRICTED nao devem ser compartilhados
com os avaliadores.
"""

# ============================================================
# 10. GRAVACAO SOMENTE APOS VALIDACOES
# ============================================================

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S_%fZ")

OUT = (
    EXP / f"24A_human_validation_corrected_{STAMP}"
)

OUT.mkdir(parents=True, exist_ok=False)

selected.to_csv(
    OUT / "RESTRICTED_selected_focals.csv",
    index=False
)

pairs.to_csv(
    OUT / "RESTRICTED_sampling_decisions.csv",
    index=False
)

candidate_audit.to_parquet(
    OUT / "RESTRICTED_candidate_rankings.parquet",
    index=False
)

provenance.to_parquet(
    OUT / "RESTRICTED_algorithmic_provenance.parquet",
    index=False
)

contexts.to_parquet(
    OUT / "RESTRICTED_pair_contexts.parquet",
    index=False
)

pd.DataFrame(relaxation_records).to_csv(
    OUT / "RESTRICTED_selection_relaxations.csv",
    index=False
)

for evaluator, form in forms.items():

    form.to_csv(
        OUT / f"EVALUATOR_{evaluator}_BLIND.csv",
        index=False,
        encoding="utf-8-sig"
    )

(OUT / "EVALUATION_PROTOCOL.md").write_text(
    protocol.strip(),
    encoding="utf-8"
)

manifest = {
    "stage": "24A_corrected",
    "n_grand_areas": 9,
    "n_focals": 18,
    "pairs_per_focal": 6,
    "n_pairs": 108,
    "n_evaluators_planned": 2,
    "n_evaluations_planned": 216,
    "seed": SEED,
    "ranking_window": "historical",
    "n_relaxed_method_slots": len(
        relaxation_records
    ),
    "method_slot_relaxation": (
        "Non-exclusive Top30 accepted only when "
        "exclusive Top30 unavailable, with "
        "explicit RESTRICTED audit trail."
    ),
    "sampling_inference": (
        "Exploratory stratified sample; "
        "not a national precision estimate."
    ),
    "human_evaluations_completed": False,
    "source_semantic": str(SEMANTIC),
    "source_bm25": str(BM25)
}

(OUT / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ============================================================
# 11. AUDITORIA FINAL
# ============================================================

print("\n" + "=" * 68)
print("ETAPA 24A — RESULTADOS FINAIS")
print("=" * 68)

print("\n1. AMOSTRA")
print("Grandes areas:", selected["GRANDE_AREA"].nunique())
print("Documentos focais:", len(selected))
print("Pares selecionados:", len(pairs))
print("Avaliacoes previstas:", len(pairs) * 2)

print("\n2. ESTRATOS")
print(
    pairs.groupby([
        "SLOT", "SELECTION_RULE"
    ]).size().rename("N").to_string()
)

print("\n3. RELAXAMENTOS")
print("Total:", len(relaxation_records))

if relaxation_records:
    print(
        pd.DataFrame(relaxation_records)
        .to_string(index=False)
    )
else:
    print("Nenhum relaxamento necessario.")

print("\n4. AMOSTRA DISCIPLINAR")
print(
    selected.groupby("GRANDE_AREA")
    .agg(
        N_FOCALS=("FOCAL_ID", "size"),
        YEARS=("ANO", lambda x:
            ", ".join(map(str, sorted(x)))
        )
    )
    .to_string()
)

print("\n5. ARQUIVOS DE AVALIACAO")
print("EVALUATOR_A_BLIND.csv")
print("EVALUATOR_B_BLIND.csv")
print("EVALUATION_PROTOCOL.md")

print("\n6. INTEGRIDADE")
print("Identificadores unicos: PASS")
print("Temporalidade dos antecedentes: PASS")
print("Proveniencia preservada: PASS")
print("Cegamento dos formularios: PASS")
print("Status: PASS")

print("\nDiretorio:", OUT)
print("ETAPA 24A CONCLUIDA")


ETAPA 24A — PREPARACAO CORRIGIDA

1. HARMONIZACAO DOS FOCAIS
Documentos focais: 108
Grandes areas: 9
Colunas: ['FOCAL_ID', 'ID_TRABALHO', 'ANO', 'GRANDE_AREA', 'AREA_HARMONIZADA']
Status: PASS

2. AMOSTRA ESTRATIFICADA
Focais selecionados: 18
Focais de 2016: 0
NAT-0025: 6 pares | ano=2020 | relaxamentos acumulados=0
NAT-0103: 6 pares | ano=2023 | relaxamentos acumulados=0
NAT-0021: 6 pares | ano=2017 | relaxamentos acumulados=0
NAT-0031: 6 pares | ano=2020 | relaxamentos acumulados=0
NAT-0059: 6 pares | ano=2023 | relaxamentos acumulados=0
NAT-0072: 6 pares | ano=2017 | relaxamentos acumulados=0
NAT-0069: 6 pares | ano=2018 | relaxamentos acumulados=0
NAT-0104: 6 pares | ano=2017 | relaxamentos acumulados=0
NAT-0050: 6 pares | ano=2020 | relaxamentos acumulados=0
NAT-0058: 6 pares | ano=2018 | relaxamentos acumulados=0
NAT-0093: 6 pares | ano=2017 | relaxamentos acumulados=0
NAT-0098: 6 pares | ano=2019 | relaxamentos acumulados=0
NAT-0014: 6 pares | ano=2018 | relaxamentos acumulados=

In [19]:

# ============================================================
# CAPES — ETAPA 24B
# INSTRUMENTOS DE AVALIACAO HUMANA CEGA
#
# Entrada: Etapa 24A congelada
# Saida: dois formularios XLSX independentes
# Nao preenche respostas humanas
# Nao altera arquivos RESTRICTED ou CSV originais
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

from openpyxl import Workbook, load_workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

EXP = (
    ROOT / "08_epistemic_measurement_validation"
    / "05_national_expansion"
)

SOURCE = (
    EXP /
    "24A_human_validation_corrected_20261021T030554_024945Z"
)

assert SOURCE.is_dir(), "Pasta 24A nao encontrada."

OUT = EXP / "24B_human_evaluation_instruments_v1"
OUT.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "A": SOURCE / "EVALUATOR_A_BLIND.csv",
    "B": SOURCE / "EVALUATOR_B_BLIND.csv"
}

OUTPUTS = {
    "A": OUT / "EVALUATOR_A_FORM.xlsx",
    "B": OUT / "EVALUATOR_B_FORM.xlsx"
}

# Nao sobrescrever formularios existentes:
# podem conter julgamentos de avaliadores.
for path in OUTPUTS.values():
    assert not path.exists(), (
        f"Arquivo ja existe, preservado: {path}"
    )

REQUIRED = [
    "EVALUATOR",
    "PAIR_ID",
    "GRANDE_AREA",
    "FOCAL_YEAR",
    "ANTECEDENT_YEAR",
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]

RATINGS = [
    "RELEVANCIA_TEMATICA_0_3",
    "RELACAO_CONCEITUAL_0_3",
    "RELACAO_METODOLOGICA_0_3",
    "PLAUSIBILIDADE_ANTECEDENTE_0_3"
]

OTHER_FIELDS = [
    "EVIDENCIA_INSUFICIENTE_SIM_NAO",
    "JUSTIFICATIVA",
    "OBSERVACOES"
]

frames = {}

for evaluator, path in INPUTS.items():

    assert path.is_file(), path

    df = pd.read_csv(
        path,
        dtype={"PAIR_ID": str},
        keep_default_na=False
    )

    assert len(df) == 108
    assert df["PAIR_ID"].is_unique
    assert df["EVALUATOR"].eq(evaluator).all()
    assert set(REQUIRED).issubset(df.columns)

    forbidden = {
        "SLOT", "SELECTION_RULE",
        "RELAXED", "ANTECEDENT_ID",
        "FOCAL_DOCUMENT_ID", "BM25_SCORE"
    }

    assert not (forbidden & set(df.columns))

    frames[evaluator] = df

assert set(frames["A"]["PAIR_ID"]) == set(
    frames["B"]["PAIR_ID"]
)

# ------------------------------------------------------------
# FUNCAO PARA TEXTOS SEGUROS
# ------------------------------------------------------------

def safe_cell(value):

    if pd.isna(value):
        return ""

    if isinstance(value, str):

        # Evitar interpretacao de conteudo textual
        # como formula em programas de planilha.
        if value.startswith(("=", "+", "-", "@")):
            value = "'" + value

        assert len(value) <= 32767, (
            "Texto excede limite de celula Excel."
        )

    return value

# ------------------------------------------------------------
# CRIAR UM FORMULARIO POR AVALIADOR
# ------------------------------------------------------------

NAVY = "183153"
BLUE = "2563A6"
PALE = "EAF2FA"
WHITE = "FFFFFF"
GRAY = "526377"

for evaluator, df in frames.items():

    wb = Workbook()

    guide = wb.active
    guide.title = "Instrucoes"

    instructions = [
        ("PROTOCOLO DE AVALIACAO CEGA", ""),
        ("Avaliador", evaluator),
        ("Numero de pares", 108),
        (
            "Objetivo",
            "Julgar a pertinencia cientifica de "
            "documentos anteriores como possiveis "
            "antecedentes do trabalho focal."
        ),
        (
            "Relevancia tematica",
            "Proximidade de problema, objeto e tema."
        ),
        (
            "Relacao conceitual",
            "Proximidade de conceitos, teorias "
            "e perspectivas."
        ),
        (
            "Relacao metodologica",
            "Proximidade de desenhos, metodos "
            "e procedimentos."
        ),
        (
            "Plausibilidade do antecedente",
            "Pertinencia cientifica do documento "
            "anterior em relacao ao focal."
        ),
        (
            "Escala 0",
            "Relacao ausente ou nao identificavel."
        ),
        (
            "Escala 1",
            "Relacao fraca ou generica."
        ),
        (
            "Escala 2",
            "Relacao substantiva."
        ),
        (
            "Escala 3",
            "Relacao direta e central."
        ),
        (
            "Evidencia insuficiente",
            "Marcar SIM quando os metadados "
            "nao permitirem julgamento responsavel. "
            "Deixar em branco criterios nao avaliaveis."
        ),
        (
            "Justificativa",
            "Fundamentar a pontuacao utilizando "
            "elementos dos titulos, resumos ou "
            "palavras-chave apresentados."
        ),
        (
            "Independencia",
            "Nao consultar o outro avaliador "
            "antes de encerrar todas as avaliacoes."
        ),
        (
            "Restricao",
            "Nao alterar os identificadores PAIR_ID."
        ),
        (
            "Interpretacao",
            "Avaliacao de plausibilidade, nao "
            "comprovacao de citacao ou influencia."
        )
    ]

    for row_no, (label, content) in enumerate(
        instructions, start=1
    ):
        guide.cell(row_no, 1, label)
        guide.cell(row_no, 2, content)

        guide.cell(row_no, 1).font = Font(
            bold=True,
            color=NAVY
        )

        guide.cell(row_no, 2).alignment = Alignment(
            wrap_text=True,
            vertical="top"
        )

        guide.row_dimensions[row_no].height = 38

    guide.column_dimensions["A"].width = 31
    guide.column_dimensions["B"].width = 90
    guide.freeze_panes = "B3"

    ws = wb.create_sheet("Avaliacoes")

    columns = REQUIRED + RATINGS + OTHER_FIELDS

    for col_no, name in enumerate(columns, start=1):

        cell = ws.cell(1, col_no, name)
        cell.fill = PatternFill(
            "solid", fgColor=NAVY
        )
        cell.font = Font(
            bold=True, color=WHITE
        )
        cell.alignment = Alignment(
            wrap_text=True,
            vertical="center"
        )

    ws.row_dimensions[1].height = 44

    for row_no, values in enumerate(
        df[REQUIRED].itertuples(
            index=False, name=None
        ),
        start=2
    ):

        for col_no, value in enumerate(
            values, start=1
        ):
            cell = ws.cell(
                row_no, col_no, safe_cell(value)
            )
            cell.alignment = Alignment(
                wrap_text=True,
                vertical="top"
            )

            if row_no % 2 == 0:
                cell.fill = PatternFill(
                    "solid", fgColor="F5F8FC"
                )

        for col_no in range(
            len(REQUIRED) + 1,
            len(columns) + 1
        ):
            cell = ws.cell(row_no, col_no)
            cell.fill = PatternFill(
                "solid", fgColor=PALE
            )
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

        ws.row_dimensions[row_no].height = 72

    # Validacao: escores 0, 1, 2, 3 ou vazio.
    dv_score = DataValidation(
        type="whole",
        operator="between",
        formula1=0,
        formula2=3,
        allow_blank=True
    )

    dv_score.error = (
        "Digite somente 0, 1, 2 ou 3."
    )
    dv_score.errorTitle = "Pontuacao invalida"
    dv_score.showErrorMessage = True

    ws.add_data_validation(dv_score)

    for field in RATINGS:
        col_no = columns.index(field) + 1
        letter = get_column_letter(col_no)

        dv_score.add(
            f"{letter}2:{letter}109"
        )

    dv_insufficient = DataValidation(
        type="list",
        formula1='"SIM,NAO"',
        allow_blank=True
    )

    dv_insufficient.showErrorMessage = True
    dv_insufficient.error = (
        "Selecione SIM ou NAO."
    )

    ws.add_data_validation(dv_insufficient)

    insufficient_col = get_column_letter(
        columns.index(
            "EVIDENCIA_INSUFICIENTE_SIM_NAO"
        ) + 1
    )

    dv_insufficient.add(
        f"{insufficient_col}2:{insufficient_col}109"
    )

    for col_no, field in enumerate(
        columns, start=1
    ):

        letter = get_column_letter(col_no)

        if field in [
            "FOCAL_ABSTRACT",
            "ANTECEDENT_ABSTRACT"
        ]:
            width = 55

        elif field in [
            "FOCAL_TITLE",
            "ANTECEDENT_TITLE"
        ]:
            width = 43

        elif field in [
            "FOCAL_KEYWORDS",
            "ANTECEDENT_KEYWORDS"
        ]:
            width = 32

        elif field in [
            "JUSTIFICATIVA",
            "OBSERVACOES"
        ]:
            width = 48

        elif field in RATINGS:
            width = 22

        else:
            width = 19

        ws.column_dimensions[letter].width = width

    ws.freeze_panes = "D2"
    ws.auto_filter.ref = (
        f"A1:{get_column_letter(len(columns))}109"
    )

    ws.sheet_view.showGridLines = False

    wb.save(OUTPUTS[evaluator])

# ------------------------------------------------------------
# VERIFICACAO DOS ARQUIVOS PRODUZIDOS
# ------------------------------------------------------------

for evaluator, path in OUTPUTS.items():

    assert path.is_file()

    wb_check = load_workbook(
        path, read_only=True
    )

    assert wb_check.sheetnames == [
        "Instrucoes", "Avaliacoes"
    ]

    sheet = wb_check["Avaliacoes"]

    assert sheet.max_row == 109
    assert sheet.max_column == len(columns)

    actual_ids = {
        sheet.cell(row, 2).value
        for row in range(2, 110)
    }

    assert actual_ids == set(
        frames[evaluator]["PAIR_ID"]
    )

    wb_check.close()

print("=" * 65)
print("ETAPA 24B — INSTRUMENTOS GERADOS")
print("=" * 65)

print("Avaliadores:", len(OUTPUTS))
print("Pares por avaliador: 108")
print("Avaliacoes humanas realizadas: 0")

for evaluator, path in OUTPUTS.items():
    print(
        f"\nAvaliador {evaluator}:"
        f"\n{path}"
    )

print("\nCegamento preservado: PASS")
print("Identificadores preservados: PASS")
print("Escalas controladas: PASS")
print("Status: PASS")
print("\nETAPA 24B CONCLUIDA")


ETAPA 24B — INSTRUMENTOS GERADOS
Avaliadores: 2
Pares por avaliador: 108
Avaliacoes humanas realizadas: 0

Avaliador A:
/content/capes_drive_clean_20261020/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion/24B_human_evaluation_instruments_v1/EVALUATOR_A_FORM.xlsx

Avaliador B:
/content/capes_drive_clean_20261020/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion/24B_human_evaluation_instruments_v1/EVALUATOR_B_FORM.xlsx

Cegamento preservado: PASS
Identificadores preservados: PASS
Escalas controladas: PASS
Status: PASS

ETAPA 24B CONCLUIDA


In [20]:

# ============================================================
# SCIENTOMETRICS 2026 — ETAPA 24C-1
# PRE-ESPECIFICACAO DA VALIDACAO HUMANA
#
# Objetivos:
# 1. Auditar os formularios cegos A e B.
# 2. Validar os 108 pares e os seis estratos.
# 3. Registrar desfechos e regras de analise.
# 4. Congelar o plano antes da avaliacao humana.
#
# CPU | SOMENTE LEITURA DAS ETAPAS ANTERIORES
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
from openpyxl import load_workbook

ROOT = Path(
    "/content/capes_drive_clean_20261020"
    "/MyDrive/CAPES_Epistemic_Change"
)

EXP = (
    ROOT
    / "08_epistemic_measurement_validation"
    / "05_national_expansion"
)

SOURCE_24A = (
    EXP
    / "24A_human_validation_corrected_20261021T030554_024945Z"
)

SOURCE_24B = (
    EXP
    / "24B_human_evaluation_instruments_v1"
)

OUT = EXP / "24C_human_validation_analysis_plan_v1"

assert SOURCE_24A.is_dir()
assert SOURCE_24B.is_dir()

FORM_A = SOURCE_24B / "EVALUATOR_A_FORM.xlsx"
FORM_B = SOURCE_24B / "EVALUATOR_B_FORM.xlsx"

PROVENANCE = (
    SOURCE_24A
    / "RESTRICTED_algorithmic_provenance.parquet"
)

for path in [FORM_A, FORM_B, PROVENANCE]:
    assert path.is_file(), path

# ------------------------------------------------------------
# 1. LER INSTRUMENTOS, SEM ALTERAR OS ARQUIVOS
# ------------------------------------------------------------

ratings = [
    "RELEVANCIA_TEMATICA_0_3",
    "RELACAO_CONCEITUAL_0_3",
    "RELACAO_METODOLOGICA_0_3",
    "PLAUSIBILIDADE_ANTECEDENTE_0_3"
]

other_fields = [
    "EVIDENCIA_INSUFICIENTE_SIM_NAO",
    "JUSTIFICATIVA",
    "OBSERVACOES"
]

def read_form(path, evaluator):

    wb = load_workbook(
        path,
        read_only=True,
        data_only=True
    )

    assert "Avaliacoes" in wb.sheetnames

    ws = wb["Avaliacoes"]

    rows = list(ws.values)

    wb.close()

    assert len(rows) == 109

    headers = list(rows[0])

    assert len(headers) == len(set(headers)), (
        "Cabecalhos duplicados."
    )

    df = pd.DataFrame(
        rows[1:],
        columns=headers
    )

    required = [
        "EVALUATOR",
        "PAIR_ID",
        "GRANDE_AREA",
        "FOCAL_TITLE",
        "ANTECEDENT_TITLE"
    ] + ratings + other_fields

    assert set(required).issubset(df.columns)

    assert len(df) == 108
    assert df["PAIR_ID"].notna().all()
    assert df["PAIR_ID"].is_unique

    assert df["EVALUATOR"].eq(
        evaluator
    ).all()

    forbidden = {
        "SLOT",
        "SELECTION_RULE",
        "RELAXED",
        "ANTECEDENT_ID",
        "FOCAL_DOCUMENT_ID",
        "BM25_SCORE"
    }

    assert not (forbidden & set(df.columns))

    return df

a = read_form(FORM_A, "A")
b = read_form(FORM_B, "B")

assert set(a["PAIR_ID"]) == set(b["PAIR_ID"])

# ------------------------------------------------------------
# 2. VERIFICAR IDENTIDADE DOS PARES
# ------------------------------------------------------------

context_fields = [
    "PAIR_ID",
    "GRANDE_AREA",
    "FOCAL_YEAR",
    "ANTECEDENT_YEAR",
    "FOCAL_TITLE",
    "FOCAL_ABSTRACT",
    "FOCAL_KEYWORDS",
    "ANTECEDENT_TITLE",
    "ANTECEDENT_ABSTRACT",
    "ANTECEDENT_KEYWORDS"
]

assert set(context_fields).issubset(a.columns)
assert set(context_fields).issubset(b.columns)

# String canonica dos contextos para
# assegurar que ambos julgam o mesmo material.
def context_fingerprint(df):

    ordered = df[
        context_fields
    ].sort_values("PAIR_ID")

    canonical = ordered.fillna("").astype(
        str
    ).to_json(
        orient="records",
        force_ascii=False
    )

    return hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()

hash_a = context_fingerprint(a)
hash_b = context_fingerprint(b)

assert hash_a == hash_b, (
    "Os formularios apresentam diferencas "
    "nos contextos documentais."
)

provenance = pd.read_parquet(PROVENANCE)

assert len(provenance) == 108
assert provenance["PAIR_ID"].is_unique

assert set(provenance["PAIR_ID"]) == set(
    a["PAIR_ID"]
)

assert provenance["FOCAL_ID"].nunique() == 18

slot_counts = provenance.groupby("SLOT").size()

assert len(slot_counts) == 6
assert slot_counts.eq(18).all()

# ------------------------------------------------------------
# 3. AUDITAR ESTADO ATUAL DAS AVALIACOES
# ------------------------------------------------------------

def nonempty(value):

    if pd.isna(value):
        return False

    return str(value).strip() != ""

response_fields = ratings + other_fields

def count_filled(df):

    return {
        field: int(
            df[field].map(nonempty).sum()
        )
        for field in response_fields
    }

counts_a = count_filled(a)
counts_b = count_filled(b)

filled_total = (
    sum(counts_a.values())
    + sum(counts_b.values())
)

# Esta etapa e feita antes da coleta.
# Se ja houver respostas, preservar o registro
# e interromper o congelamento retrospectivo.
assert filled_total == 0, (
    "Foram encontradas respostas preenchidas. "
    "Nao e apropriado registrar este plano como "
    "pre-especificado antes da coleta. "
    "Preserve os formularios e documente "
    "a data real das decisoes analiticas."
)

# ------------------------------------------------------------
# 4. PLANO ESTATISTICO PRE-ESPECIFICADO
# ------------------------------------------------------------

plan = {
    "stage": "24C-1",
    "status": "frozen_before_human_ratings",
    "design": {
        "focal_documents": 18,
        "grand_areas": 9,
        "pairs": 108,
        "evaluators": 2,
        "planned_judgments": 216,
        "selection": (
            "Purposive balanced sampling of "
            "algorithmic retrieval profiles"
        ),
        "sampling_inference": False
    },
    "primary_outcome": (
        "PLAUSIBILIDADE_ANTECEDENTE_0_3"
    ),
    "secondary_outcomes": ratings[:3],
    "ordinal_scale": [0, 1, 2, 3],
    "insufficient_evidence": {
        "indicator": (
            "EVIDENCIA_INSUFICIENTE_SIM_NAO"
        ),
        "allowed_values": ["SIM", "NAO"],
        "rule": (
            "Missing score due to insufficient "
            "evidence is not recoded as zero."
        )
    },
    "unit_of_human_judgment": "pair_evaluator",
    "unit_of_agreement": "pair",
    "unit_of_sampling_cluster": "focal_document",
    "primary_reliability": {
        "metric": "quadratic_weighted_cohen_kappa",
        "scope": (
            "Pairs rated by both evaluators "
            "on the primary ordinal outcome"
        ),
        "report_with": [
            "exact_agreement",
            "absolute_rating_difference",
            "contingency_table",
            "paired_sample_size",
            "non_evaluable_pairs"
        ]
    },
    "secondary_reliability": [
        "weighted_kappa_per_dimension",
        "exact_agreement_per_dimension"
    ],
    "descriptive_comparisons": [
        "rating_distribution_by_sampling_slot",
        "insufficient_evidence_by_sampling_slot",
        "ratings_by_grand_area_descriptive_only",
        "shared_vs_exclusive_vs_rank31_50"
    ],
    "score_aggregation": (
        "Report each evaluator separately; "
        "do not automatically average ratings "
        "to create ground truth."
    ),
    "disagreement_rule": (
        "Flag differences >= 2 categories "
        "on primary outcome for later review. "
        "Do not silently overwrite independent ratings."
    ),
    "sensitivity": [
        "complete_case_pairs",
        "insufficient_evidence_as_separate_category"
    ],
    "inferential_limitations": [
        "Non-probability sample of pairs",
        "Only two focal documents per grand area",
        "Selection favors distinctive retrieval profiles",
        "Human ratings assess plausibility, not proven influence",
        "No unbiased national precision or recall estimates"
    ],
    "human_judgments_present_at_freeze": False,
    "context_sha256": hash_a,
    "source_stage_24A": str(SOURCE_24A),
    "source_stage_24B": str(SOURCE_24B)
}

# ------------------------------------------------------------
# 5. DESCRICAO LEGIVEL DO PLANO
# ------------------------------------------------------------

protocol_text = """
# CAPES — Plano de analise da validacao humana

## 1. Desenho

Amostra exploratoria de 108 pares documento focal–
antecedente, provenientes de 18 focais distribuidos
igualmente entre nove grandes areas CAPES.

Cada focal contribui com seis perfis de recuperacao.
Os pares sao julgados independentemente por dois
avaliadores cegos para a origem algoritmica.

## 2. Desfecho principal

Plausibilidade do antecedente cientifico:
escala ordinal de 0 a 3.

O julgamento indica pertinencia cientifica plausivel
a partir de titulo, resumo e palavras-chave.
Nao constitui prova de citacao ou influencia.

## 3. Desfechos secundarios

Relevancia tematica.
Relacao conceitual.
Relacao metodologica.

Cada dimensao possui escala ordinal 0–3.

## 4. Evidencia insuficiente

Registrar a frequencia de julgamentos com
evidencia insuficiente.

Notas nao avaliaveis permanecem ausentes.
Nunca converter ausencia de evidencia em nota zero.

## 5. Confiabilidade interavaliadores

Indicador principal:
kappa de Cohen com ponderacao quadratica.

Relatar conjuntamente:
- Numero de pares validos para comparacao.
- Concordancia exata.
- Distribuicao de diferencas absolutas.
- Tabela de contingencia das notas.
- Frequencia de informacao insuficiente.

Em distribuicoes degeneradas, registrar
a impossibilidade de estimar kappa em vez
de forcar um valor numerico.

## 6. Resolucao de divergencias

Diferencas de duas ou mais categorias no
desfecho principal serao sinalizadas.

As avaliacoes independentes originais serao
preservadas, inclusive se houver adjudicacao.

## 7. Comparacoes exploratorias

Descrever os escores pelos seis estratos de
selecao, mantendo as categorias originais.

Nao tratar os estratos como grupos aleatorios
representativos de cada algoritmo.

## 8. Limitacoes

A amostra foi selecionada deliberadamente para
cobrir perfis de recuperacao.

Nao permite estimar precisao ou recall nacional.

A avaliacao mede plausibilidade de antecedencia,
nao influencia historica demonstrada.

## 9. Congelamento

O plano e registrado antes da coleta humana.
Alteracoes posteriores deverao ser documentadas
como emendas ao plano original.
"""

# ------------------------------------------------------------
# 6. GRAVACAO SEM SOBRESCREVER VERSOES ANTERIORES
# ------------------------------------------------------------

assert not OUT.exists(), (
    "O plano 24C-1 ja existe. "
    "Preservar a versao congelada, "
    "sem sobrescrita automatica."
)

OUT.mkdir(parents=True, exist_ok=False)

(OUT / "statistical_analysis_plan.json").write_text(
    json.dumps(
        plan,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

(OUT / "statistical_analysis_plan.md").write_text(
    protocol_text.strip(),
    encoding="utf-8"
)

audit = pd.DataFrame([
    {
        "EVALUATOR": "A",
        "N_PAIRS": len(a),
        "N_DISTINCT_PAIRS": a["PAIR_ID"].nunique(),
        "N_FILLED_RESPONSE_CELLS": (
            sum(counts_a.values())
        ),
        "CONTEXT_SHA256": hash_a
    },
    {
        "EVALUATOR": "B",
        "N_PAIRS": len(b),
        "N_DISTINCT_PAIRS": b["PAIR_ID"].nunique(),
        "N_FILLED_RESPONSE_CELLS": (
            sum(counts_b.values())
        ),
        "CONTEXT_SHA256": hash_b
    }
])

audit.to_csv(
    OUT / "pre_collection_audit.csv",
    index=False
)

# ------------------------------------------------------------
# 7. RESULTADO
# ------------------------------------------------------------

print("=" * 68)
print("ETAPA 24C-1 — PLANO ESTATISTICO CONGELADO")
print("=" * 68)

print("\n1. INTEGRIDADE")
print("Focais:", provenance["FOCAL_ID"].nunique())
print("Pares:", provenance["PAIR_ID"].nunique())
print("Avaliadores:", 2)
print("Estratos:", len(slot_counts))

print("\n2. ESTADO DOS FORMULARIOS")
print(audit[[
    "EVALUATOR",
    "N_PAIRS",
    "N_FILLED_RESPONSE_CELLS"
]].to_string(index=False))

print("\n3. PLANO ESTATISTICO")
print("Desfecho principal: plausibilidade 0-3")
print("Confiabilidade: kappa ponderado quadratico")
print("Dados ausentes: nao imputar zero")
print("Inferencia nacional: nao autorizada")
print("Plano registrado antes dos julgamentos: SIM")

print("\n4. AUDITORIA")
print("Identidade dos pares A/B: PASS")
print("Cegamento algoritmico: PASS")
print("Estratos da Etapa 24A: PASS")
print("Formulario sem julgamentos: PASS")
print("Status: PASS")

print("\nDiretorio:", OUT)
print("ETAPA 24C-1 CONCLUIDA")


ETAPA 24C-1 — PLANO ESTATISTICO CONGELADO

1. INTEGRIDADE
Focais: 18
Pares: 108
Avaliadores: 2
Estratos: 6

2. ESTADO DOS FORMULARIOS
EVALUATOR  N_PAIRS  N_FILLED_RESPONSE_CELLS
        A      108                        0
        B      108                        0

3. PLANO ESTATISTICO
Desfecho principal: plausibilidade 0-3
Confiabilidade: kappa ponderado quadratico
Dados ausentes: nao imputar zero
Inferencia nacional: nao autorizada
Plano registrado antes dos julgamentos: SIM

4. AUDITORIA
Identidade dos pares A/B: PASS
Cegamento algoritmico: PASS
Estratos da Etapa 24A: PASS
Formulario sem julgamentos: PASS
Status: PASS

Diretorio: /content/capes_drive_clean_20261020/MyDrive/CAPES_Epistemic_Change/08_epistemic_measurement_validation/05_national_expansion/24C_human_validation_analysis_plan_v1
ETAPA 24C-1 CONCLUIDA
